# Full-GPU UAV training

This notebook is a standalone full-GPU/CPU-tensor training notebook.

The batched environment state, reset/step logic, sensing, belief map, communication, energy, reward, observations, replay buffer, actor/critic inference, and gradient updates are designed to run on the selected torch device. All learner/reference definitions needed by this notebook are embedded below, so `test.ipynb` is not required.

Run the cells top-to-bottom, then call `benchmark_env(...)` or `train_full_gpu(...)` explicitly.

## How to read this notebook

This notebook has two execution modes over the same tensorized implementation:

1. **CPU debug/regression**: use execution_mode="cpu" or device="cpu" with strict_cuda=False.
2. **GPU production**: use execution_mode="gpu"; CUDA is required and the tensorized environment/learner path stays on CUDA.
3. **Auto mode**: execution_mode="auto" chooses GPU when CUDA is available, otherwise CPU.

Recommended reading order:

setup -> FullGpuUAVBatchEnv -> GpuReplayBuffer -> project/learner loaders -> action helpers -> network/parity -> single-GPU training -> DDP/multi-GPU -> auto device selection -> profiling/benchmarks

For readability, every **top-level function** below is kept in its own code cell. A Python class remains one class cell because its methods must stay inside the class body to preserve normal Python class semantics.


## Function Map — đọc file này theo thứ tự nào?

### 1. Device guard

| Function | Vai trò |
|---|---|
| `_require_cuda(device)` | Kiểm tra device thực sự là CUDA và CUDA đang khả dụng; dùng cho các path bắt buộc GPU. |

### 2. Core environment: `FullGpuUAVBatchEnv`

Đây là phần quan trọng nhất của simulator tensorized. Đọc theo thứ tự: `__init__ -> reset -> _motion -> _sense -> _network -> _energy -> _observation/_state -> step`.

| Method | Vai trò |
|---|---|
| `__init__` | Khởi tạo batch environment, device, RNG và toàn bộ tensor state cố định. |
| `attach_external_network` | Gắn adapter network ngoài, ví dụ bridge tới UavNetSim CPU/SimPy. |
| `detach_external_network` | Tháo network adapter ngoài khỏi environment. |
| `close` | Đóng/cleanup adapter hoặc tài nguyên liên quan. |
| `_rand` | Sinh random tensor bằng generator của environment trên device đang chọn. |
| `_sample_world` | Sinh batch world: UAV, target, obstacle và state ban đầu. |
| `reset` | Reset toàn bộ hoặc một subset environment bằng mask. |
| `_belief_entropy` | Tính entropy của belief map. |
| `_belief_potential` | Chuyển entropy/belief thành potential dùng cho reward/information progress. |
| `_project_unit_ball` | Project/clamp vector action về unit ball hợp lệ. |
| `_segment_cylinder_blocked` | Kiểm tra segment có cắt obstacle hình trụ hay không; dùng cho collision/LoS. |
| `_motion` | Cập nhật chuyển động UAV và áp các ràng buộc boundary/obstacle/safety. |
| `_sensing_profile` | Tính sensing profile theo altitude, gồm các tham số detection/false positive liên quan. |
| `_circle_sqrt_integral` | Helper toán học để tính chính xác phần giao giữa camera footprint hình tròn và grid cell. |
| `_cell_coverage_fraction` | Tính tỷ lệ một grid cell nằm trong camera footprint. |
| `_sense` | Thực hiện sensing, Bayes/belief update, confirmation/report-related state. |
| `_flush_pending` | Chuyển report pending vào buffer khi còn dung lượng. |
| `_destination_mask` | Tạo mask các destination communication hợp lệ cho mỗi UAV. |
| `_commit_completed_peer_receipts` | Commit peer transfer đã hoàn tất khi buffer UAV nhận còn chỗ. |
| `_uavnetsim_gpu_radio` | GPU tensor surrogate cho A2A PHY/SINR theo contract UavNetSim. |
| `_network` | Xử lý action communication, link feasibility, peer relay/GCS delivery và network metrics. |
| `_energy` | Tính energy cho motion + communication và cập nhật battery. |
| `_expire_reports` | Expire report theo TTL và cập nhật penalty/cleanup state. |
| `_observation` | Xây per-agent observation tensor. |
| `_report_slots` | Pack report state thành fixed-size feature slots. |
| `_state` | Xây centralized/global state cho critic/CTDE. |
| `step` | Một bước simulation hoàn chỉnh: action -> motion/sense/network/energy/reward -> next obs/state. |
| `assert_all_state_on_cuda` | Assert tất cả tensor state của environment đang nằm trên CUDA; chỉ dùng khi kiểm GPU residency. |

### 3. Replay: `GpuReplayBuffer`

| Method | Vai trò |
|---|---|
| `__init__` | Allocate replay storage trên device được chọn. |
| `__len__` | Trả số transition hiện có. |
| `add_batch` | Ghi một batch transition vào circular replay buffer. |
| `sample` | Sample minibatch cho learner trên device yêu cầu. |

### 4. Load project + learner + action helpers

| Function | Vai trò |
|---|---|
| `load_project_namespace(repo)` | Trả về project namespace đã nhúng trong chính notebook này; không đọc `test.ipynb`. |
| `_trainer(repo, algorithm, device, seed)` | Tạo MASAC hoặc MATD3 learner hiện có mà không dựng CPU mission env. |
| `make_project_gpu_replay_buffer(...)` | Tạo replay buffer tương thích trainer/project nhưng storage nằm trên selected device. |
| `_matd3_actions(...)` | Sinh continuous + destination action cho MATD3, gồm exploration/mask logic. |
| `_random_exploration_actions(...)` | Sinh random action hợp lệ cho giai đoạn warmup/learning-starts. |

### 5. Environment/hybrid benchmarks

| Function | Vai trò |
|---|---|
| `benchmark_env(...)` | Benchmark riêng tensorized environment trên CPU hoặc GPU tùy `device`. |
| `_make_hybrid_vector_env(...)` | Tạo CPU AsyncVectorEnv từ CPU reference environment. |
| `benchmark_hybrid_env(...)` | Đo throughput của CPU vectorized reference env không learner update. |
| `benchmark_hybrid_train(...)` | Benchmark path CPU env + learner trên device được chọn, dùng làm baseline so với full tensor path. |

### 6. Official UavNetSim bridge + parity

#### `GpuUavNetSimBridge`

| Method | Vai trò |
|---|---|
| `__init__` | Tạo bridge giữa tensor mission env và official UavNetSim CPU/SimPy. |
| `np` | Trả NumPy namespace/helper dùng trong bridge. |
| `_tensor_numpy` | Chuyển tensor cần thiết sang NumPy cho CPU simulator. |
| `_build_mission_objects` | Tạo object UAV/report/network state mà UavNetSim cần. |
| `_sync_gpu_to_cpu` | Đồng bộ state cần thiết từ tensor env sang CPU UavNetSim. |
| `_sync_cpu_to_gpu` | Đồng bộ kết quả network từ CPU simulator về tensor env. |
| `reset_from_env` | Reset UavNetSim bridge theo state hiện tại của env. |
| `step` | Chạy một network step bằng official UavNetSim. |
| `metrics` | Trả network metrics. |
| `timing_metrics` | Trả thời gian GPU->CPU, simulator, CPU->GPU để đo bottleneck. |
| `close` | Cleanup bridge/UavNetSim resources. |

#### Parity/validation functions

| Function | Vai trò |
|---|---|
| `copy_cpu_reference_state_to_gpu(cpu_env, gpu_env)` | Copy một scenario CPU sang tensor env để so apples-to-apples. |
| `benchmark_uavnetsim_full_flow(...)` | So full-flow CPU UavNetSim với tensor mission + UavNetSim trên cùng scenario/action trace. |
| `validate_uavnetsim_gpu_direct_link(...)` | Validation controlled single direct UAV->GCS link, tránh ambiguity do contention/collision. |
| `validate_uavnetsim_network_parity(...)` | Drive hai backend UavNetSim bằng cùng state/intents để kiểm network parity. |

### 7. Single-device training + optimizer helpers

| Function | Vai trò |
|---|---|
| `_rebuild_cuda_optimizers_with_fused_adam(...)` | Rebuild optimizer bằng fused Adam cho benchmark CUDA trước optimizer step đầu tiên. |
| `_compile_trainer_modules(...)` | Dùng `torch.compile` cho actor/critic fixed-shape modules khi benchmark. |
| `train_full_gpu(...)` | Training loop tensorized trên một selected device. Tên giữ lịch sử là “full_gpu”, nhưng có thể chạy CPU debug khi truyền `device="cpu", strict_cuda=False`. |

### 8. DDP actor wrappers

| Class/method | Vai trò |
|---|---|
| `_MASACDDPActorForward.__init__` | Bọc MASAC actor/project metadata cho DDP forward. |
| `_MASACDDPActorForward.forward` | Forward MASAC actor theo observation + destination mask. |
| `_MATD3DDPActorForward.__init__` | Bọc MATD3 actor/project metadata cho DDP forward. |
| `_MATD3DDPActorForward.forward` | Forward MATD3 actor theo observation + destination mask. |

### 9. W&B + DDP update helpers

| Function | Vai trò |
|---|---|
| `_gpu_wandb_api_key_from_kaggle()` | Đọc W&B API key từ Kaggle secret/environment khi có. |
| `_init_gpu_wandb_run(...)` | Khởi tạo W&B run theo config; DDP chỉ rank phù hợp mới log. |
| `_ddp_actor_sample_masac(...)` | Lấy MASAC actions qua DDP actor. |
| `_ddp_actor_actions_matd3(...)` | Lấy MATD3 actions qua DDP actor. |
| `_allreduce_parameter_grad(parameter)` | All-reduce gradient của một parameter giữa DDP ranks. |
| `_ddp_update_masac(...)` | Một learner update MASAC trong DDP. |
| `_ddp_update_matd3(...)` | Một learner update MATD3 trong DDP. |
| `_train_full_gpu_ddp_worker(...)` | Worker process cho một DDP rank: env shard + learner/update loop. |
| `_write_ddp_launcher_script(path)` | Ghi temporary launcher script để spawn DDP workers. |
| `train_full_gpu_ddp(...)` | Entry point training 2-GPU DDP. |

### 10. CPU / GPU / auto selection

| Function | Vai trò |
|---|---|
| `resolve_full_execution_mode(execution_mode=None)` | Resolve `cpu/gpu/auto`; `gpu` fail-fast nếu CUDA không có, `auto` fallback CPU. |
| `resolve_full_gpu_devices(max_gpus=None)` | Liệt kê CUDA devices dùng cho auto training, hiện cap tối đa 2 GPU. |
| `_split_env_counts(total_envs, num_shards)` | Chia số environment gần đều cho các GPU shards. |
| `train_full_gpu_sharded(...)` | Multi-GPU env sharding: env chia GPU0/GPU1 nhưng learner chính nằm GPU0. |
| `train_full_gpu_auto(...)` | Entry point nên dùng: CPU mode -> CPU tensor path; GPU mode -> 1 GPU hoặc 2-GPU strategy; auto -> tự chọn. |

### 11. Profiling và performance benchmarks

| Function | Vai trò |
|---|---|
| `benchmark_full_gpu_training_optimizations(...)` | Sweep compile/fused Adam/update ratio/env count để tìm config nhanh mà không đổi numerical path. |
| `profile_full_gpu_step(...)` | PyTorch profiler cho actor inference + env/network step. |
| `benchmark_gpu_network_models(...)` | So chi phí các GPU network models như packet approximation và uavnetsim_gpu. |
| `benchmark_simple_scaling(...)` | So scaling CPU env counts và GPU env counts. |
| `_uavnetsim_cpu_worker(payload)` | Worker function cho parallel official UavNetSim CPU benchmark. |
| `benchmark_uavnetsim_cpu_worker_scaling(...)` | Đo scaling khi tăng số CPU UavNetSim workers. |
| `_scalarize_metrics(metrics)` | Chuyển metrics về scalar Python values để compare/log/serialize dễ hơn. |
| `_trainer_parameter_fingerprint(trainer)` | SHA256 toàn bộ parameter/buffer để kiểm deterministic model state độc lập device. |
| `benchmark_uavnetsim_training_compare(...)` | So end-to-end CPU-mission và GPU/tensor-mission training khi cả hai dùng UavNetSim validation path. |

### Flow nên dùng khi chạy thật

- **Laptop debug:** `train_full_gpu_auto(..., execution_mode="cpu")`
- **1 GPU:** `train_full_gpu_auto(..., execution_mode="gpu")` -> single-device training
- **2 GPU:** `train_full_gpu_auto(..., execution_mode="gpu")` -> strategy từ CONFIG, hiện production config dùng DDP
- **Không chắc máy có GPU hay không:** `execution_mode="auto"`

Official UavNetSim là SimPy/CPU backend; `uavnetsim_gpu` là tensorized training surrogate để giữ mission/network training path trên selected torch device.



In [ ]:
from __future__ import annotations

import ast
import random
import sys
import types
import json
import math
import time
from pathlib import Path

import torch
import torch.nn.functional as F

# config.py is the single source of truth.
if not Path("config.py").is_file() and Path("/kaggle/input").is_dir():
    _gpu_config_candidates = sorted(
        Path("/kaggle/input").glob("**/config.py")
    )
    if len(_gpu_config_candidates) != 1:
        raise FileNotFoundError(
            "config.py is required. Upload it alongside test_gpu.ipynb or "
            "attach exactly one Kaggle input containing config.py. Found "
            f"{len(_gpu_config_candidates)} candidates."
        )
    _gpu_config_dir = str(_gpu_config_candidates[0].parent)
    if _gpu_config_dir not in sys.path:
        sys.path.insert(0, _gpu_config_dir)

from config import CONFIG


def _require_cuda(device: torch.device) -> None:
    if device.type != "cuda" or not torch.cuda.is_available():
        raise RuntimeError("test_gpu.ipynb requires a CUDA device for full-GPU mode")


In [ ]:
import os as _bootstrap_os
import sys as _bootstrap_sys
from pathlib import Path as _BootstrapPath
import copy
import csv
import importlib
import json
import os
import random
import secrets
import subprocess
import tempfile
import sys
import time
from abc import ABC, abstractmethod
from itertools import pairwise
from pathlib import Path
from typing import ClassVar
import gymnasium as gym
import numpy as np
import torch
import torch.nn.functional as F
from torch import nn
from torch.distributions import Normal
from config import CONFIG


In [ ]:
def set_seed(seed=44):
    seed = int(seed)

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.use_deterministic_algorithms(True)

    if hasattr(
        torch.backends,
        "cudnn",
    ):
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True

        if hasattr(
            torch.backends.cudnn,
            "allow_tf32",
        ):
            torch.backends.cudnn.allow_tf32 = False

    if (
        hasattr(
            torch.backends,
            "cuda",
        )
        and hasattr(
            torch.backends.cuda,
            "matmul",
        )
    ):
        torch.backends.cuda.matmul.allow_tf32 = False


In [ ]:
from dataclasses import asdict, dataclass, is_dataclass


In [ ]:
@dataclass
class UAV:
    id: int
    position: np.ndarray
    velocity: np.ndarray
    battery_j: float

    active: bool = True


In [ ]:
@dataclass
class Target:
    id: int
    position: np.ndarray
    confirmed: bool = False


In [ ]:
@dataclass
class Obstacle:
    position: np.ndarray
    radius: float
    height: float

    def __post_init__(self):
        self.position = np.asarray(
            self.position,
            dtype=np.float64,
        )
        self.radius = float(self.radius)
        self.height = float(self.height)

        if self.position.shape != (2,):
            raise ValueError(
                "Obstacle.position must be the XY center with shape (2,)"
            )

        if not np.all(np.isfinite(self.position)):
            raise ValueError(
                "Obstacle.position must contain only finite values"
            )

        if not np.isfinite(self.radius) or self.radius <= 0.0:
            raise ValueError(
                "Obstacle.radius must be finite and > 0"
            )

        if not np.isfinite(self.height) or self.height <= 0.0:
            raise ValueError(
                "Obstacle.height must be finite and > 0"
            )


In [ ]:
@dataclass
class Report:
    target_id: int
    source_uav: int
    created_step: int
    size_bytes: int
    ttl_s: float
    delivered_bytes: int = 0

    def __post_init__(self):
        for name, value in (
            ("target_id", self.target_id),
            ("source_uav", self.source_uav),
            ("created_step", self.created_step),
            ("size_bytes", self.size_bytes),
            ("delivered_bytes", self.delivered_bytes),
        ):
            if (
                isinstance(value, (bool, np.bool_))
                or not isinstance(value, (int, np.integer))
            ):
                raise TypeError(
                    f"{name} must be an integer"
                )

        self.target_id = int(self.target_id)
        self.source_uav = int(self.source_uav)
        self.created_step = int(self.created_step)
        self.size_bytes = int(self.size_bytes)
        self.delivered_bytes = int(self.delivered_bytes)
        self.ttl_s = float(self.ttl_s)

        if self.target_id < 0:
            raise ValueError(
                "target_id must be >= 0"
            )

        if self.source_uav < 0:
            raise ValueError(
                "source_uav must be >= 0"
            )

        if self.created_step < 0:
            raise ValueError(
                "created_step must be >= 0"
            )

        if self.size_bytes <= 0:
            raise ValueError(
                "size_bytes must be > 0"
            )

        if (
            not np.isfinite(self.ttl_s)
            or self.ttl_s <= 0.0
        ):
            raise ValueError(
                "ttl_s must be finite and > 0"
            )

        if not (
            0
            <= self.delivered_bytes
            <= self.size_bytes
        ):
            raise ValueError(
                "delivered_bytes must be in "
                "[0, size_bytes]"
            )


In [ ]:
def create_uav(rng):
    if not isinstance(rng, np.random.Generator):
        raise TypeError("rng must be numpy.random.Generator")

    num_uavs = int(CONFIG["num_uavs"])
    gcs = np.asarray(CONFIG["gcs_position"], dtype=np.float64)
    launch_radius = float(CONFIG["launch_radius_m"])
    safety_distance = float(CONFIG["safety_distance"])
    launch_min_spacing = float(CONFIG["launch_min_spacing_m"])
    map_size = float(CONFIG["map_size"])
    altitude = float(CONFIG["altitude_min"])

    if num_uavs < 1:
        raise ValueError("num_uavs must be >= 1")

    if gcs.shape != (3,) or not np.all(np.isfinite(gcs)):
        raise ValueError("gcs_position must be a finite 3D position")

    if not (
        0.0 <= gcs[0] <= map_size
        and 0.0 <= gcs[1] <= map_size
    ):
        raise ValueError("gcs_position must lie inside the map")

    if not np.isfinite(launch_radius) or launch_radius <= 0.0:
        raise ValueError("launch_radius_m must be finite and > 0")

    if not np.isfinite(safety_distance) or safety_distance < 0.0:
        raise ValueError("safety_distance must be finite and >= 0")

    if (
        not np.isfinite(launch_min_spacing)
        or launch_min_spacing < safety_distance
    ):
        raise ValueError(
            "launch_min_spacing_m must be finite and >= safety_distance"
        )

    if not np.isfinite(map_size) or map_size <= 0.0:
        raise ValueError("map_size must be finite and > 0")

    if not np.isfinite(altitude):
        raise ValueError("altitude_min must be finite")

    uavs = []
    max_attempts = max(10_000, 1_000 * num_uavs)
    attempts = 0
    launch_radius_sq = launch_radius * launch_radius

    while len(uavs) < num_uavs:
        attempts += 1

        if attempts > max_attempts:
            raise RuntimeError(
                "Could not place all UAVs inside the launch region while "
                "respecting map bounds and launch_min_spacing_m. "
                "Increase launch_radius_m or reduce num_uavs/launch_min_spacing_m."
            )

        offset_xy = rng.uniform(
            -launch_radius,
            launch_radius,
            size=2,
        ).astype(np.float64)

        if float(offset_xy @ offset_xy) > launch_radius_sq:
            continue

        candidate_xy = gcs[:2] + offset_xy

        if not (
            0.0 <= candidate_xy[0] <= map_size
            and 0.0 <= candidate_xy[1] <= map_size
        ):
            continue

        too_close = any(
            np.linalg.norm(candidate_xy - uav.position[:2])
            < launch_min_spacing
            for uav in uavs
        )

        if too_close:
            continue

        position = np.array(
            [candidate_xy[0], candidate_xy[1], altitude],
            dtype=np.float64,
        )

        uavs.append(
            UAV(
                id=len(uavs),
                position=position,
                velocity=np.zeros(3, dtype=np.float64),
                battery_j=CONFIG["battery_j"],
                active=True,
            )
        )

    return uavs


In [ ]:
def point_inside_obstacle(point, obstacles, margin=0.0):
    point = np.asarray(point, dtype=np.float64)
    margin = float(margin)

    if not np.isfinite(margin) or margin < 0.0:
        raise ValueError("margin must be finite and >= 0")

    if point.shape == (2,):
        point_xy = point
        z = 0.0
    elif point.shape == (3,):
        point_xy = point[:2]
        z = float(point[2])
    else:
        raise ValueError("point must have shape (2,) or (3,)")

    if not np.all(np.isfinite(point)):
        raise ValueError("point must contain only finite values")

    for obs in obstacles:
        horizontal_distance = np.linalg.norm(
            point_xy - obs.position
        )

        inside_horizontal = (
            horizontal_distance
            <= obs.radius + margin
        )
        inside_vertical = (
            0.0 <= z <= obs.height + margin
        )

        if inside_horizontal and inside_vertical:
            return True

    return False


In [ ]:
def create_obstacle(rng, uavs):
    obstacles = []

    map_size = float(CONFIG["map_size"])
    r_min = float(CONFIG["obstacle_radius_min_m"])
    r_max = float(CONFIG["obstacle_radius_max_m"])
    h_min = float(CONFIG["obstacle_height_min_m"])
    h_max = float(CONFIG["obstacle_height_max_m"])

    safety_margin = float(CONFIG["safety_distance"])
    gcs_xy = np.asarray(CONFIG["gcs_position"][:2], dtype=np.float64)
    gcs_exclusion = float(CONFIG["gcs_exclusion_radius_m"])

    if r_min <= 0.0 or r_max < r_min:
        raise ValueError("obstacle radius range is invalid")
    if 2.0 * r_max > map_size:
        raise ValueError(
            "obstacle_radius_max_m must be <= map_size / 2"
        )

    max_attempts = 100000
    attempts = 0

    while len(obstacles) < CONFIG["num_obstacles"]:
        attempts += 1

        if attempts > max_attempts:
            raise RuntimeError(
                "Could not place all obstacles with the current constraints."
            )

        radius = float(rng.uniform(r_min, r_max))
        height = float(rng.uniform(h_min, h_max))

        x = float(rng.uniform(radius, map_size - radius))
        y = float(rng.uniform(radius, map_size - radius))
        center = np.array([x, y], dtype=np.float64)

        distance_to_gcs = np.linalg.norm(center - gcs_xy)
        if distance_to_gcs <= gcs_exclusion + radius:
            continue

        overlaps_launch = any(
            np.linalg.norm(center - uav.position[:2])
            <= radius + safety_margin
            for uav in uavs
        )
        if overlaps_launch:
            continue

        overlaps_obstacle = any(
            np.linalg.norm(center - other.position)
            <= radius + other.radius
            for other in obstacles
        )
        if overlaps_obstacle:
            continue

        obstacles.append(
            Obstacle(
                position=center,
                radius=radius,
                height=height,
            )
        )

    return obstacles


In [ ]:
def create_target(rng, obstacles):
    targets = []
    occupied_cells = set()

    map_size = float(CONFIG["map_size"])
    cell_size = float(CONFIG["grid_cell_m"])
    gcs_xy = np.asarray(CONFIG["gcs_position"][:2], dtype=np.float64)
    target_exclusion = float(CONFIG["target_exclusion_radius_m"])

    max_attempts = 100000
    attempts = 0

    while len(targets) < CONFIG["num_targets"]:
        attempts += 1

        if attempts > max_attempts:
            raise RuntimeError(
                "Could not place all targets with the current constraints."
            )

        position = rng.uniform(0.0,map_size,size=2,).astype(np.float64)

        if point_inside_obstacle(position, obstacles):
            continue

        if np.linalg.norm(position - gcs_xy) <= target_exclusion:
            continue

        gx = int(np.floor(position[0] / cell_size))
        gy = int(np.floor(position[1] / cell_size))
        cell = (gx, gy)

        if cell in occupied_cells:
            continue

        targets.append(
            Target(
                id=len(targets),
                position=position,
                confirmed=False,
            )
        )
        occupied_cells.add(cell)

    return targets


In [ ]:
def create_world(seed):
    rng = np.random.default_rng(seed)
    uavs = create_uav(rng)
    obstacles = create_obstacle(rng, uavs)
    targets = create_target(rng, obstacles)

    return rng, uavs, targets, obstacles


In [ ]:
def segment_intersects_obstacle(
    start_position,
    end_position,
    obstacles,
    margin=0.0,
):
    start = np.asarray(
        start_position,
        dtype=np.float64,
    )

    end = np.asarray(
        end_position,
        dtype=np.float64,
    )

    if start.shape != (3,) or end.shape != (3,):
        raise ValueError(
            "start_position and end_position must have shape (3,)"
        )

    if (
        not np.all(np.isfinite(start))
        or not np.all(np.isfinite(end))
    ):
        raise ValueError(
            "start_position and end_position "
            "must contain only finite values"
        )

    margin = float(margin)

    if not np.isfinite(margin) or margin < 0.0:
        raise ValueError(
            "margin must be finite and >= 0"
        )

    direction = end - start
    eps = 1e-12

    for obs in obstacles:

        center = np.asarray(obs.position,dtype=np.float64,)

        radius = float(obs.radius) + float(margin)
        height = float(obs.height) + float(margin)

        dz = float(direction[2])

        if abs(dz) <= eps:

            if not (
                0.0 <= start[2] <= height
            ):
                continue

            z_enter = 0.0
            z_exit = 1.0

        else:

            t_ground = (0.0 - start[2]) / dz
            t_top = (height - start[2]) / dz

            z_enter = max(0.0,min(t_ground, t_top))
            z_exit = min(1.0,max(t_ground, t_top))

            if z_enter > z_exit:
                continue

        relative_xy = start[:2] - center[:2]
        direction_xy = direction[:2]

        a = direction_xy@direction_xy
        c = (relative_xy@relative_xy)- radius * radius

        if a <= eps:

            if c > 0:
                continue

            xy_enter = 0
            xy_exit = 1

        else:

            b = 2 * relative_xy@direction_xy
            discriminant = (b * b- 4.0 * a * c)

            if discriminant < 0.0:
                continue

            root = np.sqrt(max(discriminant, 0.0))

            t1 = (-b - root) / (2.0 * a)
            t2 = (-b + root) / (2.0 * a)

            xy_enter = max(0.0,min(t1, t2))
            xy_exit = min(1.0,max(t1, t2))

            if xy_enter > xy_exit:
                continue

        enter = max(z_enter,xy_enter)
        exit_ = min(z_exit,xy_exit)

        if enter <= exit_:
            return True

    return False


In [ ]:
def project_motion_action_np(action):
    """Project one or more 3-D motion actions onto the unit ball."""
    action = np.asarray(
        action,
        dtype=np.float64,
    )

    if action.ndim < 1 or action.shape[-1] != 3:
        raise ValueError(
            "motion action must have trailing shape (3,)"
        )

    if not np.all(np.isfinite(action)):
        raise ValueError(
            "motion action must contain only finite values"
        )

    clipped = np.clip(
        action,
        -1.0,
        1.0,
    )
    norms = np.linalg.norm(
        clipped,
        axis=-1,
        keepdims=True,
    )
    safe_norms = np.maximum(
        norms,
        1.0,
    )

    return (
        clipped / safe_norms
    ).astype(
        np.float64,
        copy=False,
    )


In [ ]:
def compute_motion_candidate(
    uav,
    action,
    dt=None,
):
    if dt is None:
        dt = CONFIG["dt"]

    dt = float(dt)

    if not np.isfinite(dt) or dt <= 0.0:
        raise ValueError(
            "dt must be finite and > 0"
        )

    action = np.asarray(
        action,
        dtype=np.float64,
    )

    if action.shape != (3,):
        raise ValueError("action must have shape (3,)")

    if not np.all(np.isfinite(action)):
        raise ValueError("action must contain only finite values")

    if not uav.active:
        return (
            uav.position.copy(),
            np.zeros(3, dtype=np.float64),
            False,
            False,
            False,
        )

    action = project_motion_action_np(
        action
    )

    acceleration = action * CONFIG["max_accel"]

    candidate_velocity = (
        uav.velocity
        + acceleration * dt
    )

    speed = np.linalg.norm(candidate_velocity)

    if speed > CONFIG["max_speed"]:
        candidate_velocity = (candidate_velocity/ speed* CONFIG["max_speed"])

    old_position = uav.position.copy()

    raw_candidate_position = old_position+ candidate_velocity * dt

    candidate_position = raw_candidate_position.copy()

    candidate_position[0] = np.clip(
        candidate_position[0],
        0.0,
        CONFIG["map_size"],
    )

    candidate_position[1] = np.clip(
        candidate_position[1],
        0.0,
        CONFIG["map_size"],
    )

    candidate_position[2] = np.clip(
        candidate_position[2],
        CONFIG["altitude_min"],
        CONFIG["altitude_max"],
    )

    horizontal_boundary_clipped = not np.allclose(
        raw_candidate_position[:2],
        candidate_position[:2],
    )
    altitude_clipped = not np.isclose(
        raw_candidate_position[2],
        candidate_position[2],
    )
    boundary_clipped = bool(
        horizontal_boundary_clipped
        or altitude_clipped
    )

    actual_velocity = (
        candidate_position
        - old_position
    ) / dt

    return (
        candidate_position,
        actual_velocity,
        boundary_clipped,
        bool(horizontal_boundary_clipped),
        bool(altitude_clipped),
    )


In [ ]:
def minimum_distance_during_motion(
    start_a,
    end_a,
    start_b,
    end_b,
):
    start_a = np.asarray(start_a,dtype=np.float64,)
    end_a = np.asarray(end_a,dtype=np.float64,)
    start_b = np.asarray(start_b,dtype=np.float64,)
    end_b = np.asarray(end_b,dtype=np.float64,)
    for point in (
        start_a,
        end_a,
        start_b,
        end_b,
    ):
        if point.shape != (3,):
            raise ValueError(
                "all positions must have shape (3,)"
            )

        if not np.all(np.isfinite(point)):
            raise ValueError(
                "all positions must contain only finite values"
            )
    relative_start = start_a - start_b
    displacement_a = end_a - start_a
    displacement_b = end_b - start_b

    relative_motion = displacement_a - displacement_b

    denominator = relative_motion@relative_motion
    if denominator <= 1e-12:
        return np.linalg.norm(
                relative_start
            )
        

    t_closest = -(relative_start@relative_motion)/ denominator
    t_closest = np.clip(t_closest,0.0,1.0,)

    relative_at_closest = (
        relative_start
        + t_closest * relative_motion
    )

    return float(
        np.linalg.norm(
            relative_at_closest
        )
    )


In [ ]:
def apply_swarm_motion(
    uavs,
    actions,
    obstacles=None,
    dt=None,
):
    if obstacles is None:
        obstacles = []

    if dt is None:
        dt = CONFIG["dt"]
    dt = float(dt)


    if not np.isfinite(dt) or dt <= 0.0:
        raise ValueError(
            "dt must be finite and > 0"
        )

    num_uavs = len(uavs)

    if num_uavs == 0:
        raise ValueError(
            "uavs must not be empty"
        )

    actions = np.asarray(actions,dtype=np.float64)

    if actions.shape != (num_uavs, 3):
        raise ValueError(
            f"actions must have shape "
            f"({num_uavs}, 3)"
        )

    if not np.all(np.isfinite(actions)):
        raise ValueError(
            "actions must contain only finite values"
        )

    old_positions = np.stack([
        uav.position.copy()
        for uav in uavs
    ])
    old_velocities = np.stack([
        uav.velocity.copy()
        for uav in uavs
    ])

    safety_distance = float(CONFIG["safety_distance"])

    for i in range(num_uavs):
        if not uavs[i].active:
            continue

        for j in range(i + 1,num_uavs):
            if not uavs[j].active:
                continue

            distance = np.linalg.norm(old_positions[i]- old_positions[j])

            if distance < safety_distance:
                raise ValueError(
                    "initial active UAV positions "
                    "violate safety_distance"
                )

    candidate_positions = []
    candidate_velocities = []
    boundary_clipped = np.zeros(
        num_uavs,
        dtype=bool,
    )
    horizontal_boundary_clipped = np.zeros(
        num_uavs,
        dtype=bool,
    )
    altitude_clipped = np.zeros(
        num_uavs,
        dtype=bool,
    )

    for i, (uav, action) in enumerate(zip(uavs, actions)):
        
        (
            position,
            velocity,
            clipped,
            horizontal_clipped,
            vertical_clipped,
        ) = compute_motion_candidate(
            uav,
            action,
            dt=dt,
        )

        candidate_positions.append(position)
        candidate_velocities.append(velocity)

        boundary_clipped[i] = clipped
        horizontal_boundary_clipped[i] = (
            horizontal_clipped
        )
        altitude_clipped[i] = (
            vertical_clipped
        )

    candidate_positions = np.stack(candidate_positions)
    candidate_velocities = np.stack(candidate_velocities)

    blocked_by_obstacle = np.zeros(
        num_uavs,
        dtype=bool,
    )

    for i, uav in enumerate(uavs):
        if not uav.active:
            continue

        if segment_intersects_obstacle(
            old_positions[i],
            candidate_positions[i],
            obstacles,
            margin=float(
                CONFIG[
                    "obstacle_clearance_m"
                ]
            ),
        ):
            blocked_by_obstacle[i] = True

    blocked_by_peer = np.zeros(num_uavs,dtype=bool)
    blocked = blocked_by_obstacle.copy()
    

    while True:
        effective_positions = candidate_positions.copy()
        effective_positions[blocked] = old_positions[blocked]

        next_blocked = blocked.copy()

        for i in range(num_uavs):
            if not uavs[i].active:
                continue

            for j in range(i + 1,num_uavs):
                if not uavs[j].active:
                    continue

                distance = minimum_distance_during_motion(
                        old_positions[i],
                        effective_positions[i],
                        old_positions[j],
                        effective_positions[j],
                    )
                
                if distance < safety_distance:
                    next_blocked[i] = True
                    next_blocked[j] = True

                    blocked_by_peer[i] = True
                    blocked_by_peer[j] = True

        if np.array_equal(next_blocked,blocked):
            break

        blocked = next_blocked

    for i, uav in enumerate(uavs):
        if not uav.active:
            uav.velocity = np.zeros(3, dtype=np.float64)
            continue

        if blocked[i]:
            uav.position = old_positions[i].copy()
            uav.velocity = np.zeros(3,dtype=np.float64)

        else:
            uav.position = candidate_positions[i].copy()
            uav.velocity = candidate_velocities[i].copy()

    realized_acceleration_mps2 = (
        np.stack([
            uav.velocity.copy()
            for uav in uavs
        ])
        - old_velocities
    ) / dt

    return {
        "blocked": blocked,
        "blocked_by_obstacle":blocked_by_obstacle,
        "blocked_by_peer":blocked_by_peer,
        "boundary_clipped":boundary_clipped,
        "horizontal_boundary_clipped": (
            horizontal_boundary_clipped
        ),
        "altitude_clipped": altitude_clipped,
        "realized_acceleration_mps2": (
            realized_acceleration_mps2
        ),
    }


In [ ]:
def sensing_profile(altitude):
    altitude = float(altitude)

    if not np.isfinite(altitude):
        raise ValueError("altitude must be finite")

    altitude = np.clip(altitude,CONFIG["altitude_min"],CONFIG["altitude_max"])
    altitude_anchors = np.asarray(CONFIG["altitude_anchors"])

    pd_anchors = np.asarray(CONFIG["pd"])
    pf_anchors = np.asarray(CONFIG["pf"])

    pd = np.interp(altitude,altitude_anchors,pd_anchors)
    pf = np.interp(altitude,altitude_anchors,pf_anchors)

    full_fov = CONFIG["camera_full_fov_deg"]
    half_fov_rad = np.deg2rad(full_fov / 2.0)
    fov_radius = altitude* np.tan(half_fov_rad)

    return (
        float(pd),
        float(pf),
        float(fov_radius)
    )


In [ ]:
def create_belief_maps():
    grid_n = int(np.ceil(CONFIG["map_size"] / CONFIG["grid_cell_m"]))
    belief_maps = np.full((CONFIG["num_uavs"],grid_n,grid_n),CONFIG["belief_prior"])
    return belief_maps


In [ ]:
def world_to_grid(position_xy):
    position_xy = np.asarray(position_xy, dtype=np.float64)

    if position_xy.ndim != 1 or position_xy.size < 2:
        raise ValueError(
            "position_xy must be a 1D array containing at least x and y"
        )

    x = float(position_xy[0])
    y = float(position_xy[1])

    if not np.isfinite(x) or not np.isfinite(y):
        raise ValueError("position must be finite")

    map_size = float(CONFIG["map_size"])
    cell_size = float(CONFIG["grid_cell_m"])

    if not np.isfinite(map_size) or map_size <= 0.0:
        raise ValueError("map_size must be finite and > 0")

    if not np.isfinite(cell_size) or cell_size <= 0.0:
        raise ValueError("grid_cell_m must be finite and > 0")

    if not (
        0.0 <= x <= map_size
        and 0.0 <= y <= map_size
    ):
        raise ValueError("position is outside the map")

    grid_n = int(np.ceil(map_size / cell_size))

    gx = int(np.floor(x / cell_size))
    gy = int(np.floor(y / cell_size))

    gx = min(gx, grid_n - 1)
    gy = min(gy, grid_n - 1)

    return gx, gy


In [ ]:
def cell_intersects_fov(
    gx,
    gy,
    uav_xy,
    fov_radius,
):
    cell_size = float(CONFIG["grid_cell_m"])
    map_size = float(CONFIG["map_size"])

    x_min = gx * cell_size
    x_max = min((gx + 1) * cell_size,map_size)

    y_min = gy * cell_size
    y_max = min((gy + 1) * cell_size,map_size)

    closest_x = np.clip(uav_xy[0],x_min,x_max)
    closest_y = np.clip(uav_xy[1],y_min,y_max)

    dx = float(uav_xy[0] - closest_x)
    dy = float(uav_xy[1] - closest_y)

    return (
        dx * dx + dy * dy <= fov_radius * fov_radius
    )


In [ ]:
import math


In [ ]:
def _circle_sqrt_integral(x, radius):
    """Antiderivative of sqrt(radius^2 - x^2) on [-radius, radius]."""
    x = max(-radius, min(radius, float(x)))
    root = math.sqrt(max(0.0, radius * radius - x * x))
    return 0.5 * (
        x * root
        + radius * radius * math.asin(x / radius)
    )


In [ ]:
def _circle_rectangle_intersection_area(
    circle_x,
    circle_y,
    radius,
    x_min,
    x_max,
    y_min,
    y_max,
):
    """Exact area of a circle intersected with an axis-aligned rectangle."""
    radius = float(radius)
    if radius <= 0.0 or x_max <= x_min or y_max <= y_min:
        return 0.0

    # Translate the circle center to the origin.
    x_min = float(x_min) - float(circle_x)
    x_max = float(x_max) - float(circle_x)
    y_min = float(y_min) - float(circle_y)
    y_max = float(y_max) - float(circle_y)

    left = max(x_min, -radius)
    right = min(x_max, radius)

    if right <= left or y_max <= -radius or y_min >= radius:
        return 0.0

    # The integrand changes form only where a horizontal rectangle edge
    # intersects the circle. Split at those x values, then integrate the
    # circle arc analytically on each interval.
    cuts = [left, right]
    for y_edge in (y_min, y_max):
        if abs(y_edge) < radius:
            x_cross = math.sqrt(
                max(0.0, radius * radius - y_edge * y_edge)
            )
            for cut in (-x_cross, x_cross):
                if left < cut < right:
                    cuts.append(cut)

    cuts = sorted(set(cuts))
    area = 0.0

    for a, b in pairwise(cuts):
        if b <= a:
            continue

        midpoint = 0.5 * (a + b)
        half_height = math.sqrt(
            max(0.0, radius * radius - midpoint * midpoint)
        )

        upper = min(y_max, half_height)
        lower = max(y_min, -half_height)
        if upper <= lower:
            continue

        arc_integral = (
            _circle_sqrt_integral(b, radius)
            - _circle_sqrt_integral(a, radius)
        )

        if y_max < half_height:
            upper_integral = y_max * (b - a)
        else:
            upper_integral = arc_integral

        if y_min > -half_height:
            lower_integral = y_min * (b - a)
        else:
            lower_integral = -arc_integral

        area += upper_integral - lower_integral

    return max(0.0, float(area))


In [ ]:
def cell_fov_coverage_fraction(gx, gy, uav_xy, fov_radius):
    """Return the fraction of a belief cell covered by the circular FOV.

    Geometry depends only on UAV/FOV and grid cell; target ground truth never
    changes measurement support. The circle/rectangle area is analytic, which
    is both deterministic and much cheaper than per-cell numerical quadrature.
    """
    if isinstance(gx, (bool, np.bool_)) or not isinstance(gx, (int, np.integer)):
        raise TypeError("gx must be an integer")
    if isinstance(gy, (bool, np.bool_)) or not isinstance(gy, (int, np.integer)):
        raise TypeError("gy must be an integer")

    gx = int(gx)
    gy = int(gy)
    uav_xy = np.asarray(uav_xy, dtype=np.float64)
    fov_radius = float(fov_radius)

    if uav_xy.shape != (2,) or not np.all(np.isfinite(uav_xy)):
        raise ValueError("uav_xy must be a finite shape-(2,) position")
    if not np.isfinite(fov_radius) or fov_radius < 0.0:
        raise ValueError("fov_radius must be finite and >= 0")
    if fov_radius == 0.0:
        return 0.0

    cell_size = float(CONFIG["grid_cell_m"])
    map_size = float(CONFIG["map_size"])
    grid_n = int(np.ceil(map_size / cell_size))

    if not (0 <= gx < grid_n and 0 <= gy < grid_n):
        raise ValueError("grid cell index out of range")

    x_min = gx * cell_size
    x_max = min((gx + 1) * cell_size, map_size)
    y_min = gy * cell_size
    y_max = min((gy + 1) * cell_size, map_size)

    if not cell_intersects_fov(gx, gy, uav_xy, fov_radius):
        return 0.0

    intersection_area = _circle_rectangle_intersection_area(
        circle_x=float(uav_xy[0]),
        circle_y=float(uav_xy[1]),
        radius=fov_radius,
        x_min=x_min,
        x_max=x_max,
        y_min=y_min,
        y_max=y_max,
    )
    cell_area = (x_max - x_min) * (y_max - y_min)

    if cell_area <= 0.0:
        return 0.0

    return float(np.clip(intersection_area / cell_area, 0.0, 1.0))


In [ ]:
def cells_with_fov_coverage(uav):
    if not uav.active:
        return []

    altitude = float(uav.position[2])

    if altitude <= 0.0:
        return []

    _, _, fov_radius = sensing_profile(altitude)
    cell_size = float(CONFIG["grid_cell_m"])
    grid_n = int(np.ceil(CONFIG["map_size"] / cell_size))
    uav_xy = np.asarray(uav.position[:2], dtype=np.float64)
    uav_x = float(uav_xy[0])
    uav_y = float(uav_xy[1])

    gx_min = max(0, int(np.floor((uav_x - fov_radius) / cell_size)))
    gx_max = min(grid_n - 1, int(np.floor((uav_x + fov_radius) / cell_size)))
    gy_min = max(0, int(np.floor((uav_y - fov_radius) / cell_size)))
    gy_max = min(grid_n - 1, int(np.floor((uav_y + fov_radius) / cell_size)))

    visible = []
    for gy in range(gy_min, gy_max + 1):
        for gx in range(gx_min, gx_max + 1):
            coverage = cell_fov_coverage_fraction(
                gx,
                gy,
                uav_xy,
                fov_radius,
            )
            if coverage > 0.0:
                visible.append((gx, gy, coverage))

    return visible


In [ ]:
def cells_in_fov(uav):
    return [
        (gx, gy)
        for gx, gy, _ in cells_with_fov_coverage(uav)
    ]


In [ ]:
def get_target_cells(targets):
    target_cells = {}

    for target in targets:
        gx, gy = world_to_grid(target.position)
        target_cells.setdefault((gx, gy), []).append(target.id)

    return target_cells


In [ ]:
def sample_sensor_measurement(has_target,pd,pf,rng):
    if has_target:
        positive_probability = pd

    else:
        positive_probability = pf
    observation = (rng.random()< positive_probability)

    return int(observation)


In [ ]:
def bayes_update(prior, observation, pd, pf, eps=1e-8):
    eps = float(eps)

    if (
        not np.isfinite(eps)
        or not 0.0 < eps < 0.5
    ):
        raise ValueError(
            "eps must be finite and in (0, 0.5)"
        )

    prior = float(prior)
    pd = float(pd)
    pf = float(pf)

    for name, value in (
        ("prior", prior),
        ("pd", pd),
        ("pf", pf),
    ):
        if not np.isfinite(value):
            raise ValueError(
                f"{name} must be finite"
            )

        if not 0.0 <= value <= 1.0:
            raise ValueError(
                f"{name} must be in [0, 1]"
            )

    prior = np.clip(prior,eps,1.0 - eps)

    if observation == 1:
        numerator = pd * prior
        denominator = pd * prior + pf * (1.0 - prior)
    elif observation == 0:
        numerator = (1.0 - pd) * prior
        denominator = (
            (1.0 - pd) * prior
            + (1.0 - pf) * (1.0 - prior)
        )
    else:
        raise ValueError("observation must be 0 or 1")

    denominator = max(float(denominator), eps)
    posterior = numerator / denominator

    return float(np.clip(posterior, eps, 1.0 - eps))


In [ ]:
def camera_ground_point_has_los(
    uav,
    ground_xy,
    obstacles=None,
):
    if obstacles is None:
        obstacles = []

    ground_xy = np.asarray(
        ground_xy,
        dtype=np.float64,
    )

    if (
        ground_xy.shape != (2,)
        or not np.all(
            np.isfinite(
                ground_xy
            )
        )
    ):
        raise ValueError(
            "ground_xy must be a "
            "finite shape-(2,) point"
        )

    ground_position = np.array(
        [
            float(
                ground_xy[0]
            ),
            float(
                ground_xy[1]
            ),
            0.0,
        ],
        dtype=np.float64,
    )

    return not segment_intersects_obstacle(
        uav.position,
        ground_position,
        obstacles,
    )


In [ ]:
def camera_target_has_los(
    uav,
    target,
    obstacles=None,
):
    return camera_ground_point_has_los(
        uav,
        target.position,
        obstacles=obstacles,
    )


In [ ]:
def sense_and_update(
    uav,
    belief_map,
    targets,
    rng,
    obstacles=None,
):
    if not uav.active:
        return []

    if obstacles is None:
        obstacles = []

    altitude = float(
        uav.position[2]
    )

    if altitude <= 0.0:
        return []

    (
        base_pd,
        base_pf,
        fov_radius,
    ) = sensing_profile(
        altitude
    )

    visible_cells = (
        cells_with_fov_coverage(
            uav
        )
    )
    target_cells = (
        get_target_cells(
            targets
        )
    )
    targets_by_id = {
        target.id: target
        for target in targets
    }

    cell_size = float(
        CONFIG[
            "grid_cell_m"
        ]
    )
    map_size = float(
        CONFIG[
            "map_size"
        ]
    )

    # A visible cell center may extend half a cell diagonal beyond the FOV.
    # Cylinders outside this disk cannot intersect any of the camera rays.
    ray_radius = fov_radius + np.sqrt(0.5) * cell_size
    los_obstacles = [
        obstacle for obstacle in obstacles
        if np.linalg.norm(obstacle.position - uav.position[:2])
        <= ray_radius + obstacle.radius
    ]

    sensing_log = []

    for (
        gx,
        gy,
        coverage_fraction,
    ) in visible_cells:
        cell_center = np.array(
            [
                min(
                    (
                        float(gx)
                        + 0.5
                    )
                    * cell_size,
                    map_size,
                ),
                min(
                    (
                        float(gy)
                        + 0.5
                    )
                    * cell_size,
                    map_size,
                ),
            ],
            dtype=np.float64,
        )

        if (
            bool(
                CONFIG[
                    "sensing_obstacle_occlusion"
                ]
            )
            and not camera_ground_point_has_los(
                uav,
                cell_center,
                obstacles=los_obstacles,
            )
        ):
            continue

        candidate_target_ids = (
            target_cells.get(
                (gx, gy),
                [],
            )
        )

        target_ids_in_fov = []

        for target_id in (
            candidate_target_ids
        ):
            target = targets_by_id[
                target_id
            ]

            in_footprint = (
                np.linalg.norm(
                    target.position
                    - uav.position[:2]
                )
                <= fov_radius
            )

            target_visible = (
                in_footprint
                and (
                    not bool(
                        CONFIG[
                            "sensing_obstacle_occlusion"
                        ]
                    )
                    or camera_target_has_los(
                        uav,
                        target,
                        obstacles=los_obstacles,
                    )
                )
            )

            if target_visible:
                target_ids_in_fov.append(
                    target_id
                )

        has_target = bool(
            target_ids_in_fov
        )

        effective_pf = float(
            1.0
            - (
                1.0 - base_pf
            )
            ** coverage_fraction
        )
        effective_pd = float(
            coverage_fraction
            * base_pd
            + (
                1.0
                - coverage_fraction
            )
            * effective_pf
        )

        if has_target:
            positive_probability = (
                base_pd
            )
        else:
            positive_probability = (
                effective_pf
            )

        observation = int(
            rng.random()
            < positive_probability
        )

        prior = float(
            belief_map[
                gy,
                gx,
            ]
        )
        posterior = bayes_update(
            prior,
            observation,
            effective_pd,
            effective_pf,
        )
        belief_map[
            gy,
            gx,
        ] = posterior

        sensing_log.append(
            {
                "gx": gx,
                "gy": gy,
                "target_ids": (
                    target_ids_in_fov
                ),
                "has_target": (
                    has_target
                ),
                "observation": (
                    observation
                ),
                "prior": prior,
                "posterior": (
                    posterior
                ),
                "pd": effective_pd,
                "pf": effective_pf,
                "base_pd": base_pd,
                "base_pf": base_pf,
                "coverage_fraction": (
                    float(
                        coverage_fraction
                    )
                ),
            }
        )

    return sensing_log


In [ ]:
def check_confirmation(
    uav,
    sensing_record,
):
    if not uav.active:
        return "none", []

    posterior = float(
        sensing_record[
            "posterior"
        ]
    )
    observation = int(
        sensing_record[
            "observation"
        ]
    )
    target_ids = list(
        sensing_record[
            "target_ids"
        ]
    )

    if (
        float(uav.position[2])
        > float(
            CONFIG[
                "fine_altitude"
            ]
        )
    ):
        return "none", []

    if (
        posterior
        < float(
            CONFIG[
                "confirmation_threshold"
            ]
        )
    ):
        return "none", []

    if observation != 1:
        return "none", []

    if target_ids:
        return (
            "true_confirmation",
            target_ids,
        )

    return (
        "false_confirmation",
        [],
    )


In [ ]:
@dataclass
class ConfirmationEvent:
    target_id: int | None
    uav_id: int
    step: int

    cell: tuple
    uav_position: np.ndarray

    altitude: float
    belief: float
    observation: int

    confirmation_type: str
    report_status: str | None = None


In [ ]:
def create_confirmation_events(
    uav,
    sensing_record,
    step
):
    confirmation_type, target_ids = check_confirmation(
        uav,
        sensing_record
    )

    if confirmation_type == "none":
        return []

    common = {
        "uav_id": int(uav.id),
        "step": int(step),
        "cell": (
            int(sensing_record["gx"]),
            int(sensing_record["gy"]),
        ),
        "uav_position": uav.position.copy(),
        "altitude": float(uav.position[2]),
        "belief": float(sensing_record["posterior"]),
        "observation": int(sensing_record["observation"]),
        "confirmation_type": confirmation_type,
    }

    if confirmation_type == "false_confirmation":
        return [ConfirmationEvent(target_id=None,**common,)]

    return [ConfirmationEvent(target_id=int(target_id),**common,) for target_id in target_ids]


In [ ]:
def pending_source_used_bytes(
    pending_reports,
    source_uav,
):
    return sum(
        int(report.size_bytes)
        for report
        in pending_reports
        if int(report.source_uav)
        == int(source_uav)
    )


In [ ]:
def enqueue_pending_report(
    pending_reports,
    report,
):
    if not isinstance(
        pending_reports,
        list,
    ):
        raise TypeError(
            "pending_reports must be a list"
        )

    if not isinstance(
        report,
        Report,
    ):
        raise TypeError(
            "report must be a Report"
        )

    duplicate = any(
        (
            item.target_id
            == report.target_id
            and item.source_uav
            == report.source_uav
        )
        for item
        in pending_reports
    )

    if duplicate:
        return False, "duplicate"

    used_bytes = (
        pending_source_used_bytes(
            pending_reports,
            report.source_uav,
        )
    )

    capacity = int(
        CONFIG[
            "pending_buffer_bytes"
        ]
    )

    if (
        used_bytes
        + int(
            report.size_bytes
        )
        > capacity
    ):
        return False, "pending_full"

    pending_reports.append(
        report
    )

    return True, "enqueued"


In [ ]:
def process_confirmation_events(
    events,
    targets,
    report_buffers,
    pending_reports,
    gcs_received_target_ids,
):
    reports = []
    event_log = []

    if not isinstance(
        report_buffers,
        list,
    ):
        raise TypeError(
            "report_buffers must be a list"
        )

    if not isinstance(
        pending_reports,
        list,
    ):
        raise TypeError(
            "pending_reports must be a list"
        )

    if not isinstance(
        gcs_received_target_ids,
        set,
    ):
        raise TypeError(
            "gcs_received_target_ids "
            "must be a set"
        )

    targets_by_id = {
        target.id: target
        for target in targets
    }

    generated_keys = set()

    for event in events:
        event_log.append(
            event
        )

        if (
            event.confirmation_type
            != "true_confirmation"
        ):
            continue

        if (
            event.target_id
            not in targets_by_id
        ):
            raise ValueError(
                "confirmation event "
                "target_id not found"
            )

        target = targets_by_id[
            event.target_id
        ]
        target.confirmed = True

        current_step = int(
            event.step
        )

        for buffer in (
            report_buffers
        ):
            remove_expired_reports(
                buffer,
                current_step,
            )

        remove_expired_reports(
            pending_reports,
            current_step,
        )

        if (
            target.id
            in gcs_received_target_ids
        ):
            event.report_status = (
                "already_delivered"
            )
            continue

        source_uav = int(
            event.uav_id
        )

        if not (
            0
            <= source_uav
            < len(
                report_buffers
            )
        ):
            raise ValueError(
                "confirmation event "
                "uav_id out of range"
            )

        generation_key = (
            source_uav,
            int(target.id),
        )

        report_already_buffered = (
            report_exists(
                report_buffers[
                    source_uav
                ],
                target.id,
            )
        )

        report_already_pending = any(
            (
                report.target_id
                == target.id
                and report.source_uav
                == source_uav
            )
            for report
            in pending_reports
        )

        if (
            report_already_buffered
            or report_already_pending
            or generation_key
            in generated_keys
        ):
            event.report_status = (
                "duplicate"
            )
            continue

        report = Report(
            target_id=target.id,
            source_uav=source_uav,
            created_step=(
                event.step
            ),
            size_bytes=(
                CONFIG[
                    "report_bytes"
                ]
            ),
            ttl_s=(
                CONFIG[
                    "report_ttl"
                ]
            ),
        )

        enqueued, reason = (
            enqueue_report(
                report_buffers[
                    source_uav
                ],
                report,
            )
        )

        stored = bool(
            enqueued
        )

        if enqueued:
            event.report_status = (
                "buffered"
            )

        if (
            not enqueued
            and reason
            == "buffer_full"
        ):
            (
                pending_enqueued,
                pending_reason,
            ) = enqueue_pending_report(
                pending_reports,
                report,
            )

            stored = bool(
                pending_enqueued
            )

            if pending_enqueued:
                event.report_status = (
                    "pending"
                )
            elif (
                pending_reason
                == "pending_full"
            ):
                event.report_status = (
                    "dropped"
                )
            elif (
                pending_reason
                == "duplicate"
            ):
                event.report_status = (
                    "duplicate"
                )

            if (
                not pending_enqueued
                and pending_reason
                not in {
                    "duplicate",
                    "pending_full",
                }
            ):
                raise RuntimeError(
                    "unexpected pending "
                    "enqueue result: "
                    f"{pending_reason}"
                )

        elif (
            not enqueued
            and reason
            == "duplicate"
        ):
            event.report_status = (
                "duplicate"
            )

        elif (
            not enqueued
            and reason
            != "duplicate"
        ):
            raise RuntimeError(
                "unexpected enqueue "
                f"result: {reason}"
            )

        if stored:
            reports.append(
                report
            )

        generated_keys.add(
            generation_key
        )

    return reports, event_log


In [ ]:
def create_report_buffers():
    return [[] for _ in range(CONFIG["num_uavs"])]


In [ ]:
def create_pending_reports():
    return []


In [ ]:
def create_gcs_received_target_ids():
    return set()


In [ ]:
def report_remaining_bytes(report):
    remaining = report.size_bytes - report.delivered_bytes
    return max(0, int(remaining))


In [ ]:
def report_is_complete(report):
    return report_remaining_bytes(report) == 0


In [ ]:
def buffer_used_bytes(buffer):
    return sum(report.size_bytes for report in buffer)


In [ ]:
def report_exists(buffer, target_id):
    return any(
        report.target_id == target_id for report in buffer
    )


In [ ]:
def report_exists_in_buffers(report_buffers, target_id, source_uav):
    return any(
        report.target_id == target_id
        and int(report.source_uav) == int(source_uav)
        for buffer in report_buffers
        for report in buffer
    )


In [ ]:
def pending_report_exists(pending_reports, target_id):
    return any(
        report.target_id == target_id
        for report in pending_reports
    )


In [ ]:
def enqueue_report(buffer, report):
    if not isinstance(buffer, list):
        raise TypeError("buffer must be a list")
    if not isinstance(report, Report):
        raise TypeError("report must be a Report")

    # A report whose bytes have all reached the GCS is retained until
    # mark_target_delivered_to_gcs() records mission-level delivery and
    # removes the report. This prevents losing delivery state between steps.
    if report_exists(buffer, report.target_id):
        return False, "duplicate"

    used_bytes = buffer_used_bytes(buffer)
    new_used_bytes = used_bytes + report.size_bytes

    if new_used_bytes > CONFIG["buffer_bytes"]:
        return False, "buffer_full"

    buffer.append(report)
    return True, "enqueued"


In [ ]:
def report_age_s(report, current_step):
    if (
        isinstance(current_step, (bool, np.bool_))
        or not isinstance(current_step, (int, np.integer))
    ):
        raise TypeError("current_step must be an integer")

    created_step = report.created_step

    if (
        isinstance(created_step, (bool, np.bool_))
        or not isinstance(created_step, (int, np.integer))
    ):
        raise TypeError("report.created_step must be an integer")

    current_step = int(current_step)
    created_step = int(created_step)

    if created_step < 0:
        raise ValueError("report.created_step must be >= 0")

    if current_step < created_step:
        raise ValueError(
            "current_step must be >= report.created_step"
        )

    dt = float(CONFIG["dt"])

    if not np.isfinite(dt) or dt <= 0.0:
        raise ValueError("CONFIG['dt'] must be finite and > 0")

    return float(
        (current_step - created_step) * dt
    )


In [ ]:
def report_is_expired(report, current_step):
    age_s = report_age_s(report,current_step)

    return age_s >= report.ttl_s


In [ ]:
def remove_expired_reports(
    buffer,
    current_step,
):
    kept_reports = []
    expired_reports = []

    for report in buffer:
        if report_is_expired(report, current_step):
            expired_reports.append(report)
        else:
            kept_reports.append(report)

    buffer[:] = kept_reports
    return expired_reports


In [ ]:
def remove_completed_reports(buffer):
    completed_reports = [
        report
        for report in buffer
        if report_is_complete(report)
    ]
    buffer[:] = [
        report
        for report in buffer
        if not report_is_complete(report)
    ]
    return completed_reports


In [ ]:
def cleanup_report_buffer(buffer, current_step):
    expired = remove_expired_reports(buffer, current_step)
    completed = [
        report
        for report in buffer
        if report_is_complete(report)
    ]
    # Completed reports are intentionally retained until the GCS-delivery
    # state is marked. FIFO selection skips them without deleting them.
    return {
        "expired": expired,
        "completed": completed,
    }


In [ ]:
def flush_pending_reports(
    pending_reports,
    report_buffers,
    current_step,
    gcs_received_target_ids,
    uavs,
):
    if not isinstance(pending_reports, list):
        raise TypeError("pending_reports must be a list")
    if not isinstance(report_buffers, list):
        raise TypeError("report_buffers must be a list")
    if not isinstance(gcs_received_target_ids, set):
        raise TypeError("gcs_received_target_ids must be a set")
    if not isinstance(uavs, list):
        raise TypeError("uavs must be a list")

    uavs_by_id = {}
    for uav in uavs:
        if not isinstance(uav, UAV):
            raise TypeError("uavs must contain only UAV objects")
        uav_id = int(uav.id)
        if uav_id in uavs_by_id:
            raise ValueError("UAV ids must be unique")
        uavs_by_id[uav_id] = uav

    for buffer in report_buffers:
        cleanup_report_buffer(buffer, current_step)

    kept_pending = []
    enqueued_target_ids = []
    expired_target_ids = []
    discarded_target_ids = []
    deferred_inactive_target_ids = []

    for report in pending_reports:
        if report.target_id in gcs_received_target_ids:
            discarded_target_ids.append(report.target_id)
            continue

        if report_is_expired(report, current_step):
            expired_target_ids.append(report.target_id)
            continue

        source_uav = int(report.source_uav)
        if not 0 <= source_uav < len(report_buffers):
            raise ValueError("pending report source_uav out of range")
        if source_uav not in uavs_by_id:
            raise ValueError("pending report source_uav not found")

        # Duplicate suppression uses the logical report key
        # (target_id, source_uav) across all replicated swarm buffers.
        if report_exists_in_buffers(
            report_buffers,
            report.target_id,
            source_uav,
        ):
            discarded_target_ids.append(report.target_id)
            continue

        if not uavs_by_id[source_uav].active:
            kept_pending.append(report)
            deferred_inactive_target_ids.append(report.target_id)
            continue

        enqueued, reason = enqueue_report(
            report_buffers[source_uav],
            report,
        )

        if enqueued:
            enqueued_target_ids.append(report.target_id)
        elif reason == "buffer_full":
            kept_pending.append(report)
        elif reason == "duplicate":
            discarded_target_ids.append(report.target_id)
        else:
            raise RuntimeError(f"unexpected enqueue result: {reason}")

    pending_reports[:] = kept_pending

    return {
        "enqueued_target_ids": enqueued_target_ids,
        "expired_target_ids": expired_target_ids,
        "discarded_target_ids": discarded_target_ids,
        "deferred_inactive_target_ids": deferred_inactive_target_ids,
    }


In [ ]:
def mark_target_delivered_to_gcs(
    target_id,
    gcs_received_target_ids,
    report_buffers,
    pending_reports,
):
    if isinstance(target_id, (bool, np.bool_)) or not isinstance(
        target_id,
        (int, np.integer),
    ):
        raise TypeError("target_id must be an integer")

    target_id = int(target_id)
    if target_id < 0:
        raise ValueError("target_id must be >= 0")
    if not isinstance(gcs_received_target_ids, set):
        raise TypeError("gcs_received_target_ids must be a set")

    gcs_received_target_ids.add(target_id)

    removed_from_buffers = 0
    for buffer in report_buffers:
        before = len(buffer)
        buffer[:] = [
            report
            for report in buffer
            if report.target_id != target_id
        ]
        removed_from_buffers += before - len(buffer)

    before_pending = len(pending_reports)
    pending_reports[:] = [
        report
        for report in pending_reports
        if report.target_id != target_id
    ]

    return {
        "removed_from_buffers": removed_from_buffers,
        "removed_from_pending": before_pending - len(pending_reports),
    }


In [ ]:
def distance_3d(position_a, position_b):
    position_a = np.asarray(position_a,dtype=np.float64)
    position_b = np.asarray(position_b, dtype=np.float64)

    if position_a.shape != (3,):
        raise ValueError("position_a must have shape (3,)")

    if position_b.shape != (3,):
        raise ValueError("position_b must have shape (3,)")

    if not np.all(np.isfinite(position_a)):
        raise ValueError("position_a must contain only finite values")

    if not np.all(np.isfinite(position_b)):
        raise ValueError("position_b must contain only finite values")

    return np.linalg.norm(position_a - position_b)


In [ ]:
def positions_can_communicate(position_a,position_b,max_range_m):
    max_range_m = float(max_range_m)

    if (
        not np.isfinite(max_range_m)
        or max_range_m <= 0.0
    ):
        raise ValueError(
            "max_range_m must be finite and > 0"
        )

    return (distance_3d(position_a,position_b)<= max_range_m)


In [ ]:
def uavs_can_communicate(uav_a,uav_b):
    if uav_a.id == uav_b.id:
        return False

    if not uav_a.active:
        return False

    if not uav_b.active:
        return False

    return positions_can_communicate(
        uav_a.position,
        uav_b.position,
        CONFIG["peer_contact_range_m"]
    )


In [ ]:
def uav_can_reach_gcs(uav):
    if not uav.active:
        return False

    return positions_can_communicate(
        uav.position,
        CONFIG["gcs_position"],
        CONFIG["gcs_contact_range_m"],
    )


In [ ]:
def get_uav_neighbors(uav,uavs):
    neighbors = []

    for other in uavs:
        if uavs_can_communicate(uav,other):
            neighbors.append(other.id)

    return neighbors


In [ ]:
GCS_NODE = -1


In [ ]:
SILENT_DESTINATION = None


In [ ]:
def communication_choices(sender_id,num_uavs=None):
    if num_uavs is None:
        num_uavs = CONFIG["num_uavs"]

    if (isinstance(sender_id, bool) or not isinstance(sender_id,(int, np.integer))):
        raise TypeError("sender_id must be an integer")

    if (isinstance(num_uavs, bool) or not isinstance(num_uavs,(int, np.integer))):
        raise TypeError("num_uavs must be an integer")

    sender_id = int(sender_id)
    num_uavs = int(num_uavs)

    if num_uavs < 1:
        raise ValueError("num_uavs must be >= 1")

    if not 0 <= sender_id < num_uavs:
        raise ValueError("sender_id out of range")

    peers = tuple(
        uav_id
        for uav_id in range(num_uavs)
        if uav_id != sender_id
    )

    return (SILENT_DESTINATION,*peers,GCS_NODE)


In [ ]:
def decode_destination(
    sender_id,
    destination_index,
    num_uavs=None,
):
    if num_uavs is None:
        num_uavs = CONFIG["num_uavs"]

    if (
        isinstance(
            destination_index,
            (bool, np.bool_),
        )
        or not isinstance(
            destination_index,
            (int, np.integer),
        )
    ):
        raise TypeError(
            "destination_index must be an integer"
        )

    destination_index = int(destination_index)

    choices = communication_choices(
        sender_id,
        num_uavs,
    )

    if not 0<= destination_index< len(choices):
        raise ValueError(
            "destination_index out of range"
        )

    return choices[destination_index]


In [ ]:
def decode_tx_power_w(power_action):
    power_array = np.asarray(power_action)

    if power_array.shape not in [(), (1,)]:
        raise ValueError(
            "power_action must be a scalar or have shape (1,)"
        )

    if (
        np.issubdtype(power_array.dtype, np.bool_)
        or not np.issubdtype(power_array.dtype, np.number)
    ):
        raise TypeError(
            "power_action must be numeric and not boolean"
        )

    power_value = float(power_array.item())

    if not np.isfinite(power_value):
        raise ValueError(
            "power_action must be finite"
        )

    if not -1.0 <= power_value <= 1.0:
        raise ValueError(
            "power_action must be within [-1, 1]"
        )

    power_min = float(CONFIG["tx_power_min_w"])
    power_max = float(CONFIG["tx_power_max_w"])

    if (
        not np.isfinite(power_min)
        or not np.isfinite(power_max)
        or power_min <= 0.0
        or power_max < power_min
    ):
        raise ValueError(
            "invalid TX power range"
        )

    normalized = (power_value + 1.0) / 2.0

    return float(
        power_min
        + normalized
        * (power_max - power_min)
    )


In [ ]:
@dataclass
class HybridAction:
    movement: np.ndarray
    destination: int | None
    tx_power_w: float


In [ ]:
def decode_hybrid_action(
    sender_id,
    action,
    num_uavs=None,
):
    if num_uavs is None:
        num_uavs = CONFIG["num_uavs"]

    if not isinstance(action, dict):
        raise TypeError(
            "action must be a dict"
        )

    required_keys = {
        "motion",
        "destination",
        "power",
    }

    if set(action.keys()) != required_keys:
        raise ValueError(
            "action must contain exactly "
            "motion, destination, power"
        )

    motion_object = np.asarray(
        action["motion"],
        dtype=object,
    )

    if motion_object.shape != (3,):
        raise ValueError(
            "motion must have shape (3,)"
        )

    for value in motion_object:
        if isinstance(value, (bool, np.bool_)):
            raise TypeError(
                "motion must be numeric and not boolean"
            )

        if not isinstance(
            value,
            (int, float, np.integer, np.floating),
        ):
            raise TypeError(
                "motion must be numeric and not boolean"
            )

    motion = motion_object.astype(
        np.float64,
    )

    if not np.all(np.isfinite(motion)):
        raise ValueError(
            "motion must contain only finite values"
        )

    if (
        np.any(motion < -1.0)
        or np.any(motion > 1.0)
    ):
        raise ValueError(
            "motion must be within [-1, 1]"
        )

    destination = decode_destination(
        sender_id,
        action["destination"],
        num_uavs,
    )

    decoded_power_w = decode_tx_power_w(action["power"])

    if destination is SILENT_DESTINATION:
        tx_power_w = 0.0
    else:
        tx_power_w = decoded_power_w

    return HybridAction(
        movement=motion.copy(),
        destination=destination,
        tx_power_w=tx_power_w,
    )


In [ ]:
def select_report_for_transmission(
    buffer,
    current_step,
):
    if (
        isinstance(current_step, (bool, np.bool_))
        or not isinstance(current_step, (int, np.integer))
    ):
        raise TypeError("current_step must be an integer")

    current_step = int(current_step)

    if not isinstance(buffer, list):
        raise TypeError("buffer must be a list")

    for report in buffer:
        if not isinstance(report, Report):
            raise TypeError("buffer must contain only Report objects")

    cleanup_report_buffer(buffer, current_step)

    for report in buffer:
        if not report_is_complete(report):
            return report

    return None


In [ ]:
@dataclass(frozen=True)
class TransmissionIntent:
    sender: int
    recipient: int
    target_id: int
    requested_bytes: int
    tx_power_w: float

    def __post_init__(self):
        for name, value in (
            ("sender", self.sender),
            ("recipient", self.recipient),
            ("target_id", self.target_id),
            ("requested_bytes", self.requested_bytes),
        ):
            if (
                isinstance(value, (bool, np.bool_))
                or not isinstance(
                    value,
                    (int, np.integer),
                )
            ):
                raise TypeError(
                    f"{name} must be an integer"
                )

        sender = int(self.sender)
        recipient = int(self.recipient)
        target_id = int(self.target_id)
        requested_bytes = int(
            self.requested_bytes
        )
        tx_power_w = float(
            self.tx_power_w
        )

        num_uavs = int(
            CONFIG["num_uavs"]
        )

        if not 0 <= sender < num_uavs:
            raise ValueError(
                "sender out of range"
            )

        if (
            recipient != GCS_NODE
            and not 0 <= recipient < num_uavs
        ):
            raise ValueError(
                "recipient must be GCS_NODE "
                "or a valid UAV id"
            )

        if recipient == sender:
            raise ValueError(
                "sender and recipient "
                "must be different"
            )

        if target_id < 0:
            raise ValueError(
                "target_id must be >= 0"
            )

        if requested_bytes <= 0:
            raise ValueError(
                "requested_bytes must be > 0"
            )

        power_min = float(
            CONFIG["tx_power_min_w"]
        )
        power_max = float(
            CONFIG["tx_power_max_w"]
        )

        if (
            not np.isfinite(tx_power_w)
            or not power_min
            <= tx_power_w
            <= power_max
        ):
            raise ValueError(
                "tx_power_w outside "
                "configured range"
            )

        object.__setattr__(
            self,
            "sender",
            sender,
        )
        object.__setattr__(
            self,
            "recipient",
            recipient,
        )
        object.__setattr__(
            self,
            "target_id",
            target_id,
        )
        object.__setattr__(
            self,
            "requested_bytes",
            requested_bytes,
        )
        object.__setattr__(
            self,
            "tx_power_w",
            tx_power_w,
        )


In [ ]:
def build_transmission_intent(
    sender_id,
    hybrid_action,
    buffer,
    current_step,
    peer_transfer_states=None,
):
    if (
        isinstance(
            sender_id,
            (bool, np.bool_),
        )
        or not isinstance(
            sender_id,
            (int, np.integer),
        )
    ):
        raise TypeError(
            "sender_id must be an integer"
        )

    sender_id = int(sender_id)

    num_uavs = int(
        CONFIG["num_uavs"]
    )

    if not 0 <= sender_id < num_uavs:
        raise ValueError(
            "sender_id out of range"
        )

    if not isinstance(
        hybrid_action,
        HybridAction,
    ):
        raise TypeError(
            "hybrid_action must be HybridAction"
        )

    if (
        isinstance(current_step, (bool, np.bool_))
        or not isinstance(
            current_step,
            (int, np.integer),
        )
    ):
        raise TypeError(
            "current_step must be an integer"
        )

    current_step = int(current_step)

    if not isinstance(
        buffer,
        list,
    ):
        raise TypeError(
            "buffer must be a list"
        )

    for report in buffer:
        if not isinstance(
            report,
            Report,
        ):
            raise TypeError(
                "buffer must contain only "
                "Report objects"
            )

    if (
        peer_transfer_states is not None
        and not isinstance(
            peer_transfer_states,
            dict,
        )
    ):
        raise TypeError(
            "peer_transfer_states must be "
            "a dict or None"
        )

    if (
        hybrid_action.destination
        is SILENT_DESTINATION
    ):
        return None

    cleanup_report_buffer(
        buffer,
        current_step,
    )

    destination = (
        hybrid_action.destination
    )

    if destination == GCS_NODE:
        for report in buffer:
            if report_is_complete(
                report
            ):
                continue

            requested_bytes = (
                report_remaining_bytes(
                    report
                )
            )

            if requested_bytes <= 0:
                continue

            return TransmissionIntent(
                sender=sender_id,
                recipient=destination,
                target_id=report.target_id,
                requested_bytes=requested_bytes,
                tx_power_w=(
                    hybrid_action.tx_power_w
                ),
            )

        return None

    # Peer destination: preserve FIFO order, but skip any report
    # that this exact peer already received completely.
    for report in buffer:
        if report_is_complete(
            report
        ):
            continue

        requested_bytes = int(
            report.size_bytes
        )

        if peer_transfer_states is not None:
            key = peer_transfer_key(
                sender_id,
                destination,
                report.target_id,
            )

            state = (
                peer_transfer_states.get(
                    key
                )
            )

            if state is not None:
                if not isinstance(
                    state,
                    PeerTransferState,
                ):
                    raise TypeError(
                        "peer_transfer_states "
                        "values must be "
                        "PeerTransferState"
                    )

                if peer_transfer_is_expired(
                    state,
                    current_step,
                ):
                    del peer_transfer_states[
                        key
                    ]
                    state = None

            if state is not None:
                if (
                    state.source_uav
                    != report.source_uav
                    or state.created_step
                    != report.created_step
                    or state.size_bytes
                    != report.size_bytes
                    or not np.isclose(
                        state.ttl_s,
                        report.ttl_s,
                    )
                ):
                    raise ValueError(
                        "peer transfer state "
                        "does not match "
                        "sender report"
                    )

                requested_bytes = (
                    peer_transfer_remaining_bytes(
                        state
                    )
                )

                if requested_bytes <= 0:
                    # This peer already has the full report.
                    # Continue FIFO search so later reports are
                    # not head-of-line blocked.
                    continue

        return TransmissionIntent(
            sender=sender_id,
            recipient=destination,
            target_id=report.target_id,
            requested_bytes=(
                requested_bytes
            ),
            tx_power_w=(
                hybrid_action.tx_power_w
            ),
        )

    return None


In [ ]:
def transmission_is_feasible(
    intent,
    uavs,
):
    if not isinstance(
        intent,
        TransmissionIntent,
    ):
        raise TypeError(
            "intent must be TransmissionIntent"
        )

    if not isinstance(uavs, list):
        raise TypeError(
            "uavs must be a list"
        )

    num_uavs = int(
        CONFIG["num_uavs"]
    )

    if len(uavs) != num_uavs:
        raise ValueError(
            "uavs length does not match "
            "CONFIG['num_uavs']"
        )

    for uav in uavs:
        if not isinstance(uav, UAV):
            raise TypeError(
                "uavs must contain only UAV objects"
            )

        if (
            isinstance(uav.id, (bool, np.bool_))
            or not isinstance(
                uav.id,
                (int, np.integer),
            )
        ):
            raise TypeError(
                "each UAV id must be an integer"
            )

    ids = [
        int(uav.id)
        for uav in uavs
    ]

    if len(set(ids)) != num_uavs:
        raise ValueError(
            "UAV ids must be unique"
        )

    if set(ids) != set(range(num_uavs)):
        raise ValueError(
            "UAV ids must be exactly "
            "0..num_uavs-1"
        )

    uavs_by_id = {
        int(uav.id): uav
        for uav in uavs
    }

    sender = uavs_by_id[
        intent.sender
    ]

    if not sender.active:
        return False

    if intent.recipient == GCS_NODE:
        return bool(
            uav_can_reach_gcs(
                sender
            )
        )

    recipient = uavs_by_id[
        intent.recipient
    ]

    return bool(
        uavs_can_communicate(
            sender,
            recipient,
        )
    )


In [ ]:
LOS_LINK = "los"


In [ ]:
NLOS_LINK = "nlos"


In [ ]:
def positions_have_los(
    position_a,
    position_b,
    obstacles,
):
    if obstacles is None:
        obstacles = []

    if not isinstance(obstacles, list):
        raise TypeError("obstacles must be a list")

    for obstacle in obstacles:
        if not isinstance(obstacle, Obstacle):
            raise TypeError(
                "obstacles must contain only Obstacle objects"
            )

    return not segment_intersects_obstacle(
        position_a,
        position_b,
        obstacles,
    )


In [ ]:
def classify_link_state(
    position_a,
    position_b,
    obstacles,
):
    if positions_have_los(
        position_a,
        position_b,
        obstacles,
    ):
        return LOS_LINK

    return NLOS_LINK


In [ ]:
def db_to_linear(db_value):
    db_value = float(db_value)

    if not np.isfinite(db_value):
        raise ValueError(
            "db_value must be finite"
        )

    return float(
        10.0 ** (db_value / 10.0)
    )


In [ ]:
def dbm_to_w(dbm_value):
    dbm_value = float(dbm_value)

    if not np.isfinite(dbm_value):
        raise ValueError(
            "dbm_value must be finite"
        )

    return float(
        10.0 ** (
            (dbm_value - 30.0) / 10.0
        )
    )


In [ ]:
def calculate_link_snr(
    tx_power_w,
    distance_m,
    additional_loss_db=0.0,
):
    tx_power_w = float(tx_power_w)
    distance_m = float(distance_m)
    additional_loss_db = float(additional_loss_db)

    if (
        not np.isfinite(tx_power_w)
        or tx_power_w <= 0.0
    ):
        raise ValueError(
            "tx_power_w must be finite and > 0"
        )

    if (
        not np.isfinite(distance_m)
        or distance_m < 0.0
    ):
        raise ValueError(
            "distance_m must be finite and >= 0"
        )

    if (
        not np.isfinite(additional_loss_db)
        or additional_loss_db < 0.0
    ):
        raise ValueError(
            "additional_loss_db must be finite and >= 0"
        )

    power_min = float(
        CONFIG["tx_power_min_w"]
    )
    power_max = float(
        CONFIG["tx_power_max_w"]
    )

    if not (
        power_min
        <= tx_power_w
        <= power_max
    ):
        raise ValueError(
            "tx_power_w outside configured range"
        )

    reference_distance_m = float(
        CONFIG["comm_reference_distance_m"]
    )
    path_loss_exponent = float(
        CONFIG["comm_path_loss_exponent"]
    )

    if (
        not np.isfinite(reference_distance_m)
        or reference_distance_m <= 0.0
    ):
        raise ValueError(
            "comm_reference_distance_m "
            "must be finite and > 0"
        )

    if (
        not np.isfinite(path_loss_exponent)
        or path_loss_exponent <= 0.0
    ):
        raise ValueError(
            "comm_path_loss_exponent "
            "must be finite and > 0"
        )

    reference_gain_linear = db_to_linear(
        CONFIG["comm_reference_gain_db"]
    )

    noise_power_w = dbm_to_w(
        CONFIG["comm_noise_power_dbm"]
    )

    if (
        reference_gain_linear <= 0.0
        or noise_power_w <= 0.0
    ):
        raise ValueError(
            "invalid communication model"
        )

    effective_distance_m = max(
        distance_m,
        reference_distance_m,
    )

    channel_power_gain = (
        reference_gain_linear
        * (
            reference_distance_m
            / effective_distance_m
        )
        ** path_loss_exponent
    )

    additional_gain = db_to_linear(
        -additional_loss_db
    )

    received_power_w = (
        tx_power_w
        * channel_power_gain
        * additional_gain
    )

    snr_linear = (
        received_power_w
        / noise_power_w
    )

    return float(snr_linear)


In [ ]:
def snr_linear_to_db(snr_linear):
    snr_linear = float(snr_linear)

    if not np.isfinite(snr_linear) or snr_linear < 0.0:
        raise ValueError(
            "snr_linear must be finite and >= 0"
        )

    if snr_linear == 0.0:
        return float("-inf")

    return float(
        10.0 * np.log10(snr_linear)
    )


In [ ]:
def transmission_distance_m(
    intent,
    uavs,
):
    if not isinstance(
        intent,
        TransmissionIntent,
    ):
        raise TypeError(
            "intent must be TransmissionIntent"
        )

    if not isinstance(uavs, list):
        raise TypeError(
            "uavs must be a list"
        )

    uavs_by_id = {
        int(uav.id): uav
        for uav in uavs
        if isinstance(uav, UAV)
    }

    if intent.sender not in uavs_by_id:
        raise ValueError(
            "sender UAV not found"
        )

    sender = uavs_by_id[
        intent.sender
    ]

    if intent.recipient == GCS_NODE:
        return distance_3d(
            sender.position,
            CONFIG["gcs_position"],
        )

    if intent.recipient not in uavs_by_id:
        raise ValueError(
            "recipient UAV not found"
        )

    recipient = uavs_by_id[
        intent.recipient
    ]

    return distance_3d(
        sender.position,
        recipient.position,
    )


In [ ]:
def calculate_intent_snr(
    intent,
    uavs,
    obstacles=None,
):
    if obstacles is None:
        obstacles = []

    if not transmission_is_feasible(
        intent,
        uavs,
    ):
        return 0.0

    distance_m = transmission_distance_m(
        intent,
        uavs,
    )

    uavs_by_id = {
        int(uav.id): uav
        for uav in uavs
        if isinstance(uav, UAV)
    }

    sender_position = uavs_by_id[
        intent.sender
    ].position

    if intent.recipient == GCS_NODE:
        recipient_position = np.asarray(
            CONFIG["gcs_position"],
            dtype=np.float64,
        )
    else:
        recipient_position = uavs_by_id[
            intent.recipient
        ].position

    link_state = classify_link_state(
        sender_position,
        recipient_position,
        obstacles,
    )

    if link_state == LOS_LINK:
        additional_loss_db = 0.0
    else:
        additional_loss_db = float(
            CONFIG["comm_nlos_additional_loss_db"]
        )

    return calculate_link_snr(
        intent.tx_power_w,
        distance_m,
        additional_loss_db=additional_loss_db,
    )


In [ ]:
def calculate_link_rate_bps(
    snr_linear,
):
    snr_linear = float(
        snr_linear
    )

    if (
        not np.isfinite(
            snr_linear
        )
        or snr_linear < 0.0
    ):
        raise ValueError(
            "snr_linear must be "
            "finite and >= 0"
        )

    bandwidth_hz = float(
        CONFIG[
            "comm_bandwidth_hz"
        ]
    )
    max_rate_bps = float(
        CONFIG[
            "comm_max_link_rate_bps"
        ]
    )

    if (
        not np.isfinite(
            bandwidth_hz
        )
        or bandwidth_hz <= 0.0
    ):
        raise ValueError(
            "comm_bandwidth_hz "
            "must be finite and > 0"
        )

    if (
        not np.isfinite(
            max_rate_bps
        )
        or max_rate_bps <= 0.0
    ):
        raise ValueError(
            "comm_max_link_rate_bps "
            "must be finite and > 0"
        )

    if snr_linear == 0.0:
        return 0.0

    shannon_rate_bps = float(
        bandwidth_hz
        * np.log1p(
            snr_linear
        )
        / np.log(2.0)
    )

    return min(
        shannon_rate_bps,
        max_rate_bps,
    )


In [ ]:
def calculate_intent_rate_bps(
    intent,
    uavs,
    obstacles=None,
):
    snr_linear = (
        calculate_intent_snr(
            intent,
            uavs,
            obstacles=obstacles,
        )
    )

    threshold_db = float(
        CONFIG[
            "comm_snr_threshold_db"
        ]
    )

    if (
        not np.isfinite(
            threshold_db
        )
    ):
        raise ValueError(
            "comm_snr_threshold_db "
            "must be finite"
        )

    if (
        snr_linear_to_db(
            snr_linear
        )
        < threshold_db
    ):
        return 0.0

    return calculate_link_rate_bps(
        snr_linear
    )


In [ ]:
def calculate_intent_tx_bytes(
    intent,
    uavs,
    obstacles=None,
    dt=None,
):
    if dt is None:
        dt = CONFIG["dt"]

    dt = float(dt)

    if not np.isfinite(dt) or dt <= 0.0:
        raise ValueError(
            "dt must be finite and > 0"
        )

    rate_bps = calculate_intent_rate_bps(
        intent,
        uavs,
        obstacles=obstacles,
    )

    capacity_bytes = (
        rate_bps
        * dt
        / 8.0
    )

    transferable_bytes = int(
        np.floor(capacity_bytes)
    )

    return min(
        int(intent.requested_bytes),
        transferable_bytes,
    )


In [ ]:
@dataclass
class PeerTransferState:
    sender: int
    recipient: int
    target_id: int
    source_uav: int
    created_step: int
    size_bytes: int
    ttl_s: float
    received_bytes: int = 0

    def __post_init__(self):
        for name, value in (
            ("sender", self.sender),
            ("recipient", self.recipient),
            ("target_id", self.target_id),
            ("source_uav", self.source_uav),
            ("created_step", self.created_step),
            ("size_bytes", self.size_bytes),
            ("received_bytes", self.received_bytes),
        ):
            if (
                isinstance(value, (bool, np.bool_))
                or not isinstance(value, (int, np.integer))
            ):
                raise TypeError(
                    f"{name} must be an integer"
                )

        self.sender = int(self.sender)
        self.recipient = int(self.recipient)
        self.target_id = int(self.target_id)
        self.source_uav = int(self.source_uav)
        self.created_step = int(self.created_step)
        self.size_bytes = int(self.size_bytes)
        self.received_bytes = int(self.received_bytes)
        self.ttl_s = float(self.ttl_s)

        num_uavs = int(CONFIG["num_uavs"])

        if not 0 <= self.sender < num_uavs:
            raise ValueError("sender out of range")

        if not 0 <= self.recipient < num_uavs:
            raise ValueError("recipient out of range")

        if self.sender == self.recipient:
            raise ValueError(
                "sender and recipient must be different"
            )

        if self.target_id < 0:
            raise ValueError(
                "target_id must be >= 0"
            )

        if not 0 <= self.source_uav < num_uavs:
            raise ValueError(
                "source_uav out of range"
            )

        if self.created_step < 0:
            raise ValueError(
                "created_step must be >= 0"
            )

        if self.size_bytes <= 0:
            raise ValueError(
                "size_bytes must be > 0"
            )

        if (
            not np.isfinite(self.ttl_s)
            or self.ttl_s <= 0.0
        ):
            raise ValueError(
                "ttl_s must be finite and > 0"
            )

        if not (
            0
            <= self.received_bytes
            <= self.size_bytes
        ):
            raise ValueError(
                "received_bytes must be in "
                "[0, size_bytes]"
            )


In [ ]:
def make_info_pickle_safe(value):
    """Convert notebook-defined dataclasses in public info payloads to plain data."""
    if is_dataclass(value) and not isinstance(value, type):
        return {
            key: make_info_pickle_safe(item)
            for key, item in asdict(value).items()
        }

    if isinstance(value, dict):
        return {
            key: make_info_pickle_safe(item)
            for key, item in value.items()
        }

    if isinstance(value, list):
        return [
            make_info_pickle_safe(item)
            for item in value
        ]

    if isinstance(value, tuple):
        return tuple(
            make_info_pickle_safe(item)
            for item in value
        )

    if isinstance(value, set):
        return [
            make_info_pickle_safe(item)
            for item in sorted(
                value,
                key=repr,
            )
        ]

    if isinstance(value, np.generic):
        return value.item()

    return value


In [ ]:
def create_peer_transfer_states():
    return {}


In [ ]:
def peer_transfer_key(
    sender,
    recipient,
    target_id,
):
    for name, value in (
        ("sender", sender),
        ("recipient", recipient),
        ("target_id", target_id),
    ):
        if (
            isinstance(value, (bool, np.bool_))
            or not isinstance(value, (int, np.integer))
        ):
            raise TypeError(
                f"{name} must be an integer"
            )

    sender = int(sender)
    recipient = int(recipient)
    target_id = int(target_id)

    num_uavs = int(CONFIG["num_uavs"])

    if not 0 <= sender < num_uavs:
        raise ValueError(
            "sender out of range"
        )

    if not 0 <= recipient < num_uavs:
        raise ValueError(
            "recipient out of range"
        )

    if sender == recipient:
        raise ValueError(
            "sender and recipient must be different"
        )

    if target_id < 0:
        raise ValueError(
            "target_id must be >= 0"
        )

    return (
        sender,
        recipient,
        target_id,
    )


In [ ]:
def peer_transfer_remaining_bytes(state):
    if not isinstance(
        state,
        PeerTransferState,
    ):
        raise TypeError(
            "state must be PeerTransferState"
        )

    return max(
        0,
        int(
            state.size_bytes
            - state.received_bytes
        ),
    )


In [ ]:
def peer_transfer_is_complete(state):
    return (peer_transfer_remaining_bytes(state)== 0)


In [ ]:
def peer_transfer_age_s(
    state,
    current_step,
):
    if not isinstance(
        state,
        PeerTransferState,
    ):
        raise TypeError(
            "state must be PeerTransferState"
        )

    if (
        isinstance(current_step, (bool, np.bool_))
        or not isinstance(
            current_step,
            (int, np.integer),
        )
    ):
        raise TypeError(
            "current_step must be an integer"
        )

    current_step = int(current_step)

    if current_step < state.created_step:
        raise ValueError(
            "current_step must be >= "
            "state.created_step"
        )

    dt = float(CONFIG["dt"])

    if (
        not np.isfinite(dt)
        or dt <= 0.0
    ):
        raise ValueError(
            "CONFIG['dt'] must be finite and > 0"
        )

    return float((current_step - state.created_step)* dt)


In [ ]:
def peer_transfer_is_expired(
    state,
    current_step,
):
    return peer_transfer_age_s(state,current_step)>= state.ttl_s


In [ ]:
def find_report_in_buffer(
    buffer,
    target_id,
):
    if not isinstance(buffer, list):
        raise TypeError("buffer must be a list")

    if (
        isinstance(target_id, (bool, np.bool_))
        or not isinstance(
            target_id,
            (int, np.integer),
        )
    ):
        raise TypeError(
            "target_id must be an integer"
        )

    target_id = int(target_id)

    for report in buffer:
        if not isinstance(
            report,
            Report,
        ):
            raise TypeError(
                "buffer must contain only "
                "Report objects"
            )

        if report.target_id == target_id:
            return report

    return None


In [ ]:
def get_or_create_peer_transfer_state(
    intent,
    sender_buffer,
    peer_transfer_states,
    current_step,
):
    if not isinstance(
        intent,
        TransmissionIntent,
    ):
        raise TypeError(
            "intent must be TransmissionIntent"
        )

    if intent.recipient == GCS_NODE:
        raise ValueError(
            "peer transfer state is only "
            "for UAV-to-UAV transmissions"
        )

    if not isinstance(
        peer_transfer_states,
        dict,
    ):
        raise TypeError(
            "peer_transfer_states must be a dict"
        )

    report = find_report_in_buffer(
        sender_buffer,
        intent.target_id,
    )

    if report is None:
        raise ValueError(
            "intent target report not found "
            "in sender buffer"
        )

    if report_is_expired(
        report,
        current_step,
    ):
        raise ValueError(
            "cannot create transfer state "
            "for an expired report"
        )

    key = peer_transfer_key(
        intent.sender,
        intent.recipient,
        intent.target_id,
    )

    state = peer_transfer_states.get(
        key
    )

    if state is not None:
        if not isinstance(
            state,
            PeerTransferState,
        ):
            raise TypeError(
                "peer_transfer_states values "
                "must be PeerTransferState"
            )

        # A stale partial copy must not block a fresh report with the
        # same target_id after the original report lifetime expires.
        if peer_transfer_is_expired(
            state,
            current_step,
        ):
            del peer_transfer_states[key]
            state = None

    if state is not None:
        if (
            state.source_uav
            != report.source_uav
            or state.created_step
            != report.created_step
            or state.size_bytes
            != report.size_bytes
            or not np.isclose(
                state.ttl_s,
                report.ttl_s,
            )
        ):
            raise ValueError(
                "existing peer transfer state "
                "does not match sender report"
            )

        return state

    state = PeerTransferState(
        sender=int(intent.sender),
        recipient=int(intent.recipient),
        target_id=int(intent.target_id),
        source_uav=int(report.source_uav),
        created_step=int(report.created_step),
        size_bytes=int(report.size_bytes),
        ttl_s=float(report.ttl_s),
        received_bytes=0,
    )

    peer_transfer_states[key] = state

    return state


In [ ]:
def commit_peer_transfer_bytes(
    intent,
    tx_bytes,
    sender_buffer,
    peer_transfer_states,
    current_step,
):
    if not isinstance(
        intent,
        TransmissionIntent,
    ):
        raise TypeError(
            "intent must be TransmissionIntent"
        )

    if (
        isinstance(tx_bytes, (bool, np.bool_))
        or not isinstance(
            tx_bytes,
            (int, np.integer),
        )
    ):
        raise TypeError(
            "tx_bytes must be an integer"
        )

    tx_bytes = int(tx_bytes)

    if tx_bytes < 0:
        raise ValueError(
            "tx_bytes must be >= 0"
        )

    if tx_bytes > intent.requested_bytes:
        raise ValueError(
            "tx_bytes cannot exceed "
            "intent.requested_bytes"
        )

    state = get_or_create_peer_transfer_state(
        intent,
        sender_buffer,
        peer_transfer_states,
        current_step,
    )

    if peer_transfer_is_expired(
        state,
        current_step,
    ):
        raise ValueError(
            "cannot commit bytes to an "
            "expired peer transfer"
        )

    remaining = (
        peer_transfer_remaining_bytes(
            state
        )
    )

    committed_bytes = min(
        tx_bytes,
        remaining,
    )

    state.received_bytes += (
        committed_bytes
    )

    return {
        "state": state,
        "committed_bytes": (
            committed_bytes
        ),
        "remaining_bytes": (
            peer_transfer_remaining_bytes(
                state
            )
        ),
        "complete": (
            peer_transfer_is_complete(
                state
            )
        ),
    }


In [ ]:
def cleanup_peer_transfer_states(
    peer_transfer_states,
    current_step,
):
    if not isinstance(
        peer_transfer_states,
        dict,
    ):
        raise TypeError(
            "peer_transfer_states must be a dict"
        )

    expired_keys = []

    for key, state in list(
        peer_transfer_states.items()
    ):
        if not isinstance(
            state,
            PeerTransferState,
        ):
            raise TypeError(
                "peer_transfer_states values "
                "must be PeerTransferState"
            )

        if peer_transfer_is_expired(
            state,
            current_step,
        ):
            expired_keys.append(key)
            del peer_transfer_states[key]

    return expired_keys


In [ ]:
def known_gcs_delivered_bytes_for_transfer(
    state,
    report_buffers,
):
    if not isinstance(
        state,
        PeerTransferState,
    ):
        raise TypeError(
            "state must be PeerTransferState"
        )

    if not isinstance(
        report_buffers,
        list,
    ):
        raise TypeError(
            "report_buffers must be a list"
        )

    known_bytes = 0

    for buffer in report_buffers:
        if not isinstance(
            buffer,
            list,
        ):
            raise TypeError(
                "each report buffer must be a list"
            )

        for report in buffer:
            if not isinstance(
                report,
                Report,
            ):
                raise TypeError(
                    "report buffers must contain "
                    "only Report objects"
                )

            if report.target_id != state.target_id:
                continue

            # Only merge progress from the same report generation.
            if (
                report.source_uav
                != state.source_uav
                or report.created_step
                != state.created_step
                or report.size_bytes
                != state.size_bytes
                or not np.isclose(
                    report.ttl_s,
                    state.ttl_s,
                )
            ):
                continue

            known_bytes = max(
                known_bytes,
                int(
                    report.delivered_bytes
                ),
            )

    return min(
        known_bytes,
        int(state.size_bytes),
    )


In [ ]:
def commit_completed_peer_transfer_to_receiver(
    state_key,
    peer_transfer_states,
    report_buffers,
    current_step,
    gcs_received_target_ids=None,
):
    if not isinstance(
        peer_transfer_states,
        dict,
    ):
        raise TypeError(
            "peer_transfer_states must be a dict"
        )

    if not isinstance(
        report_buffers,
        list,
    ):
        raise TypeError(
            "report_buffers must be a list"
        )

    num_uavs = int(
        CONFIG["num_uavs"]
    )

    if len(report_buffers) != num_uavs:
        raise ValueError(
            "report_buffers length does not match "
            "CONFIG['num_uavs']"
        )

    for buffer in report_buffers:
        if not isinstance(buffer, list):
            raise TypeError(
                "each report buffer must be a list"
            )

    if gcs_received_target_ids is None:
        gcs_received_target_ids = set()

    if not isinstance(
        gcs_received_target_ids,
        set,
    ):
        raise TypeError(
            "gcs_received_target_ids must be a set"
        )

    if (
        not isinstance(state_key, tuple)
        or len(state_key) != 3
    ):
        raise TypeError(
            "state_key must be a "
            "(sender, recipient, target_id) tuple"
        )

    key = peer_transfer_key(
        state_key[0],
        state_key[1],
        state_key[2],
    )

    state = peer_transfer_states.get(
        key
    )

    if state is None:
        raise ValueError(
            "peer transfer state not found"
        )

    if not isinstance(
        state,
        PeerTransferState,
    ):
        raise TypeError(
            "peer_transfer_states values must "
            "be PeerTransferState"
        )

    if (
        isinstance(current_step, (bool, np.bool_))
        or not isinstance(
            current_step,
            (int, np.integer),
        )
    ):
        raise TypeError(
            "current_step must be an integer"
        )

    current_step = int(current_step)

    if not peer_transfer_is_complete(
        state
    ):
        return {
            "status": "incomplete",
            "report": None,
            "remaining_bytes": (
                peer_transfer_remaining_bytes(
                    state
                )
            ),
        }

    if peer_transfer_is_expired(
        state,
        current_step,
    ):
        del peer_transfer_states[key]

        return {
            "status": "expired",
            "report": None,
            "remaining_bytes": 0,
        }

    if (
        state.target_id
        in gcs_received_target_ids
    ):
        del peer_transfer_states[key]

        return {
            "status": "already_delivered",
            "report": None,
            "remaining_bytes": 0,
        }

    receiver_buffer = report_buffers[
        state.recipient
    ]

    cleanup_report_buffer(
        receiver_buffer,
        current_step,
    )

    known_gcs_delivered_bytes = (
        known_gcs_delivered_bytes_for_transfer(
            state,
            report_buffers,
        )
    )

    existing_report = find_report_in_buffer(
        receiver_buffer,
        state.target_id,
    )

    if existing_report is not None:
        existing_report.delivered_bytes = max(
            int(existing_report.delivered_bytes),
            int(known_gcs_delivered_bytes),
        )
        # The receiver already has a full copy. Keep the completed
        # transfer state so this sender does not retransmit it.
        return {
            "status": "already_present",
            "report": existing_report,
            "remaining_bytes": 0,
        }

    received_report = Report(
        target_id=state.target_id,
        source_uav=state.source_uav,
        created_step=state.created_step,
        size_bytes=state.size_bytes,
        ttl_s=state.ttl_s,
        delivered_bytes=(
            known_gcs_delivered_bytes
        ),
    )

    enqueued, reason = enqueue_report(
        receiver_buffer,
        received_report,
    )

    if enqueued:
        # Keep the completed state as a lightweight receipt that this
        # sender already transferred the full report to this recipient.
        # build_transmission_intent() will therefore return None for the
        # same hop instead of sending the same report again.
        return {
            "status": "enqueued",
            "report": received_report,
            "remaining_bytes": 0,
        }

    if reason == "buffer_full":
        return {
            "status": "buffer_full",
            "report": None,
            "remaining_bytes": 0,
        }

    if reason == "duplicate":
        existing_report = find_report_in_buffer(
            receiver_buffer,
            state.target_id,
        )

        return {
            "status": "already_present",
            "report": existing_report,
            "remaining_bytes": 0,
        }

    raise RuntimeError(
        f"unexpected enqueue result: {reason}"
    )


In [ ]:
def reports_are_same_generation(
    report_a,
    report_b,
):
    if not isinstance(
        report_a,
        Report,
    ):
        raise TypeError(
            "report_a must be Report"
        )

    if not isinstance(
        report_b,
        Report,
    ):
        raise TypeError(
            "report_b must be Report"
        )

    return bool(
        report_a.target_id
        == report_b.target_id
        and report_a.source_uav
        == report_b.source_uav
        and report_a.created_step
        == report_b.created_step
        and report_a.size_bytes
        == report_b.size_bytes
        and np.isclose(
            report_a.ttl_s,
            report_b.ttl_s,
        )
    )


In [ ]:
def remove_peer_transfer_states_for_target(
    peer_transfer_states,
    target_id,
):
    if not isinstance(
        peer_transfer_states,
        dict,
    ):
        raise TypeError(
            "peer_transfer_states must be a dict"
        )

    if (
        isinstance(target_id, (bool, np.bool_))
        or not isinstance(
            target_id,
            (int, np.integer),
        )
    ):
        raise TypeError(
            "target_id must be an integer"
        )

    target_id = int(target_id)

    if target_id < 0:
        raise ValueError(
            "target_id must be >= 0"
        )

    removed_keys = []

    for key, state in list(
        peer_transfer_states.items()
    ):
        if not isinstance(
            state,
            PeerTransferState,
        ):
            raise TypeError(
                "peer_transfer_states values "
                "must be PeerTransferState"
            )

        if state.target_id == target_id:
            removed_keys.append(key)
            del peer_transfer_states[key]

    return removed_keys


In [ ]:
def commit_gcs_transfer_bytes(
    intent,
    tx_bytes,
    report_buffers,
    pending_reports,
    gcs_received_target_ids,
    peer_transfer_states,
    current_step,
):
    if not isinstance(
        intent,
        TransmissionIntent,
    ):
        raise TypeError(
            "intent must be TransmissionIntent"
        )

    if intent.recipient != GCS_NODE:
        raise ValueError(
            "commit_gcs_transfer_bytes "
            "requires a GCS intent"
        )

    if (
        isinstance(tx_bytes, (bool, np.bool_))
        or not isinstance(
            tx_bytes,
            (int, np.integer),
        )
    ):
        raise TypeError(
            "tx_bytes must be an integer"
        )

    tx_bytes = int(tx_bytes)

    if tx_bytes < 0:
        raise ValueError(
            "tx_bytes must be >= 0"
        )

    if tx_bytes > intent.requested_bytes:
        raise ValueError(
            "tx_bytes cannot exceed "
            "intent.requested_bytes"
        )

    if not isinstance(
        report_buffers,
        list,
    ):
        raise TypeError(
            "report_buffers must be a list"
        )

    if len(report_buffers) != int(
        CONFIG["num_uavs"]
    ):
        raise ValueError(
            "report_buffers length does not "
            "match CONFIG['num_uavs']"
        )

    for buffer in report_buffers:
        if not isinstance(
            buffer,
            list,
        ):
            raise TypeError(
                "each report buffer "
                "must be a list"
            )

        for report in buffer:
            if not isinstance(
                report,
                Report,
            ):
                raise TypeError(
                    "report buffers must contain "
                    "only Report objects"
                )

    if not isinstance(
        pending_reports,
        list,
    ):
        raise TypeError(
            "pending_reports must be a list"
        )

    for report in pending_reports:
        if not isinstance(
            report,
            Report,
        ):
            raise TypeError(
                "pending_reports must contain "
                "only Report objects"
            )

    if not isinstance(
        gcs_received_target_ids,
        set,
    ):
        raise TypeError(
            "gcs_received_target_ids "
            "must be a set"
        )

    if not isinstance(
        peer_transfer_states,
        dict,
    ):
        raise TypeError(
            "peer_transfer_states must be a dict"
        )

    if (
        isinstance(current_step, (bool, np.bool_))
        or not isinstance(
            current_step,
            (int, np.integer),
        )
    ):
        raise TypeError(
            "current_step must be an integer"
        )

    current_step = int(
        current_step
    )

    if current_step < 0:
        raise ValueError(
            "current_step must be >= 0"
        )

    if (
        intent.target_id
        in gcs_received_target_ids
    ):
        # Make this path idempotent: if stale network copies remain for
        # any reason, calling the commit again restores the invariant
        # that a GCS-delivered target has no buffered/pending copies.
        cleanup_result = (
            mark_target_delivered_to_gcs(
                intent.target_id,
                gcs_received_target_ids,
                report_buffers,
                pending_reports,
            )
        )

        removed_peer_states = (
            remove_peer_transfer_states_for_target(
                peer_transfer_states,
                intent.target_id,
            )
        )

        return {
            "status": "already_delivered",
            "committed_bytes": 0,
            "delivered_bytes": 0,
            "remaining_bytes": 0,
            "removed_peer_states": (
                removed_peer_states
            ),
            "cleanup_result": (
                cleanup_result
            ),
        }

    sender_buffer = report_buffers[
        intent.sender
    ]

    sender_report = (
        find_report_in_buffer(
            sender_buffer,
            intent.target_id,
        )
    )

    if sender_report is None:
        raise ValueError(
            "intent target report "
            "not found in sender buffer"
        )

    if report_is_expired(
        sender_report,
        current_step,
    ):
        raise ValueError(
            "cannot commit an "
            "expired report to GCS"
        )

    report_copies = []

    for buffer in report_buffers:
        for report in buffer:
            if reports_are_same_generation(
                report,
                sender_report,
            ):
                report_copies.append(
                    report
                )

    if not report_copies:
        raise RuntimeError(
            "no matching report copies found"
        )

    known_delivered_bytes = max(
        int(report.delivered_bytes)
        for report in report_copies
    )

    remaining_bytes = max(
        0,
        int(sender_report.size_bytes)
        - known_delivered_bytes,
    )

    committed_bytes = min(
        tx_bytes,
        remaining_bytes,
    )

    new_delivered_bytes = (
        known_delivered_bytes
        + committed_bytes
    )

    for report in report_copies:
        report.delivered_bytes = (
            new_delivered_bytes
        )

    complete = (
        new_delivered_bytes
        >= sender_report.size_bytes
    )

    removed_peer_states = []

    if complete:
        mark_target_delivered_to_gcs(
            intent.target_id,
            gcs_received_target_ids,
            report_buffers,
            pending_reports,
        )

        removed_peer_states = (
            remove_peer_transfer_states_for_target(
                peer_transfer_states,
                intent.target_id,
            )
        )

    return {
        "status": (
            "delivered"
            if complete
            else "partial"
        ),
        "committed_bytes": (
            committed_bytes
        ),
        "delivered_bytes": (
            new_delivered_bytes
        ),
        "remaining_bytes": max(
            0,
            int(sender_report.size_bytes)
            - new_delivered_bytes,
        ),
        "removed_peer_states": (
            removed_peer_states
        ),
    }


In [ ]:
def retry_completed_peer_transfers(
    peer_transfer_states,
    report_buffers,
    current_step,
    gcs_received_target_ids,
):
    if not isinstance(
        peer_transfer_states,
        dict,
    ):
        raise TypeError(
            "peer_transfer_states must be a dict"
        )

    if not isinstance(
        report_buffers,
        list,
    ):
        raise TypeError(
            "report_buffers must be a list"
        )

    if len(report_buffers) != int(
        CONFIG["num_uavs"]
    ):
        raise ValueError(
            "report_buffers length does not "
            "match CONFIG['num_uavs']"
        )

    for buffer in report_buffers:
        if not isinstance(buffer, list):
            raise TypeError(
                "each report buffer must be a list"
            )

    if not isinstance(
        gcs_received_target_ids,
        set,
    ):
        raise TypeError(
            "gcs_received_target_ids "
            "must be a set"
        )

    if (
        isinstance(current_step, (bool, np.bool_))
        or not isinstance(
            current_step,
            (int, np.integer),
        )
    ):
        raise TypeError(
            "current_step must be an integer"
        )

    current_step = int(current_step)

    if current_step < 0:
        raise ValueError(
            "current_step must be >= 0"
        )

    results = []

    for key, state in list(
        peer_transfer_states.items()
    ):
        if not isinstance(
            state,
            PeerTransferState,
        ):
            raise TypeError(
                "peer_transfer_states values "
                "must be PeerTransferState"
            )

        if not peer_transfer_is_complete(
            state
        ):
            continue

        receiver_buffer = report_buffers[
            state.recipient
        ]

        # Remove stale receiver copies before deciding whether the
        # completed state can act as a receipt.
        cleanup_report_buffer(
            receiver_buffer,
            current_step,
        )

        # Expiration and global GCS delivery take precedence over receipt
        # retention. commit_completed_peer_transfer_to_receiver() removes
        # the completed state in both cases.
        if (
            peer_transfer_is_expired(
                state,
                current_step,
            )
            or state.target_id
            in gcs_received_target_ids
        ):
            result = (
                commit_completed_peer_transfer_to_receiver(
                    key,
                    peer_transfer_states,
                    report_buffers,
                    current_step,
                    gcs_received_target_ids,
                )
            )

            results.append(
                {
                    "key": key,
                    "status": result["status"],
                    "result": result,
                }
            )
            continue

        # A non-expired completed state is also used as a receipt after
        # the receiver has a full copy. Do not repeatedly re-commit it.
        existing_report = find_report_in_buffer(
            receiver_buffer,
            state.target_id,
        )

        if (
            existing_report is not None
            and reports_are_same_generation(
                existing_report,
                Report(
                    target_id=state.target_id,
                    source_uav=state.source_uav,
                    created_step=state.created_step,
                    size_bytes=state.size_bytes,
                    ttl_s=state.ttl_s,
                    delivered_bytes=0,
                ),
            )
        ):
            results.append(
                {
                    "key": key,
                    "status": "receipt_present",
                }
            )
            continue

        result = (
            commit_completed_peer_transfer_to_receiver(
                key,
                peer_transfer_states,
                report_buffers,
                current_step,
                gcs_received_target_ids,
            )
        )

        results.append(
            {
                "key": key,
                "status": result["status"],
                "result": result,
            }
        )

    return results


In [ ]:
def execute_transmission_intent(
    intent,
    uavs,
    report_buffers,
    pending_reports,
    gcs_received_target_ids,
    peer_transfer_states,
    current_step,
    obstacles=None,
    dt=None,
):
    if not isinstance(
        intent,
        TransmissionIntent,
    ):
        raise TypeError(
            "intent must be TransmissionIntent"
        )

    if not isinstance(uavs, list):
        raise TypeError(
            "uavs must be a list"
        )

    if not isinstance(
        report_buffers,
        list,
    ):
        raise TypeError(
            "report_buffers must be a list"
        )

    if len(report_buffers) != int(
        CONFIG["num_uavs"]
    ):
        raise ValueError(
            "report_buffers length does not "
            "match CONFIG['num_uavs']"
        )

    if not isinstance(
        pending_reports,
        list,
    ):
        raise TypeError(
            "pending_reports must be a list"
        )

    if not isinstance(
        gcs_received_target_ids,
        set,
    ):
        raise TypeError(
            "gcs_received_target_ids "
            "must be a set"
        )

    if not isinstance(
        peer_transfer_states,
        dict,
    ):
        raise TypeError(
            "peer_transfer_states must be a dict"
        )

    if (
        isinstance(current_step, (bool, np.bool_))
        or not isinstance(
            current_step,
            (int, np.integer),
        )
    ):
        raise TypeError(
            "current_step must be an integer"
        )

    current_step = int(current_step)

    if current_step < 0:
        raise ValueError(
            "current_step must be >= 0"
        )

    if obstacles is None:
        obstacles = []

    if not isinstance(obstacles, list):
        raise TypeError(
            "obstacles must be a list"
        )

    retry_results = (
        retry_completed_peer_transfers(
            peer_transfer_states,
            report_buffers,
            current_step,
            gcs_received_target_ids,
        )
    )

    # A stale intent can survive after another copy reaches GCS.
    # Restore the global-delivery invariant before doing any radio work.
    if (
        intent.target_id
        in gcs_received_target_ids
    ):
        cleanup_result = (
            mark_target_delivered_to_gcs(
                intent.target_id,
                gcs_received_target_ids,
                report_buffers,
                pending_reports,
            )
        )

        removed_peer_states = (
            remove_peer_transfer_states_for_target(
                peer_transfer_states,
                intent.target_id,
            )
        )

        return {
            "status": "already_delivered",
            "tx_bytes": 0,
            "retry_results": retry_results,
            "cleanup_result": cleanup_result,
            "removed_peer_states": (
                removed_peer_states
            ),
        }

    tx_bytes = calculate_intent_tx_bytes(
        intent,
        uavs,
        obstacles=obstacles,
        dt=dt,
    )

    if tx_bytes <= 0:
        return {
            "status": "no_transfer",
            "tx_bytes": 0,
            "retry_results": retry_results,
        }

    if intent.recipient == GCS_NODE:
        commit_result = (
            commit_gcs_transfer_bytes(
                intent,
                tx_bytes,
                report_buffers,
                pending_reports,
                gcs_received_target_ids,
                peer_transfer_states,
                current_step,
            )
        )

        return {
            "status": (
                "gcs_"
                + commit_result["status"]
            ),
            "tx_bytes": tx_bytes,
            "retry_results": retry_results,
            "commit_result": commit_result,
        }

    sender_buffer = report_buffers[
        intent.sender
    ]

    peer_result = (
        commit_peer_transfer_bytes(
            intent,
            tx_bytes,
            sender_buffer,
            peer_transfer_states,
            current_step,
        )
    )

    custody_result = None

    if peer_result["complete"]:
        key = peer_transfer_key(
            intent.sender,
            intent.recipient,
            intent.target_id,
        )

        custody_result = (
            commit_completed_peer_transfer_to_receiver(
                key,
                peer_transfer_states,
                report_buffers,
                current_step,
                gcs_received_target_ids,
            )
        )

    if custody_result is None:
        status = "peer_partial"
    else:
        status = (
            "peer_"
            + custody_result["status"]
        )

    return {
        "status": status,
        "tx_bytes": tx_bytes,
        "retry_results": retry_results,
        "peer_result": peer_result,
        "custody_result": custody_result,
    }


In [ ]:
class NetworkBackend(ABC):
    """Backend-neutral interface for mission-level network simulation."""

    @abstractmethod
    def reset(
        self,
        *,
        uavs,
        gcs_position,
        obstacles,
        report_buffers,
        pending_reports,
        gcs_received_target_ids,
    ):
        """Reset backend state and bind the current mission state."""
        raise NotImplementedError

    @abstractmethod
    def sync_positions(self, uavs):
        """Synchronize backend node positions with mission state."""
        raise NotImplementedError

    @abstractmethod
    def step(
        self,
        requests,
        dt,
        current_step,
    ):
        """Execute one mission-step worth of network requests."""
        raise NotImplementedError

    @abstractmethod
    def metrics(self):
        """Return backend metrics without mutating backend state."""
        raise NotImplementedError

    @abstractmethod
    def last_step_communication_energy_by_uav(self):
        """Return communication energy used by each mission UAV last step."""
        raise NotImplementedError


In [ ]:
class SimpleNetworkBackend(NetworkBackend):
    """Thin backend wrapper around the existing simple-network primitives."""

    def __init__(self):
        self._initialized = False
        self._uavs = None
        self._gcs_position = None
        self._obstacles = None
        self._report_buffers = None
        self._pending_reports = None
        self._gcs_received_target_ids = None
        self._peer_transfer_states = create_peer_transfer_states()
        self._metric_state = self._new_metric_state()
        self._last_step_comm_energy_by_uav = np.zeros(
            int(CONFIG["num_uavs"]),
            dtype=np.float64,
        )

    @staticmethod
    def _new_metric_state():
        return {
            "steps": 0,
            "requests": 0,
            "tx_bytes": 0,
            "communication_energy_j": 0.0,
            "expired_peer_states": 0,
            "custody_retry_events": 0,
            "status_counts": {},
        }

    @staticmethod
    def _validate_uavs(uavs):
        if not isinstance(uavs, list):
            raise TypeError("uavs must be a list")

        num_uavs = int(CONFIG["num_uavs"])

        if len(uavs) != num_uavs:
            raise ValueError(
                "uavs length does not match CONFIG['num_uavs']"
            )

        for expected_id, uav in enumerate(uavs):
            if not isinstance(uav, UAV):
                raise TypeError(
                    "uavs must contain only UAV objects"
                )

            if int(uav.id) != expected_id:
                raise ValueError(
                    "uavs must be ordered by contiguous UAV id"
                )

    @staticmethod
    def _validate_obstacles(obstacles):
        if not isinstance(obstacles, list):
            raise TypeError("obstacles must be a list")

        for obstacle in obstacles:
            if not isinstance(obstacle, Obstacle):
                raise TypeError(
                    "obstacles must contain only Obstacle objects"
                )

    @staticmethod
    def _validate_report_state(
        report_buffers,
        pending_reports,
        gcs_received_target_ids,
    ):
        if not isinstance(report_buffers, list):
            raise TypeError(
                "report_buffers must be a list"
            )

        if len(report_buffers) != int(
            CONFIG["num_uavs"]
        ):
            raise ValueError(
                "report_buffers length does not match "
                "CONFIG['num_uavs']"
            )

        for buffer in report_buffers:
            if not isinstance(buffer, list):
                raise TypeError(
                    "each report buffer must be a list"
                )

            for report in buffer:
                if not isinstance(report, Report):
                    raise TypeError(
                        "report buffers must contain "
                        "only Report objects"
                    )

        if not isinstance(pending_reports, list):
            raise TypeError(
                "pending_reports must be a list"
            )

        for report in pending_reports:
            if not isinstance(report, Report):
                raise TypeError(
                    "pending_reports must contain "
                    "only Report objects"
                )

        if not isinstance(
            gcs_received_target_ids,
            set,
        ):
            raise TypeError(
                "gcs_received_target_ids must be a set"
            )

    @staticmethod
    def _validated_gcs_position(gcs_position):
        position = np.asarray(
            gcs_position,
            dtype=np.float64,
        )

        if position.shape != (3,):
            raise ValueError(
                "gcs_position must have shape (3,)"
            )

        if not np.all(np.isfinite(position)):
            raise ValueError(
                "gcs_position must contain only finite values"
            )

        configured_position = np.asarray(
            CONFIG["gcs_position"],
            dtype=np.float64,
        )

        # Existing simple-network primitives read GCS position from
        # CONFIG directly. Reject a mismatch instead of silently using
        # two different GCS locations.
        if not np.allclose(
            position,
            configured_position,
            rtol=0.0,
            atol=1e-12,
        ):
            raise ValueError(
                "SimpleNetworkBackend gcs_position must match "
                "CONFIG['gcs_position']"
            )

        return position.copy()

    def reset(
        self,
        *,
        uavs,
        gcs_position,
        obstacles,
        report_buffers,
        pending_reports,
        gcs_received_target_ids,
    ):
        self._validate_uavs(uavs)
        self._validate_obstacles(obstacles)
        self._validate_report_state(
            report_buffers,
            pending_reports,
            gcs_received_target_ids,
        )

        self._uavs = uavs
        self._gcs_position = (
            self._validated_gcs_position(
                gcs_position
            )
        )
        self._obstacles = obstacles
        self._report_buffers = report_buffers
        self._pending_reports = pending_reports
        self._gcs_received_target_ids = (
            gcs_received_target_ids
        )
        self._peer_transfer_states = (
            create_peer_transfer_states()
        )
        self._metric_state = (
            self._new_metric_state()
        )
        self._last_step_comm_energy_by_uav = np.zeros(
            int(CONFIG["num_uavs"]),
            dtype=np.float64,
        )
        self._initialized = True

    def sync_positions(self, uavs):
        if not self._initialized:
            raise RuntimeError(
                "backend must be reset before sync_positions"
            )

        self._validate_uavs(uavs)
        self._uavs = uavs

    def step(
        self,
        requests,
        dt,
        current_step,
    ):
        if not self._initialized:
            raise RuntimeError(
                "backend must be reset before step"
            )

        if (
            isinstance(current_step, (bool, np.bool_))
            or not isinstance(
                current_step,
                (int, np.integer),
            )
        ):
            raise TypeError(
                "current_step must be an integer"
            )

        current_step = int(current_step)

        if current_step < 0:
            raise ValueError(
                "current_step must be >= 0"
            )

        if isinstance(dt, (bool, np.bool_)):
            raise TypeError(
                "dt must be numeric and not boolean"
            )

        dt = float(dt)

        if not np.isfinite(dt) or dt <= 0.0:
            raise ValueError(
                "dt must be finite and > 0"
            )

        # Report/peer TTL helpers currently use CONFIG['dt'].
        # Enforce one mission clock until those helpers are made
        # backend-neutral.
        if not np.isclose(
            dt,
            float(CONFIG["dt"]),
            rtol=0.0,
            atol=1e-12,
        ):
            raise ValueError(
                "SimpleNetworkBackend dt must match CONFIG['dt']"
            )

        if requests is None:
            request_list = []
        elif isinstance(
            requests,
            TransmissionIntent,
        ):
            request_list = [requests]
        elif isinstance(requests, (list, tuple)):
            request_list = list(requests)
        else:
            raise TypeError(
                "requests must be a TransmissionIntent, "
                "a list/tuple of intents, or None"
            )

        for request in request_list:
            if not isinstance(
                request,
                TransmissionIntent,
            ):
                raise TypeError(
                    "requests must contain only "
                    "TransmissionIntent objects"
                )

        sender_ids = [
            int(request.sender)
            for request in request_list
        ]

        if len(sender_ids) != len(
            set(sender_ids)
        ):
            raise ValueError(
                "each sender may submit at most one "
                "transmission request per network step"
            )

        expired_peer_states = (
            cleanup_peer_transfer_states(
                self._peer_transfer_states,
                current_step,
            )
        )
        custody_retry_results = (
            retry_completed_peer_transfers(
                self._peer_transfer_states,
                self._report_buffers,
                current_step,
                self._gcs_received_target_ids,
            )
        )

        self._metric_state[
            "expired_peer_states"
        ] += len(expired_peer_states)
        self._metric_state[
            "custody_retry_events"
        ] += len(custody_retry_results)

        self._last_step_comm_energy_by_uav.fill(
            0.0
        )

        results = []

        for request in request_list:
            result = execute_transmission_intent(
                request,
                self._uavs,
                self._report_buffers,
                self._pending_reports,
                self._gcs_received_target_ids,
                self._peer_transfer_states,
                current_step,
                obstacles=self._obstacles,
                dt=dt,
            )

            results.append(result)

            status = str(
                result.get(
                    "status",
                    "unknown",
                )
            )
            status_counts = self._metric_state[
                "status_counts"
            ]
            status_counts[status] = (
                status_counts.get(status, 0)
                + 1
            )
            tx_bytes = int(
                result.get(
                    "tx_bytes",
                    0,
                )
            )
            self._metric_state["tx_bytes"] += tx_bytes

            # TX attempts consume radio energy even if SNR is too low
            # and no payload bytes are delivered.
            if (
                status != "already_delivered"
                and int(request.requested_bytes) > 0
                and request.tx_power_w > 0.0
            ):
                phy_rate_bps = float(
                    CONFIG[
                        "comm_max_link_rate_bps"
                    ]
                )
                attempted_bytes = min(
                    int(request.requested_bytes),
                    int(
                        np.floor(
                            phy_rate_bps
                            * dt
                            / 8.0
                        )
                    ),
                )
                duration_s = min(
                    dt,
                    attempted_bytes
                    * 8.0
                    / phy_rate_bps,
                )
                energy_j = (
                    float(request.tx_power_w)
                    * duration_s
                )

                self._last_step_comm_energy_by_uav[
                    request.sender
                ] += energy_j
                self._metric_state[
                    "communication_energy_j"
                ] += energy_j

        self._metric_state["steps"] += 1
        self._metric_state["requests"] += len(
            request_list
        )

        return results

    @property
    def peer_transfer_states(self):
        if not self._initialized:
            raise RuntimeError(
                "backend must be reset before accessing "
                "peer_transfer_states"
            )

        return self._peer_transfer_states

    def metrics(self):
        status_counts = dict(
            self._metric_state[
                "status_counts"
            ]
        )

        return {
            "steps": int(
                self._metric_state["steps"]
            ),
            "requests": int(
                self._metric_state["requests"]
            ),
            "tx_bytes": int(
                self._metric_state["tx_bytes"]
            ),
            "communication_energy_j": float(
                self._metric_state[
                    "communication_energy_j"
                ]
            ),
            "expired_peer_states": int(
                self._metric_state[
                    "expired_peer_states"
                ]
            ),
            "custody_retry_events": int(
                self._metric_state[
                    "custody_retry_events"
                ]
            ),
            "status_counts": status_counts,
            "active_peer_transfers": len(
                self._peer_transfer_states
            ),
        }

    def last_step_communication_energy_by_uav(self):
        if not self._initialized:
            raise RuntimeError(
                "backend must be reset before "
                "reading communication energy"
            )

        return (
            self._last_step_comm_energy_by_uav
            .copy()
        )


In [ ]:
def resolve_uavnetsim_repo_path(repo_path=None):
    candidates = [
        repo_path,
        CONFIG.get("uavnetsim_path"),
        os.environ.get("UAVNETSIM_PATH"),
    ]

    # Also support a normal pip-installed UavNetSim distribution. The
    # package exposes simulator/, entities/, and utils/ as top-level packages,
    # so there is intentionally no importable top-level "uavnetsim" module.
    try:
        from importlib.metadata import distribution
        installed_root = Path(
            distribution("uavnetsim").locate_file("")
        ).resolve()
        candidates.append(installed_root)
    except Exception:
        pass

    checked = []

    for candidate in candidates:
        if candidate in (None, ""):
            continue

        path = Path(candidate).expanduser().resolve()
        checked.append(str(path))

        required = (
            path / "simulator" / "simulator.py",
            path / "entities" / "packet.py",
            path / "utils" / "config.py",
        )

        if all(item.is_file() for item in required):
            return path

    suffix = (
        f" Checked: {checked}."
        if checked
        else ""
    )

    raise FileNotFoundError(
        "UavNetSim repository was not found. "
        "Pass repo_path=... to UavNetSimBackend, set "
        "CONFIG['uavnetsim_path'], or set UAVNETSIM_PATH."
        + suffix
    )


In [ ]:
def uavnetsim_available(repo_path=None):
    try:
        resolve_uavnetsim_repo_path(
            repo_path
        )
    except FileNotFoundError:
        return False

    return True


In [ ]:
def _uavnetsim_scene_feature(
    obstacle,
    obstacle_id,
):
    if not isinstance(obstacle, Obstacle):
        raise TypeError(
            "obstacle must be an Obstacle"
        )

    sides = int(
        CONFIG[
            "uavnetsim_obstacle_polygon_sides"
        ]
    )

    if sides < 8:
        raise ValueError(
            "uavnetsim_obstacle_polygon_sides "
            "must be >= 8"
        )

    cx = float(obstacle.position[0])
    cy = float(obstacle.position[1])
    radius = float(obstacle.radius)

    footprint = []

    for index in range(sides):
        angle = (
            2.0
            * np.pi
            * index
            / sides
        )

        footprint.append(
            {
                "x": float(
                    cx
                    + radius
                    * np.cos(angle)
                ),
                "y": float(
                    cy
                    + radius
                    * np.sin(angle)
                ),
                "z": 0.0,
            }
        )

    return {
        "id": f"mission-obstacle-{obstacle_id}",
        "category": "building",
        "height": float(obstacle.height),
        "material": "itu_concrete",
        "source": "uav_search_target",
        "footprint": footprint,
    }


In [ ]:
def build_uavnetsim_scene_payload(
    obstacles,
):
    if not isinstance(obstacles, list):
        raise TypeError(
            "obstacles must be a list"
        )

    features = [
        _uavnetsim_scene_feature(
            obstacle,
            obstacle_id,
        )
        for obstacle_id, obstacle
        in enumerate(obstacles)
    ]

    map_size = float(
        CONFIG["map_size"]
    )

    if (
        not np.isfinite(map_size)
        or map_size <= 0.0
    ):
        raise ValueError(
            "CONFIG['map_size'] must be "
            "finite and > 0"
        )

    return {
        "schema_version": 1,
        "name": "uav-search-target",
        "anchor": {
            "latitude": 0.0,
            "longitude": 0.0,
        },
        "size_x": map_size,
        "size_y": map_size,
        "features": features,
    }


In [ ]:
def write_uavnetsim_scene(
    obstacles,
    directory,
):
    directory = Path(directory)
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    scene_path = (
        directory / "scene.json"
    )

    payload = (
        build_uavnetsim_scene_payload(
            obstacles
        )
    )

    scene_path.write_text(
        json.dumps(
            payload,
            indent=2,
        ),
        encoding="utf-8",
    )

    return scene_path


In [ ]:
class MarlSelectedNextHopRouting:
    """Routing shim that preserves the MARL-selected mission next hop."""

    def __init__(
        self,
        simulator,
        my_drone,
    ):
        self.simulator = simulator
        self.my_drone = my_drone

    def next_hop_selection(
        self,
        packet,
    ):
        enquire = False

        next_hop_id = getattr(
            packet,
            "forced_next_hop_id",
            packet.dst_drone.identifier,
        )

        try:
            next_hop_id = int(
                next_hop_id
            )
        except (TypeError, ValueError):
            return (
                False,
                packet,
                enquire,
            )

        if not (
            0
            <= next_hop_id
            < len(self.simulator.drones)
        ):
            return (
                False,
                packet,
                enquire,
            )

        if (
            next_hop_id
            == self.my_drone.identifier
        ):
            return (
                False,
                packet,
                enquire,
            )

        packet.next_hop_id = (
            next_hop_id
        )

        if (
            self.my_drone.identifier
            not in packet.intermediate_drones
        ):
            packet.intermediate_drones.append(
                self.my_drone.identifier
            )

        return (
            True,
            packet,
            enquire,
        )

    def packet_reception(
        self,
        packet,
        src_drone_id,
    ):
        import copy

        from entities.packet import (
            AckPacket,
            DataPacket,
        )
        from utils import config as uav_config

        if isinstance(
            packet,
            DataPacket,
        ):
            packet_copy = copy.copy(
                packet
            )

            destination_id = (
                packet_copy
                .dst_drone
                .identifier
            )

            if (
                destination_id
                == self.my_drone.identifier
            ):
                if (
                    packet_copy.packet_id
                    not in (
                        self.simulator
                        .metrics
                        .datapacket_arrived
                    )
                ):
                    self.simulator.metrics.calculate_metrics(
                        packet_copy
                    )

                uav_config.GL_ID_ACK_PACKET += 1

                ack_packet = AckPacket(
                    src_drone=self.my_drone,
                    dst_drone=(
                        self.simulator
                        .drones[src_drone_id]
                    ),
                    ack_packet_id=(
                        uav_config
                        .GL_ID_ACK_PACKET
                    ),
                    ack_packet_length=(
                        uav_config
                        .ACK_PACKET_LENGTH
                    ),
                    ack_packet=packet_copy,
                    simulator=self.simulator,
                    channel_id=(
                        packet_copy.channel_id
                    ),
                )

                yield self.simulator.env.timeout(
                    uav_config.SIFS_DURATION
                )

                ack_packet.increase_ttl()

                self.my_drone.mac_protocol.phy.unicast(
                    ack_packet,
                    src_drone_id,
                )

                yield self.simulator.env.timeout(
                    ack_packet.packet_length
                    / uav_config.BIT_RATE
                    * 1e6
                )

                return

            if (
                self.my_drone
                .transmitting_queue
                .qsize()
                < self.my_drone.max_queue_size
            ):
                self.my_drone.transmitting_queue.put(
                    packet_copy
                )

            return

        if isinstance(
            packet,
            AckPacket,
        ):
            data_packet = (
                packet.ack_packet
            )

            if (
                getattr(
                    data_packet,
                    "first_attempt_time",
                    None,
                )
                is not None
            ):
                self.simulator.metrics.mac_delay.append(
                    (
                        self.simulator.env.now
                        - data_packet.first_attempt_time
                    )
                    / 1e3
                )

            self.my_drone.remove_from_queue(
                data_packet
            )

            key = (
                f"wait_ack"
                f"{self.my_drone.identifier}"
                f"_{data_packet.packet_id}"
            )

            finish_state = (
                self.my_drone
                .mac_protocol
                .wait_ack_process_finish
                .get(
                    key,
                    1,
                )
            )

            if finish_state == 0:
                process = (
                    self.my_drone
                    .mac_protocol
                    .wait_ack_process_dict
                    .get(key)
                )

                if (
                    process is not None
                    and not process.triggered
                ):
                    self.my_drone.mac_protocol.wait_ack_process_finish[
                        key
                    ] = 1

                    process.interrupt()

            return

    def penalize(
        self,
        packet,
    ):
        return None


In [ ]:
class UavNetSimBackend(NetworkBackend):
    """Adapter from mission reports to UavNetSim packet-level networking."""

    def __init__(
        self,
        repo_path=None,
        seed=None,
    ):
        self.repo_path = repo_path
        self.seed = (
            int(CONFIG.get("seed", 44))
            if seed is None
            else int(seed)
        )

        self._initialized = False
        self._uavs = None
        self._gcs_position = None
        self._obstacles = None
        self._report_buffers = None
        self._pending_reports = None
        self._gcs_received_target_ids = None
        self._peer_transfer_states = (
            create_peer_transfer_states()
        )

        self._repo = None
        self._simpy = None
        self._simulator_class = None
        self._data_packet_class = None
        self._uav_config = None

        self._scene_directory = None
        self._environment = None
        self._simulator = None
        self._gcs_node_id = int(
            CONFIG["num_uavs"]
        )

        self._packet_records = {}
        self._metric_state = (
            self._new_metric_state()
        )
        self._last_step_comm_energy_by_uav = np.zeros(
            int(CONFIG["num_uavs"]),
            dtype=np.float64,
        )
        self._step_energy_budget_j = None

    @staticmethod
    def _new_metric_state():
        return {
            "steps": 0,
            "requests": 0,
            "packets_injected": 0,
            "packets_delivered": 0,
            "packets_failed": 0,
            "queue_limit_events": 0,
            "payload_bytes_deferred": 0,
            "payload_bytes_injected": 0,
            "payload_bytes_delivered": 0,
            "communication_energy_j": 0.0,
            "status_counts": {},
        }

    @staticmethod
    def _validate_backend_config():
        payload_bytes = int(
            CONFIG[
                "uavnetsim_payload_bytes"
            ]
        )

        if payload_bytes <= 0:
            raise ValueError(
                "uavnetsim_payload_bytes "
                "must be > 0"
            )

        packet_lifetime_s = float(
            CONFIG[
                "uavnetsim_packet_lifetime_s"
            ]
        )

        if (
            not np.isfinite(
                packet_lifetime_s
            )
            or packet_lifetime_s <= 0.0
        ):
            raise ValueError(
                "uavnetsim_packet_lifetime_s "
                "must be finite and > 0"
            )

        max_queue_size = int(
            CONFIG[
                "uavnetsim_max_queue_size"
            ]
        )

        if max_queue_size <= 0:
            raise ValueError(
                "uavnetsim_max_queue_size "
                "must be > 0"
            )

    def _load_uavnetsim(self):
        self._repo = (
            resolve_uavnetsim_repo_path(
                self.repo_path
            )
        )

        repo_string = str(
            self._repo
        )

        if repo_string not in sys.path:
            sys.path.insert(
                0,
                repo_string,
            )

        # UavNetSim's simulator.log configures a relative
        # running_log.log file at import time when the root logger has
        # no handlers. Add a temporary NullHandler so importing the
        # external simulator never writes files into this project.
        import logging

        root_logger = logging.getLogger()
        temporary_handler = None

        if not root_logger.handlers:
            temporary_handler = (
                logging.NullHandler()
            )
            root_logger.addHandler(
                temporary_handler
            )

        try:
            self._simpy = (
                importlib.import_module(
                    "simpy"
                )
            )
            simulator_module = (
                importlib.import_module(
                    "simulator.simulator"
                )
            )
            packet_module = (
                importlib.import_module(
                    "entities.packet"
                )
            )
            self._uav_config = (
                importlib.import_module(
                    "utils.config"
                )
            )
        finally:
            if (
                temporary_handler
                is not None
            ):
                root_logger.removeHandler(
                    temporary_handler
                )

        simulator_file = Path(
            simulator_module.__file__
        ).resolve()

        if (
            self._repo
            not in simulator_file.parents
        ):
            raise RuntimeError(
                "A different simulator package "
                "named 'simulator' is already "
                "loaded in this Python process"
            )

        self._simulator_class = (
            simulator_module.Simulator
        )
        self._data_packet_class = (
            packet_module.DataPacket
        )

    def _configure_uavnetsim(
        self,
        scene_directory,
    ):
        uav_config = self._uav_config

        num_network_nodes = (
            int(CONFIG["num_uavs"])
            + 1
        )

        uav_config.NUMBER_OF_DRONES = (
            num_network_nodes
        )
        uav_config.MAX_TTL = (
            num_network_nodes + 1
        )

        uav_config.ROUTING_PROTOCOL = (
            "DRL"
        )
        uav_config.DRL_ROUTING_PROTOCOL_CLASS = (
            MarlSelectedNextHopRouting
        )
        uav_config.DRL_HELLO_PACKET_CLASS = (
            None
        )
        uav_config.DRL_NEIGHBOR_TABLE_CLASS = (
            None
        )

        uav_config.MAC_PROTOCOL = str(
            CONFIG[
                "uavnetsim_mac_protocol"
            ]
        )
        uav_config.MOBILITY_MODEL = (
            "GaussMarkov3D"
        )

        uav_config.CHANNEL_MODE = str(
            CONFIG[
                "uavnetsim_channel_mode"
            ]
        )
        uav_config.LOS_A2A_MODEL = str(
            CONFIG[
                "uavnetsim_los_model"
            ]
        )
        uav_config.NLOS_A2A_MODEL = str(
            CONFIG[
                "uavnetsim_nlos_model"
            ]
        )

        uav_config.STATIC_CASE = 1
        uav_config.HETEROGENEOUS = 0

        uav_config.TRAFFIC_PATTERN = (
            "UNIFORM"
        )
        uav_config.PACKET_ARRIVAL_RATE = (
            1e-12
        )

        uav_config.MAP_LENGTH = float(
            CONFIG["map_size"]
        )
        uav_config.MAP_WIDTH = float(
            CONFIG["map_size"]
        )
        uav_config.MAP_HEIGHT = float(
            CONFIG["altitude_max"]
        )
        uav_config.UAV_MIN_ALTITUDE = (
            float(
                CONFIG[
                    "altitude_min"
                ]
            )
        )
        uav_config.UAV_MAX_ALTITUDE = (
            float(
                CONFIG[
                    "altitude_max"
                ]
            )
        )
        uav_config.UAV_BOUNDARY_CLEARANCE = (
            0.0
        )
        uav_config.UAV_BUILDING_CLEARANCE = (
            0.0
        )

        uav_config.MAX_QUEUE_SIZE = int(
            CONFIG[
                "uavnetsim_max_queue_size"
            ]
        )

        uav_config.AVERAGE_PAYLOAD_LENGTH = (
            int(
                CONFIG[
                    "uavnetsim_payload_bytes"
                ]
            )
            * 8
        )
        uav_config.VARIABLE_PAYLOAD_LENGTH = (
            0
        )
        uav_config.PACKET_LIFETIME = (
            float(
                CONFIG[
                    "uavnetsim_packet_lifetime_s"
                ]
            )
            * 1e6
        )

        uav_config.INITIAL_ENERGY = max(
            float(CONFIG["battery_j"]),
            10_000.0,
        )
        uav_config.ENERGY_THRESHOLD = (
            0.0
        )

        uav_config.TRANSMITTING_POWER = (
            float(
                CONFIG[
                    "tx_power_min_w"
                ]
            )
        )

        # Pin UavNetSim radio/MAC values to the project config. This avoids
        # silent behavior changes when the external simulator is upgraded and
        # keeps the CUDA surrogate on exactly the same physical/MAC contract.
        uav_config.CARRIER_FREQUENCY = float(
            CONFIG["uavnetsim_carrier_frequency_hz"]
        )
        uav_config.BANDWIDTH = float(
            CONFIG["uavnetsim_bandwidth_hz"]
        )
        uav_config.BIT_RATE = float(
            CONFIG["uavnetsim_bit_rate_bps"]
        )
        uav_config.SINR_THRESHOLD_DB = float(
            CONFIG["uavnetsim_sinr_threshold_db"]
        )
        uav_config.CCA_THRESHOLD_DBM = float(
            CONFIG["uavnetsim_cca_threshold_dbm"]
        )
        uav_config.THERMAL_NOISE_DENSITY_DBM_HZ = float(
            CONFIG["uavnetsim_thermal_noise_density_dbm_hz"]
        )
        uav_config.RECEIVER_NOISE_FIGURE_DB = float(
            CONFIG["uavnetsim_receiver_noise_figure_db"]
        )
        uav_config.MAX_RETRANSMISSION_ATTEMPT = int(
            CONFIG["uavnetsim_max_retransmission_attempt"]
        )
        uav_config.CW_MIN = int(
            CONFIG["full_gpu_packet_cw_min"]
        )
        uav_config.SLOT_DURATION = float(
            CONFIG["full_gpu_packet_slot_us"]
        )
        uav_config.DIFS_DURATION = float(
            CONFIG["full_gpu_packet_difs_us"]
        )
        uav_config.SIFS_DURATION = float(
            CONFIG["full_gpu_packet_sifs_us"]
        )
        uav_config.ACK_PACKET_LENGTH = int(
            CONFIG["full_gpu_packet_ack_bits"]
        )

        uav_config.SIONNA_SCENE_PATH = str(
            Path(scene_directory)
            / "scene.xml"
        )

    def _patch_dynamic_tx_power(self):
        original_transmit = (
            self._simulator
            .channel
            .transmit
        )
        uav_config = self._uav_config

        def transmit_with_packet_power(
            packet,
            transmitter_id,
            receiver_ids,
        ):
            previous_power = (
                uav_config
                .TRANSMITTING_POWER
            )

            packet_power = float(
                getattr(
                    packet,
                    "mission_tx_power_w",
                    previous_power,
                )
            )

            uav_config.TRANSMITTING_POWER = (
                packet_power
            )

            try:
                return original_transmit(
                    packet,
                    transmitter_id,
                    receiver_ids,
                )
            finally:
                uav_config.TRANSMITTING_POWER = (
                    previous_power
                )

        self._simulator.channel.transmit = (
            transmit_with_packet_power
        )

        for drone in self._simulator.drones:
            phy = (
                drone
                .mac_protocol
                .phy
            )

            def consume_energy(
                packet,
                phy=phy,
            ):
                power_w = float(
                    getattr(
                        packet,
                        "mission_tx_power_w",
                        uav_config
                        .TRANSMITTING_POWER,
                    )
                )

                duration_s = (
                    packet.packet_length
                    / uav_config.BIT_RATE
                )

                phy.my_drone.residual_energy = max(
                    0.0,
                    (
                        phy.my_drone
                        .residual_energy
                        - duration_s
                        * power_w
                    ),
                )

            phy._consume_transmit_energy = (
                consume_energy
            )

            def unicast_with_energy_budget(
                packet,
                next_hop_id,
                phy=phy,
                consume_energy=consume_energy,
            ):
                power_w = float(
                    getattr(
                        packet,
                        "mission_tx_power_w",
                        uav_config.TRANSMITTING_POWER,
                    )
                )
                duration_s = (
                    float(packet.packet_length)
                    / float(uav_config.BIT_RATE)
                )
                required_j = duration_s * power_w
                if (
                    float(phy.my_drone.residual_energy)
                    + 1e-12
                    < required_j
                ):
                    packet.mission_energy_blocked = True
                    return False

                consume_energy(packet)
                phy.my_drone.simulator.channel.transmit(
                    packet,
                    phy.my_drone.identifier,
                    [next_hop_id],
                )
                return True

            phy.unicast = unicast_with_energy_budget

    def _freeze_external_mobility(self):
        total_us = (
            (
                int(CONFIG["max_steps"])
                + 2
            )
            * float(CONFIG["dt"])
            * 1e6
        )

        freeze_interval = (
            total_us + 1e6
        )

        for drone in self._simulator.drones:
            drone.speed = 0.0
            drone.velocity = [
                0.0,
                0.0,
                0.0,
            ]
            drone.velocity_mean = 0.0

            model = getattr(
                drone,
                "mobility_model",
                None,
            )

            if model is not None:
                if hasattr(
                    model,
                    "position_update_interval",
                ):
                    model.position_update_interval = (
                        freeze_interval
                    )

                if hasattr(
                    model,
                    "direction_update_interval",
                ):
                    model.direction_update_interval = (
                        freeze_interval
                    )

    def reset(
        self,
        *,
        uavs,
        gcs_position,
        obstacles,
        report_buffers,
        pending_reports,
        gcs_received_target_ids,
    ):
        self._validate_backend_config()

        SimpleNetworkBackend._validate_uavs(
            uavs
        )
        SimpleNetworkBackend._validate_obstacles(
            obstacles
        )
        SimpleNetworkBackend._validate_report_state(
            report_buffers,
            pending_reports,
            gcs_received_target_ids,
        )

        validated_gcs = (
            SimpleNetworkBackend
            ._validated_gcs_position(
                gcs_position
            )
        )

        if self._initialized:
            self.close()

        self._load_uavnetsim()

        self._scene_directory = (
            tempfile.TemporaryDirectory(
                prefix="uav_search_uavnetsim_"
            )
        )

        write_uavnetsim_scene(
            obstacles,
            self._scene_directory.name,
        )

        self._configure_uavnetsim(
            self._scene_directory.name
        )

        self._environment = (
            self._simpy.Environment()
        )

        total_us = (
            (
                int(CONFIG["max_steps"])
                + 2
            )
            * float(CONFIG["dt"])
            * 1e6
        )

        self._simulator = (
            self._simulator_class(
                seed=self.seed,
                env=self._environment,
                n_drones=(
                    int(
                        CONFIG[
                            "num_uavs"
                        ]
                    )
                    + 1
                ),
                total_simulation_time=(
                    total_us
                ),
                drone_speed=0.0,
            )
        )

        self._freeze_external_mobility()
        self._patch_dynamic_tx_power()

        self._uavs = uavs
        self._gcs_position = (
            validated_gcs
        )
        self._obstacles = obstacles
        self._report_buffers = (
            report_buffers
        )
        self._pending_reports = (
            pending_reports
        )
        self._gcs_received_target_ids = (
            gcs_received_target_ids
        )

        self._peer_transfer_states = (
            create_peer_transfer_states()
        )
        self._packet_records = {}
        self._metric_state = (
            self._new_metric_state()
        )
        self._last_step_comm_energy_by_uav = np.zeros(
            int(CONFIG["num_uavs"]),
            dtype=np.float64,
        )
        self._step_energy_budget_j = None

        self._initialized = True

        self.sync_positions(
            uavs
        )

        # Execute all processes scheduled at t=0 once.
        self._environment.run(
            until=1.0
        )

        # UavNetSim mobility is deliberately frozen; re-apply the
        # mission positions after its t=0 initialization.
        self.sync_positions(
            uavs
        )

    def sync_positions(
        self,
        uavs,
    ):
        if not self._initialized:
            raise RuntimeError(
                "backend must be reset before "
                "sync_positions"
            )

        SimpleNetworkBackend._validate_uavs(
            uavs
        )

        map_size = float(
            CONFIG["map_size"]
        )
        altitude_min = float(
            CONFIG["altitude_min"]
        )
        altitude_max = float(
            CONFIG["altitude_max"]
        )

        for mission_uav in uavs:
            position = np.asarray(
                mission_uav.position,
                dtype=np.float64,
            )
            velocity = np.asarray(
                mission_uav.velocity,
                dtype=np.float64,
            )

            if (
                position.shape != (3,)
                or velocity.shape != (3,)
            ):
                raise ValueError(
                    "UAV position and velocity "
                    "must have shape (3,)"
                )

            if (
                not np.all(
                    np.isfinite(position)
                )
                or not np.all(
                    np.isfinite(velocity)
                )
            ):
                raise ValueError(
                    "UAV position and velocity "
                    "must be finite"
                )

            if not (
                0.0
                <= position[0]
                <= map_size
                and 0.0
                <= position[1]
                <= map_size
                and altitude_min
                <= position[2]
                <= altitude_max
            ):
                raise ValueError(
                    "UAV position is outside "
                    "mission bounds"
                )

            external_drone = (
                self._simulator
                .drones[mission_uav.id]
            )

            # Mission motion remains the source of truth. Directly
            # synchronize coordinates instead of invoking UavNetSim
            # mobility/collision resolution.
            external_drone._coords = [
                float(value)
                for value in position
            ]
            external_drone.velocity = [
                float(value)
                for value in velocity
            ]
            external_drone.sleep = bool(
                not mission_uav.active
            )

        gcs_drone = (
            self._simulator
            .drones[
                self._gcs_node_id
            ]
        )
        gcs_drone._coords = [
            float(value)
            for value
            in self._gcs_position
        ]
        gcs_drone.velocity = [
            0.0,
            0.0,
            0.0,
        ]

        self._uavs = uavs

    @staticmethod
    def _validate_step_clock(
        dt,
        current_step,
    ):
        if (
            isinstance(
                current_step,
                (bool, np.bool_),
            )
            or not isinstance(
                current_step,
                (int, np.integer),
            )
        ):
            raise TypeError(
                "current_step must be an integer"
            )

        current_step = int(
            current_step
        )

        if current_step < 0:
            raise ValueError(
                "current_step must be >= 0"
            )

        if isinstance(
            dt,
            (bool, np.bool_),
        ):
            raise TypeError(
                "dt must be numeric and not boolean"
            )

        dt = float(dt)

        if (
            not np.isfinite(dt)
            or dt <= 0.0
        ):
            raise ValueError(
                "dt must be finite and > 0"
            )

        if not np.isclose(
            dt,
            float(CONFIG["dt"]),
            rtol=0.0,
            atol=1e-12,
        ):
            raise ValueError(
                "UavNetSimBackend dt must match "
                "CONFIG['dt']"
            )

        return dt, current_step

    @staticmethod
    def _normalize_requests(
        requests,
    ):
        if requests is None:
            request_list = []
        elif isinstance(
            requests,
            TransmissionIntent,
        ):
            request_list = [
                requests
            ]
        elif isinstance(
            requests,
            (list, tuple),
        ):
            request_list = list(
                requests
            )
        else:
            raise TypeError(
                "requests must be a "
                "TransmissionIntent, a list/"
                "tuple of intents, or None"
            )

        for request in request_list:
            if not isinstance(
                request,
                TransmissionIntent,
            ):
                raise TypeError(
                    "requests must contain only "
                    "TransmissionIntent objects"
                )

        sender_ids = [
            int(request.sender)
            for request in request_list
        ]

        if len(sender_ids) != len(
            set(sender_ids)
        ):
            raise ValueError(
                "each sender may submit at most "
                "one transmission request per "
                "network step"
            )

        return request_list

    def _report_generation_key(
        self,
        intent,
        report,
    ):
        return (
            int(intent.sender),
            int(intent.recipient),
            int(report.target_id),
            int(report.source_uav),
            int(report.created_step),
            int(report.size_bytes),
            float(report.ttl_s),
        )

    def _report_for_intent(
        self,
        intent,
        current_step,
    ):
        if (
            intent.target_id
            in self._gcs_received_target_ids
        ):
            return None

        sender_buffer = (
            self._report_buffers[
                intent.sender
            ]
        )

        cleanup_report_buffer(
            sender_buffer,
            current_step,
        )

        report = find_report_in_buffer(
            sender_buffer,
            intent.target_id,
        )

        if report is None:
            return None

        if report_is_expired(
            report,
            current_step,
        ):
            return None

        return report

    def _inflight_bytes(
        self,
        transfer_key,
    ):
        return sum(
            int(record["payload_bytes"])
            for record
            in self._packet_records.values()
            if (
                record["status"]
                == "in_flight"
                and record["transfer_key"]
                == transfer_key
            )
        )

    def _recipient_node_id(
        self,
        recipient,
    ):
        if recipient == GCS_NODE:
            return self._gcs_node_id

        return int(recipient)

    def _request_is_active(
        self,
        intent,
    ):
        if not self._uavs[
            intent.sender
        ].active:
            return False

        if intent.recipient == GCS_NODE:
            return True

        return bool(
            self._uavs[
                intent.recipient
            ].active
        )

    def _request_is_within_contact_range(
        self,
        intent,
    ):
        sender_position = np.asarray(
            self._uavs[
                intent.sender
            ].position,
            dtype=np.float64,
        )

        if intent.recipient == GCS_NODE:
            recipient_position = np.asarray(
                self._gcs_position,
                dtype=np.float64,
            )
            max_range = float(
                CONFIG[
                    "gcs_contact_range_m"
                ]
            )
        else:
            recipient_position = np.asarray(
                self._uavs[
                    intent.recipient
                ].position,
                dtype=np.float64,
            )
            max_range = float(
                CONFIG[
                    "peer_contact_range_m"
                ]
            )

        distance = float(
            np.linalg.norm(
                sender_position
                - recipient_position
            )
        )

        return bool(
            distance <= max_range
        )

    def _inject_request(
        self,
        intent,
        current_step,
    ):
        if not self._request_is_active(
            intent
        ):
            return {
                "transfer_key": None,
                "injected_bytes": 0,
                "injected_packets": 0,
                "reason": "inactive_node",
            }

        if not self._request_is_within_contact_range(
            intent
        ):
            return {
                "transfer_key": None,
                "injected_bytes": 0,
                "injected_packets": 0,
                "reason": "out_of_contact_range",
            }

        report = self._report_for_intent(
            intent,
            current_step,
        )

        if report is None:
            return {
                "transfer_key": None,
                "injected_bytes": 0,
                "injected_packets": 0,
                "reason": "no_report",
            }

        transfer_key = (
            self._report_generation_key(
                intent,
                report,
            )
        )

        inflight_bytes = (
            self._inflight_bytes(
                transfer_key
            )
        )

        bytes_to_inject = max(
            0,
            int(intent.requested_bytes)
            - inflight_bytes,
        )

        if bytes_to_inject <= 0:
            return {
                "transfer_key": transfer_key,
                "injected_bytes": 0,
                "injected_packets": 0,
                "reason": "already_in_flight",
            }

        recipient_node_id = (
            self._recipient_node_id(
                intent.recipient
            )
        )

        source_drone = (
            self._simulator
            .drones[intent.sender]
        )
        destination_drone = (
            self._simulator
            .drones[recipient_node_id]
        )

        payload_limit = int(
            CONFIG[
                "uavnetsim_payload_bytes"
            ]
        )

        injected_bytes = 0
        injected_packets = 0

        remaining = (
            bytes_to_inject
        )

        while remaining > 0:
            payload_bytes = min(
                payload_limit,
                remaining,
            )

            (
                self._uav_config
                .GL_ID_DATA_PACKET
            ) += 1

            packet_id = int(
                self._uav_config
                .GL_ID_DATA_PACKET
            )

            packet_length_bits = int(
                self._uav_config
                .IP_HEADER_LENGTH
                + self._uav_config
                .MAC_HEADER_LENGTH
                + self._uav_config
                .PHY_HEADER_LENGTH
                + payload_bytes
                * 8
            )

            channel_id = (
                source_drone
                .channel_assigner
                .channel_assign()
            )

            packet = (
                self._data_packet_class(
                    src_drone=source_drone,
                    dst_drone=destination_drone,
                    creation_time=(
                        self._environment.now
                    ),
                    data_packet_id=(
                        packet_id
                    ),
                    data_packet_length=(
                        packet_length_bits
                    ),
                    simulator=(
                        self._simulator
                    ),
                    channel_id=(
                        channel_id
                    ),
                )
            )

            packet.transmission_mode = 0
            packet.forced_next_hop_id = (
                recipient_node_id
            )
            packet.mission_tx_power_w = (
                float(
                    intent.tx_power_w
                )
            )

            packet.waiting_start_time = (
                self._environment.now
            )

            if (
                source_drone
                .transmitting_queue
                .qsize()
                >= source_drone.max_queue_size
            ):
                self._metric_state[
                    "queue_limit_events"
                ] += 1
                self._metric_state[
                    "payload_bytes_deferred"
                ] += int(remaining)
                break

            self._simulator.metrics.record_generated(
                packet
            )

            source_drone.transmitting_queue.put(
                packet
            )

            self._packet_records[
                packet_id
            ] = {
                "packet": packet,
                "transfer_key": transfer_key,
                "sender": int(
                    intent.sender
                ),
                "recipient": int(
                    intent.recipient
                ),
                "target_id": int(
                    intent.target_id
                ),
                "source_uav": int(
                    report.source_uav
                ),
                "created_step": int(
                    report.created_step
                ),
                "size_bytes": int(
                    report.size_bytes
                ),
                "ttl_s": float(
                    report.ttl_s
                ),
                "payload_bytes": int(
                    payload_bytes
                ),
                "tx_power_w": float(
                    intent.tx_power_w
                ),
                "status": "in_flight",
            }

            injected_bytes += int(
                payload_bytes
            )
            injected_packets += 1
            remaining -= int(
                payload_bytes
            )

        self._metric_state[
            "packets_injected"
        ] += injected_packets
        self._metric_state[
            "payload_bytes_injected"
        ] += injected_bytes

        deferred_bytes = int(
            remaining
        )

        return {
            "transfer_key": transfer_key,
            "injected_bytes": (
                injected_bytes
            ),
            "injected_packets": (
                injected_packets
            ),
            "deferred_bytes": (
                deferred_bytes
            ),
            "reason": (
                "queue_limited"
                if deferred_bytes > 0
                else "injected"
            ),
        }

    def _collect_packet_outcomes(self):
        delivered_ids = (
            self._simulator
            .metrics
            .datapacket_arrived
        )

        delivered_by_key = {}
        failed_by_key = {}

        max_attempts = int(
            self._uav_config
            .MAX_RETRANSMISSION_ATTEMPT
        )

        for (
            packet_id,
            record,
        ) in self._packet_records.items():
            if (
                record["status"]
                != "in_flight"
            ):
                continue

            packet = record["packet"]
            transfer_key = (
                record["transfer_key"]
            )

            if packet_id in delivered_ids:
                recipient = int(
                    record["recipient"]
                )

                if (
                    recipient != GCS_NODE
                    and not self._uavs[
                        recipient
                    ].active
                ):
                    record["status"] = (
                        "failed"
                    )
                    failed_by_key[
                        transfer_key
                    ] = (
                        failed_by_key.get(
                            transfer_key,
                            0,
                        )
                        + int(
                            record[
                                "payload_bytes"
                            ]
                        )
                    )
                    self._metric_state[
                        "packets_failed"
                    ] += 1
                    continue

                record["status"] = (
                    "delivered"
                )

                delivered_by_key[
                    transfer_key
                ] = (
                    delivered_by_key.get(
                        transfer_key,
                        0,
                    )
                    + int(
                        record[
                            "payload_bytes"
                        ]
                    )
                )

                self._metric_state[
                    "packets_delivered"
                ] += 1
                self._metric_state[
                    "payload_bytes_delivered"
                ] += int(
                    record[
                        "payload_bytes"
                    ]
                )

                continue

            attempts = int(
                packet
                .number_retransmission_attempt
                .get(
                    record["sender"],
                    0,
                )
            )

            expired = bool(
                self._environment.now
                >= (
                    packet.creation_time
                    + packet.deadline
                )
            )

            if (
                attempts
                >= max_attempts
                or expired
            ):
                record["status"] = (
                    "failed"
                )
                failed_by_key[
                    transfer_key
                ] = (
                    failed_by_key.get(
                        transfer_key,
                        0,
                    )
                    + int(
                        record[
                            "payload_bytes"
                        ]
                    )
                )
                self._metric_state[
                    "packets_failed"
                ] += 1

        return (
            delivered_by_key,
            failed_by_key,
        )

    def _generation_matches_record(
        self,
        report,
        record,
    ):
        return bool(
            report.target_id
            == record["target_id"]
            and report.source_uav
            == record["source_uav"]
            and report.created_step
            == record["created_step"]
            and report.size_bytes
            == record["size_bytes"]
            and np.isclose(
                report.ttl_s,
                record["ttl_s"],
            )
        )

    def _commit_delivered_bytes(
        self,
        transfer_key,
        delivered_bytes,
        current_step,
    ):
        delivered_bytes = int(
            delivered_bytes
        )

        if delivered_bytes <= 0:
            return None

        matching_records = [
            record
            for record
            in self._packet_records.values()
            if (
                record["transfer_key"]
                == transfer_key
            )
        ]

        if not matching_records:
            return None

        record = (
            matching_records[0]
        )

        target_id = int(
            record["target_id"]
        )

        if (
            target_id
            in self._gcs_received_target_ids
        ):
            return {
                "status": "already_delivered",
                "committed_bytes": 0,
            }

        sender = int(
            record["sender"]
        )

        sender_report = find_report_in_buffer(
            self._report_buffers[
                sender
            ],
            target_id,
        )

        if (
            sender_report is None
            or not self._generation_matches_record(
                sender_report,
                record,
            )
        ):
            return {
                "status": "stale_generation",
                "committed_bytes": 0,
            }

        intent = TransmissionIntent(
            sender=sender,
            recipient=int(
                record["recipient"]
            ),
            target_id=target_id,
            requested_bytes=(
                delivered_bytes
            ),
            tx_power_w=float(
                record["tx_power_w"]
            ),
        )

        if (
            intent.recipient
            == GCS_NODE
        ):
            result = (
                commit_gcs_transfer_bytes(
                    intent,
                    delivered_bytes,
                    self._report_buffers,
                    self._pending_reports,
                    self._gcs_received_target_ids,
                    self._peer_transfer_states,
                    current_step,
                )
            )

            return {
                "status": (
                    "gcs_delivered"
                    if (
                        result["status"]
                        == "delivered"
                    )
                    else (
                        "gcs_partial"
                        if (
                            result["status"]
                            == "partial"
                        )
                        else result[
                            "status"
                        ]
                    )
                ),
                "committed_bytes": int(
                    result.get(
                        "committed_bytes",
                        0,
                    )
                ),
                "commit_result": result,
            }

        peer_commit = commit_peer_transfer_bytes(
            intent,
            delivered_bytes,
            self._report_buffers[
                sender
            ],
            self._peer_transfer_states,
            current_step,
        )

        state = peer_commit["state"]
        committed_bytes = int(
            peer_commit[
                "committed_bytes"
            ]
        )

        if not peer_commit["complete"]:
            return {
                "status": "peer_partial",
                "committed_bytes": (
                    committed_bytes
                ),
                "peer_state": state,
                "peer_commit_result": (
                    peer_commit
                ),
            }

        key = peer_transfer_key(
            sender,
            intent.recipient,
            target_id,
        )

        custody_result = (
            commit_completed_peer_transfer_to_receiver(
                key,
                self._peer_transfer_states,
                self._report_buffers,
                current_step,
                self._gcs_received_target_ids,
            )
        )

        status_map = {
            "enqueued": "peer_enqueued",
            "buffer_full": (
                "peer_buffer_full"
            ),
            "already_present": (
                "peer_already_present"
            ),
            "expired": "expired",
            "already_delivered": (
                "already_delivered"
            ),
        }

        return {
            "status": status_map.get(
                custody_result["status"],
                custody_result["status"],
            ),
            "committed_bytes": (
                committed_bytes
            ),
            "peer_state": state,
            "peer_commit_result": (
                peer_commit
            ),
            "custody_result": (
                custody_result
            ),
        }

    def _has_inflight_for_key(
        self,
        transfer_key,
    ):
        return any(
            (
                record["status"]
                == "in_flight"
                and record[
                    "transfer_key"
                ]
                == transfer_key
            )
            for record
            in self._packet_records.values()
        )

    def _cleanup_terminal_packet_records(
        self,
    ):
        for (
            packet_id,
            record,
        ) in list(
            self._packet_records.items()
        ):
            if (
                record["status"]
                == "in_flight"
            ):
                continue

            del self._packet_records[
                packet_id
            ]

    def set_step_energy_budget_j(
        self,
        energy_budget_j,
    ):
        if not self._initialized:
            raise RuntimeError(
                "backend must be reset before setting energy budget"
            )
        values = np.asarray(
            energy_budget_j,
            dtype=np.float64,
        )
        expected = (
            int(CONFIG["num_uavs"]),
        )
        if values.shape != expected:
            raise ValueError(
                "energy_budget_j has wrong shape"
            )
        if (
            not np.all(np.isfinite(values))
            or np.any(values < 0.0)
        ):
            raise ValueError(
                "energy_budget_j must be finite and >= 0"
            )
        self._step_energy_budget_j = (
            values.copy()
        )

    def step(
        self,
        requests,
        dt,
        current_step,
    ):
        if not self._initialized:
            raise RuntimeError(
                "backend must be reset before step"
            )

        dt, current_step = (
            self._validate_step_clock(
                dt,
                current_step,
            )
        )

        request_list = (
            self._normalize_requests(
                requests
            )
        )

        self.sync_positions(
            self._uavs
        )

        cleanup_peer_transfer_states(
            self._peer_transfer_states,
            current_step,
        )

        retry_completed_peer_transfers(
            self._peer_transfer_states,
            self._report_buffers,
            current_step,
            self._gcs_received_target_ids,
        )

        if self._step_energy_budget_j is not None:
            for drone, budget_j in zip(
                self._simulator.drones[
                    : int(CONFIG["num_uavs"])
                ],
                self._step_energy_budget_j,
            ):
                drone.residual_energy = float(
                    budget_j
                )
            self._step_energy_budget_j = None

        request_contexts = [
            (
                request,
                self._inject_request(
                    request,
                    current_step,
                ),
            )
            for request
            in request_list
        ]

        energy_before = [
            float(
                drone.residual_energy
            )
            for drone
            in self._simulator.drones[
                : int(
                    CONFIG["num_uavs"]
                )
            ]
        ]

        target_time = (
            self._environment.now
            + dt
            * 1e6
        )

        self._environment.run(
            until=target_time
        )

        energy_after = [
            float(
                drone.residual_energy
            )
            for drone
            in self._simulator.drones[
                : int(
                    CONFIG["num_uavs"]
                )
            ]
        ]

        self._last_step_comm_energy_by_uav = np.asarray(
            [
                max(
                    0.0,
                    before - after,
                )
                for before, after
                in zip(
                    energy_before,
                    energy_after,
                )
            ],
            dtype=np.float64,
        )

        communication_energy = float(
            np.sum(
                self._last_step_comm_energy_by_uav
            )
        )

        self._metric_state[
            "communication_energy_j"
        ] += communication_energy

        (
            delivered_by_key,
            failed_by_key,
        ) = self._collect_packet_outcomes()

        commit_results = {}

        for (
            transfer_key,
            delivered_bytes,
        ) in delivered_by_key.items():
            commit_results[
                transfer_key
            ] = (
                self._commit_delivered_bytes(
                    transfer_key,
                    delivered_bytes,
                    current_step,
                )
            )

        results = []

        for (
            request,
            context,
        ) in request_contexts:
            transfer_key = (
                context["transfer_key"]
            )

            if transfer_key is None:
                result = {
                    "status": "no_transfer",
                    "tx_bytes": 0,
                    "injected_bytes": 0,
                    "injected_packets": 0,
                    "deferred_bytes": int(
                        context.get(
                            "deferred_bytes",
                            0,
                        )
                    ),
                    "reason": (
                        context["reason"]
                    ),
                }
            else:
                delivered_bytes = int(
                    delivered_by_key.get(
                        transfer_key,
                        0,
                    )
                )
                failed_bytes = int(
                    failed_by_key.get(
                        transfer_key,
                        0,
                    )
                )
                commit_result = (
                    commit_results.get(
                        transfer_key
                    )
                )

                if commit_result is not None:
                    status = (
                        commit_result[
                            "status"
                        ]
                    )
                    committed_bytes = int(
                        commit_result.get(
                            "committed_bytes",
                            0,
                        )
                    )
                elif self._has_inflight_for_key(
                    transfer_key
                ):
                    status = "in_flight"
                    committed_bytes = 0
                elif failed_bytes > 0:
                    status = "packet_failed"
                    committed_bytes = 0
                else:
                    status = "no_transfer"
                    committed_bytes = 0

                result = {
                    "status": status,
                    "tx_bytes": (
                        committed_bytes
                    ),
                    "delivered_payload_bytes": (
                        delivered_bytes
                    ),
                    "failed_payload_bytes": (
                        failed_bytes
                    ),
                    "injected_bytes": int(
                        context[
                            "injected_bytes"
                        ]
                    ),
                    "injected_packets": int(
                        context[
                            "injected_packets"
                        ]
                    ),
                    "deferred_bytes": int(
                        context.get(
                            "deferred_bytes",
                            0,
                        )
                    ),
                    "reason": (
                        context["reason"]
                    ),
                    "commit_result": (
                        commit_result
                    ),
                }

            results.append(
                result
            )

            status = str(
                result["status"]
            )

            status_counts = (
                self._metric_state[
                    "status_counts"
                ]
            )

            status_counts[status] = (
                status_counts.get(
                    status,
                    0,
                )
                + 1
            )

        self._metric_state[
            "steps"
        ] += 1
        self._metric_state[
            "requests"
        ] += len(
            request_list
        )

        self._cleanup_terminal_packet_records()

        return results

    @property
    def peer_transfer_states(self):
        if not self._initialized:
            raise RuntimeError(
                "backend must be reset before "
                "accessing peer_transfer_states"
            )

        return self._peer_transfer_states

    def metrics(self):
        network_metrics = {}

        if self._simulator is not None:
            network_metrics = (
                self._simulator
                .metrics
                .snapshot()
            )

        return {
            "backend": "uavnetsim",
            "steps": int(
                self._metric_state[
                    "steps"
                ]
            ),
            "requests": int(
                self._metric_state[
                    "requests"
                ]
            ),
            "packets_injected": int(
                self._metric_state[
                    "packets_injected"
                ]
            ),
            "packets_delivered": int(
                self._metric_state[
                    "packets_delivered"
                ]
            ),
            "packets_failed": int(
                self._metric_state[
                    "packets_failed"
                ]
            ),
            "queue_limit_events": int(
                self._metric_state[
                    "queue_limit_events"
                ]
            ),
            "payload_bytes_deferred": int(
                self._metric_state[
                    "payload_bytes_deferred"
                ]
            ),
            "payload_bytes_injected": int(
                self._metric_state[
                    "payload_bytes_injected"
                ]
            ),
            "payload_bytes_delivered": int(
                self._metric_state[
                    "payload_bytes_delivered"
                ]
            ),
            "communication_energy_j": float(
                self._metric_state[
                    "communication_energy_j"
                ]
            ),
            "in_flight_packets": sum(
                record["status"]
                == "in_flight"
                for record
                in self._packet_records.values()
            ),
            "active_peer_transfers": len(
                self._peer_transfer_states
            ),
            "status_counts": dict(
                self._metric_state[
                    "status_counts"
                ]
            ),
            "network_metrics": dict(
                network_metrics
            ),
        }

    def last_step_communication_energy_by_uav(self):
        if not self._initialized:
            raise RuntimeError(
                "backend must be reset before "
                "reading communication energy"
            )

        return (
            self._last_step_comm_energy_by_uav
            .copy()
        )

    def close(self):
        if self._simulator is not None:
            self._simulator.close()

        if (
            self._scene_directory
            is not None
        ):
            self._scene_directory.cleanup()

        self._scene_directory = None
        self._environment = None
        self._simulator = None
        self._packet_records = {}
        self._initialized = False


In [ ]:
def create_network_backend(
    backend_name=None,
    **kwargs,
):
    if backend_name is None:
        backend_name = CONFIG.get(
            "network_backend",
            "simple",
        )

    normalized = str(
        backend_name
    ).strip().lower()

    if normalized == "simple":
        if kwargs:
            unexpected = ", ".join(
                sorted(kwargs)
            )
            raise TypeError(
                "SimpleNetworkBackend does not "
                f"accept: {unexpected}"
            )

        return SimpleNetworkBackend()

    if normalized in {
        "uavnetsim",
        "uav_net_sim",
    }:
        return UavNetSimBackend(
            **kwargs
        )

    raise ValueError(
        "network backend must be "
        "'simple' or 'uavnetsim'"
    )


In [ ]:
def binary_entropy_bits(probability):
    probability = float(
        np.clip(
            probability,
            1e-12,
            1.0 - 1e-12,
        )
    )

    return float(
        -probability
        * np.log2(probability)
        - (1.0 - probability)
        * np.log2(
            1.0 - probability
        )
    )


In [ ]:
def sensing_information_gain_bits(
    sensing_records,
):
    entropy_reduction = 0.0

    for record in (
        sensing_records
    ):
        prior_entropy = (
            binary_entropy_bits(
                record[
                    "prior"
                ]
            )
        )
        posterior_entropy = (
            binary_entropy_bits(
                record[
                    "posterior"
                ]
            )
        )

        # Signed realized entropy reduction.
        # A later increase in uncertainty cancels earlier shaping
        # instead of allowing repeated positive-only reward.
        entropy_reduction += (
            prior_entropy
            - posterior_entropy
        )

    return float(
        entropy_reduction
    )


In [ ]:
def belief_certainty_potential(
    belief_maps,
):
    values = np.asarray(
        belief_maps,
        dtype=np.float64,
    )

    if values.ndim != 3:
        raise ValueError(
            "belief_maps must have shape "
            "(num_uavs, height, width)"
        )

    probabilities = np.clip(
        values,
        1e-12,
        1.0 - 1e-12,
    )
    entropy = (
        -probabilities
        * np.log2(
            probabilities
        )
        - (
            1.0
            - probabilities
        )
        * np.log2(
            1.0
            - probabilities
        )
    )

    mean_entropy = float(
        np.mean(
            entropy
        )
    )

    return float(
        np.clip(
            1.0 - mean_entropy,
            0.0,
            1.0,
        )
    )


In [ ]:
def propulsion_energy_j(
    uav,
    hybrid_action,
    dt=None,
    realized_acceleration_mps2=None,
):
    if not isinstance(uav, UAV):
        raise TypeError(
            "uav must be UAV"
        )

    if not isinstance(
        hybrid_action,
        HybridAction,
    ):
        raise TypeError(
            "hybrid_action must be HybridAction"
        )

    if dt is None:
        dt = CONFIG["dt"]

    dt = float(dt)

    if (
        not np.isfinite(dt)
        or dt <= 0.0
    ):
        raise ValueError(
            "dt must be finite and > 0"
        )

    if not uav.active:
        return 0.0

    speed = float(
        np.linalg.norm(
            uav.velocity
        )
    )

    commanded_accel = float(
        np.linalg.norm(
            hybrid_action.movement
        )
        * float(
            CONFIG["max_accel"]
        )
    )

    if realized_acceleration_mps2 is None:
        acceleration_for_energy = (
            commanded_accel
        )
    else:
        realized_acceleration_mps2 = (
            np.asarray(
                realized_acceleration_mps2,
                dtype=np.float64,
            )
        )
        if realized_acceleration_mps2.shape != (3,):
            raise ValueError(
                "realized_acceleration_mps2 must "
                "have shape (3,)"
            )
        if not np.all(
            np.isfinite(
                realized_acceleration_mps2
            )
        ):
            raise ValueError(
                "realized_acceleration_mps2 must "
                "contain only finite values"
            )
        acceleration_for_energy = min(
            float(
                np.linalg.norm(
                    realized_acceleration_mps2
                )
            ),
            float(CONFIG["max_accel"]),
        )

    if (
        uav.position[2] <= 0.0
        and speed <= 1e-12
        and commanded_accel <= 1e-12
    ):
        power_w = float(
            CONFIG[
                "energy_idle_power_w"
            ]
        )
    else:
        power_w = (
            float(
                CONFIG[
                    "energy_hover_power_w"
                ]
            )
            + float(
                CONFIG[
                    "energy_speed_sq_coeff"
                ]
            )
            * speed
            * speed
            + float(
                CONFIG[
                    "energy_accel_sq_coeff"
                ]
            )
            * acceleration_for_energy
            * acceleration_for_energy
        )

    if (
        not np.isfinite(power_w)
        or power_w < 0.0
    ):
        raise ValueError(
            "computed propulsion power "
            "must be finite and >= 0"
        )

    return float(
        power_w * dt
    )


In [ ]:
def apply_uav_energy_budget(
    uavs,
    hybrid_actions,
    communication_energy_j,
    dt=None,
    realized_accelerations_mps2=None,
    propulsion_energy_override_j=None,
):
    if dt is None:
        dt = CONFIG["dt"]

    if len(uavs) != len(hybrid_actions):
        raise ValueError(
            "uavs and hybrid_actions must have the same length"
        )

    communication_energy_j = np.asarray(
        communication_energy_j,
        dtype=np.float64,
    ).copy()

    if communication_energy_j.shape != (len(uavs),):
        raise ValueError(
            "communication_energy_j has the wrong shape"
        )

    if (
        not np.all(np.isfinite(communication_energy_j))
        or np.any(communication_energy_j < 0.0)
    ):
        raise ValueError(
            "communication energy must be finite and >= 0"
        )

    if realized_accelerations_mps2 is not None:
        realized_accelerations_mps2 = np.asarray(
            realized_accelerations_mps2,
            dtype=np.float64,
        )
        if realized_accelerations_mps2.shape != (
            len(uavs),
            3,
        ):
            raise ValueError(
                "realized_accelerations_mps2 has the wrong shape"
            )
        if not np.all(
            np.isfinite(realized_accelerations_mps2)
        ):
            raise ValueError(
                "realized_accelerations_mps2 must contain only finite values"
            )

    if propulsion_energy_override_j is not None:
        propulsion_energy_override_j = np.asarray(
            propulsion_energy_override_j,
            dtype=np.float64,
        )
        if propulsion_energy_override_j.shape != (
            len(uavs),
        ):
            raise ValueError(
                "propulsion_energy_override_j has the wrong shape"
            )
        if (
            not np.all(
                np.isfinite(propulsion_energy_override_j)
            )
            or np.any(
                propulsion_energy_override_j < 0.0
            )
        ):
            raise ValueError(
                "propulsion override must be finite and >= 0"
            )

    propulsion = np.zeros(
        len(uavs),
        dtype=np.float64,
    )
    total = np.zeros(
        len(uavs),
        dtype=np.float64,
    )
    depleted_ids = []

    for index, (uav, hybrid_action) in enumerate(
        zip(uavs, hybrid_actions)
    ):
        if propulsion_energy_override_j is None:
            propulsion[index] = propulsion_energy_j(
                uav,
                hybrid_action,
                dt=dt,
                realized_acceleration_mps2=(
                    None
                    if realized_accelerations_mps2 is None
                    else realized_accelerations_mps2[index]
                ),
            )
        else:
            propulsion[index] = float(
                propulsion_energy_override_j[index]
            )

        available = max(
            0.0,
            float(uav.battery_j),
        )

        # Propulsion has priority because it has already determined the
        # physical trajectory for this decision step. Communication can
        # consume only what remains after that propulsion allocation.
        propulsion[index] = min(
            propulsion[index],
            available,
        )
        communication_energy_j[index] = min(
            communication_energy_j[index],
            max(
                0.0,
                available - propulsion[index],
            ),
        )
        total[index] = (
            propulsion[index]
            + communication_energy_j[index]
        )

        uav.battery_j = max(
            0.0,
            available - total[index],
        )

        if (
            uav.active
            and uav.battery_j <= 1e-12
        ):
            uav.active = False
            uav.velocity = np.zeros(
                3,
                dtype=np.float64,
            )
            depleted_ids.append(
                int(uav.id)
            )

    return {
        "propulsion_j": propulsion,
        "communication_j": communication_energy_j.copy(),
        "total_j": total,
        "depleted_uav_ids": depleted_ids,
    }


In [ ]:
def apply_battery_limited_motion(
    uavs,
    hybrid_actions,
    motion_result,
    positions_before,
    velocities_before,
    battery_before_j,
    active_before,
    dt=None,
):
    if dt is None:
        dt = CONFIG["dt"]
    dt = float(dt)
    if not np.isfinite(dt) or dt <= 0.0:
        raise ValueError("dt must be finite and > 0")

    count = len(uavs)
    positions_before = np.asarray(
        positions_before,
        dtype=np.float64,
    )
    velocities_before = np.asarray(
        velocities_before,
        dtype=np.float64,
    )
    battery_before_j = np.asarray(
        battery_before_j,
        dtype=np.float64,
    )
    active_before = np.asarray(
        active_before,
        dtype=bool,
    )

    if positions_before.shape != (count, 3):
        raise ValueError("positions_before has the wrong shape")
    if velocities_before.shape != (count, 3):
        raise ValueError("velocities_before has the wrong shape")
    if battery_before_j.shape != (count,):
        raise ValueError("battery_before_j has the wrong shape")
    if active_before.shape != (count,):
        raise ValueError("active_before has the wrong shape")

    original_realized = np.asarray(
        motion_result["realized_acceleration_mps2"],
        dtype=np.float64,
    )
    if original_realized.shape != (count, 3):
        raise ValueError(
            "motion_result realized acceleration has the wrong shape"
        )

    full_positions = np.stack(
        [
            uav.position.copy()
            for uav in uavs
        ],
        axis=0,
    )
    full_velocities = np.stack(
        [
            uav.velocity.copy()
            for uav in uavs
        ],
        axis=0,
    )

    full_required = np.zeros(
        count,
        dtype=np.float64,
    )
    battery_fraction = np.ones(
        count,
        dtype=np.float64,
    )
    battery_limited = np.zeros(
        count,
        dtype=bool,
    )

    for index, (uav, hybrid_action) in enumerate(
        zip(uavs, hybrid_actions)
    ):
        if not active_before[index]:
            continue
        required = propulsion_energy_j(
            uav,
            hybrid_action,
            dt=dt,
            realized_acceleration_mps2=(
                original_realized[index]
            ),
        )
        available = max(
            0.0,
            float(battery_before_j[index]),
        )
        full_required[index] = required
        if required > available + 1e-12:
            fraction = (
                0.0
                if required <= 0.0
                else float(
                    np.clip(
                        available / required,
                        0.0,
                        1.0,
                    )
                )
            )
            battery_fraction[index] = fraction
            battery_limited[index] = True
            # The discrete-time model treats this as a uniformly scaled
            # segment over the same dt, then the UAV is inactive at the end.
            uav.position = (
                positions_before[index]
                + fraction
                * (
                    full_positions[index]
                    - positions_before[index]
                )
            )
            uav.velocity = (
                full_velocities[index]
                * fraction
            )

    # Scaling one path can create a peer conflict that did not exist for the
    # original full-length simultaneous segments. Re-run the same monotone
    # safety rule on the realized scaled segments before committing them.
    blocked = np.asarray(
        motion_result["blocked"],
        dtype=bool,
    ).copy()
    blocked_by_peer = np.asarray(
        motion_result["blocked_by_peer"],
        dtype=bool,
    ).copy()
    safety_distance = float(
        CONFIG["safety_distance"]
    )

    while True:
        effective_positions = np.stack(
            [
                (
                    positions_before[index]
                    if blocked[index]
                    else uavs[index].position
                )
                for index in range(count)
            ],
            axis=0,
        )
        next_blocked = blocked.copy()

        for left in range(count):
            if not active_before[left]:
                continue
            for right in range(
                left + 1,
                count,
            ):
                if not active_before[right]:
                    continue
                distance = minimum_distance_during_motion(
                    positions_before[left],
                    effective_positions[left],
                    positions_before[right],
                    effective_positions[right],
                )
                if distance < safety_distance:
                    next_blocked[left] = True
                    next_blocked[right] = True
                    blocked_by_peer[left] = True
                    blocked_by_peer[right] = True

        if np.array_equal(
            next_blocked,
            blocked,
        ):
            break
        blocked = next_blocked

    for index, uav in enumerate(uavs):
        if not active_before[index]:
            uav.velocity = np.zeros(
                3,
                dtype=np.float64,
            )
            continue
        if blocked[index]:
            uav.position = (
                positions_before[index].copy()
            )
            uav.velocity = np.zeros(
                3,
                dtype=np.float64,
            )

    final_velocities = np.stack(
        [
            uav.velocity.copy()
            for uav in uavs
        ],
        axis=0,
    )
    final_realized = (
        final_velocities
        - velocities_before
    ) / dt
    motion_result[
        "realized_acceleration_mps2"
    ] = final_realized
    motion_result["blocked"] = blocked
    motion_result[
        "blocked_by_peer"
    ] = blocked_by_peer

    propulsion_required = np.zeros(
        count,
        dtype=np.float64,
    )
    propulsion_charged = np.zeros(
        count,
        dtype=np.float64,
    )

    for index, (uav, hybrid_action) in enumerate(
        zip(uavs, hybrid_actions)
    ):
        if not active_before[index]:
            continue
        available = max(
            0.0,
            float(battery_before_j[index]),
        )
        final_required = propulsion_energy_j(
            uav,
            hybrid_action,
            dt=dt,
            realized_acceleration_mps2=(
                final_realized[index]
            ),
        )
        propulsion_required[index] = (
            final_required
        )
        if battery_limited[index]:
            propulsion_charged[index] = (
                available
            )
        elif final_required > (
            available + 1e-12
        ):
            # A newly blocked path can require a larger braking impulse than
            # the originally feasible segment. It consumes the remaining
            # battery and becomes inactive without re-introducing motion.
            battery_limited[index] = True
            battery_fraction[index] = 0.0
            propulsion_charged[index] = (
                available
            )
            uav.position = (
                positions_before[index].copy()
            )
            uav.velocity = np.zeros(
                3,
                dtype=np.float64,
            )
            final_realized[index] = (
                -velocities_before[index]
                / dt
            )
        else:
            propulsion_charged[index] = (
                final_required
            )

    motion_result[
        "realized_acceleration_mps2"
    ] = final_realized
    motion_result[
        "battery_limited"
    ] = battery_limited
    motion_result[
        "battery_fraction"
    ] = battery_fraction

    for index, uav in enumerate(uavs):
        if battery_limited[index]:
            uav.active = False
            uav.velocity = np.zeros(
                3,
                dtype=np.float64,
            )

    communication_budget_j = np.maximum(
        0.0,
        battery_before_j - propulsion_charged,
    )
    communication_budget_j[
        battery_limited
    ] = 0.0

    return {
        "propulsion_required_j": propulsion_required,
        "propulsion_charged_j": propulsion_charged,
        "communication_budget_j": communication_budget_j,
        "battery_limited": battery_limited,
        "battery_fraction": battery_fraction,
    }


In [ ]:
def transmission_intent_with_energy_budget(
    intent,
    energy_budget_j,
    dt,
    backend_name,
):
    if intent is None:
        return None
    if not isinstance(intent, TransmissionIntent):
        raise TypeError("intent must be TransmissionIntent or None")

    energy_budget_j = float(energy_budget_j)
    dt = float(dt)
    if (
        not np.isfinite(energy_budget_j)
        or energy_budget_j < 0.0
    ):
        raise ValueError(
            "energy_budget_j must be finite and >= 0"
        )
    if not np.isfinite(dt) or dt <= 0.0:
        raise ValueError("dt must be finite and > 0")
    if energy_budget_j <= 0.0:
        return None

    tx_power_w = float(intent.tx_power_w)
    if tx_power_w <= 0.0:
        return None

    full_step_tx_energy = tx_power_w * dt
    if energy_budget_j >= full_step_tx_energy - 1e-12:
        return intent

    backend = str(backend_name).strip().lower()
    if backend in {"uavnetsim", "uav_net_sim"}:
        bit_rate_bps = float(
            CONFIG["uavnetsim_bit_rate_bps"]
        )
    else:
        bit_rate_bps = float(
            CONFIG["comm_max_link_rate_bps"]
        )

    affordable_duration_s = min(
        dt,
        energy_budget_j / tx_power_w,
    )
    max_bytes = int(
        np.floor(
            bit_rate_bps
            * affordable_duration_s
            / 8.0
            + 1e-9
        )
    )
    if max_bytes <= 0:
        return None
    if max_bytes >= int(intent.requested_bytes):
        return intent

    return TransmissionIntent(
        sender=int(intent.sender),
        recipient=int(intent.recipient),
        target_id=int(intent.target_id),
        requested_bytes=max_bytes,
        tx_power_w=tx_power_w,
    )


In [ ]:
def _normalized_relative_vector(
    source_position,
    destination_position,
):
    source_position = np.asarray(
        source_position,
        dtype=np.float64,
    )
    destination_position = np.asarray(
        destination_position,
        dtype=np.float64,
    )

    if (
        source_position.shape != (3,)
        or destination_position.shape
        != (3,)
    ):
        raise ValueError(
            "positions must have shape (3,)"
        )

    altitude_span = max(
        1e-9,
        float(CONFIG["altitude_max"])
        - float(CONFIG["altitude_min"]),
    )

    scale = np.array(
        [
            float(CONFIG["map_size"]),
            float(CONFIG["map_size"]),
            altitude_span,
        ],
        dtype=np.float64,
    )

    return np.clip(
        (
            destination_position
            - source_position
        )
        / scale,
        -1.0,
        1.0,
    ).astype(
        np.float32
    )


In [ ]:
def extract_belief_patch(
    belief_map,
    position,
):
    patch_cells = int(
        CONFIG["belief_patch_cells"]
    )

    if (
        patch_cells < 1
        or patch_cells % 2 == 0
    ):
        raise ValueError(
            "belief_patch_cells must be "
            "a positive odd integer"
        )

    belief_map = np.asarray(
        belief_map,
        dtype=np.float64,
    )

    if belief_map.ndim != 2:
        raise ValueError(
            "belief_map must be 2-D"
        )

    gx, gy = world_to_grid(
        position
    )

    radius = (
        patch_cells // 2
    )

    patch = np.full(
        (
            patch_cells,
            patch_cells,
        ),
        float(
            CONFIG["belief_prior"]
        ),
        dtype=np.float32,
    )

    height, width = (
        belief_map.shape
    )

    for patch_y in range(
        patch_cells
    ):
        map_y = (
            gy
            + patch_y
            - radius
        )

        if not 0 <= map_y < height:
            continue

        for patch_x in range(
            patch_cells
        ):
            map_x = (
                gx
                + patch_x
                - radius
            )

            if not 0 <= map_x < width:
                continue

            patch[
                patch_y,
                patch_x,
            ] = float(
                belief_map[
                    map_y,
                    map_x,
                ]
            )

    return patch


In [ ]:
def build_destination_mask(
    sender_id,
    uavs,
    buffer=None,
):
    choices = communication_choices(
        sender_id,
        len(uavs),
    )

    mask = np.zeros(
        len(choices),
        dtype=np.int8,
    )

    # Silent is always valid.
    mask[0] = 1

    sender = uavs[
        sender_id
    ]

    if not sender.active:
        return mask

    if buffer is not None:
        has_transmittable_report = any(
            not report_is_complete(
                report
            )
            for report in buffer
        )

        if not has_transmittable_report:
            return mask

    gcs_position = np.asarray(
        CONFIG[
            "gcs_position"
        ],
        dtype=np.float64,
    )

    for index, destination in enumerate(
        choices[1:],
        start=1,
    ):
        if destination == GCS_NODE:
            distance = float(
                np.linalg.norm(
                    sender.position
                    - gcs_position
                )
            )
            mask[index] = int(
                distance
                <= float(
                    CONFIG[
                        "gcs_contact_range_m"
                    ]
                )
            )
            continue

        peer = uavs[
            destination
        ]

        if not peer.active:
            continue

        distance = float(
            np.linalg.norm(
                sender.position
                - peer.position
            )
        )

        mask[index] = int(
            distance
            <= float(
                CONFIG[
                    "peer_contact_range_m"
                ]
            )
        )

    return mask


In [ ]:
def build_obstacle_features(
    uav,
    obstacles,
):
    count = int(
        CONFIG[
            "observation_nearest_obstacles"
        ]
    )

    if count < 0:
        raise ValueError(
            "observation_nearest_obstacles "
            "must be >= 0"
        )

    features = np.zeros(
        (count, 5),
        dtype=np.float32,
    )

    if count == 0:
        return features

    map_size = max(
        1e-9,
        float(
            CONFIG[
                "map_size"
            ]
        ),
    )
    altitude_max = max(
        1e-9,
        float(
            CONFIG[
                "altitude_max"
            ]
        ),
    )
    diagonal = max(
        1e-9,
        np.sqrt(2.0)
        * map_size,
    )

    ranked = sorted(
        obstacles,
        key=lambda obstacle: (
            float(
                np.linalg.norm(
                    np.asarray(
                        obstacle.position,
                        dtype=np.float64,
                    )
                    - uav.position[:2]
                )
            )
        ),
    )

    for row_index, obstacle in enumerate(
        ranked[:count]
    ):
        relative_xy = (
            np.asarray(
                obstacle.position,
                dtype=np.float64,
            )
            - uav.position[:2]
        )
        horizontal_distance = float(
            np.linalg.norm(
                relative_xy
            )
        )

        features[
            row_index
        ] = np.array(
            [
                np.clip(
                    relative_xy[0]
                    / map_size,
                    -1.0,
                    1.0,
                ),
                np.clip(
                    relative_xy[1]
                    / map_size,
                    -1.0,
                    1.0,
                ),
                np.clip(
                    horizontal_distance
                    / diagonal,
                    0.0,
                    1.0,
                ),
                np.clip(
                    float(
                        obstacle.radius
                    )
                    / map_size,
                    0.0,
                    1.0,
                ),
                np.clip(
                    float(
                        obstacle.height
                    )
                    / altitude_max,
                    0.0,
                    1.0,
                ),
            ],
            dtype=np.float32,
        )

    return features


In [ ]:
def coarse_belief_uncertainty_map(
    belief_map,
    output_cells=None,
):
    values = np.asarray(
        belief_map,
        dtype=np.float64,
    )

    if values.ndim != 2:
        raise ValueError(
            "belief_map must be 2-D"
        )

    if output_cells is None:
        output_cells = int(
            CONFIG[
                "belief_coarse_cells"
            ]
        )

    output_cells = int(
        output_cells
    )

    if output_cells < 1:
        raise ValueError(
            "belief_coarse_cells "
            "must be >= 1"
        )

    if (
        output_cells
        > values.shape[0]
        or output_cells
        > values.shape[1]
    ):
        raise ValueError(
            "belief_coarse_cells cannot "
            "exceed belief-map size"
        )

    probabilities = np.clip(
        values,
        1e-12,
        1.0 - 1e-12,
    )
    entropy = (
        -probabilities
        * np.log2(
            probabilities
        )
        - (
            1.0
            - probabilities
        )
        * np.log2(
            1.0
            - probabilities
        )
    )

    if (
        entropy.shape[0] % output_cells == 0
        and entropy.shape[1] % output_cells == 0
    ):
        row_size = entropy.shape[0] // output_cells
        col_size = entropy.shape[1] // output_cells
        pooled = (
            entropy.reshape(
                output_cells,
                row_size,
                output_cells,
                col_size,
            )
            .mean(axis=(1, 3))
            .astype(np.float32)
        )
        return np.clip(
            pooled,
            0.0,
            1.0,
        )

    row_groups = np.array_split(
        np.arange(
            entropy.shape[0]
        ),
        output_cells,
    )
    col_groups = np.array_split(
        np.arange(
            entropy.shape[1]
        ),
        output_cells,
    )

    pooled = np.zeros(
        (
            output_cells,
            output_cells,
        ),
        dtype=np.float32,
    )

    for row_index, rows in enumerate(
        row_groups
    ):
        for col_index, cols in enumerate(
            col_groups
        ):
            pooled[
                row_index,
                col_index,
            ] = float(
                np.mean(
                    entropy[
                        np.ix_(
                            rows,
                            cols,
                        )
                    ]
                )
            )

    return np.clip(
        pooled,
        0.0,
        1.0,
    )


In [ ]:
def build_agent_observation(
    uav_id,
    uavs,
    belief_maps,
    obstacles,
    report_buffers,
    pending_reports,
    current_step,
):
    uav = uavs[uav_id]

    map_size = float(
        CONFIG["map_size"]
    )
    altitude_min = float(
        CONFIG["altitude_min"]
    )
    altitude_span = max(
        1e-9,
        float(
            CONFIG[
                "altitude_max"
            ]
        )
        - altitude_min,
    )
    max_speed = max(
        1e-9,
        float(
            CONFIG[
                "max_speed"
            ]
        ),
    )
    battery_capacity = max(
        1e-9,
        float(
            CONFIG[
                "battery_j"
            ]
        ),
    )
    time_fraction = float(
        np.clip(
            float(
                current_step
            )
            / max(
                1.0,
                float(
                    CONFIG[
                        "max_steps"
                    ]
                ),
            ),
            0.0,
            1.0,
        )
    )

    self_state = np.array(
        [
            np.clip(
                uav.position[0]
                / map_size,
                0.0,
                1.0,
            ),
            np.clip(
                uav.position[1]
                / map_size,
                0.0,
                1.0,
            ),
            np.clip(
                (
                    uav.position[2]
                    - altitude_min
                )
                / altitude_span,
                0.0,
                1.0,
            ),
            np.clip(
                uav.velocity[0]
                / max_speed,
                -1.0,
                1.0,
            ),
            np.clip(
                uav.velocity[1]
                / max_speed,
                -1.0,
                1.0,
            ),
            np.clip(
                uav.velocity[2]
                / max_speed,
                -1.0,
                1.0,
            ),
            np.clip(
                uav.battery_j
                / battery_capacity,
                0.0,
                1.0,
            ),
            float(
                uav.active
            ),
            time_fraction,
        ],
        dtype=np.float32,
    )

    gcs_relative = (
        _normalized_relative_vector(
            uav.position,
            np.asarray(
                CONFIG[
                    "gcs_position"
                ],
                dtype=np.float64,
            ),
        )
    )

    peer_rows = []

    for peer_id, peer in enumerate(
        uavs
    ):
        if peer_id == uav_id:
            continue

        row = np.zeros(
            5,
            dtype=np.float32,
        )

        distance = float(
            np.linalg.norm(
                peer.position
                - uav.position
            )
        )

        visible = bool(
            uav.active
            and peer.active
            and distance
            <= float(
                CONFIG[
                    "peer_contact_range_m"
                ]
            )
        )

        if visible:
            row[:3] = (
                _normalized_relative_vector(
                    uav.position,
                    peer.position,
                )
            )
            row[3] = float(
                np.clip(
                    distance
                    / max(
                        1e-9,
                        float(
                            CONFIG[
                                "peer_contact_range_m"
                            ]
                        ),
                    ),
                    0.0,
                    1.0,
                )
            )
            row[4] = 1.0

        peer_rows.append(
            row
        )

    neighbors = np.stack(
        peer_rows,
        axis=0,
    ).astype(
        np.float32
    )

    obstacle_features = (
        build_obstacle_features(
            uav,
            obstacles,
        )
    )

    buffer = report_buffers[
        uav_id
    ]

    max_buffer_reports = max(
        1.0,
        float(
            CONFIG[
                "buffer_bytes"
            ]
        )
        / max(
            1.0,
            float(
                CONFIG[
                    "report_bytes"
                ]
            ),
        ),
    )

    used_fraction = np.clip(
        buffer_used_bytes(
            buffer
        )
        / max(
            1.0,
            float(
                CONFIG[
                    "buffer_bytes"
                ]
            ),
        ),
        0.0,
        1.0,
    )

    report_fraction = np.clip(
        len(buffer)
        / max_buffer_reports,
        0.0,
        1.0,
    )

    oldest_age_fraction = 0.0

    for report in buffer:
        if report.ttl_s <= 0.0:
            continue

        oldest_age_fraction = max(
            oldest_age_fraction,
            float(
                np.clip(
                    report_age_s(
                        report,
                        current_step,
                    )
                    / report.ttl_s,
                    0.0,
                    1.0,
                )
            ),
        )

    pending_bytes = (
        pending_source_used_bytes(
            pending_reports,
            uav_id,
        )
    )

    pending_fraction = np.clip(
        pending_bytes
        / max(
            1.0,
            float(
                CONFIG[
                    "pending_buffer_bytes"
                ]
            ),
        ),
        0.0,
        1.0,
    )

    head_delivery_fraction = 0.0

    for report in buffer:
        if report.size_bytes <= 0:
            continue

        head_delivery_fraction = float(
            np.clip(
                float(
                    report.delivered_bytes
                )
                / float(
                    report.size_bytes
                ),
                0.0,
                1.0,
            )
        )
        break

    buffer_state = np.array(
        [
            used_fraction,
            report_fraction,
            oldest_age_fraction,
            pending_fraction,
            head_delivery_fraction,
        ],
        dtype=np.float32,
    )

    return {
        "self_state": (
            self_state
        ),
        "gcs_relative": (
            gcs_relative
        ),
        "neighbors": (
            neighbors
        ),
        "obstacles": (
            obstacle_features
        ),
        "belief_patch": (
            extract_belief_patch(
                belief_maps[
                    uav_id
                ],
                uav.position,
            )
        ),
        "belief_coarse": (
            coarse_belief_uncertainty_map(
                belief_maps[
                    uav_id
                ]
            )
        ),
        "buffer": (
            buffer_state
        ),
        "destination_mask": (
            build_destination_mask(
                uav_id,
                uavs,
                buffer=buffer,
            )
        ),
    }


In [ ]:
def belief_map_summary(
    belief_map,
):
    values = np.asarray(
        belief_map,
        dtype=np.float64,
    )

    if values.ndim != 2:
        raise ValueError(
            "belief_map must be 2-D"
        )

    probabilities = np.clip(
        values,
        1e-12,
        1.0 - 1e-12,
    )

    entropy = (
        -probabilities
        * np.log2(
            probabilities
        )
        - (
            1.0
            - probabilities
        )
        * np.log2(
            1.0
            - probabilities
        )
    )

    mean_entropy = float(
        np.mean(
            entropy
        )
    )

    known_fraction = float(
        np.mean(
            np.abs(
                probabilities
                - float(
                    CONFIG[
                        "belief_prior"
                    ]
                )
            )
            > 0.05
        )
    )

    return np.array(
        [
            np.clip(
                mean_entropy,
                0.0,
                1.0,
            ),
            np.clip(
                known_fraction,
                0.0,
                1.0,
            ),
        ],
        dtype=np.float32,
    )


In [ ]:
def pool_belief_map(
    belief_map,
    output_cells=None,
):
    values = np.asarray(
        belief_map,
        dtype=np.float64,
    )

    if values.ndim != 2:
        raise ValueError(
            "belief_map must be 2-D"
        )

    if output_cells is None:
        output_cells = int(
            CONFIG[
                "critic_belief_grid_cells"
            ]
        )

    output_cells = int(
        output_cells
    )

    if output_cells < 1:
        raise ValueError(
            "critic_belief_grid_cells "
            "must be >= 1"
        )

    if (
        output_cells
        > values.shape[0]
        or output_cells
        > values.shape[1]
    ):
        raise ValueError(
            "critic_belief_grid_cells "
            "cannot exceed belief-map size"
        )

    if (
        values.shape[0] % output_cells == 0
        and values.shape[1] % output_cells == 0
    ):
        row_size = values.shape[0] // output_cells
        col_size = values.shape[1] // output_cells
        pooled = (
            values.reshape(
                output_cells,
                row_size,
                output_cells,
                col_size,
            )
            .mean(axis=(1, 3))
            .astype(np.float32)
        )
        return np.clip(
            pooled,
            0.0,
            1.0,
        )

    row_groups = np.array_split(
        np.arange(
            values.shape[0]
        ),
        output_cells,
    )
    col_groups = np.array_split(
        np.arange(
            values.shape[1]
        ),
        output_cells,
    )

    pooled = np.zeros(
        (
            output_cells,
            output_cells,
        ),
        dtype=np.float32,
    )

    for row_index, rows in enumerate(
        row_groups
    ):
        for col_index, cols in enumerate(
            col_groups
        ):
            pooled[
                row_index,
                col_index,
            ] = float(
                np.mean(
                    values[
                        np.ix_(
                            rows,
                            cols,
                        )
                    ]
                )
            )

    return np.clip(
        pooled,
        0.0,
        1.0,
    )


In [ ]:
def report_state_features(
    report,
    current_step,
    num_targets,
):
    if not isinstance(
        report,
        Report,
    ):
        raise TypeError(
            "report must be Report"
        )

    target_scale = max(
        1.0,
        float(
            num_targets
            - 1
        ),
    )
    age_fraction = float(
        np.clip(
            report_age_s(
                report,
                current_step,
            )
            / float(
                report.ttl_s
            ),
            0.0,
            1.0,
        )
    )
    delivery_fraction = float(
        np.clip(
            float(
                report.delivered_bytes
            )
            / max(
                1.0,
                float(
                    report.size_bytes
                ),
            ),
            0.0,
            1.0,
        )
    )

    return np.asarray(
        [
            1.0,
            np.clip(
                float(
                    report.target_id
                )
                / target_scale,
                0.0,
                1.0,
            ),
            age_fraction,
            delivery_fraction,
        ],
        dtype=np.float32,
    )


In [ ]:
def fixed_report_slots(
    reports,
    slot_count,
    current_step,
    num_targets,
):
    slot_count = int(
        slot_count
    )

    if slot_count < 0:
        raise ValueError(
            "slot_count must be >= 0"
        )

    features = np.zeros(
        (
            slot_count,
            4,
        ),
        dtype=np.float32,
    )

    ordered = sorted(
        reports,
        key=lambda report: (
            int(
                report.created_step
            ),
            int(
                report.target_id
            ),
        ),
    )

    for index, report in enumerate(
        ordered[:slot_count]
    ):
        features[index] = (
            report_state_features(
                report,
                current_step,
                num_targets,
            )
        )

    return features


In [ ]:
def build_global_state(
    uavs,
    targets,
    belief_maps,
    obstacles,
    report_buffers,
    pending_reports,
    gcs_received_target_ids,
    current_step,
    observations=None,
):
    values = []

    observation_keys = (
        "self_state",
        "gcs_relative",
        "neighbors",
        "obstacles",
        "belief_patch",
        "belief_coarse",
        "buffer",
        "destination_mask",
    )

    for uav_id in range(
        len(uavs)
    ):
        if observations is None:
            observation = (
                build_agent_observation(
                    uav_id,
                    uavs,
                    belief_maps,
                    obstacles,
                    report_buffers,
                    pending_reports,
                    current_step,
                )
            )
        else:
            observation = observations[
                f"uav_{uav_id}"
            ]

        for key in (
            observation_keys
        ):
            values.extend(
                np.asarray(
                    observation[
                        key
                    ],
                    dtype=np.float32,
                )
                .reshape(-1)
                .tolist()
            )

        buffer_slot_count = int(
            CONFIG[
                "buffer_bytes"
            ]
            // CONFIG[
                "report_bytes"
            ]
        )
        pending_slot_count = int(
            CONFIG[
                "pending_buffer_bytes"
            ]
            // CONFIG[
                "report_bytes"
            ]
        )

        values.extend(
            fixed_report_slots(
                report_buffers[
                    uav_id
                ],
                buffer_slot_count,
                current_step,
                len(targets),
            )
            .reshape(-1)
            .tolist()
        )

        source_pending = [
            report
            for report
            in pending_reports
            if int(
                report.source_uav
            )
            == int(
                uav_id
            )
        ]

        values.extend(
            fixed_report_slots(
                source_pending,
                pending_slot_count,
                current_step,
                len(targets),
            )
            .reshape(-1)
            .tolist()
        )

        values.extend(
            belief_map_summary(
                belief_maps[
                    uav_id
                ]
            ).tolist()
        )

        values.extend(
            pool_belief_map(
                belief_maps[
                    uav_id
                ]
            )
            .reshape(-1)
            .tolist()
        )

    map_size = max(
        1e-9,
        float(
            CONFIG[
                "map_size"
            ]
        ),
    )
    altitude_max = max(
        1e-9,
        float(
            CONFIG[
                "altitude_max"
            ]
        ),
    )

    for target in targets:
        values.extend(
            [
                np.clip(
                    target.position[0]
                    / map_size,
                    0.0,
                    1.0,
                ),
                np.clip(
                    target.position[1]
                    / map_size,
                    0.0,
                    1.0,
                ),
                float(
                    target.confirmed
                ),
                float(
                    target.id
                    in gcs_received_target_ids
                ),
            ]
        )

    for obstacle in obstacles:
        values.extend(
            [
                np.clip(
                    obstacle.position[0]
                    / map_size,
                    0.0,
                    1.0,
                ),
                np.clip(
                    obstacle.position[1]
                    / map_size,
                    0.0,
                    1.0,
                ),
                np.clip(
                    float(
                        obstacle.radius
                    )
                    / map_size,
                    0.0,
                    1.0,
                ),
                np.clip(
                    float(
                        obstacle.height
                    )
                    / altitude_max,
                    0.0,
                    1.0,
                ),
            ]
        )

    values.append(
        np.clip(
            float(
                current_step
            )
            / max(
                1.0,
                float(
                    CONFIG[
                        "max_steps"
                    ]
                ),
            ),
            0.0,
            1.0,
        )
    )

    return np.asarray(
        values,
        dtype=np.float32,
    )


In [ ]:
_CHECKPOINT_TYPE_KEY = "__uav_checkpoint_type__"


In [ ]:
def _checkpoint_pack(value):
    """Convert environment state to weights-only-safe checkpoint containers."""
    if isinstance(value, np.ndarray):
        array = np.ascontiguousarray(
            value
        )
        return {
            _CHECKPOINT_TYPE_KEY: "ndarray",
            "dtype": array.dtype.str,
            "shape": list(
                array.shape
            ),
            "data": array.tobytes(
                order="C"
            ),
        }

    if isinstance(value, np.generic):
        return value.item()

    if (
        is_dataclass(value)
        and not isinstance(value, type)
    ):
        return {
            _CHECKPOINT_TYPE_KEY: "dataclass",
            "name": type(value).__name__,
            "fields": _checkpoint_pack(
                asdict(value)
            ),
        }

    if isinstance(value, dict):
        return {
            _CHECKPOINT_TYPE_KEY: "dict",
            "items": [
                [
                    _checkpoint_pack(key),
                    _checkpoint_pack(item),
                ]
                for key, item in value.items()
            ],
        }

    if isinstance(value, list):
        return {
            _CHECKPOINT_TYPE_KEY: "list",
            "items": [
                _checkpoint_pack(item)
                for item in value
            ],
        }

    if isinstance(value, tuple):
        return {
            _CHECKPOINT_TYPE_KEY: "tuple",
            "items": [
                _checkpoint_pack(item)
                for item in value
            ],
        }

    if isinstance(value, set):
        return {
            _CHECKPOINT_TYPE_KEY: "set",
            "items": [
                _checkpoint_pack(item)
                for item in sorted(
                    value,
                    key=repr,
                )
            ],
        }

    if value is None or isinstance(
        value,
        (bool, int, float, str),
    ):
        return value

    raise TypeError(
        "unsupported checkpoint value "
        f"{type(value).__name__}"
    )


In [ ]:
def _checkpoint_unpack(value):
    """Restore values produced by _checkpoint_pack."""
    if not isinstance(value, dict):
        return value

    kind = value.get(
        _CHECKPOINT_TYPE_KEY
    )

    if kind == "ndarray":
        data = value["data"]
        if not isinstance(
            data,
            (bytes, bytearray),
        ):
            raise TypeError(
                "checkpoint ndarray payload "
                "must be bytes-like"
            )
        array = np.frombuffer(
            data,
            dtype=np.dtype(
                value["dtype"]
            ),
        ).copy()
        return array.reshape(
            tuple(
                int(size)
                for size
                in value["shape"]
            )
        )

    if kind == "dict":
        return {
            _checkpoint_unpack(key): (
                _checkpoint_unpack(item)
            )
            for key, item
            in value["items"]
        }

    if kind == "list":
        return [
            _checkpoint_unpack(item)
            for item in value["items"]
        ]

    if kind == "tuple":
        return tuple(
            _checkpoint_unpack(item)
            for item in value["items"]
        )

    if kind == "set":
        return {
            _checkpoint_unpack(item)
            for item in value["items"]
        }

    if kind == "dataclass":
        classes = {
            "UAV": UAV,
            "Target": Target,
            "Obstacle": Obstacle,
            "Report": Report,
            "PeerTransferState": (
                PeerTransferState
            ),
        }
        class_name = value["name"]
        if class_name not in classes:
            raise ValueError(
                "unsupported checkpoint "
                f"dataclass {class_name!r}"
            )
        fields = _checkpoint_unpack(
            value["fields"]
        )
        return classes[class_name](
            **fields
        )

    raise ValueError(
        "invalid packed checkpoint value"
    )


In [ ]:
class UAVSearchEnv(gym.Env):
    """Cooperative multi-UAV search environment with structured MARL actions."""

    metadata: ClassVar[dict] = {
        "render_modes": [],
    }

    def __init__(
        self,
        backend_name=None,
        network_backend_kwargs=None,
    ):
        super().__init__()

        self.num_uavs = int(
            CONFIG["num_uavs"]
        )
        self.num_targets = int(
            CONFIG["num_targets"]
        )

        if self.num_uavs < 1:
            raise ValueError(
                "num_uavs must be >= 1"
            )

        patch_cells = int(
            CONFIG["belief_patch_cells"]
        )

        if (
            patch_cells < 1
            or patch_cells % 2 == 0
        ):
            raise ValueError(
                "belief_patch_cells must be "
                "a positive odd integer"
            )

        self.agent_ids = tuple(
            f"uav_{uav_id}"
            for uav_id
            in range(self.num_uavs)
        )

        self.backend_name = (
            CONFIG.get(
                "network_backend",
                "simple",
            )
            if backend_name is None
            else backend_name
        )

        self.network_backend_kwargs = dict(
            network_backend_kwargs
            or {}
        )

        destination_count = (
            self.num_uavs + 1
        )

        self.action_space = (
            gym.spaces.Dict(
                {
                    agent_id: (
                        gym.spaces.Dict(
                            {
                                "motion": (
                                    gym.spaces.Box(
                                        low=-1.0,
                                        high=1.0,
                                        shape=(3,),
                                        dtype=np.float32,
                                    )
                                ),
                                "destination": (
                                    gym.spaces.Discrete(
                                        destination_count
                                    )
                                ),
                                "power": (
                                    gym.spaces.Box(
                                        low=-1.0,
                                        high=1.0,
                                        shape=(1,),
                                        dtype=np.float32,
                                    )
                                ),
                            }
                        )
                    )
                    for agent_id
                    in self.agent_ids
                }
            )
        )

        self.observation_space = (
            gym.spaces.Dict(
                {
                    agent_id: (
                        gym.spaces.Dict(
                            {
                                "self_state": (
                                    gym.spaces.Box(
                                        low=-1.0,
                                        high=1.0,
                                        shape=(9,),
                                        dtype=np.float32,
                                    )
                                ),
                                "gcs_relative": (
                                    gym.spaces.Box(
                                        low=-1.0,
                                        high=1.0,
                                        shape=(3,),
                                        dtype=np.float32,
                                    )
                                ),
                                "neighbors": (
                                    gym.spaces.Box(
                                        low=-1.0,
                                        high=1.0,
                                        shape=(
                                            self.num_uavs
                                            - 1,
                                            5,
                                        ),
                                        dtype=np.float32,
                                    )
                                ),
                                "obstacles": (
                                    gym.spaces.Box(
                                        low=-1.0,
                                        high=1.0,
                                        shape=(
                                            int(
                                                CONFIG[
                                                    "observation_nearest_obstacles"
                                                ]
                                            ),
                                            5,
                                        ),
                                        dtype=np.float32,
                                    )
                                ),
                                "belief_patch": (
                                    gym.spaces.Box(
                                        low=0.0,
                                        high=1.0,
                                        shape=(
                                            patch_cells,
                                            patch_cells,
                                        ),
                                        dtype=np.float32,
                                    )
                                ),
                                "belief_coarse": (
                                    gym.spaces.Box(
                                        low=0.0,
                                        high=1.0,
                                        shape=(
                                            int(
                                                CONFIG[
                                                    "belief_coarse_cells"
                                                ]
                                            ),
                                            int(
                                                CONFIG[
                                                    "belief_coarse_cells"
                                                ]
                                            ),
                                        ),
                                        dtype=np.float32,
                                    )
                                ),
                                "buffer": (
                                    gym.spaces.Box(
                                        low=0.0,
                                        high=1.0,
                                        shape=(5,),
                                        dtype=np.float32,
                                    )
                                ),
                                "destination_mask": (
                                    gym.spaces.MultiBinary(
                                        destination_count
                                    )
                                ),
                            }
                        )
                    )
                    for agent_id
                    in self.agent_ids
                }
            )
        )

        local_observation_size = (
            9
            + 3
            + (
                self.num_uavs
                - 1
            )
            * 5
            + int(
                CONFIG[
                    "observation_nearest_obstacles"
                ]
            )
            * 5
            + patch_cells
            * patch_cells
            + int(
                CONFIG[
                    "belief_coarse_cells"
                ]
            )
            ** 2
            + 5
            + destination_count
        )

        report_slots_per_uav = (
            int(
                CONFIG[
                    "buffer_bytes"
                ]
                // CONFIG[
                    "report_bytes"
                ]
            )
            + int(
                CONFIG[
                    "pending_buffer_bytes"
                ]
                // CONFIG[
                    "report_bytes"
                ]
            )
        )

        state_size = (
            self.num_uavs
            * (
                local_observation_size
                + report_slots_per_uav
                * 4
                + 2
                + int(
                    CONFIG[
                        "critic_belief_grid_cells"
                    ]
                )
                ** 2
            )
            + self.num_targets
            * 4
            + int(
                CONFIG[
                    "num_obstacles"
                ]
            )
            * 4
            + 1
        )

        self.state_space = (
            gym.spaces.Box(
                low=-1.0,
                high=1.0,
                shape=(state_size,),
                dtype=np.float32,
            )
        )

        self.current_step = 0
        self.episode_seed = None
        self.rng = None
        self.uavs = None
        self.targets = None
        self.obstacles = None
        self.belief_maps = None
        self.report_buffers = None
        self.pending_reports = None
        self.gcs_received_target_ids = None
        self.network_backend = None
        self._episode_done = False
        self._pending_checkpoint_state = (
            None
        )

    def _make_network_backend(
        self,
        seed,
    ):
        kwargs = dict(
            self.network_backend_kwargs
        )

        normalized = str(
            self.backend_name
        ).strip().lower()

        if normalized in {
            "uavnetsim",
            "uav_net_sim",
        }:
            kwargs.setdefault(
                "seed",
                int(seed),
            )

        return create_network_backend(
            self.backend_name,
            **kwargs,
        )

    def _canonical_actions(
        self,
        actions,
    ):
        if not isinstance(
            actions,
            dict,
        ):
            raise TypeError(
                "actions must be a dict "
                "keyed by UAV agent id"
            )

        if set(actions) != set(
            self.agent_ids
        ):
            raise ValueError(
                "actions must contain exactly "
                "one entry for every UAV"
            )

        canonical = {}

        for agent_id in self.agent_ids:
            action = actions[
                agent_id
            ]

            if not isinstance(
                action,
                dict,
            ):
                raise TypeError(
                    f"{agent_id} action "
                    "must be a dict"
                )

            if set(action) != {
                "motion",
                "destination",
                "power",
            }:
                raise ValueError(
                    f"{agent_id} action must "
                    "contain exactly motion, "
                    "destination, power"
                )

            destination = action[
                "destination"
            ]

            if isinstance(
                destination,
                (bool, np.bool_),
            ):
                raise TypeError(
                    "destination must be "
                    "an integer"
                )

            if not isinstance(
                destination,
                (int, np.integer),
            ):
                raise TypeError(
                    "destination must be "
                    "an integer"
                )

            motion = np.asarray(
                action["motion"],
                dtype=np.float32,
            )

            power = np.asarray(
                action["power"],
                dtype=np.float32,
            )

            if power.shape == ():
                power = power.reshape(
                    1
                )

            candidate = {
                "motion": motion,
                "destination": int(
                    destination
                ),
                "power": power,
            }

            subspace = (
                self.action_space.spaces[
                    agent_id
                ]
            )

            if not subspace.contains(
                candidate
            ):
                raise ValueError(
                    f"{agent_id} action is "
                    "outside action_space"
                )

            canonical[
                agent_id
            ] = candidate

        return canonical

    def _decode_actions(
        self,
        actions,
    ):
        return [
            decode_hybrid_action(
                uav_id,
                actions[
                    self.agent_ids[
                        uav_id
                    ]
                ],
                self.num_uavs,
            )
            for uav_id
            in range(self.num_uavs)
        ]

    def _observations(self):
        observations = {
            agent_id: (
                build_agent_observation(
                    uav_id,
                    self.uavs,
                    self.belief_maps,
                    self.obstacles,
                    self.report_buffers,
                    self.pending_reports,
                    self.current_step,
                )
            )
            for uav_id, agent_id
            in enumerate(
                self.agent_ids
            )
        }

        if not self.observation_space.contains(
            observations
        ):
            raise RuntimeError(
                "generated observation is "
                "outside observation_space"
            )

        return observations

    def global_state(
        self,
        observations=None,
    ):
        if self.uavs is None:
            raise RuntimeError(
                "reset must be called before "
                "global_state"
            )

        state = build_global_state(
            self.uavs,
            self.targets,
            self.belief_maps,
            self.obstacles,
            self.report_buffers,
            self.pending_reports,
            self.gcs_received_target_ids,
            self.current_step,
            observations=observations,
        )

        if not self.state_space.contains(
            state
        ):
            raise RuntimeError(
                "generated global state is "
                "outside state_space"
            )

        return state

    def checkpoint_state(self):
        """Return exact simple-backend state in a torch-safe representation."""
        if self.uavs is None:
            raise RuntimeError(
                "reset must be called before "
                "checkpoint_state"
            )

        normalized = str(
            self.backend_name
        ).strip().lower()
        if (
            normalized != "simple"
            or not isinstance(
                self.network_backend,
                SimpleNetworkBackend,
            )
        ):
            raise ValueError(
                "exact environment checkpointing "
                "supports simple backend only"
            )

        state = {
            "backend_name": "simple",
            "current_step": int(
                self.current_step
            ),
            "episode_seed": int(
                self.episode_seed
            ),
            "episode_done": bool(
                self._episode_done
            ),
            "belief_potential": float(
                self._belief_potential
            ),
            "rng_state_json": json.dumps(
                self.rng.bit_generator.state
            ),
            "uavs": self.uavs,
            "targets": self.targets,
            "obstacles": self.obstacles,
            "belief_maps": (
                self.belief_maps
            ),
            "report_buffers": (
                self.report_buffers
            ),
            "pending_reports": (
                self.pending_reports
            ),
            "gcs_received_target_ids": (
                self.gcs_received_target_ids
            ),
            "network_peer_transfer_states": (
                self.network_backend
                ._peer_transfer_states
            ),
            "network_metric_state": (
                self.network_backend
                ._metric_state
            ),
            "network_last_energy": (
                self.network_backend
                ._last_step_comm_energy_by_uav
            ),
        }
        return {
            "format_version": 1,
            "payload": _checkpoint_pack(
                state
            ),
        }

    def restore_checkpoint_state(
        self,
        checkpoint_state,
    ):
        """Restore a state returned by checkpoint_state without resetting it."""
        if not isinstance(
            checkpoint_state,
            dict,
        ):
            raise TypeError(
                "checkpoint_state must be "
                "a dict"
            )

        if int(
            checkpoint_state.get(
                "format_version",
                -1,
            )
        ) != 1:
            raise ValueError(
                "unsupported environment "
                "checkpoint format"
            )

        state = _checkpoint_unpack(
            checkpoint_state["payload"]
        )
        if (
            state.get("backend_name")
            != "simple"
            or str(
                self.backend_name
            ).strip().lower()
            != "simple"
        ):
            raise ValueError(
                "environment checkpoint backend "
                "does not match simple backend"
            )

        if self.network_backend is not None:
            close_method = getattr(
                self.network_backend,
                "close",
                None,
            )
            if callable(close_method):
                close_method()

        self.current_step = int(
            state["current_step"]
        )
        self.episode_seed = int(
            state["episode_seed"]
        )
        self._episode_done = bool(
            state["episode_done"]
        )
        self._belief_potential = float(
            state["belief_potential"]
        )
        self.uavs = state["uavs"]
        self.targets = state["targets"]
        self.obstacles = state["obstacles"]
        self.belief_maps = np.asarray(
            state["belief_maps"],
            dtype=np.float64,
        ).copy()
        self.report_buffers = (
            state["report_buffers"]
        )
        self.pending_reports = (
            state["pending_reports"]
        )
        self.gcs_received_target_ids = (
            state[
                "gcs_received_target_ids"
            ]
        )

        self.rng = np.random.default_rng()
        self.rng.bit_generator.state = (
            json.loads(
                state["rng_state_json"]
            )
        )
        super().reset(
            seed=self.episode_seed
        )

        self.network_backend = (
            self._make_network_backend(
                self.episode_seed
            )
        )
        if not isinstance(
            self.network_backend,
            SimpleNetworkBackend,
        ):
            raise TypeError(
                "restored backend must be "
                "SimpleNetworkBackend"
            )

        self.network_backend.reset(
            uavs=self.uavs,
            gcs_position=CONFIG[
                "gcs_position"
            ],
            obstacles=self.obstacles,
            report_buffers=(
                self.report_buffers
            ),
            pending_reports=(
                self.pending_reports
            ),
            gcs_received_target_ids=(
                self.gcs_received_target_ids
            ),
        )
        self.network_backend._peer_transfer_states = (
            state[
                "network_peer_transfer_states"
            ]
        )
        self.network_backend._metric_state = (
            state["network_metric_state"]
        )
        self.network_backend._last_step_comm_energy_by_uav = (
            np.asarray(
                state[
                    "network_last_energy"
                ],
                dtype=np.float64,
            ).copy()
        )

        observations = self._observations()
        info = {
            "step": self.current_step,
            "seed": self.episode_seed,
            "backend": str(
                self.backend_name
            ),
            "state": self.global_state(
                observations=observations
            ),
        }
        return observations, info

    def restore_pending_checkpoint_state(
        self,
    ):
        """Worker-callable bridge for per-env AsyncVectorEnv restoration."""
        if (
            self._pending_checkpoint_state
            is None
        ):
            raise RuntimeError(
                "no pending checkpoint state"
            )

        checkpoint_state = (
            self._pending_checkpoint_state
        )
        self._pending_checkpoint_state = (
            None
        )
        return self.restore_checkpoint_state(
            checkpoint_state
        )

    def reset(
        self,
        *,
        seed=None,
        options=None,
    ):
        del options

        if seed is None:
            seed = int(
                CONFIG.get(
                    "seed",
                    44,
                )
            )

        seed = int(seed)

        super().reset(
            seed=seed
        )

        if self.network_backend is not None:
            close_method = getattr(
                self.network_backend,
                "close",
                None,
            )

            if callable(
                close_method
            ):
                close_method()

        # The world owns its RNG; resetting it must not reseed the learner.

        (
            self.rng,
            self.uavs,
            self.targets,
            self.obstacles,
        ) = create_world(seed)

        self.belief_maps = (
            create_belief_maps()
        )
        self._belief_potential = (
            belief_certainty_potential(
                self.belief_maps
            )
        )
        self.report_buffers = (
            create_report_buffers()
        )
        self.pending_reports = (
            create_pending_reports()
        )
        self.gcs_received_target_ids = (
            create_gcs_received_target_ids()
        )

        self.current_step = 0
        self.episode_seed = seed
        self._episode_done = False

        self.network_backend = (
            self._make_network_backend(
                seed
            )
        )

        self.network_backend.reset(
            uavs=self.uavs,
            gcs_position=CONFIG[
                "gcs_position"
            ],
            obstacles=self.obstacles,
            report_buffers=(
                self.report_buffers
            ),
            pending_reports=(
                self.pending_reports
            ),
            gcs_received_target_ids=(
                self.gcs_received_target_ids
            ),
        )

        observations = (
            self._observations()
        )

        info = {
            "step": 0,
            "seed": seed,
            "backend": str(
                self.backend_name
            ),
            "state": (
                self.global_state(
                    observations=observations,
                )
            ),
        }

        return observations, info

    def step(
        self,
        actions,
    ):
        if self.uavs is None:
            raise RuntimeError(
                "reset must be called before step"
            )

        if self._episode_done:
            raise RuntimeError(
                "episode is done; call reset "
                "before stepping again"
            )

        canonical_actions = (
            self._canonical_actions(
                actions
            )
        )
        hybrid_actions = (
            self._decode_actions(
                canonical_actions
            )
        )

        next_step = (
            self.current_step + 1
        )

        expired_target_ids = set()

        for buffer in self.report_buffers:
            expired_target_ids.update(
                report.target_id
                for report
                in remove_expired_reports(
                    buffer,
                    next_step,
                )
            )

        expired_target_ids.update(
            report.target_id
            for report
            in remove_expired_reports(
                self.pending_reports,
                next_step,
            )
        )

        flush_before = (
            flush_pending_reports(
                self.pending_reports,
                self.report_buffers,
                next_step,
                self.gcs_received_target_ids,
                self.uavs,
            )
        )

        expired_target_ids.update(
            flush_before[
                "expired_target_ids"
            ]
        )

        confirmed_before = {
            target.id
            for target in self.targets
            if target.confirmed
        }

        delivered_before = set(
            self.gcs_received_target_ids
        )

        motion_actions = np.stack(
            [
                action.movement
                for action
                in hybrid_actions
            ],
            axis=0,
        )

        positions_before_motion = np.stack(
            [
                uav.position.copy()
                for uav in self.uavs
            ],
            axis=0,
        )
        velocities_before_motion = np.stack(
            [
                uav.velocity.copy()
                for uav in self.uavs
            ],
            axis=0,
        )
        battery_before_step = np.asarray(
            [
                float(uav.battery_j)
                for uav in self.uavs
            ],
            dtype=np.float64,
        )
        active_before_step = np.asarray(
            [
                bool(uav.active)
                for uav in self.uavs
            ],
            dtype=bool,
        )

        motion_result = (
            apply_swarm_motion(
                self.uavs,
                motion_actions,
                obstacles=self.obstacles,
                dt=CONFIG["dt"],
            )
        )
        battery_motion = (
            apply_battery_limited_motion(
                self.uavs,
                hybrid_actions,
                motion_result,
                positions_before_motion,
                velocities_before_motion,
                battery_before_step,
                active_before_step,
                dt=CONFIG["dt"],
            )
        )

        sensing_record_count = 0
        positive_observation_count = 0
        target_positive_observation_count = 0
        targets_in_fov_ids = set()
        information_gain_bits = 0.0
        confirmation_events = []

        belief_potential_before = float(
            self._belief_potential
        )

        for uav_id, uav in enumerate(
            self.uavs
        ):
            records = sense_and_update(
                uav,
                self.belief_maps[
                    uav_id
                ],
                self.targets,
                self.rng,
                obstacles=(
                    self.obstacles
                ),
            )

            sensing_record_count += len(
                records
            )
            information_gain_bits += (
                sensing_information_gain_bits(
                    records
                )
            )

            for record in records:
                observation_positive = bool(
                    record.get(
                        "observation",
                        0,
                    )
                )
                has_target = bool(
                    record.get(
                        "has_target",
                        False,
                    )
                )
                positive_observation_count += int(
                    observation_positive
                )
                target_positive_observation_count += int(
                    observation_positive
                    and has_target
                )
                if has_target:
                    targets_in_fov_ids.update(
                        int(target_id)
                        for target_id in record.get(
                            "target_ids",
                            [],
                        )
                    )
                confirmation_events.extend(
                    create_confirmation_events(
                        uav,
                        record,
                        next_step,
                    )
                )

        _, event_log = (
            process_confirmation_events(
                confirmation_events,
                self.targets,
                self.report_buffers,
                self.pending_reports,
                self.gcs_received_target_ids,
            )
        )

        for event in event_log:
            if (
                event.confirmation_type
                != "false_confirmation"
            ):
                continue

            gx, gy = event.cell

            self.belief_maps[
                int(
                    event.uav_id
                ),
                int(gy),
                int(gx),
            ] = float(
                CONFIG[
                    "verified_empty_belief"
                ]
            )

        belief_potential_after = (
            belief_certainty_potential(
                self.belief_maps
            )
        )
        self._belief_potential = float(
            belief_potential_after
        )

        flush_after_confirmation = (
            flush_pending_reports(
                self.pending_reports,
                self.report_buffers,
                next_step,
                self.gcs_received_target_ids,
                self.uavs,
            )
        )

        expired_target_ids.update(
            flush_after_confirmation[
                "expired_target_ids"
            ]
        )

        intents = []

        peer_transfer_states = (
            self.network_backend
            .peer_transfer_states
        )

        for uav_id, (
            uav,
            hybrid_action,
        ) in enumerate(
            zip(
                self.uavs,
                hybrid_actions,
            )
        ):
            if not uav.active:
                continue

            intent = (
                build_transmission_intent(
                    uav_id,
                    hybrid_action,
                    self.report_buffers[
                        uav_id
                    ],
                    next_step,
                    peer_transfer_states=(
                        peer_transfer_states
                    ),
                )
            )

            intent = (
                transmission_intent_with_energy_budget(
                    intent,
                    battery_motion[
                        "communication_budget_j"
                    ][uav_id],
                    CONFIG["dt"],
                    self.backend_name,
                )
            )

            if intent is not None:
                intents.append(
                    intent
                )

        self.network_backend.sync_positions(
            self.uavs
        )
        set_energy_budget = getattr(
            self.network_backend,
            "set_step_energy_budget_j",
            None,
        )
        if callable(set_energy_budget):
            set_energy_budget(
                battery_motion[
                    "communication_budget_j"
                ]
            )

        network_results = (
            self.network_backend.step(
                intents,
                dt=CONFIG["dt"],
                current_step=next_step,
            )
        )

        flush_after_network = (
            flush_pending_reports(
                self.pending_reports,
                self.report_buffers,
                next_step,
                self.gcs_received_target_ids,
                self.uavs,
            )
        )

        expired_target_ids.update(
            flush_after_network[
                "expired_target_ids"
            ]
        )

        communication_energy = (
            self.network_backend
            .last_step_communication_energy_by_uav()
        )
        communication_energy = np.minimum(
            np.asarray(
                communication_energy,
                dtype=np.float64,
            ),
            battery_motion[
                "communication_budget_j"
            ],
        )

        energy_result = (
            apply_uav_energy_budget(
                self.uavs,
                hybrid_actions,
                communication_energy,
                dt=CONFIG["dt"],
                realized_accelerations_mps2=(
                    motion_result[
                        "realized_acceleration_mps2"
                    ]
                ),
                propulsion_energy_override_j=(
                    battery_motion[
                        "propulsion_charged_j"
                    ]
                ),
            )
        )
        energy_result[
            "depleted_uav_ids"
        ] = sorted(
            set(
                int(value)
                for value in energy_result[
                    "depleted_uav_ids"
                ]
            )
            | set(
                int(value)
                for value in np.flatnonzero(
                    battery_motion[
                        "battery_limited"
                    ]
                )
            )
        )

        confirmed_after = {
            target.id
            for target in self.targets
            if target.confirmed
        }

        delivered_after = set(
            self.gcs_received_target_ids
        )

        newly_confirmed = sorted(
            confirmed_after
            - confirmed_before
        )
        newly_delivered = sorted(
            delivered_after
            - delivered_before
        )

        false_confirmation_count = sum(
            event.confirmation_type
            == "false_confirmation"
            for event in event_log
        )

        dropped_report_count = sum(
            event.report_status
            == "dropped"
            for event in event_log
        )

        blocked_count = int(
            np.count_nonzero(
                motion_result[
                    "blocked"
                ]
            )
        )
        boundary_count = int(
            np.count_nonzero(
                motion_result[
                    "boundary_clipped"
                ]
            )
        )

        success = (
            len(
                self.gcs_received_target_ids
            )
            == self.num_targets
        )

        all_inactive = not any(
            uav.active
            for uav in self.uavs
        )

        episode_will_end = bool(
            success
            or all_inactive
            or next_step
            >= int(
                CONFIG[
                    "max_steps"
                ]
            )
        )

        shaping_next_potential = (
            0.0
            if episode_will_end
            else belief_potential_after
        )

        information_shaping = float(
            CONFIG[
                "reward_info_gain"
            ]
        ) * (
            float(
                CONFIG[
                    "reward_shaping_gamma"
                ]
            )
            * shaping_next_potential
            - belief_potential_before
        )

        reward_components = {
            "information_gain": (
                information_shaping
            ),
            "confirmation": (
                float(
                    CONFIG[
                        "reward_confirmation"
                    ]
                )
                * len(
                    newly_confirmed
                )
            ),
            "delivery": (
                float(
                    CONFIG[
                        "reward_delivery"
                    ]
                )
                * len(
                    newly_delivered
                )
            ),
            "false_confirmation": (
                -float(
                    CONFIG[
                        "reward_false_confirmation"
                    ]
                )
                * false_confirmation_count
            ),
            "blocked_motion": (
                -float(
                    CONFIG[
                        "reward_blocked"
                    ]
                )
                * blocked_count
            ),
            "boundary": (
                -float(
                    CONFIG[
                        "reward_boundary"
                    ]
                )
                * boundary_count
            ),
            "expired_report": (
                -float(
                    CONFIG[
                        "reward_expired_report"
                    ]
                )
                * len(
                    expired_target_ids
                )
            ),
            "dropped_report": (
                -float(
                    CONFIG[
                        "reward_dropped_report"
                    ]
                )
                * dropped_report_count
            ),
            "energy": (
                -float(
                    CONFIG[
                        "reward_energy_per_kj"
                    ]
                )
                * float(
                    np.sum(
                        energy_result[
                            "total_j"
                        ]
                    )
                )
                / 1000.0
            ),
            "step": (
                -float(
                    CONFIG[
                        "reward_step_penalty"
                    ]
                )
            ),
            "success_bonus": (
                float(
                    CONFIG[
                        "reward_all_delivered_bonus"
                    ]
                )
                if success
                else 0.0
            ),
        }

        reward = float(
            sum(
                reward_components.values()
            )
        )

        self.current_step = next_step

        horizon_reached = bool(
            self.current_step
            >= int(
                CONFIG[
                    "max_steps"
                ]
            )
        )
        # max_steps is the finite mission horizon, not an
        # external TimeLimit wrapper. The remaining-time fraction
        # is already part of each actor observation and the critic
        # state, so reaching the horizon is an MDP terminal event.
        terminated = bool(
            success
            or all_inactive
            or horizon_reached
        )
        truncated = False

        self._episode_done = bool(
            terminated
            or truncated
        )

        observations = (
            self._observations()
        )
        state = self.global_state(
            observations=observations,
        )

        agent_rewards = {
            agent_id: reward
            for agent_id
            in self.agent_ids
        }

        info = {
            "step": self.current_step,
            "backend": str(
                self.backend_name
            ),
            "state": state,
            "agent_rewards": (
                agent_rewards
            ),
            "reward_components": (
                reward_components
            ),
            "motion": motion_result,
            "network_results": (
                make_info_pickle_safe(
                    network_results
                )
            ),
            "network_metrics": (
                self.network_backend
                .metrics()
            ),
            "sensing_record_count": (
                sensing_record_count
            ),
            "positive_observation_count": int(
                positive_observation_count
            ),
            "target_positive_observation_count": int(
                target_positive_observation_count
            ),
            "targets_in_fov_ids": sorted(
                targets_in_fov_ids
            ),
            "information_gain_bits": (
                information_gain_bits
            ),
            "belief_certainty_before": (
                belief_potential_before
            ),
            "belief_certainty_after": (
                belief_potential_after
            ),
            "information_shaping": (
                information_shaping
            ),
            "newly_confirmed_target_ids": (
                newly_confirmed
            ),
            "newly_delivered_target_ids": (
                newly_delivered
            ),
            "false_confirmation_count": (
                int(
                    false_confirmation_count
                )
            ),
            "dropped_report_count": (
                int(
                    dropped_report_count
                )
            ),
            "expired_target_ids": sorted(
                expired_target_ids
            ),
            "energy": energy_result,
            "success": bool(success),
            "all_uavs_inactive": bool(
                all_inactive
            ),
        }

        return (
            observations,
            reward,
            terminated,
            truncated,
            info,
        )

    def close(self):
        if self.network_backend is None:
            return

        close_method = getattr(
            self.network_backend,
            "close",
            None,
        )

        if callable(close_method):
            close_method()

        self.network_backend = None


In [ ]:
import concurrent.futures


In [ ]:
import queue


In [ ]:
import threading


def _get_torch_sensing_tls():
    # Resolve process-local state lazily instead of capturing an unpicklable
    # threading.local in cloudpickle's dependency graph for the DDP worker.
    local_state = globals().get("_TORCH_SENSING_TLS")
    if local_state is None:
        local_state = threading.local()
        globals()["_TORCH_SENSING_TLS"] = local_state
    return local_state


_get_torch_sensing_tls()


In [ ]:
_CPU_SENSE_AND_UPDATE = sense_and_update


In [ ]:
class TorchBatchedSensingEngine:
    """Batch the expensive FOV/LOS/Bayes sensing math with torch.

    The mission/world/report/network semantics stay in UAVSearchEnv. Only the
    per-UAV sensing kernel is vectorized. CPU sensing remains the reference
    implementation and is used whenever no coordinator is active.
    """

    def __init__(
        self,
        device="auto",
        dtype="float32",
    ):
        if str(device).strip().lower() == "auto":
            device = (
                "cuda"
                if torch.cuda.is_available()
                else "cpu"
            )

        self.device = torch.device(device)

        normalized_dtype = str(dtype).strip().lower()
        if normalized_dtype == "float64":
            self.dtype = torch.float64
        elif normalized_dtype == "float32":
            self.dtype = torch.float32
        else:
            raise ValueError(
                "training_torch_sensing_dtype must be "
                "'float32' or 'float64'"
            )

        self.batch_count = 0
        self.request_count = 0
        self.candidate_count = 0
        self.kernel_seconds = 0.0

    @staticmethod
    def _circle_sqrt_integral_torch(
        x,
        radius,
    ):
        radius_safe = torch.clamp(
            radius,
            min=torch.finfo(x.dtype).eps,
        )
        x = torch.maximum(
            torch.minimum(x, radius),
            -radius,
        )
        root = torch.sqrt(
            torch.clamp(
                radius * radius - x * x,
                min=0.0,
            )
        )
        ratio = torch.clamp(
            x / radius_safe,
            -1.0,
            1.0,
        )
        return 0.5 * (
            x * root
            + radius * radius
            * torch.asin(ratio)
        )

    def _coverage_fraction(
        self,
        circle_x,
        circle_y,
        radius,
        gx,
        gy,
    ):
        cell_size = float(CONFIG["grid_cell_m"])
        map_size = float(CONFIG["map_size"])

        x_min = gx * cell_size
        x_max = torch.clamp(
            (gx + 1.0) * cell_size,
            max=map_size,
        )
        y_min = gy * cell_size
        y_max = torch.clamp(
            (gy + 1.0) * cell_size,
            max=map_size,
        )

        x_min = x_min - circle_x
        x_max = x_max - circle_x
        y_min = y_min - circle_y
        y_max = y_max - circle_y

        left = torch.maximum(
            x_min,
            -radius,
        )
        right = torch.minimum(
            x_max,
            radius,
        )

        base_valid = (
            (radius > 0.0)
            & (right > left)
            & (y_max > -radius)
            & (y_min < radius)
        )

        def _crossings(y_edge):
            has_crossing = (
                torch.abs(y_edge)
                < radius
            )
            x_cross = torch.sqrt(
                torch.clamp(
                    radius * radius
                    - y_edge * y_edge,
                    min=0.0,
                )
            )
            negative = torch.where(
                has_crossing,
                -x_cross,
                left,
            )
            positive = torch.where(
                has_crossing,
                x_cross,
                left,
            )
            negative = torch.maximum(
                left,
                torch.minimum(
                    right,
                    negative,
                ),
            )
            positive = torch.maximum(
                left,
                torch.minimum(
                    right,
                    positive,
                ),
            )
            return negative, positive

        y_min_neg, y_min_pos = _crossings(
            y_min
        )
        y_max_neg, y_max_pos = _crossings(
            y_max
        )

        cuts = torch.stack(
            (
                left,
                right,
                y_min_neg,
                y_min_pos,
                y_max_neg,
                y_max_pos,
            ),
            dim=-1,
        )
        cuts = torch.sort(
            cuts,
            dim=-1,
        ).values

        a = cuts[:, :-1]
        b = cuts[:, 1:]
        width = b - a
        interval_valid = width > 0.0
        midpoint = 0.5 * (a + b)

        radius_2d = radius[:, None]
        y_min_2d = y_min[:, None]
        y_max_2d = y_max[:, None]

        half_height = torch.sqrt(
            torch.clamp(
                radius_2d * radius_2d
                - midpoint * midpoint,
                min=0.0,
            )
        )
        upper = torch.minimum(
            y_max_2d,
            half_height,
        )
        lower = torch.maximum(
            y_min_2d,
            -half_height,
        )

        arc_integral = (
            self._circle_sqrt_integral_torch(
                b,
                radius_2d,
            )
            - self._circle_sqrt_integral_torch(
                a,
                radius_2d,
            )
        )

        upper_integral = torch.where(
            y_max_2d < half_height,
            y_max_2d * width,
            arc_integral,
        )
        lower_integral = torch.where(
            y_min_2d > -half_height,
            y_min_2d * width,
            -arc_integral,
        )

        area_terms = torch.where(
            interval_valid
            & (upper > lower),
            upper_integral - lower_integral,
            torch.zeros_like(width),
        )
        intersection_area = torch.clamp(
            area_terms.sum(dim=-1),
            min=0.0,
        )

        cell_area = torch.clamp(
            (x_max - x_min)
            * (y_max - y_min),
            min=torch.finfo(
                intersection_area.dtype
            ).eps,
        )
        fraction = torch.clamp(
            intersection_area / cell_area,
            0.0,
            1.0,
        )
        return torch.where(
            base_valid,
            fraction,
            torch.zeros_like(fraction),
        )

    @staticmethod
    def _segment_hits_any_obstacle(
        start,
        end,
        centers,
        radii,
        heights,
        obstacle_valid,
    ):
        if centers.shape[1] == 0:
            return torch.zeros(
                start.shape[0],
                dtype=torch.bool,
                device=start.device,
            )

        eps = 1e-12
        direction = end - start
        dz = direction[:, 2:3]
        start_z = start[:, 2:3]

        horizontal_z = (
            torch.abs(dz) <= eps
        )
        safe_dz = torch.where(
            horizontal_z,
            torch.ones_like(dz),
            dz,
        )

        t_ground = (
            -start_z / safe_dz
        )
        t_top = (
            heights - start_z
        ) / safe_dz

        z_enter_dynamic = torch.maximum(
            torch.zeros_like(t_top),
            torch.minimum(
                t_ground,
                t_top,
            ),
        )
        z_exit_dynamic = torch.minimum(
            torch.ones_like(t_top),
            torch.maximum(
                t_ground,
                t_top,
            ),
        )

        z_enter = torch.where(
            horizontal_z,
            torch.zeros_like(
                z_enter_dynamic
            ),
            z_enter_dynamic,
        )
        z_exit = torch.where(
            horizontal_z,
            torch.ones_like(
                z_exit_dynamic
            ),
            z_exit_dynamic,
        )
        z_valid = torch.where(
            horizontal_z,
            (
                (start_z >= 0.0)
                & (start_z <= heights)
            ),
            z_enter <= z_exit,
        )

        relative_xy = (
            start[:, None, :2]
            - centers
        )
        direction_xy = (
            direction[:, None, :2]
        )
        a = (
            direction_xy
            * direction_xy
        ).sum(dim=-1)
        c = (
            relative_xy
            * relative_xy
        ).sum(dim=-1) - radii * radii

        stationary_xy = a <= eps
        safe_a = torch.where(
            stationary_xy,
            torch.ones_like(a),
            a,
        )

        b = 2.0 * (
            relative_xy
            * direction_xy
        ).sum(dim=-1)
        discriminant = (
            b * b
            - 4.0 * safe_a * c
        )
        root = torch.sqrt(
            torch.clamp(
                discriminant,
                min=0.0,
            )
        )

        t1 = (
            -b - root
        ) / (2.0 * safe_a)
        t2 = (
            -b + root
        ) / (2.0 * safe_a)

        xy_enter_dynamic = torch.maximum(
            torch.zeros_like(t1),
            torch.minimum(t1, t2),
        )
        xy_exit_dynamic = torch.minimum(
            torch.ones_like(t1),
            torch.maximum(t1, t2),
        )

        xy_enter = torch.where(
            stationary_xy,
            torch.zeros_like(
                xy_enter_dynamic
            ),
            xy_enter_dynamic,
        )
        xy_exit = torch.where(
            stationary_xy,
            torch.ones_like(
                xy_exit_dynamic
            ),
            xy_exit_dynamic,
        )
        xy_valid = torch.where(
            stationary_xy,
            c <= 0.0,
            (
                (discriminant >= 0.0)
                & (
                    xy_enter_dynamic
                    <= xy_exit_dynamic
                )
            ),
        )

        enter = torch.maximum(
            z_enter,
            xy_enter,
        )
        exit_ = torch.minimum(
            z_exit,
            xy_exit,
        )

        intersects = (
            obstacle_valid
            & z_valid
            & xy_valid
            & (enter <= exit_)
        )
        return torch.any(
            intersects,
            dim=-1,
        )

    def _request_obstacle_tensors(
        self,
        requests,
    ):
        max_obstacles = max(
            (
                len(
                    request["obstacles"]
                )
                for request in requests
            ),
            default=0,
        )
        count = len(requests)

        centers = np.zeros(
            (
                count,
                max_obstacles,
                2,
            ),
            dtype=np.float64,
        )
        radii = np.zeros(
            (
                count,
                max_obstacles,
            ),
            dtype=np.float64,
        )
        heights = np.zeros(
            (
                count,
                max_obstacles,
            ),
            dtype=np.float64,
        )
        valid = np.zeros(
            (
                count,
                max_obstacles,
            ),
            dtype=np.bool_,
        )

        for request_index, request in enumerate(
            requests
        ):
            for obstacle_index, obstacle in enumerate(
                request["obstacles"]
            ):
                centers[
                    request_index,
                    obstacle_index,
                ] = obstacle.position
                radii[
                    request_index,
                    obstacle_index,
                ] = float(
                    obstacle.radius
                )
                heights[
                    request_index,
                    obstacle_index,
                ] = float(
                    obstacle.height
                )
                valid[
                    request_index,
                    obstacle_index,
                ] = True

        return (
            torch.as_tensor(
                centers,
                dtype=self.dtype,
                device=self.device,
            ),
            torch.as_tensor(
                radii,
                dtype=self.dtype,
                device=self.device,
            ),
            torch.as_tensor(
                heights,
                dtype=self.dtype,
                device=self.device,
            ),
            torch.as_tensor(
                valid,
                dtype=torch.bool,
                device=self.device,
            ),
        )

    def sense_batch(
        self,
        requests,
    ):
        if not requests:
            return []

        kernel_start = time.perf_counter()

        results = [
            []
            for _ in requests
        ]

        candidate_request = []
        candidate_gx = []
        candidate_gy = []
        candidate_circle_x = []
        candidate_circle_y = []
        candidate_radius = []
        candidate_base_pd = []
        candidate_base_pf = []
        candidate_start = []
        target_id = []
        target_xy = []

        target_maps = []
        request_profiles = []

        for request in requests:
            target_map = {}
            for target in request["targets"]:
                target_map[
                    world_to_grid(
                        target.position
                    )
                ] = (
                    int(target.id),
                    np.asarray(
                        target.position,
                        dtype=np.float64,
                    ),
                )
            target_maps.append(
                target_map
            )

        for request_index, request in enumerate(
            requests
        ):
            uav = request["uav"]

            if (
                not uav.active
                or float(
                    uav.position[2]
                )
                <= 0.0
            ):
                request_profiles.append(
                    None
                )
                continue

            (
                base_pd,
                base_pf,
                fov_radius,
            ) = sensing_profile(
                float(
                    uav.position[2]
                )
            )
            request_profiles.append(
                (
                    base_pd,
                    base_pf,
                    fov_radius,
                )
            )

            cell_size = float(
                CONFIG["grid_cell_m"]
            )
            grid_n = int(
                np.ceil(
                    float(
                        CONFIG[
                            "map_size"
                        ]
                    )
                    / cell_size
                )
            )
            uav_x = float(
                uav.position[0]
            )
            uav_y = float(
                uav.position[1]
            )

            gx_min = max(
                0,
                int(
                    np.floor(
                        (
                            uav_x
                            - fov_radius
                        )
                        / cell_size
                    )
                ),
            )
            gx_max = min(
                grid_n - 1,
                int(
                    np.floor(
                        (
                            uav_x
                            + fov_radius
                        )
                        / cell_size
                    )
                ),
            )
            gy_min = max(
                0,
                int(
                    np.floor(
                        (
                            uav_y
                            - fov_radius
                        )
                        / cell_size
                    )
                ),
            )
            gy_max = min(
                grid_n - 1,
                int(
                    np.floor(
                        (
                            uav_y
                            + fov_radius
                        )
                        / cell_size
                    )
                ),
            )

            for gy in range(
                gy_min,
                gy_max + 1,
            ):
                for gx in range(
                    gx_min,
                    gx_max + 1,
                ):
                    candidate_request.append(
                        request_index
                    )
                    candidate_gx.append(gx)
                    candidate_gy.append(gy)
                    candidate_circle_x.append(
                        uav_x
                    )
                    candidate_circle_y.append(
                        uav_y
                    )
                    candidate_radius.append(
                        fov_radius
                    )
                    candidate_base_pd.append(
                        base_pd
                    )
                    candidate_base_pf.append(
                        base_pf
                    )
                    candidate_start.append(
                        np.asarray(
                            uav.position,
                            dtype=np.float64,
                        )
                    )

                    target_entry = (
                        target_maps[
                            request_index
                        ].get(
                            (gx, gy)
                        )
                    )
                    if target_entry is None:
                        target_id.append(-1)
                        target_xy.append(
                            np.zeros(
                                2,
                                dtype=np.float64,
                            )
                        )
                    else:
                        (
                            current_target_id,
                            current_target_xy,
                        ) = target_entry
                        target_id.append(
                            current_target_id
                        )
                        target_xy.append(
                            current_target_xy
                        )

        if not candidate_request:
            self.batch_count += 1
            self.request_count += len(
                requests
            )
            self.kernel_seconds += (
                time.perf_counter()
                - kernel_start
            )
            return results

        request_index_np = np.asarray(
            candidate_request,
            dtype=np.int64,
        )
        gx_np = np.asarray(
            candidate_gx,
            dtype=np.int64,
        )
        gy_np = np.asarray(
            candidate_gy,
            dtype=np.int64,
        )
        target_id_np = np.asarray(
            target_id,
            dtype=np.int64,
        )

        request_index = torch.as_tensor(
            request_index_np,
            dtype=torch.long,
            device=self.device,
        )
        gx = torch.as_tensor(
            gx_np,
            dtype=self.dtype,
            device=self.device,
        )
        gy = torch.as_tensor(
            gy_np,
            dtype=self.dtype,
            device=self.device,
        )
        circle_x = torch.as_tensor(
            candidate_circle_x,
            dtype=self.dtype,
            device=self.device,
        )
        circle_y = torch.as_tensor(
            candidate_circle_y,
            dtype=self.dtype,
            device=self.device,
        )
        radius = torch.as_tensor(
            candidate_radius,
            dtype=self.dtype,
            device=self.device,
        )
        base_pd = torch.as_tensor(
            candidate_base_pd,
            dtype=self.dtype,
            device=self.device,
        )
        base_pf = torch.as_tensor(
            candidate_base_pf,
            dtype=self.dtype,
            device=self.device,
        )
        start = torch.as_tensor(
            np.asarray(
                candidate_start,
                dtype=np.float64,
            ),
            dtype=self.dtype,
            device=self.device,
        )
        target_xy_tensor = torch.as_tensor(
            np.asarray(
                target_xy,
                dtype=np.float64,
            ),
            dtype=self.dtype,
            device=self.device,
        )
        target_id_tensor = torch.as_tensor(
            target_id_np,
            dtype=torch.long,
            device=self.device,
        )

        coverage = (
            self._coverage_fraction(
                circle_x,
                circle_y,
                radius,
                gx,
                gy,
            )
        )
        visible = coverage > 0.0

        cell_size = float(
            CONFIG["grid_cell_m"]
        )
        map_size = float(
            CONFIG["map_size"]
        )
        cell_center = torch.stack(
            (
                torch.clamp(
                    (
                        gx + 0.5
                    )
                    * cell_size,
                    max=map_size,
                ),
                torch.clamp(
                    (
                        gy + 0.5
                    )
                    * cell_size,
                    max=map_size,
                ),
                torch.zeros_like(gx),
            ),
            dim=-1,
        )

        (
            obstacle_centers,
            obstacle_radii,
            obstacle_heights,
            obstacle_valid,
        ) = self._request_obstacle_tensors(
            requests
        )
        candidate_centers = (
            obstacle_centers[
                request_index
            ]
        )
        candidate_radii = (
            obstacle_radii[
                request_index
            ]
        )
        candidate_heights = (
            obstacle_heights[
                request_index
            ]
        )
        candidate_obstacle_valid = (
            obstacle_valid[
                request_index
            ]
        )

        occlusion_enabled = bool(
            CONFIG[
                "sensing_obstacle_occlusion"
            ]
        )

        if occlusion_enabled:
            cell_blocked = (
                self._segment_hits_any_obstacle(
                    start,
                    cell_center,
                    candidate_centers,
                    candidate_radii,
                    candidate_heights,
                    candidate_obstacle_valid,
                )
            )
        else:
            cell_blocked = torch.zeros_like(
                visible
            )

        cell_valid = (
            visible
            & ~cell_blocked
        )

        target_present = (
            target_id_tensor >= 0
        )
        target_delta = (
            target_xy_tensor
            - start[:, :2]
        )
        target_in_footprint = (
            torch.linalg.vector_norm(
                target_delta,
                dim=-1,
            )
            <= radius
        )

        target_end = torch.cat(
            (
                target_xy_tensor,
                torch.zeros(
                    (
                        target_xy_tensor.shape[
                            0
                        ],
                        1,
                    ),
                    dtype=self.dtype,
                    device=self.device,
                ),
            ),
            dim=-1,
        )

        if occlusion_enabled:
            target_blocked = (
                self._segment_hits_any_obstacle(
                    start,
                    target_end,
                    candidate_centers,
                    candidate_radii,
                    candidate_heights,
                    candidate_obstacle_valid,
                )
            )
        else:
            target_blocked = (
                torch.zeros_like(
                    target_present
                )
            )

        target_visible = (
            target_present
            & target_in_footprint
            & ~target_blocked
        )
        has_target = target_visible

        effective_pf = (
            1.0
            - torch.pow(
                1.0 - base_pf,
                coverage,
            )
        )
        effective_pd = (
            coverage * base_pd
            + (
                1.0 - coverage
            )
            * effective_pf
        )
        positive_probability = (
            torch.where(
                has_target,
                base_pd,
                effective_pf,
            )
        )

        valid_np = (
            cell_valid
            .detach()
            .cpu()
            .numpy()
            .astype(np.bool_)
        )

        random_values = np.zeros(
            len(candidate_request),
            dtype=np.float64,
        )
        prior_values = np.zeros(
            len(candidate_request),
            dtype=np.float64,
        )

        for request_id, request in enumerate(
            requests
        ):
            indices = np.flatnonzero(
                valid_np
                & (
                    request_index_np
                    == request_id
                )
            )
            if indices.size == 0:
                continue

            random_values[
                indices
            ] = request[
                "rng"
            ].random(
                indices.size
            )
            prior_values[
                indices
            ] = request[
                "belief_map"
            ][
                gy_np[indices],
                gx_np[indices],
            ]

        random_tensor = torch.as_tensor(
            random_values,
            dtype=self.dtype,
            device=self.device,
        )
        prior = torch.as_tensor(
            prior_values,
            dtype=self.dtype,
            device=self.device,
        )

        observation = (
            random_tensor
            < positive_probability
        )
        eps = 1e-8
        prior_clamped = torch.clamp(
            prior,
            eps,
            1.0 - eps,
        )

        positive_numerator = (
            effective_pd
            * prior_clamped
        )
        positive_denominator = (
            positive_numerator
            + effective_pf
            * (
                1.0
                - prior_clamped
            )
        )

        negative_numerator = (
            (
                1.0
                - effective_pd
            )
            * prior_clamped
        )
        negative_denominator = (
            negative_numerator
            + (
                1.0
                - effective_pf
            )
            * (
                1.0
                - prior_clamped
            )
        )

        numerator = torch.where(
            observation,
            positive_numerator,
            negative_numerator,
        )
        denominator = torch.where(
            observation,
            positive_denominator,
            negative_denominator,
        )
        denominator = torch.clamp(
            denominator,
            min=eps,
        )
        posterior = torch.clamp(
            numerator / denominator,
            eps,
            1.0 - eps,
        )

        coverage_np = (
            coverage.detach()
            .cpu()
            .numpy()
        )
        effective_pf_np = (
            effective_pf.detach()
            .cpu()
            .numpy()
        )
        effective_pd_np = (
            effective_pd.detach()
            .cpu()
            .numpy()
        )
        base_pd_np = (
            base_pd.detach()
            .cpu()
            .numpy()
        )
        base_pf_np = (
            base_pf.detach()
            .cpu()
            .numpy()
        )
        target_visible_np = (
            target_visible.detach()
            .cpu()
            .numpy()
            .astype(np.bool_)
        )
        observation_np = (
            observation.detach()
            .cpu()
            .numpy()
            .astype(np.int8)
        )
        posterior_np = (
            posterior.detach()
            .cpu()
            .numpy()
        )

        for request_id, request in enumerate(
            requests
        ):
            indices = np.flatnonzero(
                valid_np
                & (
                    request_index_np
                    == request_id
                )
            )
            if indices.size == 0:
                continue

            request[
                "belief_map"
            ][
                gy_np[indices],
                gx_np[indices],
            ] = posterior_np[
                indices
            ]

            sensing_log = []
            for index in indices:
                current_target_ids = (
                    [
                        int(
                            target_id_np[
                                index
                            ]
                        )
                    ]
                    if target_visible_np[
                        index
                    ]
                    else []
                )
                sensing_log.append(
                    {
                        "gx": int(
                            gx_np[index]
                        ),
                        "gy": int(
                            gy_np[index]
                        ),
                        "target_ids": (
                            current_target_ids
                        ),
                        "has_target": bool(
                            target_visible_np[
                                index
                            ]
                        ),
                        "observation": int(
                            observation_np[
                                index
                            ]
                        ),
                        "prior": float(
                            prior_values[
                                index
                            ]
                        ),
                        "posterior": float(
                            posterior_np[
                                index
                            ]
                        ),
                        "pd": float(
                            effective_pd_np[
                                index
                            ]
                        ),
                        "pf": float(
                            effective_pf_np[
                                index
                            ]
                        ),
                        "base_pd": float(
                            base_pd_np[
                                index
                            ]
                        ),
                        "base_pf": float(
                            base_pf_np[
                                index
                            ]
                        ),
                        "coverage_fraction": float(
                            coverage_np[
                                index
                            ]
                        ),
                    }
                )
            results[
                request_id
            ] = sensing_log

        if (
            self.device.type
            == "cuda"
        ):
            torch.cuda.synchronize(
                self.device
            )

        self.batch_count += 1
        self.request_count += len(
            requests
        )
        self.candidate_count += len(
            candidate_request
        )
        self.kernel_seconds += (
            time.perf_counter()
            - kernel_start
        )
        return results

    def metrics(self):
        return {
            "device": str(
                self.device
            ),
            "dtype": str(
                self.dtype
            ).replace(
                "torch.",
                "",
            ),
            "batches": int(
                self.batch_count
            ),
            "requests": int(
                self.request_count
            ),
            "candidates": int(
                self.candidate_count
            ),
            "kernel_seconds": float(
                self.kernel_seconds
            ),
        }


In [ ]:
class TorchSensingCoordinator:
    def __init__(
        self,
        num_envs,
        device="auto",
        dtype="float32",
        batch_timeout_s=0.002,
    ):
        self.num_envs = int(
            num_envs
        )
        self.batch_timeout_s = float(
            batch_timeout_s
        )
        self.engine = (
            TorchBatchedSensingEngine(
                device=device,
                dtype=dtype,
            )
        )
        self._queue = queue.Queue()
        self._sentinel = object()
        self._closed = False
        self._thread = threading.Thread(
            target=self._worker,
            name="torch-sensing-batcher",
            daemon=True,
        )
        self._thread.start()

    def submit(
        self,
        *,
        env_index,
        call_index,
        uav,
        belief_map,
        targets,
        rng,
        obstacles,
    ):
        if self._closed:
            raise RuntimeError(
                "sensing coordinator is closed"
            )

        future = (
            concurrent.futures.Future()
        )
        self._queue.put(
            (
                int(env_index),
                int(call_index),
                {
                    "uav": uav,
                    "belief_map": (
                        belief_map
                    ),
                    "targets": targets,
                    "rng": rng,
                    "obstacles": (
                        []
                        if obstacles is None
                        else obstacles
                    ),
                },
                future,
            )
        )
        return future.result()

    def _worker(self):
        while True:
            first = self._queue.get()
            if (
                first is self._sentinel
            ):
                return

            batch = [first]
            deadline = (
                time.perf_counter()
                + self.batch_timeout_s
            )

            while (
                len(batch)
                < self.num_envs
            ):
                remaining = (
                    deadline
                    - time.perf_counter()
                )
                if remaining <= 0.0:
                    break

                try:
                    item = self._queue.get(
                        timeout=remaining
                    )
                except queue.Empty:
                    break

                if (
                    item
                    is self._sentinel
                ):
                    self._queue.put(
                        self._sentinel
                    )
                    break
                batch.append(item)

            batch.sort(
                key=lambda item: (
                    item[1],
                    item[0],
                )
            )
            requests = [
                item[2]
                for item in batch
            ]
            futures = [
                item[3]
                for item in batch
            ]

            try:
                results = (
                    self.engine.sense_batch(
                        requests
                    )
                )
            except (
                RuntimeError,
                ValueError,
                TypeError,
                IndexError,
                KeyError,
            ) as exc:
                for future in futures:
                    future.set_exception(
                        exc
                    )
                continue

            for future, result in zip(
                futures,
                results,
            ):
                future.set_result(
                    result
                )

    def metrics(self):
        return self.engine.metrics()

    def close(self):
        if self._closed:
            return
        self._closed = True
        self._queue.put(
            self._sentinel
        )
        self._thread.join(
            timeout=5.0
        )


In [ ]:
def sense_and_update(
    uav,
    belief_map,
    targets,
    rng,
    obstacles=None,
):
    coordinator = getattr(
        _get_torch_sensing_tls(),
        "coordinator",
        None,
    )
    if coordinator is None:
        return _CPU_SENSE_AND_UPDATE(
            uav,
            belief_map,
            targets,
            rng,
            obstacles=obstacles,
        )

    env_index = getattr(
        _get_torch_sensing_tls(),
        "env_index",
        None,
    )
    if env_index is None:
        raise RuntimeError(
            "torch sensing coordinator active "
            "without env_index"
        )

    call_index = int(
        getattr(
            _get_torch_sensing_tls(),
            "call_index",
            0,
        )
    )
    _get_torch_sensing_tls().call_index = (
        call_index + 1
    )

    return coordinator.submit(
        env_index=env_index,
        call_index=call_index,
        uav=uav,
        belief_map=belief_map,
        targets=targets,
        rng=rng,
        obstacles=obstacles,
    )


In [ ]:
def _zero_like_vector_value(value):
    if isinstance(
        value,
        np.ndarray,
    ):
        return np.zeros_like(
            value
        )
    if isinstance(
        value,
        (bool, np.bool_),
    ):
        return False
    if isinstance(
        value,
        (int, np.integer),
    ):
        return 0
    if isinstance(
        value,
        (float, np.floating),
    ):
        return 0.0
    if isinstance(value, dict):
        return {
            key: _zero_like_vector_value(
                item
            )
            for key, item
            in value.items()
        }
    return None


In [ ]:
def _stack_vector_values(values):
    if all(
        isinstance(value, dict)
        for value in values
    ):
        return _stack_vector_dicts(
            values
        )

    if all(
        isinstance(value, np.ndarray)
        for value in values
    ):
        shapes = {
            value.shape
            for value in values
        }
        if len(shapes) == 1:
            return np.stack(
                values,
                axis=0,
            )

    if all(
        isinstance(
            value,
            (
                bool,
                np.bool_,
                int,
                np.integer,
                float,
                np.floating,
            ),
        )
        for value in values
    ):
        return np.asarray(
            values
        )

    result = np.empty(
        len(values),
        dtype=object,
    )
    for index, value in enumerate(
        values
    ):
        result[index] = (
            copy.deepcopy(
                value
            )
        )
    return result


In [ ]:
def _stack_vector_dicts(
    dictionaries,
):
    keys = sorted(
        {
            key
            for dictionary
            in dictionaries
            for key
            in dictionary
        },
        key=str,
    )
    result = {}

    for key in keys:
        present = np.asarray(
            [
                key in dictionary
                for dictionary
                in dictionaries
            ],
            dtype=np.bool_,
        )
        prototype = next(
            dictionary[key]
            for dictionary
            in dictionaries
            if key in dictionary
        )
        values = [
            (
                dictionary[key]
                if key in dictionary
                else _zero_like_vector_value(
                    prototype
                )
            )
            for dictionary
            in dictionaries
        ]
        result[key] = (
            _stack_vector_values(
                values
            )
        )
        if not np.all(present):
            result[
                f"_{key}"
            ] = present

    return result


In [ ]:
class TorchThreadedSimpleVectorEnv:
    """In-process vector env that batches sensing requests on one torch device."""

    def __init__(
        self,
        env_fns,
        *,
        sensing_device="auto",
        sensing_dtype="float32",
        sensing_batch_timeout_s=0.002,
    ):
        self.envs = [
            env_fn()
            for env_fn in env_fns
        ]
        if not self.envs:
            raise ValueError(
                "env_fns must not be empty"
            )

        self.num_envs = len(
            self.envs
        )
        self.coordinator = (
            TorchSensingCoordinator(
                self.num_envs,
                device=sensing_device,
                dtype=sensing_dtype,
                batch_timeout_s=(
                    sensing_batch_timeout_s
                ),
            )
        )
        self._executor = (
            concurrent.futures.ThreadPoolExecutor(
                max_workers=self.num_envs,
                thread_name_prefix=(
                    "uav-env"
                ),
            )
        )
        self._driver_executor = (
            concurrent.futures.ThreadPoolExecutor(
                max_workers=1,
                thread_name_prefix=(
                    "uav-vector-driver"
                ),
            )
        )
        self._last_observations = [
            None
        ] * self.num_envs
        self._last_infos = [
            None
        ] * self.num_envs
        self._async_future = None
        self.backend_name = (
            "torch_threaded_simple"
        )

    @property
    def sensing_metrics(self):
        return self.coordinator.metrics()

    def _run_step(
        self,
        env_index,
        action,
    ):
        sensing_tls = _get_torch_sensing_tls()
        sensing_tls.coordinator = (
            self.coordinator
        )
        sensing_tls.env_index = int(
            env_index
        )
        sensing_tls.call_index = 0
        try:
            return self.envs[
                env_index
            ].step(action)
        finally:
            for name in (
                "coordinator",
                "env_index",
                "call_index",
            ):
                if hasattr(
                    sensing_tls,
                    name,
                ):
                    delattr(
                        sensing_tls,
                        name,
                    )

    def _action_at(
        self,
        vector_actions,
        index,
    ):
        result = {}
        for agent_id, action in (
            vector_actions.items()
        ):
            result[agent_id] = {
                "motion": np.asarray(
                    action["motion"][
                        index
                    ],
                    dtype=np.float32,
                ).copy(),
                "destination": int(
                    np.asarray(
                        action[
                            "destination"
                        ]
                    )[index]
                ),
                "power": np.asarray(
                    action["power"][
                        index
                    ],
                    dtype=np.float32,
                ).copy(),
            }
        return result

    def reset(
        self,
        *,
        seed=None,
        options=None,
    ):
        if options is None:
            options = {}

        reset_mask = options.get(
            "reset_mask"
        )
        if reset_mask is None:
            reset_mask = np.ones(
                self.num_envs,
                dtype=np.bool_,
            )
        else:
            reset_mask = np.asarray(
                reset_mask,
                dtype=np.bool_,
            )
            if reset_mask.shape != (
                self.num_envs,
            ):
                raise ValueError(
                    "reset_mask has invalid shape"
                )

        if seed is None:
            seeds = [
                None
            ] * self.num_envs
        elif isinstance(
            seed,
            (int, np.integer),
        ):
            seeds = [
                int(seed) + index
                for index in range(
                    self.num_envs
                )
            ]
        else:
            seeds = list(seed)
            if len(seeds) != self.num_envs:
                raise ValueError(
                    "seed sequence length must "
                    "match num_envs"
                )

        reset_infos = [
            {}
            for _ in range(
                self.num_envs
            )
        ]

        for index in range(
            self.num_envs
        ):
            if not reset_mask[index]:
                if (
                    self._last_observations[
                        index
                    ]
                    is None
                ):
                    raise RuntimeError(
                        "partial reset before "
                        "initial reset"
                    )
                continue

            observation, info = (
                self.envs[
                    index
                ].reset(
                    seed=seeds[index]
                )
            )
            self._last_observations[
                index
            ] = observation
            self._last_infos[
                index
            ] = info
            reset_infos[index] = info

        return (
            _stack_vector_dicts(
                self._last_observations
            ),
            _stack_vector_dicts(
                reset_infos
            ),
        )

    def step(
        self,
        vector_actions,
    ):
        futures = [
            self._executor.submit(
                self._run_step,
                index,
                self._action_at(
                    vector_actions,
                    index,
                ),
            )
            for index in range(
                self.num_envs
            )
        ]
        results = [
            future.result()
            for future in futures
        ]

        observations = [
            item[0]
            for item in results
        ]
        rewards = np.asarray(
            [
                item[1]
                for item in results
            ],
            dtype=np.float64,
        )
        terminated = np.asarray(
            [
                item[2]
                for item in results
            ],
            dtype=np.bool_,
        )
        truncated = np.asarray(
            [
                item[3]
                for item in results
            ],
            dtype=np.bool_,
        )
        infos = [
            item[4]
            for item in results
        ]

        self._last_observations = (
            observations
        )
        self._last_infos = infos

        return (
            _stack_vector_dicts(
                observations
            ),
            rewards,
            terminated,
            truncated,
            _stack_vector_dicts(
                infos
            ),
        )

    def step_async(
        self,
        vector_actions,
    ):
        if self._async_future is not None:
            raise RuntimeError(
                "step_async called twice"
            )
        self._async_future = (
            self._driver_executor.submit(
                self.step,
                vector_actions,
            )
        )

    def step_wait(self):
        if self._async_future is None:
            raise RuntimeError(
                "step_wait without step_async"
            )
        future = self._async_future
        self._async_future = None
        return future.result()

    def close(self):
        if self._async_future is not None:
            try:
                self._async_future.result()
            finally:
                self._async_future = None

        self.coordinator.close()
        self._executor.shutdown(
            wait=True
        )
        self._driver_executor.shutdown(
            wait=True
        )

        for env in self.envs:
            env.close()


In [ ]:
def resolve_training_vector_runtime(
    device,
):
    if device is None:
        resolved_device = torch.device(
            "cuda"
            if torch.cuda.is_available()
            else "cpu"
        )
    else:
        resolved_device = torch.device(
            device
        )

    backend = str(
        CONFIG.get(
            "training_vector_backend",
            "auto",
        )
    ).strip().lower()

    if backend not in {
        "auto",
        "async_cpu",
        "torch_threaded",
    }:
        raise ValueError(
            "training_vector_backend must be "
            "'auto', 'async_cpu', or "
            "'torch_threaded'"
        )

    torch_enabled = bool(
        CONFIG.get(
            "training_torch_sensing_enabled",
            True,
        )
    )

    if backend == "torch_threaded":
        use_torch_vector = True
    elif backend == "async_cpu":
        use_torch_vector = False
    else:
        use_torch_vector = bool(
            torch_enabled
            and resolved_device.type
            == "cuda"
        )

    # Worker count follows the learner device for either collector.
    if resolved_device.type == "cuda":
        num_envs = int(
            CONFIG.get(
                "training_cuda_num_envs",
                CONFIG.get(
                    "training_num_envs",
                    1,
                ),
            )
        )
    else:
        num_envs = int(
            CONFIG.get(
                "training_num_envs",
                1,
            )
        )

    num_envs = max(
        1,
        num_envs,
    )

    sensing_device = CONFIG.get(
        "training_torch_sensing_device",
        "auto",
    )
    if (
        str(
            sensing_device
        ).strip().lower()
        == "auto"
    ):
        sensing_device = str(
            resolved_device
        )

    if use_torch_vector:
        overlap = bool(
            CONFIG.get(
                "training_torch_overlap_env_and_updates",
                False,
            )
        )
    else:
        overlap = bool(
            CONFIG.get(
                "training_overlap_env_and_updates",
                True,
            )
        )

    return {
        "resolved_device": (
            resolved_device
        ),
        "backend": (
            "torch_threaded"
            if use_torch_vector
            else "async_cpu"
        ),
        "use_torch_vector": bool(
            use_torch_vector
        ),
        "num_envs": int(
            num_envs
        ),
        "overlap": bool(
            overlap
        ),
        "sensing_device": str(
            sensing_device
        ),
        "sensing_dtype": str(
            CONFIG.get(
                "training_torch_sensing_dtype",
                "float64",
            )
        ),
        "sensing_batch_timeout_s": float(
            CONFIG.get(
                "training_torch_sensing_batch_timeout_s",
                0.002,
            )
        ),
    }


In [ ]:
AGENT_OBSERVATION_VECTOR_KEYS = (
    "self_state",
    "gcs_relative",
    "neighbors",
    "obstacles",
    "belief_patch",
    "belief_coarse",
    "buffer",
    "destination_mask",
)


In [ ]:
def flatten_agent_observation(
    observation,
):
    if not isinstance(
        observation,
        dict,
    ):
        raise TypeError(
            "observation must be a dict"
        )

    if set(observation) != set(
        AGENT_OBSERVATION_VECTOR_KEYS
    ):
        raise ValueError(
            "observation keys do not match "
            "the MASAC observation contract"
        )

    parts = []

    for key in (
        AGENT_OBSERVATION_VECTOR_KEYS
    ):
        values = np.asarray(
            observation[key],
            dtype=np.float32,
        ).reshape(-1)

        if not np.all(
            np.isfinite(values)
        ):
            raise ValueError(
                f"{key} contains non-finite values"
            )

        parts.append(
            values
        )

    return np.concatenate(
        parts,
        axis=0,
    ).astype(
        np.float32,
        copy=False,
    )


In [ ]:
def observations_to_masac_arrays(
    observations,
    agent_ids,
):
    if set(observations) != set(
        agent_ids
    ):
        raise ValueError(
            "observations must contain exactly "
            "the configured agent ids"
        )

    observation_vectors = np.stack(
        [
            flatten_agent_observation(
                observations[agent_id]
            )
            for agent_id
            in agent_ids
        ],
        axis=0,
    ).astype(
        np.float32
    )

    destination_masks = np.stack(
        [
            np.asarray(
                observations[
                    agent_id
                ][
                    "destination_mask"
                ],
                dtype=np.float32,
            )
            for agent_id
            in agent_ids
        ],
        axis=0,
    )

    if (
        destination_masks.ndim != 2
        or np.any(
            destination_masks.sum(
                axis=-1
            )
            < 1.0
        )
    ):
        raise ValueError(
            "every agent needs at least "
            "one valid destination"
        )

    return (
        observation_vectors,
        destination_masks,
    )


In [ ]:
def env_actions_to_masac_arrays(
    actions,
    agent_ids,
):
    continuous = []
    destination_indices = []

    for agent_id in agent_ids:
        action = actions[
            agent_id
        ]

        motion = np.asarray(
            project_motion_action_np(
                action["motion"]
            ),
            dtype=np.float32,
        )

        power = np.asarray(
            action["power"],
            dtype=np.float32,
        ).reshape(-1)

        if (
            motion.shape != (3,)
            or power.shape != (1,)
        ):
            raise ValueError(
                "invalid hybrid action shape"
            )

        continuous.append(
            np.concatenate(
                [
                    motion,
                    power,
                ],
                axis=0,
            )
        )
        destination_indices.append(
            int(
                action["destination"]
            )
        )

    return (
        np.stack(
            continuous,
            axis=0,
        ).astype(
            np.float32
        ),
        np.asarray(
            destination_indices,
            dtype=np.int64,
        ),
    )


In [ ]:
def masac_arrays_to_env_actions(
    continuous_actions,
    destination_indices,
    agent_ids,
):
    continuous_actions = np.asarray(
        continuous_actions,
        dtype=np.float32,
    )
    destination_indices = np.asarray(
        destination_indices,
        dtype=np.int64,
    )

    if continuous_actions.shape != (
        len(agent_ids),
        4,
    ):
        raise ValueError(
            "continuous_actions must have "
            "shape (num_agents, 4)"
        )

    if destination_indices.shape != (
        len(agent_ids),
    ):
        raise ValueError(
            "destination_indices must have "
            "shape (num_agents,)"
        )

    motion_actions = np.asarray(
        project_motion_action_np(
            continuous_actions[:, :3]
        ),
        dtype=np.float32,
    )

    return {
        agent_id: {
            "motion": (
                motion_actions[
                    index
                ].copy()
            ),
            "destination": int(
                destination_indices[
                    index
                ]
            ),
            "power": np.asarray(
                [
                    continuous_actions[
                        index,
                        3,
                    ]
                ],
                dtype=np.float32,
            ),
        }
        for index, agent_id
        in enumerate(agent_ids)
    }


In [ ]:
def sample_uniform_unit_ball_motion(rng):
    direction = rng.normal(
        size=3
    )
    direction_norm = max(
        float(
            np.linalg.norm(direction)
        ),
        float(
            np.finfo(np.float64).eps
        ),
    )
    radius = float(
        rng.random() ** (1.0 / 3.0)
    )

    return (
        direction
        / direction_norm
        * radius
    ).astype(
        np.float32
    )


In [ ]:
def sample_valid_random_actions(
    observations,
    agent_ids,
    rng,
):
    actions = {}

    for agent_id in agent_ids:
        mask = np.asarray(
            observations[
                agent_id
            ][
                "destination_mask"
            ],
            dtype=np.int8,
        )

        valid = np.flatnonzero(
            mask
        )

        if valid.size == 0:
            raise RuntimeError(
                "destination mask has "
                "no valid action"
            )

        actions[
            agent_id
        ] = {
            "motion": (
                sample_uniform_unit_ball_motion(
                    rng
                )
            ),
            "destination": int(
                rng.choice(
                    valid
                )
            ),
            "power": rng.uniform(
                -1.0,
                1.0,
                size=1,
            ).astype(
                np.float32
            ),
        }

    return actions


In [ ]:
def build_mlp(
    input_dim,
    hidden_dims,
    output_dim,
):
    dims = (
        int(input_dim),
        *tuple(
            int(value)
            for value
            in hidden_dims
        ),
        int(output_dim),
    )

    layers = []

    for index in range(
        len(dims) - 2
    ):
        layers.extend(
            [
                nn.Linear(
                    dims[index],
                    dims[index + 1],
                ),
                nn.ReLU(),
            ]
        )

    layers.append(
        nn.Linear(
            dims[-2],
            dims[-1],
        )
    )

    return nn.Sequential(
        *layers
    )


In [ ]:
def masked_categorical_logits(
    logits,
    destination_mask,
):
    if logits.shape != (
        destination_mask.shape
    ):
        raise ValueError(
            "logits and destination_mask "
            "must have the same shape"
        )

    mask = (
        destination_mask > 0.5
    )

    if torch.any(
        mask.sum(
            dim=-1
        )
        < 1
    ):
        raise ValueError(
            "each categorical row needs "
            "at least one valid action"
        )

    negative_large = torch.finfo(
        logits.dtype
    ).min

    return logits.masked_fill(
        ~mask,
        negative_large,
    )


In [ ]:
def straight_through_categorical_sample(
    logits,
    destination_mask,
    temperature,
    deterministic=False,
):
    temperature = float(
        temperature
    )

    if temperature <= 0.0:
        raise ValueError(
            "temperature must be > 0"
        )

    masked_logits = (
        masked_categorical_logits(
            logits,
            destination_mask,
        )
    )

    log_probabilities = (
        F.log_softmax(
            masked_logits,
            dim=-1,
        )
    )
    probabilities = torch.exp(
        log_probabilities
    )

    if deterministic:
        indices = torch.argmax(
            masked_logits,
            dim=-1,
        )
        hard = F.one_hot(
            indices,
            num_classes=(
                masked_logits.shape[-1]
            ),
        ).to(
            masked_logits.dtype
        )
        action = hard
    else:
        uniform = torch.rand_like(
            masked_logits
        ).clamp_(
            1e-6,
            1.0 - 1e-6,
        )

        gumbel_noise = -torch.log(
            -torch.log(
                uniform
            )
        )

        soft = F.softmax(
            (
                masked_logits
                + gumbel_noise
            )
            / temperature,
            dim=-1,
        )

        indices = torch.argmax(
            soft,
            dim=-1,
        )
        hard = F.one_hot(
            indices,
            num_classes=(
                masked_logits.shape[-1]
            ),
        ).to(
            masked_logits.dtype
        )

        # Straight-through estimator:
        # forward value is hard one-hot,
        # backward gradient follows soft.
        action = (
            hard
            - soft.detach()
            + soft
        )

    selected_log_probability = (
        hard
        * log_probabilities
    ).sum(
        dim=-1,
        keepdim=True,
    )

    entropy = -(
        probabilities
        * log_probabilities
    ).sum(
        dim=-1,
        keepdim=True,
    )

    return {
        "action": action,
        "hard_action": hard,
        "indices": indices,
        "log_probability": (
            selected_log_probability
        ),
        "entropy": entropy,
        "probabilities": probabilities,
        "log_probabilities": (
            log_probabilities
        ),
    }


In [ ]:
def radial_squash_motion_action(
    pre_squash,
    eps=1e-6,
):
    """Invertibly squash R^3 into the open 3-D unit ball."""
    if pre_squash.shape[-1] != 3:
        raise ValueError(
            "pre_squash must have trailing dimension 3"
        )

    radius = torch.linalg.vector_norm(
        pre_squash,
        dim=-1,
        keepdim=True,
    )
    squashed_radius = torch.tanh(
        radius
    )
    small = radius <= eps
    scale_regular = (
        squashed_radius
        / radius.clamp_min(eps)
    )
    scale_series = (
        1.0
        - radius.pow(2) / 3.0
    )
    scale = torch.where(
        small,
        scale_series,
        scale_regular,
    )
    action = pre_squash * scale

    radial_derivative = (
        1.0
        - squashed_radius.pow(2)
    ).clamp_min(eps)
    tangential_scale = scale.clamp_min(eps)
    log_abs_det_jacobian = (
        torch.log(radial_derivative)
        + 2.0
        * torch.log(tangential_scale)
    )

    return action, log_abs_det_jacobian


In [ ]:
def project_motion_action_tensor(action):
    """Project already-bounded 3-D actions onto the closed unit ball."""
    if action.shape[-1] != 3:
        raise ValueError(
            "motion action tensor must have trailing dimension 3"
        )

    norm = torch.linalg.vector_norm(
        action,
        dim=-1,
        keepdim=True,
    )
    scale = torch.where(
        norm > 1.0,
        1.0 / norm.clamp_min(1e-12),
        torch.ones_like(norm),
    )
    return action * scale


In [ ]:
class HybridMASACActor(nn.Module):
    def __init__(
        self,
        observation_dim,
        continuous_dim,
        discrete_dim,
        hidden_dims,
    ):
        super().__init__()

        hidden_dims = tuple(
            int(value)
            for value
            in hidden_dims
        )

        if not hidden_dims:
            raise ValueError(
                "hidden_dims must not be empty"
            )

        self.observation_dim = int(
            observation_dim
        )
        self.continuous_dim = int(
            continuous_dim
        )
        self.discrete_dim = int(
            discrete_dim
        )

        self.encoder = build_mlp(
            self.observation_dim,
            hidden_dims[:-1],
            hidden_dims[-1],
        )

        feature_dim = (
            hidden_dims[-1]
        )

        self.mean_head = nn.Linear(
            feature_dim,
            self.continuous_dim,
        )
        self.log_std_head = nn.Linear(
            feature_dim,
            self.continuous_dim,
        )
        self.discrete_head = nn.Linear(
            feature_dim,
            self.discrete_dim,
        )

    def forward(
        self,
        observations,
    ):
        features = self.encoder(
            observations
        )

        mean = self.mean_head(
            features
        )

        log_std = (
            self.log_std_head(
                features
            ).clamp(
                min=float(
                    CONFIG[
                        "masac_log_std_min"
                    ]
                ),
                max=float(
                    CONFIG[
                        "masac_log_std_max"
                    ]
                ),
            )
        )

        logits = self.discrete_head(
            features
        )

        return (
            mean,
            log_std,
            logits,
        )

    def sample(
        self,
        observations,
        destination_mask,
        deterministic=False,
    ):
        (
            mean,
            log_std,
            logits,
        ) = self(
            observations
        )

        std = torch.exp(
            log_std
        )
        distribution = Normal(
            mean,
            std,
        )

        if deterministic:
            pre_tanh = mean
        else:
            pre_tanh = (
                distribution.rsample()
            )

        motion_action, motion_log_det = (
            radial_squash_motion_action(
                pre_tanh[..., :3]
            )
        )
        power_action = torch.tanh(
            pre_tanh[..., 3:4]
        )
        continuous_action = torch.cat(
            [
                motion_action,
                power_action,
            ],
            dim=-1,
        )

        power_log_det = torch.log(
            1.0
            - power_action.pow(2)
            + 1e-6
        ).sum(
            dim=-1,
            keepdim=True,
        )
        log_probability = (
            distribution.log_prob(
                pre_tanh
            ).sum(
                dim=-1,
                keepdim=True,
            )
            - motion_log_det
            - power_log_det
        )

        discrete = (
            straight_through_categorical_sample(
                logits,
                destination_mask,
                temperature=CONFIG[
                    "masac_gumbel_temperature"
                ],
                deterministic=deterministic,
            )
        )

        return {
            "continuous": (
                continuous_action
            ),
            "continuous_log_probability": (
                log_probability
            ),
            "destination_one_hot": (
                discrete["action"]
            ),
            "destination_hard_one_hot": (
                discrete[
                    "hard_action"
                ]
            ),
            "destination_index": (
                discrete["indices"]
            ),
            "destination_log_probability": (
                discrete[
                    "log_probability"
                ]
            ),
            "destination_entropy": (
                discrete["entropy"]
            ),
        }


In [ ]:
class CentralizedQNetwork(nn.Module):
    def __init__(
        self,
        state_dim,
        joint_action_dim,
        hidden_dims,
    ):
        super().__init__()

        self.state_dim = int(
            state_dim
        )
        self.joint_action_dim = int(
            joint_action_dim
        )

        self.network = build_mlp(
            self.state_dim
            + self.joint_action_dim,
            hidden_dims,
            1,
        )

    def forward(
        self,
        state,
        joint_action,
    ):
        if state.ndim != 2:
            raise ValueError(
                "state must be rank 2"
            )

        if joint_action.ndim != 2:
            raise ValueError(
                "joint_action must be rank 2"
            )

        values = torch.cat(
            [
                state,
                joint_action,
            ],
            dim=-1,
        )

        return self.network(
            values
        )


In [ ]:
class HybridReplayBuffer:
    def __init__(
        self,
        capacity,
        num_agents,
        observation_dim,
        state_dim,
        continuous_dim,
        discrete_dim,
        seed,
    ):
        self.capacity = int(
            capacity
        )
        self.num_agents = int(
            num_agents
        )
        self.observation_dim = int(
            observation_dim
        )
        self.state_dim = int(
            state_dim
        )
        self.continuous_dim = int(
            continuous_dim
        )
        self.discrete_dim = int(
            discrete_dim
        )

        if self.capacity < 1:
            raise ValueError(
                "capacity must be >= 1"
            )

        self.rng = (
            np.random.default_rng(
                int(seed)
            )
        )

        self.observations = np.zeros(
            (
                self.capacity,
                self.num_agents,
                self.observation_dim,
            ),
            dtype=np.float32,
        )
        self.next_observations = (
            np.zeros_like(
                self.observations
            )
        )
        self.states = np.zeros(
            (
                self.capacity,
                self.state_dim,
            ),
            dtype=np.float32,
        )
        self.next_states = (
            np.zeros_like(
                self.states
            )
        )
        self.continuous_actions = np.zeros(
            (
                self.capacity,
                self.num_agents,
                self.continuous_dim,
            ),
            dtype=np.float32,
        )
        self.destination_indices = np.zeros(
            (
                self.capacity,
                self.num_agents,
            ),
            dtype=np.int64,
        )
        self.destination_masks = np.zeros(
            (
                self.capacity,
                self.num_agents,
                self.discrete_dim,
            ),
            dtype=np.float32,
        )
        self.next_destination_masks = (
            np.zeros_like(
                self.destination_masks
            )
        )
        self.rewards = np.zeros(
            (
                self.capacity,
                1,
            ),
            dtype=np.float32,
        )
        self.terminated = np.zeros(
            (
                self.capacity,
                1,
            ),
            dtype=np.float32,
        )
        self.truncated = np.zeros(
            (
                self.capacity,
                1,
            ),
            dtype=np.float32,
        )

        self.position = 0
        self.size = 0

    def __len__(self):
        return int(
            self.size
        )

    def add(
        self,
        observations,
        state,
        continuous_actions,
        destination_indices,
        destination_masks,
        reward,
        next_observations,
        next_state,
        next_destination_masks,
        terminated,
        truncated,
    ):
        observations = np.asarray(
            observations,
            dtype=np.float32,
        )
        next_observations = np.asarray(
            next_observations,
            dtype=np.float32,
        )
        state = np.asarray(
            state,
            dtype=np.float32,
        )
        next_state = np.asarray(
            next_state,
            dtype=np.float32,
        )
        continuous_actions = np.asarray(
            continuous_actions,
            dtype=np.float32,
        )
        destination_indices = np.asarray(
            destination_indices,
            dtype=np.int64,
        )
        destination_masks = np.asarray(
            destination_masks,
            dtype=np.float32,
        )
        next_destination_masks = np.asarray(
            next_destination_masks,
            dtype=np.float32,
        )

        expected_observation_shape = (
            self.num_agents,
            self.observation_dim,
        )
        expected_action_shape = (
            self.num_agents,
            self.continuous_dim,
        )
        expected_mask_shape = (
            self.num_agents,
            self.discrete_dim,
        )

        if observations.shape != (
            expected_observation_shape
        ):
            raise ValueError(
                "observations have wrong shape"
            )

        if next_observations.shape != (
            expected_observation_shape
        ):
            raise ValueError(
                "next_observations have wrong shape"
            )

        if state.shape != (
            self.state_dim,
        ):
            raise ValueError(
                "state has wrong shape"
            )

        if next_state.shape != (
            self.state_dim,
        ):
            raise ValueError(
                "next_state has wrong shape"
            )

        if continuous_actions.shape != (
            expected_action_shape
        ):
            raise ValueError(
                "continuous_actions have wrong shape"
            )

        if destination_indices.shape != (
            self.num_agents,
        ):
            raise ValueError(
                "destination_indices have wrong shape"
            )

        if destination_masks.shape != (
            expected_mask_shape
        ):
            raise ValueError(
                "destination_masks have wrong shape"
            )

        if next_destination_masks.shape != (
            expected_mask_shape
        ):
            raise ValueError(
                "next_destination_masks have wrong shape"
            )

        if np.any(
            destination_indices < 0
        ) or np.any(
            destination_indices
            >= self.discrete_dim
        ):
            raise ValueError(
                "destination index out of range"
            )

        slot = int(
            self.position
        )

        self.observations[
            slot
        ] = observations
        self.next_observations[
            slot
        ] = next_observations
        self.states[
            slot
        ] = state
        self.next_states[
            slot
        ] = next_state
        self.continuous_actions[
            slot
        ] = continuous_actions
        self.destination_indices[
            slot
        ] = destination_indices
        self.destination_masks[
            slot
        ] = destination_masks
        self.next_destination_masks[
            slot
        ] = next_destination_masks
        self.rewards[
            slot,
            0,
        ] = float(
            reward
        )
        self.terminated[
            slot,
            0,
        ] = float(
            bool(
                terminated
            )
        )
        self.truncated[
            slot,
            0,
        ] = float(
            bool(
                truncated
            )
        )

        self.position = (
            slot + 1
        ) % self.capacity
        self.size = min(
            self.size + 1,
            self.capacity,
        )

    def state_dict(self):
        size = int(
            self.size
        )

        def packed_tensor(
            array,
        ):
            return torch.from_numpy(
                array[:size].copy()
            )

        return {
            "format_version": 1,
            "capacity": int(
                self.capacity
            ),
            "num_agents": int(
                self.num_agents
            ),
            "observation_dim": int(
                self.observation_dim
            ),
            "state_dim": int(
                self.state_dim
            ),
            "continuous_dim": int(
                self.continuous_dim
            ),
            "discrete_dim": int(
                self.discrete_dim
            ),
            "position": int(
                self.position
            ),
            "size": size,
            "rng_state_json": json.dumps(
                self.rng
                .bit_generator
                .state
            ),
            "observations": packed_tensor(
                self.observations
            ),
            "next_observations": (
                packed_tensor(
                    self.next_observations
                )
            ),
            "states": packed_tensor(
                self.states
            ),
            "next_states": packed_tensor(
                self.next_states
            ),
            "continuous_actions": (
                packed_tensor(
                    self.continuous_actions
                )
            ),
            "destination_indices": (
                packed_tensor(
                    self.destination_indices
                )
            ),
            "destination_masks": (
                packed_tensor(
                    self.destination_masks
                )
            ),
            "next_destination_masks": (
                packed_tensor(
                    self.next_destination_masks
                )
            ),
            "rewards": packed_tensor(
                self.rewards
            ),
            "terminated": packed_tensor(
                self.terminated
            ),
            "truncated": packed_tensor(
                self.truncated
            ),
        }

    def load_state_dict(
        self,
        state,
    ):
        if not isinstance(
            state,
            dict,
        ):
            raise TypeError(
                "replay state must be a dict"
            )

        expected = {
            "capacity": self.capacity,
            "num_agents": self.num_agents,
            "observation_dim": (
                self.observation_dim
            ),
            "state_dim": self.state_dim,
            "continuous_dim": (
                self.continuous_dim
            ),
            "discrete_dim": (
                self.discrete_dim
            ),
        }

        for key, expected_value in (
            expected.items()
        ):
            actual = int(
                state[key]
            )

            if actual != int(
                expected_value
            ):
                raise ValueError(
                    f"replay {key} mismatch: "
                    f"{actual} != "
                    f"{expected_value}"
                )

        size = int(
            state["size"]
        )
        position = int(
            state["position"]
        )

        if not 0 <= size <= self.capacity:
            raise ValueError(
                "invalid replay size"
            )

        if not 0 <= position < self.capacity:
            raise ValueError(
                "invalid replay position"
            )

        if (
            size < self.capacity
            and position != size
        ):
            raise ValueError(
                "partial replay position "
                "must equal size"
            )

        tensor_keys = (
            "observations",
            "next_observations",
            "states",
            "next_states",
            "continuous_actions",
            "destination_indices",
            "destination_masks",
            "next_destination_masks",
            "rewards",
            "terminated",
            "truncated",
        )

        destination_arrays = {
            "observations": (
                self.observations
            ),
            "next_observations": (
                self.next_observations
            ),
            "states": self.states,
            "next_states": (
                self.next_states
            ),
            "continuous_actions": (
                self.continuous_actions
            ),
            "destination_indices": (
                self.destination_indices
            ),
            "destination_masks": (
                self.destination_masks
            ),
            "next_destination_masks": (
                self.next_destination_masks
            ),
            "rewards": self.rewards,
            "terminated": (
                self.terminated
            ),
            "truncated": (
                self.truncated
            ),
        }

        for key in tensor_keys:
            destination = (
                destination_arrays[
                    key
                ]
            )
            destination.fill(
                0
            )

            source_tensor = (
                state[key]
            )

            if not isinstance(
                source_tensor,
                torch.Tensor,
            ):
                raise TypeError(
                    f"replay {key} must "
                    "be a tensor"
                )

            source = (
                source_tensor
                .detach()
                .cpu()
                .numpy()
            )

            expected_shape = (
                destination[:size]
                .shape
            )

            if source.shape != (
                expected_shape
            ):
                raise ValueError(
                    f"replay {key} has "
                    "wrong shape"
                )

            destination[
                :size
            ] = source.astype(
                destination.dtype,
                copy=False,
            )

        self.position = position
        self.size = size

        self.rng.bit_generator.state = (
            json.loads(
                state[
                    "rng_state_json"
                ]
            )
        )

    def sample(
        self,
        batch_size,
        device,
    ):
        batch_size = int(
            batch_size
        )

        if batch_size < 1:
            raise ValueError(
                "batch_size must be >= 1"
            )

        if self.size < batch_size:
            raise ValueError(
                "not enough replay samples"
            )

        indices = self.rng.integers(
            0,
            self.size,
            size=batch_size,
        )

        def tensor(
            array,
            dtype=torch.float32,
        ):
            return torch.as_tensor(
                array[indices],
                dtype=dtype,
                device=device,
            )

        return {
            "observations": tensor(
                self.observations
            ),
            "next_observations": tensor(
                self.next_observations
            ),
            "states": tensor(
                self.states
            ),
            "next_states": tensor(
                self.next_states
            ),
            "continuous_actions": tensor(
                self.continuous_actions
            ),
            "destination_indices": tensor(
                self.destination_indices,
                dtype=torch.long,
            ),
            "destination_masks": tensor(
                self.destination_masks
            ),
            "next_destination_masks": tensor(
                self.next_destination_masks
            ),
            "rewards": tensor(
                self.rewards
            ),
            "terminated": tensor(
                self.terminated
            ),
            "truncated": tensor(
                self.truncated
            ),
        }


In [ ]:
class HybridMASAC:
    def __init__(
        self,
        env,
        device=None,
    ):
        if not isinstance(
            env,
            UAVSearchEnv,
        ):
            raise TypeError(
                "env must be UAVSearchEnv"
            )

        self.agent_ids = tuple(
            env.agent_ids
        )
        self.num_agents = len(
            self.agent_ids
        )

        if device is None:
            device = (
                "cuda"
                if torch.cuda.is_available()
                else "cpu"
            )

        self.device = torch.device(
            device
        )

        agent_space = (
            env.observation_space
            .spaces[
                self.agent_ids[0]
            ]
        )

        self.observation_dim = int(
            sum(
                np.prod(
                    agent_space.spaces[
                        key
                    ].shape,
                    dtype=np.int64,
                )
                for key
                in AGENT_OBSERVATION_VECTOR_KEYS
            )
        )
        self.state_dim = int(
            np.prod(
                env.state_space.shape,
                dtype=np.int64,
            )
        )
        self.continuous_dim = 4
        self.discrete_dim = int(
            env.action_space
            .spaces[
                self.agent_ids[0]
            ]
            .spaces[
                "destination"
            ]
            .n
        )

        self.joint_action_dim = (
            self.num_agents
            * (
                self.continuous_dim
                + self.discrete_dim
            )
        )

        hidden_dims = tuple(
            CONFIG[
                "masac_hidden_dims"
            ]
        )

        self.actor = (
            HybridMASACActor(
                self.observation_dim,
                self.continuous_dim,
                self.discrete_dim,
                hidden_dims,
            ).to(
                self.device
            )
        )

        self.critic_1 = (
            CentralizedQNetwork(
                self.state_dim,
                self.joint_action_dim,
                hidden_dims,
            ).to(
                self.device
            )
        )
        self.critic_2 = (
            CentralizedQNetwork(
                self.state_dim,
                self.joint_action_dim,
                hidden_dims,
            ).to(
                self.device
            )
        )

        self.target_critic_1 = (
            copy.deepcopy(
                self.critic_1
            ).to(
                self.device
            )
        )
        self.target_critic_2 = (
            copy.deepcopy(
                self.critic_2
            ).to(
                self.device
            )
        )

        for network in (
            self.target_critic_1,
            self.target_critic_2,
        ):
            network.eval()

            for parameter in (
                network.parameters()
            ):
                parameter.requires_grad_(
                    False
                )

        self.actor_optimizer = (
            torch.optim.Adam(
                self.actor.parameters(),
                lr=float(
                    CONFIG[
                        "masac_actor_lr"
                    ]
                ),
            )
        )

        self.critic_optimizer = (
            torch.optim.Adam(
                list(
                    self.critic_1.parameters()
                )
                + list(
                    self.critic_2.parameters()
                ),
                lr=float(
                    CONFIG[
                        "masac_critic_lr"
                    ]
                ),
            )
        )

        initial_alpha_continuous = float(
            CONFIG[
                "masac_initial_alpha_continuous"
            ]
        )
        initial_alpha_discrete = float(
            CONFIG[
                "masac_initial_alpha_discrete"
            ]
        )

        if (
            initial_alpha_continuous <= 0.0
            or initial_alpha_discrete <= 0.0
        ):
            raise ValueError(
                "initial alpha values must be > 0"
            )

        self.log_alpha_continuous = (
            torch.tensor(
                np.log(
                    initial_alpha_continuous
                ),
                dtype=torch.float32,
                device=self.device,
                requires_grad=True,
            )
        )
        self.log_alpha_discrete = (
            torch.tensor(
                np.log(
                    initial_alpha_discrete
                ),
                dtype=torch.float32,
                device=self.device,
                requires_grad=True,
            )
        )

        self.alpha_continuous_optimizer = (
            torch.optim.Adam(
                [
                    self.log_alpha_continuous
                ],
                lr=float(
                    CONFIG[
                        "masac_alpha_lr"
                    ]
                ),
            )
        )
        self.alpha_discrete_optimizer = (
            torch.optim.Adam(
                [
                    self.log_alpha_discrete
                ],
                lr=float(
                    CONFIG[
                        "masac_alpha_lr"
                    ]
                ),
            )
        )

        self.gamma = float(
            CONFIG["masac_gamma"]
        )
        self.tau = float(
            CONFIG["masac_tau"]
        )
        self.gradient_clip_norm = float(
            CONFIG[
                "masac_gradient_clip_norm"
            ]
        )

        self.continuous_target_entropy = (
            -float(
                CONFIG[
                    "masac_continuous_target_entropy_scale"
                ]
            )
            * self.num_agents
            * self.continuous_dim
        )

        self.discrete_target_entropy_ratio = float(
            CONFIG[
                "masac_discrete_target_entropy_ratio"
            ]
        )

        if not (
            0.0
            <= self.discrete_target_entropy_ratio
            <= 1.0
        ):
            raise ValueError(
                "discrete target entropy ratio "
                "must be within [0, 1]"
            )

        self.update_count = 0
        amp_dtype_name = str(
            CONFIG.get(
                "training_amp_dtype",
                "float16",
            )
        ).strip().lower()
        if amp_dtype_name == "float16":
            self.amp_dtype = torch.float16
        elif amp_dtype_name == "bfloat16":
            self.amp_dtype = torch.bfloat16
        else:
            raise ValueError(
                "training_amp_dtype must be float16 or bfloat16"
            )
        self.amp_enabled = (
            bool(
                CONFIG.get(
                    "training_amp_enabled",
                    False,
                )
            )
            and self.device.type == "cuda"
        )
        self.grad_scaler = torch.amp.GradScaler(
            "cuda",
            enabled=self.amp_enabled,
        )

    def _autocast(self):
        return torch.autocast(
            device_type=self.device.type,
            dtype=self.amp_dtype,
            enabled=self.amp_enabled,
        )

    @property
    def alpha_continuous(self):
        return torch.exp(
            self.log_alpha_continuous
        )

    @property
    def alpha_discrete(self):
        return torch.exp(
            self.log_alpha_discrete
        )

    def make_replay_buffer(
        self,
        seed=None,
    ):
        if seed is None:
            seed = int(
                CONFIG["seed"]
            )

        return HybridReplayBuffer(
            capacity=CONFIG[
                "masac_replay_capacity"
            ],
            num_agents=(
                self.num_agents
            ),
            observation_dim=(
                self.observation_dim
            ),
            state_dim=(
                self.state_dim
            ),
            continuous_dim=(
                self.continuous_dim
            ),
            discrete_dim=(
                self.discrete_dim
            ),
            seed=seed,
        )

    def _joint_action_tensor(
        self,
        continuous_actions,
        destination_one_hot,
    ):
        if continuous_actions.ndim != 3:
            raise ValueError(
                "continuous_actions must "
                "have rank 3"
            )

        if destination_one_hot.ndim != 3:
            raise ValueError(
                "destination_one_hot must "
                "have rank 3"
            )

        combined = torch.cat(
            [
                continuous_actions,
                destination_one_hot,
            ],
            dim=-1,
        )

        return combined.reshape(
            combined.shape[0],
            -1,
        )

    def _sample_joint_policy(
        self,
        observations,
        destination_masks,
        deterministic=False,
    ):
        if observations.ndim != 3:
            raise ValueError(
                "observations must have rank 3"
            )

        batch_size = (
            observations.shape[0]
        )

        flat_observations = (
            observations.reshape(
                batch_size
                * self.num_agents,
                self.observation_dim,
            )
        )
        flat_masks = (
            destination_masks.reshape(
                batch_size
                * self.num_agents,
                self.discrete_dim,
            )
        )

        sampled = self.actor.sample(
            flat_observations,
            flat_masks,
            deterministic=deterministic,
        )

        continuous = sampled[
            "continuous"
        ].reshape(
            batch_size,
            self.num_agents,
            self.continuous_dim,
        )
        destination_one_hot = (
            sampled[
                "destination_one_hot"
            ].reshape(
                batch_size,
                self.num_agents,
                self.discrete_dim,
            )
        )
        destination_indices = (
            sampled[
                "destination_index"
            ].reshape(
                batch_size,
                self.num_agents,
            )
        )

        continuous_log_probability = (
            sampled[
                "continuous_log_probability"
            ].reshape(
                batch_size,
                self.num_agents,
                1,
            ).sum(
                dim=1
            )
        )
        destination_log_probability = (
            sampled[
                "destination_log_probability"
            ].reshape(
                batch_size,
                self.num_agents,
                1,
            ).sum(
                dim=1
            )
        )
        destination_entropy = (
            sampled[
                "destination_entropy"
            ].reshape(
                batch_size,
                self.num_agents,
                1,
            ).sum(
                dim=1
            )
        )

        return {
            "continuous": continuous,
            "destination_one_hot": (
                destination_one_hot
            ),
            "destination_indices": (
                destination_indices
            ),
            "continuous_log_probability": (
                continuous_log_probability
            ),
            "destination_log_probability": (
                destination_log_probability
            ),
            "destination_entropy": (
                destination_entropy
            ),
        }

    def select_actions(
        self,
        observations,
        deterministic=False,
    ):
        (
            observation_vectors,
            destination_masks,
        ) = observations_to_masac_arrays(
            observations,
            self.agent_ids,
        )

        observation_tensor = (
            torch.as_tensor(
                observation_vectors,
                dtype=torch.float32,
                device=self.device,
            ).unsqueeze(
                0
            )
        )
        mask_tensor = torch.as_tensor(
            destination_masks,
            dtype=torch.float32,
            device=self.device,
        ).unsqueeze(
            0
        )

        with torch.no_grad():
            sampled = (
                self._sample_joint_policy(
                    observation_tensor,
                    mask_tensor,
                    deterministic=(
                        deterministic
                    ),
                )
            )

        continuous = (
            sampled["continuous"][
                0
            ]
            .cpu()
            .numpy()
            .astype(
                np.float32
            )
        )
        destination_indices = (
            sampled[
                "destination_indices"
            ][
                0
            ]
            .cpu()
            .numpy()
            .astype(
                np.int64
            )
        )

        return masac_arrays_to_env_actions(
            continuous,
            destination_indices,
            self.agent_ids,
        )

    def _soft_update_targets(
        self,
    ):
        with torch.no_grad():
            for target, source in (
                (
                    self.target_critic_1,
                    self.critic_1,
                ),
                (
                    self.target_critic_2,
                    self.critic_2,
                ),
            ):
                for (
                    target_parameter,
                    source_parameter,
                ) in zip(
                    target.parameters(),
                    source.parameters(),
                ):
                    target_parameter.mul_(
                        1.0 - self.tau
                    )
                    target_parameter.add_(
                        self.tau
                        * source_parameter
                    )

    def update(
        self,
        replay_buffer,
        batch_size=None,
        collect_metrics=True,
    ):
        if not isinstance(
            replay_buffer,
            HybridReplayBuffer,
        ):
            raise TypeError(
                "replay_buffer must be "
                "HybridReplayBuffer"
            )

        if batch_size is None:
            batch_size = int(
                CONFIG[
                    "masac_batch_size"
                ]
            )

        batch = replay_buffer.sample(
            batch_size,
            self.device,
        )

        destination_one_hot = (
            F.one_hot(
                batch[
                    "destination_indices"
                ],
                num_classes=(
                    self.discrete_dim
                ),
            ).to(
                torch.float32
            )
        )

        replay_joint_action = (
            self._joint_action_tensor(
                batch[
                    "continuous_actions"
                ],
                destination_one_hot,
            )
        )

        with torch.no_grad(), self._autocast():
            next_policy = (
                self._sample_joint_policy(
                    batch[
                        "next_observations"
                    ],
                    batch[
                        "next_destination_masks"
                    ],
                    deterministic=False,
                )
            )

            next_joint_action = (
                self._joint_action_tensor(
                    next_policy[
                        "continuous"
                    ],
                    next_policy[
                        "destination_one_hot"
                    ],
                )
            )

            target_q = torch.minimum(
                self.target_critic_1(
                    batch[
                        "next_states"
                    ],
                    next_joint_action,
                ),
                self.target_critic_2(
                    batch[
                        "next_states"
                    ],
                    next_joint_action,
                ),
            )

            entropy_adjusted_target = (
                target_q
                - self.alpha_continuous.detach()
                * next_policy[
                    "continuous_log_probability"
                ]
                - self.alpha_discrete.detach()
                * next_policy[
                    "destination_log_probability"
                ]
            )

            # Gymnasium truncation is a time-limit event,
            # so only true termination blocks bootstrapping.
            critic_target = (
                batch["rewards"]
                + self.gamma
                * (
                    1.0
                    - batch[
                        "terminated"
                    ]
                )
                * entropy_adjusted_target
            )

        with self._autocast():
            critic_1_value = (
                self.critic_1(
                    batch["states"],
                    replay_joint_action,
                )
            )
            critic_2_value = (
                self.critic_2(
                    batch["states"],
                    replay_joint_action,
                )
            )

            critic_loss = (
                F.mse_loss(
                    critic_1_value,
                    critic_target,
                )
                + F.mse_loss(
                    critic_2_value,
                    critic_target,
                )
            )

        self.critic_optimizer.zero_grad(
            set_to_none=True
        )
        if self.amp_enabled:
            self.grad_scaler.scale(
                critic_loss
            ).backward()
            self.grad_scaler.unscale_(
                self.critic_optimizer
            )
        else:
            critic_loss.backward()

        critic_grad_norm = (
            torch.nn.utils.clip_grad_norm_(
                list(
                    self.critic_1.parameters()
                )
                + list(
                    self.critic_2.parameters()
                ),
                self.gradient_clip_norm,
            )
        )

        if self.amp_enabled:
            self.grad_scaler.step(
                self.critic_optimizer
            )
        else:
            self.critic_optimizer.step()

        for critic in (
            self.critic_1,
            self.critic_2,
        ):
            for parameter in (
                critic.parameters()
            ):
                parameter.requires_grad_(
                    False
                )

        with self._autocast():
            policy = self._sample_joint_policy(
                batch["observations"],
                batch[
                    "destination_masks"
                ],
                deterministic=False,
            )

            policy_joint_action = (
                self._joint_action_tensor(
                    policy[
                        "continuous"
                    ],
                    policy[
                        "destination_one_hot"
                    ],
                )
            )

            policy_q = torch.minimum(
                self.critic_1(
                    batch["states"],
                    policy_joint_action,
                ),
                self.critic_2(
                    batch["states"],
                    policy_joint_action,
                ),
            )

            # Exact categorical entropy retains the probability-weight gradient.
            actor_loss = (
                self.alpha_continuous.detach()
                * policy["continuous_log_probability"]
                - self.alpha_discrete.detach()
                * policy["destination_entropy"]
                - policy_q
            ).mean()

        self.actor_optimizer.zero_grad(
            set_to_none=True
        )
        if self.amp_enabled:
            self.grad_scaler.scale(
                actor_loss
            ).backward()
            self.grad_scaler.unscale_(
                self.actor_optimizer
            )
        else:
            actor_loss.backward()

        actor_grad_norm = (
            torch.nn.utils.clip_grad_norm_(
                self.actor.parameters(),
                self.gradient_clip_norm,
            )
        )

        if self.amp_enabled:
            self.grad_scaler.step(
                self.actor_optimizer
            )
            self.grad_scaler.update()
        else:
            self.actor_optimizer.step()

        for critic in (
            self.critic_1,
            self.critic_2,
        ):
            for parameter in (
                critic.parameters()
            ):
                parameter.requires_grad_(
                    True
                )

        continuous_entropy = -policy[
            "continuous_log_probability"
        ].detach()

        continuous_target = torch.full_like(
            continuous_entropy,
            float(
                self.continuous_target_entropy
            ),
        )

        valid_counts = (
            batch[
                "destination_masks"
            ].sum(
                dim=-1
            ).clamp(
                min=1.0
            )
        )

        discrete_target = (
            self.discrete_target_entropy_ratio
            * torch.log(
                valid_counts
            ).sum(
                dim=1,
                keepdim=True,
            )
        )

        discrete_entropy = policy[
            "destination_entropy"
        ].detach()

        alpha_continuous_loss = (
            self.log_alpha_continuous
            * (
                continuous_entropy
                - continuous_target
            )
        ).mean()

        alpha_discrete_loss = (
            self.log_alpha_discrete
            * (
                discrete_entropy
                - discrete_target
            )
        ).mean()

        self.alpha_continuous_optimizer.zero_grad(
            set_to_none=True
        )
        alpha_continuous_loss.backward()
        self.alpha_continuous_optimizer.step()

        self.alpha_discrete_optimizer.zero_grad(
            set_to_none=True
        )
        alpha_discrete_loss.backward()
        self.alpha_discrete_optimizer.step()

        self._soft_update_targets()

        self.update_count += 1

        if not collect_metrics:
            return None

        metrics = {
            "critic_loss": float(
                critic_loss.detach().cpu()
            ),
            "actor_loss": float(
                actor_loss.detach().cpu()
            ),
            "alpha_continuous_loss": float(
                alpha_continuous_loss
                .detach()
                .cpu()
            ),
            "alpha_discrete_loss": float(
                alpha_discrete_loss
                .detach()
                .cpu()
            ),
            "alpha_continuous": float(
                self.alpha_continuous
                .detach()
                .cpu()
            ),
            "alpha_discrete": float(
                self.alpha_discrete
                .detach()
                .cpu()
            ),
            "continuous_entropy": float(
                continuous_entropy
                .mean()
                .cpu()
            ),
            "discrete_entropy": float(
                discrete_entropy
                .mean()
                .cpu()
            ),
            "target_q_mean": float(
                critic_target
                .mean()
                .detach()
                .cpu()
            ),
            "critic_grad_norm": float(
                torch.as_tensor(
                    critic_grad_norm
                )
                .detach()
                .cpu()
            ),
            "actor_grad_norm": float(
                torch.as_tensor(
                    actor_grad_norm
                )
                .detach()
                .cpu()
            ),
            "update_count": int(
                self.update_count
            ),
        }

        if not all(
            np.isfinite(value)
            for key, value
            in metrics.items()
            if key
            != "update_count"
        ):
            raise FloatingPointError(
                "non-finite MASAC metric"
            )

        return metrics

    @staticmethod
    def _move_optimizer_state_to_device(
        optimizer,
        device,
    ):
        for state in optimizer.state.values():
            for key, value in list(
                state.items()
            ):
                if isinstance(
                    value,
                    torch.Tensor,
                ):
                    state[key] = value.to(
                        device
                    )

    def state_dict(self):
        return {
            "actor": (
                self.actor.state_dict()
            ),
            "critic_1": (
                self.critic_1.state_dict()
            ),
            "critic_2": (
                self.critic_2.state_dict()
            ),
            "target_critic_1": (
                self.target_critic_1
                .state_dict()
            ),
            "target_critic_2": (
                self.target_critic_2
                .state_dict()
            ),
            "actor_optimizer": (
                self.actor_optimizer
                .state_dict()
            ),
            "critic_optimizer": (
                self.critic_optimizer
                .state_dict()
            ),
            "log_alpha_continuous": (
                self.log_alpha_continuous
                .detach()
                .cpu()
            ),
            "log_alpha_discrete": (
                self.log_alpha_discrete
                .detach()
                .cpu()
            ),
            "alpha_continuous_optimizer": (
                self.alpha_continuous_optimizer
                .state_dict()
            ),
            "alpha_discrete_optimizer": (
                self.alpha_discrete_optimizer
                .state_dict()
            ),
            "update_count": int(
                self.update_count
            ),
        }

    def load_state_dict(
        self,
        state,
    ):
        self.actor.load_state_dict(
            state["actor"]
        )
        self.critic_1.load_state_dict(
            state["critic_1"]
        )
        self.critic_2.load_state_dict(
            state["critic_2"]
        )
        self.target_critic_1.load_state_dict(
            state[
                "target_critic_1"
            ]
        )
        self.target_critic_2.load_state_dict(
            state[
                "target_critic_2"
            ]
        )

        self.actor_optimizer.load_state_dict(
            state[
                "actor_optimizer"
            ]
        )
        self.critic_optimizer.load_state_dict(
            state[
                "critic_optimizer"
            ]
        )

        with torch.no_grad():
            self.log_alpha_continuous.copy_(
                state[
                    "log_alpha_continuous"
                ].to(
                    self.device
                )
            )
            self.log_alpha_discrete.copy_(
                state[
                    "log_alpha_discrete"
                ].to(
                    self.device
                )
            )

        self.alpha_continuous_optimizer.load_state_dict(
            state[
                "alpha_continuous_optimizer"
            ]
        )
        self.alpha_discrete_optimizer.load_state_dict(
            state[
                "alpha_discrete_optimizer"
            ]
        )

        for optimizer in (
            self.actor_optimizer,
            self.critic_optimizer,
            self.alpha_continuous_optimizer,
            self.alpha_discrete_optimizer,
        ):
            self._move_optimizer_state_to_device(
                optimizer,
                self.device,
            )

        self.update_count = int(
            state.get(
                "update_count",
                0,
            )
        )


In [ ]:
def add_environment_transition_to_replay(
    replay_buffer,
    observations,
    state,
    actions,
    reward,
    next_observations,
    next_state,
    terminated,
    truncated,
    agent_ids,
):
    (
        observation_vectors,
        destination_masks,
    ) = observations_to_masac_arrays(
        observations,
        agent_ids,
    )

    (
        next_observation_vectors,
        next_destination_masks,
    ) = observations_to_masac_arrays(
        next_observations,
        agent_ids,
    )

    (
        continuous_actions,
        destination_indices,
    ) = env_actions_to_masac_arrays(
        actions,
        agent_ids,
    )

    replay_buffer.add(
        observations=(
            observation_vectors
        ),
        state=state,
        continuous_actions=(
            continuous_actions
        ),
        destination_indices=(
            destination_indices
        ),
        destination_masks=(
            destination_masks
        ),
        reward=reward,
        next_observations=(
            next_observation_vectors
        ),
        next_state=next_state,
        next_destination_masks=(
            next_destination_masks
        ),
        terminated=terminated,
        truncated=truncated,
    )


In [ ]:
def train_masac(
    env,
    total_steps,
    seed=None,
    trainer=None,
    replay_buffer=None,
):
    if not isinstance(
        env,
        UAVSearchEnv,
    ):
        raise TypeError(
            "env must be UAVSearchEnv"
        )

    total_steps = int(
        total_steps
    )

    if total_steps < 1:
        raise ValueError(
            "total_steps must be >= 1"
        )

    if seed is None:
        seed = int(
            CONFIG["seed"]
        )

    seed = int(
        seed
    )

    set_seed(
        seed
    )

    if trainer is None:
        trainer = HybridMASAC(
            env
        )

    if replay_buffer is None:
        replay_buffer = (
            trainer.make_replay_buffer(
                seed=seed,
            )
        )

    rng = np.random.default_rng(
        seed
    )

    observations, info = env.reset(
        seed=seed
    )
    state = info[
        "state"
    ]

    episode_return = 0.0
    episode_length = 0
    episode_index = 0
    completed_episodes = []
    latest_update_metrics = None

    learning_starts = int(
        CONFIG[
            "masac_learning_starts"
        ]
    )
    updates_per_step = int(
        CONFIG[
            "masac_updates_per_step"
        ]
    )
    batch_size = int(
        CONFIG[
            "masac_batch_size"
        ]
    )

    for global_step in range(
        1,
        total_steps + 1,
    ):
        if global_step <= learning_starts:
            actions = (
                sample_valid_random_actions(
                    observations,
                    env.agent_ids,
                    rng,
                )
            )
        else:
            actions = (
                trainer.select_actions(
                    observations,
                    deterministic=False,
                )
            )

        (
            next_observations,
            reward,
            terminated,
            truncated,
            next_info,
        ) = env.step(
            actions
        )

        next_state = next_info[
            "state"
        ]

        add_environment_transition_to_replay(
            replay_buffer,
            observations,
            state,
            actions,
            reward,
            next_observations,
            next_state,
            terminated,
            truncated,
            env.agent_ids,
        )

        episode_return += float(
            reward
        )
        episode_length += 1

        if (
            global_step > learning_starts
            and len(
                replay_buffer
            )
            >= batch_size
        ):
            for _ in range(
                updates_per_step
            ):
                latest_update_metrics = (
                    trainer.update(
                        replay_buffer,
                        batch_size=(
                            batch_size
                        ),
                    )
                )

        observations = (
            next_observations
        )
        state = next_state

        if terminated or truncated:
            completed_episodes.append(
                {
                    "episode": (
                        episode_index
                    ),
                    "return": float(
                        episode_return
                    ),
                    "length": int(
                        episode_length
                    ),
                    "terminated": bool(
                        terminated
                    ),
                    "truncated": bool(
                        truncated
                    ),
                    "success": bool(
                        next_info[
                            "success"
                        ]
                    ),
                }
            )

            episode_index += 1
            episode_return = 0.0
            episode_length = 0

            if global_step < total_steps:
                observations, info = (
                    env.reset(
                        seed=(
                            seed
                            + episode_index
                        )
                    )
                )
                state = info[
                    "state"
                ]

    return {
        "trainer": trainer,
        "replay_buffer": replay_buffer,
        "completed_episodes": (
            completed_episodes
        ),
        "latest_update_metrics": (
            latest_update_metrics
        ),
        "total_steps": total_steps,
    }


In [ ]:
def evaluate_masac(
    env,
    trainer,
    episodes=3,
    seed=None,
):
    if seed is None:
        seed = int(
            CONFIG["seed"]
        ) + 10_000

    episodes = int(
        episodes
    )

    if episodes < 1:
        raise ValueError(
            "episodes must be >= 1"
        )

    results = []

    for episode_index in range(
        episodes
    ):
        observations, _ = env.reset(
            seed=(
                int(seed)
                + episode_index
            )
        )

        episode_return = 0.0
        diagnostics = (
            new_episode_diagnostics()
        )

        while True:
            actions = trainer.select_actions(
                observations,
                deterministic=True,
            )

            (
                observations,
                reward,
                terminated,
                truncated,
                info,
            ) = env.step(
                actions
            )

            episode_return += float(
                reward
            )

            update_episode_diagnostics(
                diagnostics,
                info,
                env=env,
            )

            if (
                terminated
                or truncated
            ):
                episode_metrics = (
                    finalize_episode_diagnostics(
                        diagnostics,
                        env,
                        episode_return=(
                            episode_return
                        ),
                        episode_length=(
                            info["step"]
                        ),
                        success=(
                            info["success"]
                        ),
                    )
                )

                results.append(
                    {
                        "episode": (
                            episode_index
                        ),
                        **episode_metrics,
                        "terminated": bool(
                            terminated
                        ),
                        "truncated": bool(
                            truncated
                        ),
                    }
                )
                break

    return results


In [ ]:
def _checkpoint_config_snapshot():
    snapshot = {}

    for key, value in CONFIG.items():
        if isinstance(
            value,
            tuple,
        ):
            snapshot[key] = list(
                value
            )
        elif isinstance(
            value,
            np.ndarray,
        ):
            snapshot[key] = (
                value.tolist()
            )
        elif isinstance(
            value,
            np.generic,
        ):
            snapshot[key] = (
                value.item()
            )
        else:
            snapshot[key] = value

    return snapshot


In [ ]:
def _validate_resume_config(saved_config, algorithm):
    """Reject changes to the mission or learner during an exact resume."""
    current = _checkpoint_config_snapshot()
    other_algorithm = "matd3" if algorithm == "masac" else "masac"
    mismatches = []
    for key, value in current.items():
        if (
            key.startswith(("training_", "kaggle_", other_algorithm + "_"))
            or key in {"matd3_training_output_dir", "matd3_wandb_project"}
        ):
            continue
        if key not in saved_config or saved_config[key] != value:
            mismatches.append(key)
    if mismatches:
        raise ValueError(
            "exact resume config mismatch: " + ", ".join(sorted(mismatches))
        )


In [ ]:
def save_masac_checkpoint(
    path,
    trainer,
    replay_buffer=None,
    training_state=None,
    include_replay=None,
):
    if not isinstance(
        trainer,
        HybridMASAC,
    ):
        raise TypeError(
            "trainer must be HybridMASAC"
        )

    if include_replay is None:
        include_replay = bool(
            CONFIG[
                "training_checkpoint_include_replay"
            ]
        )

    include_replay = bool(
        include_replay
    )

    if (
        include_replay
        and replay_buffer is None
    ):
        raise ValueError(
            "replay_buffer is required "
            "when include_replay=True"
        )

    if (
        replay_buffer is not None
        and not isinstance(
            replay_buffer,
            HybridReplayBuffer,
        )
    ):
        raise TypeError(
            "replay_buffer must be "
            "HybridReplayBuffer"
        )

    path = Path(path)
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    if training_state is None:
        training_state = {}

    if not isinstance(
        training_state,
        dict,
    ):
        raise TypeError(
            "training_state must be a dict"
        )

    payload = {
        "format_version": 1,
        "trainer_state": (
            trainer.state_dict()
        ),
        "training_state": (
            copy.deepcopy(
                training_state
            )
        ),
        "config_snapshot": (
            _checkpoint_config_snapshot()
        ),
        "replay_state": (
            replay_buffer.state_dict()
            if include_replay
            else None
        ),
        "torch_rng_state": (
            torch.get_rng_state()
        ),
        "cuda_rng_states": (
            torch.cuda.get_rng_state_all()
            if torch.cuda.is_available()
            else []
        ),
    }

    temporary_path = path.with_suffix(
        path.suffix + ".tmp"
    )

    torch.save(
        payload,
        temporary_path,
    )

    temporary_path.replace(
        path
    )

    return path


In [ ]:
def load_masac_checkpoint(
    path,
    env,
    device=None,
    load_replay=True,
    restore_torch_rng=True,
):
    if not isinstance(
        env,
        UAVSearchEnv,
    ):
        raise TypeError(
            "env must be UAVSearchEnv"
        )

    path = Path(path)

    if not path.is_file():
        raise FileNotFoundError(
            str(path)
        )

    # weights_only=True is sufficient because this project checkpoint
    # contains tensors and basic Python containers, not serialized model
    # objects.
    payload = torch.load(
        path,
        map_location="cpu",
        weights_only=True,
    )

    if int(
        payload.get(
            "format_version",
            -1,
        )
    ) != 1:
        raise ValueError(
            "unsupported checkpoint "
            "format_version"
        )

    trainer = HybridMASAC(
        env,
        device=device,
    )
    trainer.load_state_dict(
        payload[
            "trainer_state"
        ]
    )

    replay_buffer = None

    replay_state = payload.get(
        "replay_state"
    )

    if (
        load_replay
        and replay_state is not None
    ):
        training_state = payload.get(
            "training_state",
            {},
        )
        replay_seed = int(
            training_state.get(
                "seed",
                CONFIG["seed"],
            )
        )

        replay_buffer = (
            trainer.make_replay_buffer(
                seed=replay_seed,
            )
        )
        replay_buffer.load_state_dict(
            replay_state
        )

    if restore_torch_rng:
        torch.set_rng_state(
            payload[
                "torch_rng_state"
            ]
        )

        cuda_states = payload.get(
            "cuda_rng_states",
            [],
        )

        if (
            torch.cuda.is_available()
            and cuda_states
        ):
            torch.cuda.set_rng_state_all(
                cuda_states
            )

    return {
        "trainer": trainer,
        "replay_buffer": replay_buffer,
        "training_state": (
            copy.deepcopy(
                payload.get(
                    "training_state",
                    {},
                )
            )
        ),
        "config_snapshot": (
            copy.deepcopy(
                payload.get(
                    "config_snapshot",
                    {},
                )
            )
        ),
        "path": path,
    }


In [ ]:
def git_experiment_metadata(
    repo_path=".",
):
    repo_path = Path(
        repo_path
    ).resolve()

    def git_output(
        *args,
    ):
        result = subprocess.run(
            [
                "git",
                "-C",
                str(repo_path),
                *args,
            ],
            capture_output=True,
            text=True,
            check=False,
        )

        if result.returncode != 0:
            return None

        value = result.stdout.strip()

        return (
            value
            if value
            else None
        )

    return {
        "git_commit": git_output(
            "rev-parse",
            "HEAD",
        ),
        "git_branch": git_output(
            "branch",
            "--show-current",
        ),
        "git_remote": git_output(
            "remote",
            "get-url",
            "origin",
        ),
    }


In [ ]:
def modeled_collision_violations(env):
    """Count post-step modeled collision/separation violations.

    Peer violations use the project's safety_distance because UAV physical
    radii are not modeled separately. Obstacle violations use penetration of
    the actual obstacle cylinder (without the extra planning clearance).
    Counts are instantaneous post-step violations, not avoidance interventions.
    """
    peer_count = 0
    obstacle_count = 0

    positions = np.stack(
        [
            np.asarray(uav.position, dtype=np.float64)
            for uav in env.uavs
        ],
        axis=0,
    )

    safety_distance = float(CONFIG["safety_distance"])
    tolerance = 1e-9

    for i in range(len(env.uavs)):
        for j in range(i + 1, len(env.uavs)):
            distance = float(
                np.linalg.norm(
                    positions[i] - positions[j]
                )
            )
            if distance < safety_distance - tolerance:
                peer_count += 1

    for uav_index, position in enumerate(positions):
        xy = position[:2]
        altitude = float(position[2])
        for obstacle in env.obstacles:
            horizontal_distance = float(
                np.linalg.norm(
                    xy
                    - np.asarray(
                        obstacle.position,
                        dtype=np.float64,
                    )
                )
            )
            if (
                horizontal_distance
                < float(obstacle.radius) - tolerance
                and altitude
                <= float(obstacle.height) + tolerance
            ):
                obstacle_count += 1

    return {
        "peer_collision_violations": int(peer_count),
        "obstacle_collision_violations": int(obstacle_count),
        "collision_violations": int(
            peer_count + obstacle_count
        ),
    }


In [ ]:
def new_episode_diagnostics():
    return {
        "information_gain_bits": 0.0,
        "sensing_records": 0,
        "positive_observations": 0,
        "target_positive_observations": 0,
        "targets_ever_in_fov_ids": set(),
        "false_confirmations": 0,
        "expired_reports": 0,
        "dropped_reports": 0,
        "blocked_motion": 0,
        "obstacle_blocks": 0,
        "peer_safety_blocks": 0,
        "boundary_clips": 0,
        "horizontal_boundary_clips": 0,
        "altitude_clips": 0,
        "communication_energy_j": 0.0,
        "total_energy_j": 0.0,
        "peer_collision_violations": 0,
        "obstacle_collision_violations": 0,
        "collision_steps": 0,
        "network_snapshot": {},
    }


In [ ]:
def update_episode_diagnostics(
    diagnostics,
    info,
    env=None,
):
    diagnostics[
        "information_gain_bits"
    ] += float(
        info[
            "information_gain_bits"
        ]
    )
    diagnostics[
        "sensing_records"
    ] += int(
        info.get(
            "sensing_record_count",
            0,
        )
    )
    diagnostics[
        "positive_observations"
    ] += int(
        info.get(
            "positive_observation_count",
            0,
        )
    )
    diagnostics[
        "target_positive_observations"
    ] += int(
        info.get(
            "target_positive_observation_count",
            0,
        )
    )
    diagnostics[
        "targets_ever_in_fov_ids"
    ].update(
        int(target_id)
        for target_id in info.get(
            "targets_in_fov_ids",
            [],
        )
    )
    diagnostics[
        "false_confirmations"
    ] += int(
        info[
            "false_confirmation_count"
        ]
    )
    diagnostics[
        "expired_reports"
    ] += len(
        info[
            "expired_target_ids"
        ]
    )
    diagnostics[
        "dropped_reports"
    ] += int(
        info[
            "dropped_report_count"
        ]
    )

    motion = info[
        "motion"
    ]

    diagnostics[
        "blocked_motion"
    ] += int(
        np.count_nonzero(
            motion[
                "blocked"
            ]
        )
    )
    diagnostics[
        "obstacle_blocks"
    ] += int(
        np.count_nonzero(
            motion[
                "blocked_by_obstacle"
            ]
        )
    )
    diagnostics[
        "peer_safety_blocks"
    ] += int(
        np.count_nonzero(
            motion[
                "blocked_by_peer"
            ]
        )
    )
    diagnostics[
        "boundary_clips"
    ] += int(
        np.count_nonzero(
            motion[
                "boundary_clipped"
            ]
        )
    )
    diagnostics[
        "horizontal_boundary_clips"
    ] += int(
        np.count_nonzero(
            motion[
                "horizontal_boundary_clipped"
            ]
        )
    )
    diagnostics[
        "altitude_clips"
    ] += int(
        np.count_nonzero(
            motion[
                "altitude_clipped"
            ]
        )
    )

    energy = info[
        "energy"
    ]

    diagnostics[
        "communication_energy_j"
    ] += float(
        np.sum(
            energy[
                "communication_j"
            ]
        )
    )
    diagnostics[
        "total_energy_j"
    ] += float(
        np.sum(
            energy[
                "total_j"
            ]
        )
    )

    if env is not None:
        collision = modeled_collision_violations(
            env
        )
        diagnostics[
            "peer_collision_violations"
        ] += int(
            collision[
                "peer_collision_violations"
            ]
        )
        diagnostics[
            "obstacle_collision_violations"
        ] += int(
            collision[
                "obstacle_collision_violations"
            ]
        )
        diagnostics[
            "collision_steps"
        ] += int(
            collision[
                "collision_violations"
            ]
            > 0
        )

    diagnostics[
        "network_snapshot"
    ] = copy.deepcopy(
        info[
            "network_metrics"
        ]
    )


In [ ]:
def extract_network_kpis(
    snapshot,
):
    metrics = {}

    nested = snapshot.get(
        "network_metrics",
        {},
    )

    key_map = {
        "pdr_percent": (
            "network_pdr_percent"
        ),
        "e2e_delay_ms": (
            "network_e2e_delay_ms"
        ),
        "throughput_kbps": (
            "network_throughput_kbps"
        ),
        "average_hops": (
            "network_average_hops"
        ),
        "phy_success_percent": (
            "network_phy_success_percent"
        ),
        "collisions": (
            "network_collisions"
        ),
    }

    for (
        source_key,
        metric_key,
    ) in key_map.items():
        value = nested.get(
            source_key
        )

        if isinstance(
            value,
            (bool, int, float, np.generic),
        ):
            metrics[
                metric_key
            ] = float(value)

    injected = snapshot.get(
        "payload_bytes_injected"
    )
    delivered = snapshot.get(
        "payload_bytes_delivered"
    )

    if (
        isinstance(
            injected,
            (int, float, np.generic),
        )
        and isinstance(
            delivered,
            (int, float, np.generic),
        )
    ):
        injected_value = float(
            injected
        )
        delivered_value = float(
            delivered
        )
        metrics[
            "network_payload_delivery_ratio"
        ] = (
            delivered_value
            / injected_value
            if injected_value > 0.0
            else 0.0
        )

    return metrics


In [ ]:
def finalize_episode_diagnostics(
    diagnostics,
    env,
    episode_return,
    episode_length,
    success,
):
    delivered_targets = len(
        env.gcs_received_target_ids
    )
    confirmed_targets = sum(
        target.confirmed
        for target
        in env.targets
    )

    target_count = max(
        1,
        int(
            env.num_targets
        ),
    )
    obstacle_count = int(
        len(env.obstacles)
    )
    targets_ever_in_fov_count = int(
        len(
            diagnostics[
                "targets_ever_in_fov_ids"
            ]
        )
    )
    possible_agent_steps = max(
        1,
        int(
            episode_length
        )
        * int(
            env.num_uavs
        ),
    )

    battery_values = np.asarray(
        [
            max(
                0.0,
                float(uav.battery_j),
            )
            for uav in env.uavs
        ],
        dtype=np.float64,
    )
    battery_capacity_j = max(
        1e-12,
        float(CONFIG["battery_j"]),
    )
    battery_mean_j = float(
        np.mean(battery_values)
    )
    metrics = {
        "return": float(
            episode_return
        ),
        "length": int(
            episode_length
        ),
        "episode_end_step": int(
            episode_length
        ),
        "success": float(
            bool(success)
        ),
        "targets_spawned": int(
            env.num_targets
        ),
        "obstacles_spawned": (
            obstacle_count
        ),
        "confirmed_targets": int(
            confirmed_targets
        ),
        "delivered_targets": int(
            delivered_targets
        ),
        "targets_ever_in_fov_count": (
            targets_ever_in_fov_count
        ),
        "target_encounter_rate": float(
            targets_ever_in_fov_count
            / target_count
        ),
        "sensing_records": int(
            diagnostics[
                "sensing_records"
            ]
        ),
        "positive_observations": int(
            diagnostics[
                "positive_observations"
            ]
        ),
        "target_positive_observations": int(
            diagnostics[
                "target_positive_observations"
            ]
        ),
        "delivery_rate": float(
            delivered_targets
            / target_count
        ),
        "confirmation_rate": float(
            confirmed_targets
            / target_count
        ),
        "false_confirmations": int(
            diagnostics[
                "false_confirmations"
            ]
        ),
        "expired_reports": int(
            diagnostics[
                "expired_reports"
            ]
        ),
        "dropped_reports": int(
            diagnostics[
                "dropped_reports"
            ]
        ),
        "blocked_motion_rate": float(
            diagnostics[
                "blocked_motion"
            ]
            / possible_agent_steps
        ),
        "obstacle_block_rate": float(
            diagnostics[
                "obstacle_blocks"
            ]
            / possible_agent_steps
        ),
        "peer_safety_block_rate": float(
            diagnostics[
                "peer_safety_blocks"
            ]
            / possible_agent_steps
        ),
        "boundary_clip_rate": float(
            diagnostics[
                "boundary_clips"
            ]
            / possible_agent_steps
        ),
        "horizontal_boundary_clip_rate": float(
            diagnostics[
                "horizontal_boundary_clips"
            ]
            / possible_agent_steps
        ),
        "altitude_clip_rate": float(
            diagnostics[
                "altitude_clips"
            ]
            / possible_agent_steps
        ),
        "total_energy_j": float(
            diagnostics[
                "total_energy_j"
            ]
        ),
        "communication_energy_j": float(
            diagnostics[
                "communication_energy_j"
            ]
        ),
        "battery_remaining_mean_j": (
            battery_mean_j
        ),
        "battery_remaining_mean_percent": float(
            100.0
            * battery_mean_j
            / battery_capacity_j
        ),
        "depleted_uav_count": int(
            np.count_nonzero(
                battery_values <= 1e-12
            )
        ),
        "collision_violation_count": int(
            diagnostics[
                "peer_collision_violations"
            ]
            + diagnostics[
                "obstacle_collision_violations"
            ]
        ),
        "collision_step_count": int(
            diagnostics[
                "collision_steps"
            ]
        ),
        "information_gain_bits": float(
            diagnostics[
                "information_gain_bits"
            ]
        ),
    }

    metrics.update(
        extract_network_kpis(
            diagnostics[
                "network_snapshot"
            ]
        )
    )

    return metrics


In [ ]:
def filter_training_metrics_for_wandb(
    algorithm,
    metrics,
):
    normalized = str(
        algorithm
    ).strip().lower()

    if normalized == "masac":
        keep = {
            "critic_loss",
            "actor_loss",
            "alpha_continuous",
            "alpha_discrete",
            "continuous_entropy",
            "discrete_entropy",
        }
    elif normalized == "matd3":
        keep = {
            "critic_loss",
            "actor_loss",
        }
    else:
        raise ValueError(
            "algorithm must be "
            "'masac' or 'matd3'"
        )

    return {
        key: value
        for key, value
        in metrics.items()
        if key in keep
        and value is not None
    }


In [ ]:
class MASACExperimentLogger:
    def __init__(
        self,
        run_dir,
        run_name,
        enable_csv=None,
        enable_tensorboard=None,
        enable_wandb=None,
        wandb_entity=None,
        wandb_project=None,
        wandb_mode=None,
        wandb_run_id=None,
        wandb_resume=None,
        algorithm=None,
        seed=None,
        backend_name=None,
        config_snapshot=None,
    ):
        self.run_dir = Path(
            run_dir
        )
        self.run_dir.mkdir(
            parents=True,
            exist_ok=True,
        )

        self.run_name = str(
            run_name
        )
        self.algorithm = (
            None
            if algorithm is None
            else str(
                algorithm
            ).lower()
        )
        self.seed = (
            None
            if seed is None
            else int(seed)
        )
        self.backend_name = (
            None
            if backend_name is None
            else str(
                backend_name
            )
        )

        if enable_csv is None:
            enable_csv = CONFIG[
                "training_enable_csv"
            ]

        if enable_tensorboard is None:
            enable_tensorboard = CONFIG[
                "training_enable_tensorboard"
            ]

        if enable_wandb is None:
            enable_wandb = CONFIG[
                "training_enable_wandb"
            ]

        if wandb_entity is None:
            wandb_entity = CONFIG.get(
                "training_wandb_entity"
            )

        if wandb_project is None:
            wandb_project = CONFIG[
                "training_wandb_project"
            ]

        if wandb_mode is None:
            wandb_mode = CONFIG[
                "training_wandb_mode"
            ]

        self.enable_csv = bool(
            enable_csv
        )
        self.enable_tensorboard = bool(
            enable_tensorboard
        )
        self.enable_wandb = bool(
            enable_wandb
        )

        self.csv_path = (
            self.run_dir
            / "metrics.csv"
        )
        self._csv_file = None
        self._csv_writer = None
        self._tensorboard_writer = None
        self._wandb_run = None

        self.tensorboard_error = None
        self.wandb_error = None
        self.wandb_run_id = None
        self.wandb_run_url = None

        if config_snapshot is None:
            config_snapshot = (
                _checkpoint_config_snapshot()
            )

        self.config_snapshot = (
            copy.deepcopy(
                config_snapshot
            )
        )

        git_metadata = (
            git_experiment_metadata()
        )

        self.config_snapshot.update(
            {
                "monitor/algorithm": (
                    self.algorithm
                ),
                "monitor/seed": self.seed,
                "monitor/backend": (
                    self.backend_name
                ),
                **{
                    (
                        f"monitor/{key}"
                    ): value
                    for key, value
                    in git_metadata.items()
                },
            }
        )

        if self.enable_csv:
            file_exists = (
                self.csv_path.is_file()
                and self.csv_path.stat().st_size
                > 0
            )

            self._csv_file = open(  # noqa: SIM115 - kept open for the run
                self.csv_path,
                "a",
                newline="",
                encoding="utf-8",
            )
            self._csv_writer = (
                csv.writer(
                    self._csv_file
                )
            )

            if not file_exists:
                self._csv_writer.writerow(
                    [
                        "step",
                        "episode",
                        "metric",
                        "value",
                    ]
                )
                self._csv_file.flush()

        if self.enable_tensorboard:
            try:
                from torch.utils.tensorboard import (
                    SummaryWriter,
                )

                self._tensorboard_writer = (
                    SummaryWriter(
                        log_dir=str(
                            self.run_dir
                            / "tensorboard"
                        )
                    )
                )
            except Exception as exc:  # noqa: BLE001 - optional logger fallback
                self.tensorboard_error = (
                    f"{type(exc).__name__}: "
                    f"{exc}"
                )

        if self.enable_wandb:
            try:
                wandb = (
                    importlib.import_module(
                        "wandb"
                    )
                )

                import secrets
                resolved_wandb_run_id = (
                    str(wandb_run_id)
                    if wandb_run_id
                    else secrets.token_hex(4)
                )
                wandb_algorithm = (
                    str(self.algorithm).upper()
                    if self.algorithm
                    else "MARL"
                )
                wandb_seed = (
                    int(self.seed)
                    if self.seed is not None
                    else int(CONFIG["seed"])
                )
                wandb_name_format = str(
                    CONFIG.get(
                        "training_wandb_name_format",
                        "{algorithm}-seed{seed}-{id}",
                    )
                )
                initial_name = wandb_name_format.format(
                    algorithm=wandb_algorithm,
                    seed=wandb_seed,
                    id=resolved_wandb_run_id,
                )

                self._wandb_run = (
                    wandb.init(
                        entity=(
                            wandb_entity
                        ),
                        project=str(
                            wandb_project
                        ),
                        name=initial_name,
                        mode=str(
                            wandb_mode
                        ),
                        id=resolved_wandb_run_id,
                        resume=wandb_resume,
                        dir=str(
                            self.run_dir
                        ),
                        reinit="finish_previous",
                        settings=wandb.Settings(
                            x_disable_stats=bool(
                                CONFIG.get(
                                    "training_wandb_disable_system_stats",
                                    True,
                                )
                            )
                        ),
                        save_code=False,
                        tags=None,
                        group=None,
                        job_type=None,
                    )
                )
                wandb_display_name = initial_name

                self.wandb_run_id = (
                    self._wandb_run.id
                )
                self.wandb_run_url = (
                    self._wandb_run.url
                )

                self._configure_wandb_metrics()
            except Exception as exc:  # noqa: BLE001 - optional logger fallback
                self.wandb_error = (
                    f"{type(exc).__name__}: "
                    f"{exc}"
                )

    def _configure_wandb_metrics(
        self,
    ):
        if self._wandb_run is None:
            return

        self._wandb_run.define_metric(
            "global_step"
        )

        for pattern in (
            "train/*",
            "episode/*",
            "evaluation/*",
        ):
            self._wandb_run.define_metric(
                pattern,
                step_metric="global_step",
            )

        for metric in (
            "evaluation/success_rate",
            "evaluation/delivery_rate",
            "evaluation/confirmation_rate",
            "evaluation/return",
            "evaluation/network_pdr_percent",
            "evaluation/network_throughput_kbps",
        ):
            self._wandb_run.define_metric(
                metric,
                summary="max",
            )

        for metric in (
            "evaluation/total_energy_j",
            "evaluation/communication_energy_j",
            "evaluation/false_confirmations",
            "evaluation/expired_reports",
            "evaluation/dropped_reports",
            "evaluation/obstacle_block_rate",
            "evaluation/peer_safety_block_rate",
            "evaluation/boundary_clip_rate",
            "evaluation/network_e2e_delay_ms",
        ):
            self._wandb_run.define_metric(
                metric,
                summary="min",
            )

        self._wandb_run.define_metric(
            "train/critic_loss",
            summary="min",
        )

    @staticmethod
    def _scalar_metrics(
        metrics,
    ):
        if not isinstance(
            metrics,
            dict,
        ):
            raise TypeError(
                "metrics must be a dict"
            )

        scalar_metrics = {}

        for key, value in (
            metrics.items()
        ):
            if isinstance(
                value,
                torch.Tensor,
            ):
                if value.numel() != 1:
                    continue

                value = float(
                    value
                    .detach()
                    .cpu()
                    .item()
                )
            elif isinstance(
                value,
                np.generic,
            ):
                value = value.item()

            if isinstance(
                value,
                (bool, int, float),
            ):
                value = float(value)

                if np.isfinite(
                    value
                ):
                    scalar_metrics[
                        str(key)
                    ] = value

        return scalar_metrics

    def log_metrics(
        self,
        prefix,
        metrics,
        step,
        episode=None,
    ):
        prefix = str(
            prefix
        ).strip(
            "/"
        )
        step = int(
            step
        )

        scalar_metrics = (
            self._scalar_metrics(
                metrics
            )
        )

        tagged = {
            (
                f"{prefix}/{key}"
                if prefix
                else key
            ): value
            for key, value
            in scalar_metrics.items()
        }

        if (
            self._csv_writer
            is not None
        ):
            for key, value in (
                tagged.items()
            ):
                self._csv_writer.writerow(
                    [
                        step,
                        (
                            ""
                            if episode is None
                            else int(
                                episode
                            )
                        ),
                        key,
                        value,
                    ]
                )

            self._csv_file.flush()

        if (
            self._tensorboard_writer
            is not None
        ):
            for key, value in (
                tagged.items()
            ):
                self._tensorboard_writer.add_scalar(
                    key,
                    value,
                    step,
                )

        if self._wandb_run is not None:
            self._wandb_run.log(
                {
                    "global_step": step,
                    **tagged,
                }
            )

        return tagged

    def update_summary(
        self,
        metrics,
        prefix="summary",
    ):
        scalar_metrics = (
            self._scalar_metrics(
                metrics
            )
        )

        if self._wandb_run is not None:
            for key, value in (
                scalar_metrics.items()
            ):
                self._wandb_run.summary[
                    f"{prefix}/{key}"
                ] = value

        return scalar_metrics

    def log_model_artifact(
        self,
        path,
        aliases,
        metadata=None,
    ):
        if self._wandb_run is None:
            return None

        path = Path(path)

        if not path.is_file():
            raise FileNotFoundError(
                str(path)
            )

        wandb = importlib.import_module(
            "wandb"
        )

        artifact_name = (
            f"{self.run_name}-"
            f"{self.algorithm or 'marl'}-model"
        )

        artifact = wandb.Artifact(
            name=artifact_name,
            type="model",
            metadata=(
                {}
                if metadata is None
                else copy.deepcopy(
                    metadata
                )
            ),
        )
        artifact.add_file(
            str(path),
            name=path.name,
        )

        self._wandb_run.log_artifact(
            artifact,
            aliases=list(
                aliases
            ),
        )

        return artifact_name

    def status(self):
        return {
            "csv": bool(
                self._csv_writer
                is not None
            ),
            "tensorboard": bool(
                self._tensorboard_writer
                is not None
            ),
            "wandb": bool(
                self._wandb_run
                is not None
            ),
            "wandb_run_id": (
                self.wandb_run_id
            ),
            "wandb_run_url": (
                self.wandb_run_url
            ),
            "algorithm": self.algorithm,
            "seed": self.seed,
            "backend": (
                self.backend_name
            ),
            "tensorboard_error": (
                self.tensorboard_error
            ),
            "wandb_error": (
                self.wandb_error
            ),
        }

    def close(self):
        if (
            self._tensorboard_writer
            is not None
        ):
            self._tensorboard_writer.flush()
            self._tensorboard_writer.close()
            self._tensorboard_writer = None

        if self._wandb_run is not None:
            self._wandb_run.finish()
            self._wandb_run = None

        if self._csv_file is not None:
            self._csv_file.flush()
            self._csv_file.close()
            self._csv_file = None
            self._csv_writer = None


In [ ]:
def summarize_evaluation_results(
    results,
):
    if not results:
        raise ValueError(
            "evaluation results must "
            "not be empty"
        )

    summary = {
        "success_rate": float(
            np.mean(
                np.asarray(
                    [
                        item[
                            "success"
                        ]
                        for item
                        in results
                    ],
                    dtype=np.float64,
                )
            )
        ),
        "return": float(
            np.mean(
                np.asarray(
                    [
                        item[
                            "return"
                        ]
                        for item
                        in results
                    ],
                    dtype=np.float64,
                )
            )
        ),
        "length": float(
            np.mean(
                np.asarray(
                    [
                        item[
                            "length"
                        ]
                        for item
                        in results
                    ],
                    dtype=np.float64,
                )
            )
        ),
    }

    keys_to_average = (
        "targets_spawned",
        "obstacles_spawned",
        "confirmed_targets",
        "delivered_targets",
        "targets_ever_in_fov_count",
        "target_encounter_rate",
        "sensing_records",
        "positive_observations",
        "target_positive_observations",
        "delivery_rate",
        "confirmation_rate",
        "false_confirmations",
        "expired_reports",
        "dropped_reports",
        "blocked_motion_rate",
        "obstacle_block_rate",
        "peer_safety_block_rate",
        "boundary_clip_rate",
        "horizontal_boundary_clip_rate",
        "altitude_clip_rate",
        "total_energy_j",
        "communication_energy_j",
        "battery_remaining_mean_j",
        "battery_remaining_mean_percent",
        "depleted_uav_count",
        "collision_violation_count",
        "collision_step_count",
        "episode_end_step",
        "information_gain_bits",
        "network_pdr_percent",
        "network_e2e_delay_ms",
        "network_throughput_kbps",
        "network_average_hops",
        "network_phy_success_percent",
        "network_collisions",
        "network_payload_delivery_ratio",
    )

    for key in keys_to_average:
        values = [
            item[key]
            for item in results
            if key in item
        ]

        if len(values) != len(
            results
        ):
            continue

        summary[key] = float(
            np.mean(
                np.asarray(
                    values,
                    dtype=np.float64,
                )
            )
        )

    return summary


In [ ]:
def evaluation_is_better(
    candidate,
    best,
):
    if best is None:
        return True

    candidate_key = (
        float(
            candidate[
                "success_rate"
            ]
        ),
        float(
            candidate.get(
                "delivery_rate",
                0.0,
            )
        ),
        float(
            candidate.get(
                "confirmation_rate",
                0.0,
            )
        ),
        float(
            candidate[
                "return"
            ]
        ),
    )
    best_key = (
        float(
            best[
                "success_rate"
            ]
        ),
        float(
            best.get(
                "delivery_rate",
                0.0,
            )
        ),
        float(
            best.get(
                "confirmation_rate",
                0.0,
            )
        ),
        float(
            best[
                "return"
            ]
        ),
    )

    return bool(
        candidate_key > best_key
    )


In [ ]:
def train_masac_experiment(
    env,
    total_steps,
    seed=None,
    run_dir=None,
    run_name=None,
    resume_checkpoint=None,
    device=None,
    enable_csv=None,
    enable_tensorboard=None,
    enable_wandb=None,
):
    if not isinstance(
        env,
        UAVSearchEnv,
    ):
        raise TypeError(
            "env must be UAVSearchEnv"
        )

    total_steps = int(
        total_steps
    )

    if total_steps < 1:
        raise ValueError(
            "total_steps must be >= 1"
        )

    if seed is None:
        seed = int(
            CONFIG["seed"]
        )

    seed = int(
        seed
    )

    if run_name is None:
        run_name = (
            make_experiment_run_name(
                "masac",
                seed,
            )
        )

    if run_dir is None:
        if resume_checkpoint is not None:
            run_dir = (
                Path(
                    resume_checkpoint
                )
                .resolve()
                .parent
                .parent
            )
        else:
            run_dir = (
                Path(
                    CONFIG[
                        "training_output_dir"
                    ]
                )
                / run_name
            )

    run_dir = Path(
        run_dir
    )
    checkpoint_dir = (
        run_dir / "checkpoints"
    )
    checkpoint_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    set_seed(
        seed
    )

    if resume_checkpoint is None:
        trainer = HybridMASAC(
            env,
            device=device,
        )
        replay_buffer = (
            trainer.make_replay_buffer(
                seed=seed,
            )
        )
        training_state = {
            "seed": seed,
            "global_step": 0,
            "episode_index": 0,
            "best_evaluation": None,
            "last_eval_step": 0,
            "last_checkpoint_step": 0,
            "training_rng_state_json": None,
            "resume_safe": True,
        }
    else:
        restored = (
            load_masac_checkpoint(
                resume_checkpoint,
                env,
                device=device,
                load_replay=True,
                restore_torch_rng=True,
            )
        )
        trainer = restored[
            "trainer"
        ]
        replay_buffer = restored[
            "replay_buffer"
        ]
        training_state = restored[
            "training_state"
        ]

        if training_state.get("vectorized", False):
            raise ValueError(
                "vector checkpoint requires the saved multi-env runtime; "
                "do not resume it through the single-env loop"
            )
        _validate_resume_config(
            restored["config_snapshot"],
            "masac",
        )

        if replay_buffer is None:
            raise ValueError(
                "resume checkpoint does not "
                "contain replay state; set "
                "training_checkpoint_include_replay=True "
                "when creating resumable checkpoints"
            )

        if not bool(
            training_state.get(
                "resume_safe",
                False,
            )
        ):
            raise ValueError(
                "checkpoint is not marked "
                "resume_safe"
            )

        saved_seed = int(
            training_state.get(
                "seed",
                seed,
            )
        )

        if saved_seed != seed:
            raise ValueError(
                "resume seed does not match "
                "checkpoint seed"
            )

    global_step = int(
        training_state.get(
            "global_step",
            0,
        )
    )
    episode_index = int(
        training_state.get(
            "episode_index",
            0,
        )
    )
    best_evaluation = copy.deepcopy(
        training_state.get(
            "best_evaluation"
        )
    )
    last_eval_step = int(
        training_state.get(
            "last_eval_step",
            0,
        )
    )
    last_checkpoint_step = int(
        training_state.get(
            "last_checkpoint_step",
            0,
        )
    )

    if global_step > total_steps:
        raise ValueError(
            "total_steps is smaller than "
            "checkpoint global_step"
        )

    rng = np.random.default_rng(
        seed
    )

    saved_rng_state = (
        training_state.get(
            "training_rng_state_json"
        )
    )

    if saved_rng_state:
        rng.bit_generator.state = (
            json.loads(
                saved_rng_state
            )
        )

    logger = MASACExperimentLogger(
        run_dir=run_dir,
        run_name=run_name,
        enable_csv=enable_csv,
        enable_tensorboard=(
            enable_tensorboard
        ),
        enable_wandb=enable_wandb,
        wandb_entity=CONFIG.get(
            "training_wandb_entity"
        ),
        wandb_project=CONFIG[
            "training_wandb_project"
        ],
        algorithm="masac",
        seed=seed,
        backend_name=(
            env.backend_name
        ),
        config_snapshot=(
            _checkpoint_config_snapshot()
        ),
    )

    eval_env = UAVSearchEnv(
        backend_name=env.backend_name,
        network_backend_kwargs=(
            env.network_backend_kwargs
        ),
    )

    observations, info = env.reset(
        seed=(
            seed + episode_index
        )
    )
    state = info[
        "state"
    ]

    episode_return = 0.0
    episode_length = 0
    episode_diagnostics = (
        new_episode_diagnostics()
    )
    completed_episodes = []
    latest_update_metrics = None
    latest_evaluation = None

    learning_starts = int(
        CONFIG[
            "masac_learning_starts"
        ]
    )
    batch_size = int(
        CONFIG[
            "masac_batch_size"
        ]
    )
    updates_per_step = int(
        CONFIG[
            "masac_updates_per_step"
        ]
    )
    log_interval = max(
        1,
        int(
            CONFIG[
                "training_log_interval_steps"
            ]
        ),
    )
    eval_interval = max(
        1,
        int(
            CONFIG[
                "training_eval_interval_steps"
            ]
        ),
    )
    checkpoint_interval = max(
        1,
        int(
            CONFIG[
                "training_checkpoint_interval_steps"
            ]
        ),
    )
    eval_episodes = max(
        1,
        int(
            CONFIG[
                "training_eval_episodes"
            ]
        ),
    )

    try:
        while global_step < total_steps:
            next_global_step = (
                global_step + 1
            )

            if (
                next_global_step
                <= learning_starts
            ):
                actions = (
                    sample_valid_random_actions(
                        observations,
                        env.agent_ids,
                        rng,
                    )
                )
            else:
                actions = (
                    trainer.select_actions(
                        observations,
                        deterministic=False,
                    )
                )

            (
                next_observations,
                reward,
                terminated,
                truncated,
                next_info,
            ) = env.step(
                actions
            )

            next_state = next_info[
                "state"
            ]

            add_environment_transition_to_replay(
                replay_buffer,
                observations,
                state,
                actions,
                reward,
                next_observations,
                next_state,
                terminated,
                truncated,
                env.agent_ids,
            )

            global_step = (
                next_global_step
            )
            episode_return += float(
                reward
            )
            episode_length += 1

            update_episode_diagnostics(
                episode_diagnostics,
                next_info,
                env=env,
            )

            if (
                global_step
                > learning_starts
                and len(
                    replay_buffer
                )
                >= batch_size
            ):
                for _ in range(
                    updates_per_step
                ):
                    latest_update_metrics = (
                        trainer.update(
                            replay_buffer,
                            batch_size=(
                                batch_size
                            ),
                        )
                    )

            if (
                latest_update_metrics
                is not None
                and global_step
                % log_interval
                == 0
            ):
                logger.log_metrics(
                    "train",
                    filter_training_metrics_for_wandb(
                        "masac",
                        latest_update_metrics,
                    ),
                    step=global_step,
                    episode=episode_index,
                )

            observations = (
                next_observations
            )
            state = next_state

            should_evaluate = (
                global_step
                - last_eval_step
                >= eval_interval
                or global_step
                >= total_steps
            )

            if should_evaluate:
                evaluation_results = (
                    evaluate_masac(
                        eval_env,
                        trainer,
                        episodes=(
                            eval_episodes
                        ),
                        seed=(
                            seed
                            + int(
                                CONFIG[
                                    "training_eval_seed_offset"
                                ]
                            )
                        ),
                    )
                )
                latest_evaluation = (
                    summarize_evaluation_results(
                        evaluation_results
                    )
                )

                logger.log_metrics(
                    "evaluation",
                    latest_evaluation,
                    step=global_step,
                    episode=episode_index,
                )
                logger.update_summary(
                    latest_evaluation,
                    prefix=(
                        "evaluation_latest"
                    ),
                )

                last_eval_step = (
                    global_step
                )

                if evaluation_is_better(
                    latest_evaluation,
                    best_evaluation,
                ):
                    best_evaluation = (
                        copy.deepcopy(
                            latest_evaluation
                        )
                    )

                    best_state = {
                        "seed": seed,
                        "global_step": (
                            global_step
                        ),
                        "episode_index": (
                            episode_index
                        ),
                        "best_evaluation": (
                            best_evaluation
                        ),
                        "last_eval_step": (
                            last_eval_step
                        ),
                        "last_checkpoint_step": (
                            last_checkpoint_step
                        ),
                        "training_rng_state_json": (
                            json.dumps(
                                rng
                                .bit_generator
                                .state
                            )
                        ),
                        "resume_safe": False,
                    }

                    best_checkpoint_path = (
                        save_masac_checkpoint(
                            checkpoint_dir
                            / "best.pt",
                            trainer,
                            replay_buffer=(
                                replay_buffer
                            ),
                            training_state=(
                                best_state
                            ),
                            include_replay=False,
                        )
                    )
                    logger.update_summary(
                        best_evaluation,
                        prefix="best",
                    )
                    logger.log_model_artifact(
                        best_checkpoint_path,
                        aliases=[
                            "best",
                        ],
                        metadata=(
                            best_evaluation
                        ),
                    )

            if not (
                terminated
                or truncated
            ):
                continue

            episode_metrics = (
                finalize_episode_diagnostics(
                    episode_diagnostics,
                    env,
                    episode_return=(
                        episode_return
                    ),
                    episode_length=(
                        episode_length
                    ),
                    success=(
                        next_info[
                            "success"
                        ]
                    ),
                )
            )

            logger.log_metrics(
                "episode",
                episode_metrics,
                step=global_step,
                episode=episode_index,
            )

            completed_episodes.append(
                {
                    "episode": (
                        episode_index
                    ),
                    **episode_metrics,
                    "terminated": bool(
                        terminated
                    ),
                    "truncated": bool(
                        truncated
                    ),
                }
            )

            episode_index += 1

            should_checkpoint = (
                global_step
                - last_checkpoint_step
                >= checkpoint_interval
                or global_step
                >= total_steps
            )

            if should_checkpoint:
                last_checkpoint_step = (
                    global_step
                )

                include_replay = bool(
                    CONFIG[
                        "training_checkpoint_include_replay"
                    ]
                )

                checkpoint_state = {
                    "seed": seed,
                    "global_step": (
                        global_step
                    ),
                    "episode_index": (
                        episode_index
                    ),
                    "best_evaluation": (
                        copy.deepcopy(
                            best_evaluation
                        )
                    ),
                    "last_eval_step": (
                        last_eval_step
                    ),
                    "last_checkpoint_step": (
                        last_checkpoint_step
                    ),
                    "training_rng_state_json": (
                        json.dumps(
                            rng
                            .bit_generator
                            .state
                        )
                    ),
                    "resume_safe": (
                        include_replay
                    ),
                }

                save_masac_checkpoint(
                    checkpoint_dir
                    / "latest.pt",
                    trainer,
                    replay_buffer=(
                        replay_buffer
                    ),
                    training_state=(
                        checkpoint_state
                    ),
                    include_replay=(
                        include_replay
                    ),
                )

            episode_return = 0.0
            episode_length = 0
            episode_diagnostics = (
                new_episode_diagnostics()
            )

            if global_step < total_steps:
                observations, info = (
                    env.reset(
                        seed=(
                            seed
                            + episode_index
                        )
                    )
                )
                state = info[
                    "state"
                ]

        final_state = {
            "seed": seed,
            "global_step": (
                global_step
            ),
            "episode_index": (
                episode_index
            ),
            "best_evaluation": (
                copy.deepcopy(
                    best_evaluation
                )
            ),
            "last_eval_step": (
                last_eval_step
            ),
            "last_checkpoint_step": (
                last_checkpoint_step
            ),
            "training_rng_state_json": (
                json.dumps(
                    rng
                    .bit_generator
                    .state
                )
            ),
            "resume_safe": False,
        }

        # Always keep a lightweight final model checkpoint. If training
        # stops mid-episode it is valid for evaluation, but resume_safe
        # is false because environment state is intentionally not serialized.
        final_checkpoint_path = (
            save_masac_checkpoint(
                checkpoint_dir
                / "final.pt",
                trainer,
                replay_buffer=(
                    replay_buffer
                ),
                training_state=(
                    final_state
                ),
                include_replay=False,
            )
        )
        logger.log_model_artifact(
            final_checkpoint_path,
            aliases=[
                "final",
            ],
            metadata={
                "global_step": (
                    global_step
                ),
                "episode_index": (
                    episode_index
                ),
            },
        )
        logger.update_summary(
            {
                "global_step": (
                    global_step
                ),
                "episodes_completed": (
                    episode_index
                ),
                "trainer_updates": int(
                    trainer.update_count
                ),
            },
            prefix="training_final",
        )

        return {
            "trainer": trainer,
            "replay_buffer": replay_buffer,
            "completed_episodes": (
                completed_episodes
            ),
            "latest_update_metrics": (
                latest_update_metrics
            ),
            "latest_evaluation": (
                latest_evaluation
            ),
            "best_evaluation": (
                best_evaluation
            ),
            "training_state": (
                final_state
            ),
            "logger_status": (
                logger.status()
            ),
            "run_dir": run_dir,
            "checkpoint_dir": (
                checkpoint_dir
            ),
        }
    finally:
        eval_env.close()
        logger.close()


In [ ]:
def straight_through_masked_argmax(
    logits,
    destination_mask,
    temperature=None,
):
    if temperature is None:
        temperature = CONFIG[
            "matd3_gumbel_temperature"
        ]

    temperature = float(
        temperature
    )

    if temperature <= 0.0:
        raise ValueError(
            "temperature must be > 0"
        )

    masked_logits = (
        masked_categorical_logits(
            logits,
            destination_mask,
        )
    )

    probabilities = F.softmax(
        masked_logits
        / temperature,
        dim=-1,
    )

    indices = torch.argmax(
        masked_logits,
        dim=-1,
    )

    hard = F.one_hot(
        indices,
        num_classes=(
            masked_logits.shape[-1]
        ),
    ).to(
        masked_logits.dtype
    )

    action = (
        hard
        - probabilities.detach()
        + probabilities
    )

    return {
        "action": action,
        "hard_action": hard,
        "indices": indices,
        "probabilities": probabilities,
    }


In [ ]:
class HybridMATD3Actor(nn.Module):
    def __init__(
        self,
        observation_dim,
        continuous_dim,
        discrete_dim,
        hidden_dims,
    ):
        super().__init__()

        hidden_dims = tuple(
            int(value)
            for value
            in hidden_dims
        )

        if not hidden_dims:
            raise ValueError(
                "hidden_dims must not be empty"
            )

        self.observation_dim = int(
            observation_dim
        )
        self.continuous_dim = int(
            continuous_dim
        )
        self.discrete_dim = int(
            discrete_dim
        )

        self.encoder = build_mlp(
            self.observation_dim,
            hidden_dims[:-1],
            hidden_dims[-1],
        )

        feature_dim = hidden_dims[-1]

        self.continuous_head = nn.Linear(
            feature_dim,
            self.continuous_dim,
        )
        self.discrete_head = nn.Linear(
            feature_dim,
            self.discrete_dim,
        )

    def forward(
        self,
        observations,
    ):
        features = self.encoder(
            observations
        )

        raw_continuous = self.continuous_head(
            features
        )
        motion, _ = radial_squash_motion_action(
            raw_continuous[..., :3]
        )
        power = torch.tanh(
            raw_continuous[..., 3:4]
        )
        continuous = torch.cat(
            [
                motion,
                power,
            ],
            dim=-1,
        )
        logits = self.discrete_head(
            features
        )

        return (
            continuous,
            logits,
        )

    def actions(
        self,
        observations,
        destination_mask,
        straight_through=True,
    ):
        (
            continuous,
            logits,
        ) = self(
            observations
        )

        discrete = (
            straight_through_masked_argmax(
                logits,
                destination_mask,
            )
        )

        destination_action = (
            discrete["action"]
            if straight_through
            else discrete[
                "hard_action"
            ]
        )

        return {
            "continuous": continuous,
            "destination_one_hot": (
                destination_action
            ),
            "destination_hard_one_hot": (
                discrete[
                    "hard_action"
                ]
            ),
            "destination_indices": (
                discrete["indices"]
            ),
            "destination_probabilities": (
                discrete[
                    "probabilities"
                ]
            ),
        }


In [ ]:
class HybridMATD3:
    def __init__(
        self,
        env,
        device=None,
        seed=None,
    ):
        if not isinstance(
            env,
            UAVSearchEnv,
        ):
            raise TypeError(
                "env must be UAVSearchEnv"
            )

        if seed is None:
            seed = int(
                CONFIG["seed"]
            )

        self.seed = int(seed)
        self.rng = (
            np.random.default_rng(
                self.seed
            )
        )

        self.agent_ids = tuple(
            env.agent_ids
        )
        self.num_agents = len(
            self.agent_ids
        )

        if device is None:
            device = (
                "cuda"
                if torch.cuda.is_available()
                else "cpu"
            )

        self.device = torch.device(
            device
        )

        agent_space = (
            env.observation_space
            .spaces[
                self.agent_ids[0]
            ]
        )

        self.observation_dim = int(
            sum(
                np.prod(
                    agent_space.spaces[
                        key
                    ].shape,
                    dtype=np.int64,
                )
                for key
                in AGENT_OBSERVATION_VECTOR_KEYS
            )
        )
        self.state_dim = int(
            np.prod(
                env.state_space.shape,
                dtype=np.int64,
            )
        )
        self.continuous_dim = 4
        self.discrete_dim = int(
            env.action_space
            .spaces[
                self.agent_ids[0]
            ]
            .spaces[
                "destination"
            ]
            .n
        )
        self.joint_action_dim = (
            self.num_agents
            * (
                self.continuous_dim
                + self.discrete_dim
            )
        )

        hidden_dims = tuple(
            CONFIG[
                "matd3_hidden_dims"
            ]
        )

        self.actor = HybridMATD3Actor(
            self.observation_dim,
            self.continuous_dim,
            self.discrete_dim,
            hidden_dims,
        ).to(
            self.device
        )

        self.target_actor = copy.deepcopy(
            self.actor
        ).to(
            self.device
        )

        self.critic_1 = (
            CentralizedQNetwork(
                self.state_dim,
                self.joint_action_dim,
                hidden_dims,
            ).to(
                self.device
            )
        )
        self.critic_2 = (
            CentralizedQNetwork(
                self.state_dim,
                self.joint_action_dim,
                hidden_dims,
            ).to(
                self.device
            )
        )

        self.target_critic_1 = (
            copy.deepcopy(
                self.critic_1
            ).to(
                self.device
            )
        )
        self.target_critic_2 = (
            copy.deepcopy(
                self.critic_2
            ).to(
                self.device
            )
        )

        for network in (
            self.target_actor,
            self.target_critic_1,
            self.target_critic_2,
        ):
            network.eval()

            for parameter in (
                network.parameters()
            ):
                parameter.requires_grad_(
                    False
                )

        self.actor_optimizer = (
            torch.optim.Adam(
                self.actor.parameters(),
                lr=float(
                    CONFIG[
                        "matd3_actor_lr"
                    ]
                ),
            )
        )
        self.critic_optimizer = (
            torch.optim.Adam(
                list(
                    self.critic_1.parameters()
                )
                + list(
                    self.critic_2.parameters()
                ),
                lr=float(
                    CONFIG[
                        "matd3_critic_lr"
                    ]
                ),
            )
        )

        self.gamma = float(
            CONFIG["matd3_gamma"]
        )
        self.tau = float(
            CONFIG["matd3_tau"]
        )
        self.policy_delay = int(
            CONFIG[
                "matd3_policy_delay"
            ]
        )
        self.target_policy_noise = float(
            CONFIG[
                "matd3_target_policy_noise"
            ]
        )
        self.target_noise_clip = float(
            CONFIG[
                "matd3_target_noise_clip"
            ]
        )
        self.exploration_noise = float(
            CONFIG[
                "matd3_exploration_noise"
            ]
        )
        self.gradient_clip_norm = float(
            CONFIG[
                "matd3_gradient_clip_norm"
            ]
        )

        if self.policy_delay < 1:
            raise ValueError(
                "matd3_policy_delay "
                "must be >= 1"
            )

        self.update_count = 0
        self.action_count = 0
        amp_dtype_name = str(
            CONFIG.get(
                "training_amp_dtype",
                "float16",
            )
        ).strip().lower()
        if amp_dtype_name == "float16":
            self.amp_dtype = torch.float16
        elif amp_dtype_name == "bfloat16":
            self.amp_dtype = torch.bfloat16
        else:
            raise ValueError(
                "training_amp_dtype must be float16 or bfloat16"
            )
        self.amp_enabled = (
            bool(
                CONFIG.get(
                    "training_amp_enabled",
                    False,
                )
            )
            and self.device.type == "cuda"
        )
        self.grad_scaler = torch.amp.GradScaler(
            "cuda",
            enabled=self.amp_enabled,
        )

    def _autocast(self):
        return torch.autocast(
            device_type=self.device.type,
            dtype=self.amp_dtype,
            enabled=self.amp_enabled,
        )

    def make_replay_buffer(
        self,
        seed=None,
    ):
        if seed is None:
            seed = self.seed

        return HybridReplayBuffer(
            capacity=CONFIG[
                "matd3_replay_capacity"
            ],
            num_agents=(
                self.num_agents
            ),
            observation_dim=(
                self.observation_dim
            ),
            state_dim=(
                self.state_dim
            ),
            continuous_dim=(
                self.continuous_dim
            ),
            discrete_dim=(
                self.discrete_dim
            ),
            seed=seed,
        )

    def _joint_action_tensor(
        self,
        continuous_actions,
        destination_one_hot,
    ):
        if continuous_actions.ndim != 3:
            raise ValueError(
                "continuous_actions must "
                "have rank 3"
            )

        if destination_one_hot.ndim != 3:
            raise ValueError(
                "destination_one_hot must "
                "have rank 3"
            )

        combined = torch.cat(
            [
                continuous_actions,
                destination_one_hot,
            ],
            dim=-1,
        )

        return combined.reshape(
            combined.shape[0],
            -1,
        )

    def _actor_joint_actions(
        self,
        observations,
        destination_masks,
        straight_through=True,
        target_actor=False,
    ):
        if observations.ndim != 3:
            raise ValueError(
                "observations must "
                "have rank 3"
            )

        batch_size = int(
            observations.shape[0]
        )

        flat_observations = (
            observations.reshape(
                batch_size
                * self.num_agents,
                self.observation_dim,
            )
        )
        flat_masks = (
            destination_masks.reshape(
                batch_size
                * self.num_agents,
                self.discrete_dim,
            )
        )

        actor = (
            self.target_actor
            if target_actor
            else self.actor
        )

        sampled = actor.actions(
            flat_observations,
            flat_masks,
            straight_through=(
                straight_through
            ),
        )

        continuous = sampled[
            "continuous"
        ].reshape(
            batch_size,
            self.num_agents,
            self.continuous_dim,
        )
        destination_one_hot = (
            sampled[
                "destination_one_hot"
            ].reshape(
                batch_size,
                self.num_agents,
                self.discrete_dim,
            )
        )
        destination_indices = (
            sampled[
                "destination_indices"
            ].reshape(
                batch_size,
                self.num_agents,
            )
        )

        return {
            "continuous": continuous,
            "destination_one_hot": (
                destination_one_hot
            ),
            "destination_indices": (
                destination_indices
            ),
        }

    def _discrete_epsilon(
        self,
        step,
    ):
        start = float(
            CONFIG[
                "matd3_discrete_epsilon_start"
            ]
        )
        end = float(
            CONFIG[
                "matd3_discrete_epsilon_end"
            ]
        )
        decay_steps = max(
            1,
            int(
                CONFIG[
                    "matd3_discrete_epsilon_decay_steps"
                ]
            ),
        )

        fraction = min(
            1.0,
            max(
                0.0,
                float(step)
                / decay_steps,
            ),
        )

        return float(
            start
            + fraction
            * (
                end - start
            )
        )

    def select_actions(
        self,
        observations,
        deterministic=False,
        exploration_step=None,
    ):
        (
            observation_vectors,
            destination_masks,
        ) = observations_to_masac_arrays(
            observations,
            self.agent_ids,
        )

        observation_tensor = (
            torch.as_tensor(
                observation_vectors,
                dtype=torch.float32,
                device=self.device,
            )
        )
        mask_tensor = torch.as_tensor(
            destination_masks,
            dtype=torch.float32,
            device=self.device,
        )

        with torch.no_grad():
            sampled = self.actor.actions(
                observation_tensor,
                mask_tensor,
                straight_through=False,
            )

        continuous = (
            sampled[
                "continuous"
            ]
            .cpu()
            .numpy()
            .astype(
                np.float32
            )
        )
        destination_indices = (
            sampled[
                "destination_indices"
            ]
            .cpu()
            .numpy()
            .astype(
                np.int64
            )
        )

        if not deterministic:
            noise = self.rng.normal(
                loc=0.0,
                scale=self.exploration_noise,
                size=continuous.shape,
            ).astype(
                np.float32
            )

            continuous = np.clip(
                continuous + noise,
                -1.0,
                1.0,
            ).astype(
                np.float32
            )
            continuous[:, :3] = np.asarray(
                project_motion_action_np(
                    continuous[:, :3]
                ),
                dtype=np.float32,
            )

            if exploration_step is None:
                exploration_step = (
                    self.action_count
                )

            epsilon = (
                self._discrete_epsilon(
                    exploration_step
                )
            )

            for agent_index in range(
                self.num_agents
            ):
                if (
                    self.rng.random()
                    >= epsilon
                ):
                    continue

                valid = np.flatnonzero(
                    destination_masks[
                        agent_index
                    ]
                    > 0.5
                )

                destination_indices[
                    agent_index
                ] = int(
                    self.rng.choice(
                        valid
                    )
                )

            self.action_count += 1

        return masac_arrays_to_env_actions(
            continuous,
            destination_indices,
            self.agent_ids,
        )

    def _soft_update(
        self,
        target,
        source,
    ):
        with torch.no_grad():
            for (
                target_parameter,
                source_parameter,
            ) in zip(
                target.parameters(),
                source.parameters(),
            ):
                target_parameter.mul_(
                    1.0 - self.tau
                )
                target_parameter.add_(
                    self.tau
                    * source_parameter
                )

    def update(
        self,
        replay_buffer,
        batch_size=None,
        collect_metrics=True,
    ):
        if not isinstance(
            replay_buffer,
            HybridReplayBuffer,
        ):
            raise TypeError(
                "replay_buffer must be "
                "HybridReplayBuffer"
            )

        if batch_size is None:
            batch_size = int(
                CONFIG[
                    "matd3_batch_size"
                ]
            )

        batch = replay_buffer.sample(
            batch_size,
            self.device,
        )

        replay_destination_one_hot = (
            F.one_hot(
                batch[
                    "destination_indices"
                ],
                num_classes=(
                    self.discrete_dim
                ),
            ).to(
                torch.float32
            )
        )

        replay_joint_action = (
            self._joint_action_tensor(
                batch[
                    "continuous_actions"
                ],
                replay_destination_one_hot,
            )
        )

        with torch.no_grad(), self._autocast():
            target_policy = (
                self._actor_joint_actions(
                    batch[
                        "next_observations"
                    ],
                    batch[
                        "next_destination_masks"
                    ],
                    straight_through=False,
                    target_actor=True,
                )
            )

            noise = (
                torch.randn_like(
                    target_policy[
                        "continuous"
                    ]
                )
                * self.target_policy_noise
            ).clamp(
                -self.target_noise_clip,
                self.target_noise_clip,
            )

            target_continuous = (
                target_policy[
                    "continuous"
                ]
                + noise
            ).clamp(
                -1.0,
                1.0,
            )
            target_continuous = torch.cat(
                [
                    project_motion_action_tensor(
                        target_continuous[..., :3]
                    ),
                    target_continuous[..., 3:4],
                ],
                dim=-1,
            )

            target_joint_action = (
                self._joint_action_tensor(
                    target_continuous,
                    target_policy[
                        "destination_one_hot"
                    ],
                )
            )

            target_q = torch.minimum(
                self.target_critic_1(
                    batch[
                        "next_states"
                    ],
                    target_joint_action,
                ),
                self.target_critic_2(
                    batch[
                        "next_states"
                    ],
                    target_joint_action,
                ),
            )

            critic_target = (
                batch["rewards"]
                + self.gamma
                * (
                    1.0
                    - batch[
                        "terminated"
                    ]
                )
                * target_q
            )

        with self._autocast():
            critic_1_value = (
                self.critic_1(
                    batch["states"],
                    replay_joint_action,
                )
            )
            critic_2_value = (
                self.critic_2(
                    batch["states"],
                    replay_joint_action,
                )
            )

            critic_loss = (
                F.mse_loss(
                    critic_1_value,
                    critic_target,
                )
                + F.mse_loss(
                    critic_2_value,
                    critic_target,
                )
            )

        self.critic_optimizer.zero_grad(
            set_to_none=True
        )
        if self.amp_enabled:
            self.grad_scaler.scale(
                critic_loss
            ).backward()
            self.grad_scaler.unscale_(
                self.critic_optimizer
            )
        else:
            critic_loss.backward()

        critic_grad_norm = (
            torch.nn.utils.clip_grad_norm_(
                list(
                    self.critic_1.parameters()
                )
                + list(
                    self.critic_2.parameters()
                ),
                self.gradient_clip_norm,
            )
        )

        if self.amp_enabled:
            self.grad_scaler.step(
                self.critic_optimizer
            )
        else:
            self.critic_optimizer.step()

        self.update_count += 1

        actor_updated = bool(
            self.update_count
            % self.policy_delay
            == 0
        )
        actor_loss_value = None
        actor_grad_norm_value = None

        if actor_updated:
            for critic in (
                self.critic_1,
                self.critic_2,
            ):
                for parameter in (
                    critic.parameters()
                ):
                    parameter.requires_grad_(
                        False
                    )

            with self._autocast():
                policy = (
                    self._actor_joint_actions(
                        batch[
                            "observations"
                        ],
                        batch[
                            "destination_masks"
                        ],
                        straight_through=True,
                        target_actor=False,
                    )
                )

                policy_joint_action = (
                    self._joint_action_tensor(
                        policy[
                            "continuous"
                        ],
                        policy[
                            "destination_one_hot"
                        ],
                    )
                )

                actor_loss = -self.critic_1(
                    batch["states"],
                    policy_joint_action,
                ).mean()

            self.actor_optimizer.zero_grad(
                set_to_none=True
            )
            if self.amp_enabled:
                self.grad_scaler.scale(
                    actor_loss
                ).backward()
                self.grad_scaler.unscale_(
                    self.actor_optimizer
                )
            else:
                actor_loss.backward()

            actor_grad_norm = (
                torch.nn.utils.clip_grad_norm_(
                    self.actor.parameters(),
                    self.gradient_clip_norm,
                )
            )

            if self.amp_enabled:
                self.grad_scaler.step(
                    self.actor_optimizer
                )
            else:
                self.actor_optimizer.step()

            for critic in (
                self.critic_1,
                self.critic_2,
            ):
                for parameter in (
                    critic.parameters()
                ):
                    parameter.requires_grad_(
                        True
                    )

            self._soft_update(
                self.target_actor,
                self.actor,
            )
            self._soft_update(
                self.target_critic_1,
                self.critic_1,
            )
            self._soft_update(
                self.target_critic_2,
                self.critic_2,
            )

            actor_loss_value = (
                actor_loss.detach()
            )
            actor_grad_norm_value = (
                torch.as_tensor(
                    actor_grad_norm
                ).detach()
            )

        if self.amp_enabled:
            self.grad_scaler.update()

        if not collect_metrics:
            return None

        if actor_loss_value is not None:
            actor_loss_value = float(
                actor_loss_value.cpu()
            )
            actor_grad_norm_value = float(
                actor_grad_norm_value.cpu()
            )

        metrics = {
            "critic_loss": float(
                critic_loss
                .detach()
                .cpu()
            ),
            "critic_1_q_mean": float(
                critic_1_value
                .mean()
                .detach()
                .cpu()
            ),
            "critic_2_q_mean": float(
                critic_2_value
                .mean()
                .detach()
                .cpu()
            ),
            "target_q_mean": float(
                critic_target
                .mean()
                .detach()
                .cpu()
            ),
            "critic_grad_norm": float(
                torch.as_tensor(
                    critic_grad_norm
                )
                .detach()
                .cpu()
            ),
            "actor_updated": (
                actor_updated
            ),
            "actor_loss": (
                actor_loss_value
            ),
            "actor_grad_norm": (
                actor_grad_norm_value
            ),
            "update_count": int(
                self.update_count
            ),
        }

        for key, value in (
            metrics.items()
        ):
            if value is None:
                continue

            if isinstance(
                value,
                bool,
            ):
                continue

            if not np.isfinite(
                value
            ):
                raise FloatingPointError(
                    f"non-finite MATD3 metric: "
                    f"{key}"
                )

        return metrics

    @staticmethod
    def _move_optimizer_state_to_device(
        optimizer,
        device,
    ):
        for state in optimizer.state.values():
            for key, value in list(
                state.items()
            ):
                if isinstance(
                    value,
                    torch.Tensor,
                ):
                    state[key] = value.to(
                        device
                    )

    def state_dict(self):
        return {
            "actor": (
                self.actor.state_dict()
            ),
            "target_actor": (
                self.target_actor
                .state_dict()
            ),
            "critic_1": (
                self.critic_1.state_dict()
            ),
            "critic_2": (
                self.critic_2.state_dict()
            ),
            "target_critic_1": (
                self.target_critic_1
                .state_dict()
            ),
            "target_critic_2": (
                self.target_critic_2
                .state_dict()
            ),
            "actor_optimizer": (
                self.actor_optimizer
                .state_dict()
            ),
            "critic_optimizer": (
                self.critic_optimizer
                .state_dict()
            ),
            "update_count": int(
                self.update_count
            ),
            "action_count": int(
                self.action_count
            ),
            "rng_state_json": json.dumps(
                self.rng
                .bit_generator
                .state
            ),
        }

    def load_state_dict(
        self,
        state,
    ):
        self.actor.load_state_dict(
            state["actor"]
        )
        self.target_actor.load_state_dict(
            state["target_actor"]
        )
        self.critic_1.load_state_dict(
            state["critic_1"]
        )
        self.critic_2.load_state_dict(
            state["critic_2"]
        )
        self.target_critic_1.load_state_dict(
            state[
                "target_critic_1"
            ]
        )
        self.target_critic_2.load_state_dict(
            state[
                "target_critic_2"
            ]
        )

        self.actor_optimizer.load_state_dict(
            state[
                "actor_optimizer"
            ]
        )
        self.critic_optimizer.load_state_dict(
            state[
                "critic_optimizer"
            ]
        )

        for optimizer in (
            self.actor_optimizer,
            self.critic_optimizer,
        ):
            self._move_optimizer_state_to_device(
                optimizer,
                self.device,
            )

        self.update_count = int(
            state.get(
                "update_count",
                0,
            )
        )
        self.action_count = int(
            state.get(
                "action_count",
                0,
            )
        )

        self.rng.bit_generator.state = (
            json.loads(
                state[
                    "rng_state_json"
                ]
            )
        )


In [ ]:
def train_matd3(
    env,
    total_steps,
    seed=None,
    trainer=None,
    replay_buffer=None,
):
    if not isinstance(
        env,
        UAVSearchEnv,
    ):
        raise TypeError(
            "env must be UAVSearchEnv"
        )

    total_steps = int(
        total_steps
    )

    if total_steps < 1:
        raise ValueError(
            "total_steps must be >= 1"
        )

    if seed is None:
        seed = int(
            CONFIG["seed"]
        )

    seed = int(seed)

    set_seed(seed)

    if trainer is None:
        trainer = HybridMATD3(
            env,
            seed=seed,
        )

    if replay_buffer is None:
        replay_buffer = (
            trainer.make_replay_buffer(
                seed=seed,
            )
        )

    rng = np.random.default_rng(
        seed
    )

    observations, info = env.reset(
        seed=seed
    )
    state = info["state"]

    episode_return = 0.0
    episode_length = 0
    episode_index = 0
    completed_episodes = []
    latest_update_metrics = None

    learning_starts = int(
        CONFIG[
            "matd3_learning_starts"
        ]
    )
    updates_per_step = int(
        CONFIG[
            "matd3_updates_per_step"
        ]
    )
    batch_size = int(
        CONFIG[
            "matd3_batch_size"
        ]
    )

    for global_step in range(
        1,
        total_steps + 1,
    ):
        if global_step <= learning_starts:
            actions = (
                sample_valid_random_actions(
                    observations,
                    env.agent_ids,
                    rng,
                )
            )
        else:
            actions = (
                trainer.select_actions(
                    observations,
                    deterministic=False,
                    exploration_step=(
                        global_step
                    ),
                )
            )

        (
            next_observations,
            reward,
            terminated,
            truncated,
            next_info,
        ) = env.step(
            actions
        )

        next_state = next_info[
            "state"
        ]

        add_environment_transition_to_replay(
            replay_buffer,
            observations,
            state,
            actions,
            reward,
            next_observations,
            next_state,
            terminated,
            truncated,
            env.agent_ids,
        )

        episode_return += float(
            reward
        )
        episode_length += 1

        if (
            global_step > learning_starts
            and len(
                replay_buffer
            )
            >= batch_size
        ):
            for _ in range(
                updates_per_step
            ):
                latest_update_metrics = (
                    trainer.update(
                        replay_buffer,
                        batch_size=(
                            batch_size
                        ),
                    )
                )

        observations = (
            next_observations
        )
        state = next_state

        if terminated or truncated:
            completed_episodes.append(
                {
                    "episode": (
                        episode_index
                    ),
                    "return": float(
                        episode_return
                    ),
                    "length": int(
                        episode_length
                    ),
                    "terminated": bool(
                        terminated
                    ),
                    "truncated": bool(
                        truncated
                    ),
                    "success": bool(
                        next_info[
                            "success"
                        ]
                    ),
                }
            )

            episode_index += 1
            episode_return = 0.0
            episode_length = 0

            if global_step < total_steps:
                observations, info = (
                    env.reset(
                        seed=(
                            seed
                            + episode_index
                        )
                    )
                )
                state = info[
                    "state"
                ]

    return {
        "trainer": trainer,
        "replay_buffer": replay_buffer,
        "completed_episodes": (
            completed_episodes
        ),
        "latest_update_metrics": (
            latest_update_metrics
        ),
        "total_steps": total_steps,
    }


In [ ]:
def evaluate_matd3(
    env,
    trainer,
    episodes=3,
    seed=None,
):
    if not isinstance(
        trainer,
        HybridMATD3,
    ):
        raise TypeError(
            "trainer must be HybridMATD3"
        )

    if seed is None:
        seed = int(
            CONFIG["seed"]
        ) + 20_000

    episodes = int(
        episodes
    )

    if episodes < 1:
        raise ValueError(
            "episodes must be >= 1"
        )

    results = []

    for episode_index in range(
        episodes
    ):
        observations, _ = env.reset(
            seed=(
                int(seed)
                + episode_index
            )
        )

        episode_return = 0.0
        diagnostics = (
            new_episode_diagnostics()
        )

        while True:
            actions = trainer.select_actions(
                observations,
                deterministic=True,
            )

            (
                observations,
                reward,
                terminated,
                truncated,
                info,
            ) = env.step(
                actions
            )

            episode_return += float(
                reward
            )

            update_episode_diagnostics(
                diagnostics,
                info,
                env=env,
            )

            if (
                terminated
                or truncated
            ):
                episode_metrics = (
                    finalize_episode_diagnostics(
                        diagnostics,
                        env,
                        episode_return=(
                            episode_return
                        ),
                        episode_length=(
                            info["step"]
                        ),
                        success=(
                            info["success"]
                        ),
                    )
                )

                results.append(
                    {
                        "episode": (
                            episode_index
                        ),
                        **episode_metrics,
                        "terminated": bool(
                            terminated
                        ),
                        "truncated": bool(
                            truncated
                        ),
                    }
                )
                break

    return results


In [ ]:
def save_matd3_checkpoint(
    path,
    trainer,
    replay_buffer=None,
    training_state=None,
    include_replay=None,
):
    if not isinstance(
        trainer,
        HybridMATD3,
    ):
        raise TypeError(
            "trainer must be HybridMATD3"
        )

    if include_replay is None:
        include_replay = bool(
            CONFIG[
                "training_checkpoint_include_replay"
            ]
        )

    include_replay = bool(
        include_replay
    )

    if (
        include_replay
        and replay_buffer is None
    ):
        raise ValueError(
            "replay_buffer is required "
            "when include_replay=True"
        )

    if training_state is None:
        training_state = {}

    path = Path(path)
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    payload = {
        "format_version": 1,
        "algorithm": "hybrid_matd3",
        "trainer_state": (
            trainer.state_dict()
        ),
        "training_state": (
            copy.deepcopy(
                training_state
            )
        ),
        "config_snapshot": (
            _checkpoint_config_snapshot()
        ),
        "replay_state": (
            replay_buffer.state_dict()
            if include_replay
            else None
        ),
        "torch_rng_state": (
            torch.get_rng_state()
        ),
        "cuda_rng_states": (
            torch.cuda.get_rng_state_all()
            if torch.cuda.is_available()
            else []
        ),
    }

    temporary_path = path.with_suffix(
        path.suffix + ".tmp"
    )

    torch.save(
        payload,
        temporary_path,
    )
    temporary_path.replace(
        path
    )

    return path


In [ ]:
def load_matd3_checkpoint(
    path,
    env,
    device=None,
    load_replay=True,
    restore_torch_rng=True,
):
    path = Path(path)

    if not path.is_file():
        raise FileNotFoundError(
            str(path)
        )

    payload = torch.load(
        path,
        map_location="cpu",
        weights_only=True,
    )

    if payload.get(
        "algorithm"
    ) != "hybrid_matd3":
        raise ValueError(
            "checkpoint algorithm "
            "is not hybrid_matd3"
        )

    trainer = HybridMATD3(
        env,
        device=device,
        seed=int(
            payload.get(
                "training_state",
                {},
            ).get(
                "seed",
                CONFIG["seed"],
            )
        ),
    )
    trainer.load_state_dict(
        payload[
            "trainer_state"
        ]
    )

    replay_buffer = None
    replay_state = payload.get(
        "replay_state"
    )

    if (
        load_replay
        and replay_state is not None
    ):
        replay_buffer = (
            trainer.make_replay_buffer(
                seed=trainer.seed
            )
        )
        replay_buffer.load_state_dict(
            replay_state
        )

    if restore_torch_rng:
        torch.set_rng_state(
            payload[
                "torch_rng_state"
            ]
        )

        cuda_states = payload.get(
            "cuda_rng_states",
            [],
        )

        if (
            torch.cuda.is_available()
            and cuda_states
        ):
            torch.cuda.set_rng_state_all(
                cuda_states
            )

    return {
        "trainer": trainer,
        "replay_buffer": replay_buffer,
        "training_state": (
            copy.deepcopy(
                payload.get(
                    "training_state",
                    {},
                )
            )
        ),
        "config_snapshot": (
            copy.deepcopy(
                payload.get(
                    "config_snapshot",
                    {},
                )
            )
        ),
        "path": path,
    }


In [ ]:
def train_matd3_experiment(
    env,
    total_steps,
    seed=None,
    run_dir=None,
    run_name=None,
    resume_checkpoint=None,
    device=None,
    enable_csv=None,
    enable_tensorboard=None,
    enable_wandb=None,
):
    if not isinstance(
        env,
        UAVSearchEnv,
    ):
        raise TypeError(
            "env must be UAVSearchEnv"
        )

    total_steps = int(
        total_steps
    )

    if total_steps < 1:
        raise ValueError(
            "total_steps must be >= 1"
        )

    if seed is None:
        seed = int(
            CONFIG["seed"]
        )

    seed = int(seed)

    if run_name is None:
        run_name = (
            make_experiment_run_name(
                "matd3",
                seed,
            )
        )

    if run_dir is None:
        if resume_checkpoint is not None:
            run_dir = (
                Path(
                    resume_checkpoint
                )
                .resolve()
                .parent
                .parent
            )
        else:
            run_dir = (
                Path(
                    CONFIG[
                        "matd3_training_output_dir"
                    ]
                )
                / run_name
            )

    run_dir = Path(
        run_dir
    )
    checkpoint_dir = (
        run_dir / "checkpoints"
    )
    checkpoint_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    set_seed(seed)

    if resume_checkpoint is None:
        trainer = HybridMATD3(
            env,
            device=device,
            seed=seed,
        )
        replay_buffer = (
            trainer.make_replay_buffer(
                seed=seed
            )
        )
        training_state = {
            "seed": seed,
            "global_step": 0,
            "episode_index": 0,
            "best_evaluation": None,
            "last_eval_step": 0,
            "last_checkpoint_step": 0,
            "training_rng_state_json": None,
            "resume_safe": True,
        }
    else:
        restored = (
            load_matd3_checkpoint(
                resume_checkpoint,
                env,
                device=device,
                load_replay=True,
                restore_torch_rng=True,
            )
        )
        trainer = restored[
            "trainer"
        ]
        replay_buffer = restored[
            "replay_buffer"
        ]
        training_state = restored[
            "training_state"
        ]

        if training_state.get("vectorized", False):
            raise ValueError(
                "vector checkpoint requires the saved multi-env runtime; "
                "do not resume it through the single-env loop"
            )
        _validate_resume_config(
            restored["config_snapshot"],
            "matd3",
        )

        if replay_buffer is None:
            raise ValueError(
                "resume checkpoint does not "
                "contain replay state; set "
                "training_checkpoint_include_replay=True"
            )

        if not bool(
            training_state.get(
                "resume_safe",
                False,
            )
        ):
            raise ValueError(
                "checkpoint is not marked "
                "resume_safe"
            )

        if int(
            training_state.get(
                "seed",
                seed,
            )
        ) != seed:
            raise ValueError(
                "resume seed does not match "
                "checkpoint seed"
            )

    global_step = int(
        training_state.get(
            "global_step",
            0,
        )
    )
    episode_index = int(
        training_state.get(
            "episode_index",
            0,
        )
    )
    best_evaluation = copy.deepcopy(
        training_state.get(
            "best_evaluation"
        )
    )
    last_eval_step = int(
        training_state.get(
            "last_eval_step",
            0,
        )
    )
    last_checkpoint_step = int(
        training_state.get(
            "last_checkpoint_step",
            0,
        )
    )

    if global_step > total_steps:
        raise ValueError(
            "total_steps is smaller than "
            "checkpoint global_step"
        )

    rng = np.random.default_rng(
        seed
    )

    saved_rng_state = (
        training_state.get(
            "training_rng_state_json"
        )
    )

    if saved_rng_state:
        rng.bit_generator.state = (
            json.loads(
                saved_rng_state
            )
        )

    logger = MASACExperimentLogger(
        run_dir=run_dir,
        run_name=run_name,
        enable_csv=enable_csv,
        enable_tensorboard=(
            enable_tensorboard
        ),
        enable_wandb=enable_wandb,
        wandb_entity=CONFIG.get(
            "training_wandb_entity"
        ),
        wandb_project=CONFIG[
            "matd3_wandb_project"
        ],
        algorithm="matd3",
        seed=seed,
        backend_name=(
            env.backend_name
        ),
        config_snapshot=(
            _checkpoint_config_snapshot()
        ),
    )

    eval_env = UAVSearchEnv(
        backend_name=env.backend_name,
        network_backend_kwargs=(
            env.network_backend_kwargs
        ),
    )

    observations, info = env.reset(
        seed=(
            seed + episode_index
        )
    )
    state = info[
        "state"
    ]

    episode_return = 0.0
    episode_length = 0
    episode_diagnostics = (
        new_episode_diagnostics()
    )
    completed_episodes = []
    latest_update_metrics = None
    latest_evaluation = None

    learning_starts = int(
        CONFIG[
            "matd3_learning_starts"
        ]
    )
    batch_size = int(
        CONFIG[
            "matd3_batch_size"
        ]
    )
    updates_per_step = int(
        CONFIG[
            "matd3_updates_per_step"
        ]
    )
    log_interval = max(
        1,
        int(
            CONFIG[
                "training_log_interval_steps"
            ]
        ),
    )
    eval_interval = max(
        1,
        int(
            CONFIG[
                "training_eval_interval_steps"
            ]
        ),
    )
    checkpoint_interval = max(
        1,
        int(
            CONFIG[
                "training_checkpoint_interval_steps"
            ]
        ),
    )
    eval_episodes = max(
        1,
        int(
            CONFIG[
                "training_eval_episodes"
            ]
        ),
    )

    try:
        while global_step < total_steps:
            next_global_step = (
                global_step + 1
            )

            if (
                next_global_step
                <= learning_starts
            ):
                actions = (
                    sample_valid_random_actions(
                        observations,
                        env.agent_ids,
                        rng,
                    )
                )
            else:
                actions = (
                    trainer.select_actions(
                        observations,
                        deterministic=False,
                        exploration_step=(
                            next_global_step
                        ),
                    )
                )

            (
                next_observations,
                reward,
                terminated,
                truncated,
                next_info,
            ) = env.step(
                actions
            )

            next_state = next_info[
                "state"
            ]

            add_environment_transition_to_replay(
                replay_buffer,
                observations,
                state,
                actions,
                reward,
                next_observations,
                next_state,
                terminated,
                truncated,
                env.agent_ids,
            )

            global_step = (
                next_global_step
            )
            episode_return += float(
                reward
            )
            episode_length += 1

            update_episode_diagnostics(
                episode_diagnostics,
                next_info,
                env=env,
            )

            if (
                global_step
                > learning_starts
                and len(
                    replay_buffer
                )
                >= batch_size
            ):
                for _ in range(
                    updates_per_step
                ):
                    latest_update_metrics = (
                        trainer.update(
                            replay_buffer,
                            batch_size=(
                                batch_size
                            ),
                        )
                    )

            if (
                latest_update_metrics
                is not None
                and global_step
                % log_interval
                == 0
            ):
                logger.log_metrics(
                    "train",
                    filter_training_metrics_for_wandb(
                        "matd3",
                        latest_update_metrics,
                    ),
                    step=global_step,
                    episode=episode_index,
                )

            observations = (
                next_observations
            )
            state = next_state

            should_evaluate = (
                global_step
                - last_eval_step
                >= eval_interval
                or global_step
                >= total_steps
            )

            if should_evaluate:
                evaluation_results = (
                    evaluate_matd3(
                        eval_env,
                        trainer,
                        episodes=(
                            eval_episodes
                        ),
                        seed=(
                            seed
                            + int(
                                CONFIG[
                                    "training_eval_seed_offset"
                                ]
                            )
                        ),
                    )
                )
                latest_evaluation = (
                    summarize_evaluation_results(
                        evaluation_results
                    )
                )

                logger.log_metrics(
                    "evaluation",
                    latest_evaluation,
                    step=global_step,
                    episode=episode_index,
                )
                logger.update_summary(
                    latest_evaluation,
                    prefix=(
                        "evaluation_latest"
                    ),
                )

                last_eval_step = (
                    global_step
                )

                if evaluation_is_better(
                    latest_evaluation,
                    best_evaluation,
                ):
                    best_evaluation = (
                        copy.deepcopy(
                            latest_evaluation
                        )
                    )

                    best_checkpoint_path = (
                        save_matd3_checkpoint(
                            checkpoint_dir
                            / "best.pt",
                            trainer,
                            replay_buffer=(
                                replay_buffer
                            ),
                            training_state={
                                "seed": seed,
                                "global_step": (
                                    global_step
                                ),
                                "episode_index": (
                                    episode_index
                                ),
                                "best_evaluation": (
                                    best_evaluation
                                ),
                                "last_eval_step": (
                                    last_eval_step
                                ),
                                "last_checkpoint_step": (
                                    last_checkpoint_step
                                ),
                                "training_rng_state_json": (
                                    json.dumps(
                                        rng
                                        .bit_generator
                                        .state
                                    )
                                ),
                                "resume_safe": False,
                            },
                            include_replay=False,
                        )
                    )
                    logger.update_summary(
                        best_evaluation,
                        prefix="best",
                    )
                    logger.log_model_artifact(
                        best_checkpoint_path,
                        aliases=[
                            "best",
                        ],
                        metadata=(
                            best_evaluation
                        ),
                    )

            if not (
                terminated
                or truncated
            ):
                continue

            episode_metrics = (
                finalize_episode_diagnostics(
                    episode_diagnostics,
                    env,
                    episode_return=(
                        episode_return
                    ),
                    episode_length=(
                        episode_length
                    ),
                    success=(
                        next_info[
                            "success"
                        ]
                    ),
                )
            )

            logger.log_metrics(
                "episode",
                episode_metrics,
                step=global_step,
                episode=episode_index,
            )

            completed_episodes.append(
                {
                    "episode": (
                        episode_index
                    ),
                    **episode_metrics,
                    "terminated": bool(
                        terminated
                    ),
                    "truncated": bool(
                        truncated
                    ),
                }
            )

            episode_index += 1

            should_checkpoint = (
                global_step
                - last_checkpoint_step
                >= checkpoint_interval
                or global_step
                >= total_steps
            )

            if should_checkpoint:
                last_checkpoint_step = (
                    global_step
                )
                include_replay = bool(
                    CONFIG[
                        "training_checkpoint_include_replay"
                    ]
                )

                save_matd3_checkpoint(
                    checkpoint_dir
                    / "latest.pt",
                    trainer,
                    replay_buffer=(
                        replay_buffer
                    ),
                    training_state={
                        "seed": seed,
                        "global_step": (
                            global_step
                        ),
                        "episode_index": (
                            episode_index
                        ),
                        "best_evaluation": (
                            copy.deepcopy(
                                best_evaluation
                            )
                        ),
                        "last_eval_step": (
                            last_eval_step
                        ),
                        "last_checkpoint_step": (
                            last_checkpoint_step
                        ),
                        "training_rng_state_json": (
                            json.dumps(
                                rng
                                .bit_generator
                                .state
                            )
                        ),
                        "resume_safe": (
                            include_replay
                        ),
                    },
                    include_replay=(
                        include_replay
                    ),
                )

            episode_return = 0.0
            episode_length = 0
            episode_diagnostics = (
                new_episode_diagnostics()
            )

            if global_step < total_steps:
                observations, info = (
                    env.reset(
                        seed=(
                            seed
                            + episode_index
                        )
                    )
                )
                state = info[
                    "state"
                ]

        final_state = {
            "seed": seed,
            "global_step": (
                global_step
            ),
            "episode_index": (
                episode_index
            ),
            "best_evaluation": (
                copy.deepcopy(
                    best_evaluation
                )
            ),
            "last_eval_step": (
                last_eval_step
            ),
            "last_checkpoint_step": (
                last_checkpoint_step
            ),
            "training_rng_state_json": (
                json.dumps(
                    rng
                    .bit_generator
                    .state
                )
            ),
            "resume_safe": False,
        }

        final_checkpoint_path = (
            save_matd3_checkpoint(
                checkpoint_dir
                / "final.pt",
                trainer,
                replay_buffer=(
                    replay_buffer
                ),
                training_state=(
                    final_state
                ),
                include_replay=False,
            )
        )
        logger.log_model_artifact(
            final_checkpoint_path,
            aliases=[
                "final",
            ],
            metadata={
                "global_step": (
                    global_step
                ),
                "episode_index": (
                    episode_index
                ),
            },
        )
        logger.update_summary(
            {
                "global_step": (
                    global_step
                ),
                "episodes_completed": (
                    episode_index
                ),
                "trainer_updates": int(
                    trainer.update_count
                ),
            },
            prefix="training_final",
        )

        return {
            "trainer": trainer,
            "replay_buffer": replay_buffer,
            "completed_episodes": (
                completed_episodes
            ),
            "latest_update_metrics": (
                latest_update_metrics
            ),
            "latest_evaluation": (
                latest_evaluation
            ),
            "best_evaluation": (
                best_evaluation
            ),
            "training_state": (
                final_state
            ),
            "logger_status": (
                logger.status()
            ),
            "run_dir": run_dir,
            "checkpoint_dir": (
                checkpoint_dir
            ),
        }
    finally:
        eval_env.close()
        logger.close()


In [ ]:
def running_on_kaggle():
    return bool(
        os.environ.get(
            "KAGGLE_KERNEL_RUN_TYPE"
        )
        or Path(
            "/kaggle/working"
        ).is_dir()
    )


In [ ]:
def kaggle_runtime_status():
    cuda_available = bool(
        torch.cuda.is_available()
    )
    device_count = (
        torch.cuda.device_count()
        if cuda_available
        else 0
    )
    gpu_names = [
        torch.cuda.get_device_name(
            index
        )
        for index in range(
            device_count
        )
    ]

    return {
        "running_on_kaggle": (
            running_on_kaggle()
        ),
        "cuda_available": (
            cuda_available
        ),
        "cuda_device_count": int(
            device_count
        ),
        "device": (
            "cuda"
            if cuda_available
            else "cpu"
        ),
        "gpu_names": gpu_names,
        "wandb_api_key_present": bool(
            os.environ.get(
                "WANDB_API_KEY"
            )
        ),
    }


In [ ]:
def configure_kaggle_wandb(
    secret_names=None,
    verify_login=True,
):
    result = {
        "running_on_kaggle": (
            running_on_kaggle()
        ),
        "configured": False,
        "secret_name": None,
        "error": None,
    }

    if not result[
        "running_on_kaggle"
    ]:
        result["error"] = (
            "not running inside a "
            "Kaggle notebook"
        )
        return result

    if secret_names is None:
        secret_names = CONFIG[
            "kaggle_wandb_secret_names"
        ]

    try:
        from kaggle_secrets import (
            UserSecretsClient,
        )

        client = UserSecretsClient()
    except Exception as exc:  # noqa: BLE001 - Kaggle-only optional API
        result["error"] = (
            f"{type(exc).__name__}: "
            f"{exc}"
        )
        return result

    api_key = os.environ.get(
        "WANDB_API_KEY"
    )
    selected_name = None

    if not api_key:
        for secret_name in (
            secret_names
        ):
            try:
                candidate = (
                    client.get_secret(
                        str(
                            secret_name
                        )
                    )
                )
            except Exception:  # noqa: BLE001,S112 - try the next configured label
                continue

            if candidate:
                api_key = candidate
                selected_name = str(
                    secret_name
                )
                break

    if not api_key:
        result["error"] = (
            "no attached W&B secret "
            "was found"
        )
        return result

    os.environ[
        "WANDB_API_KEY"
    ] = api_key

    try:
        wandb = importlib.import_module(
            "wandb"
        )

        if verify_login:
            wandb.login(
                key=api_key,
                relogin=True,
                verify=True,
            )

        result["configured"] = True
        result["secret_name"] = (
            selected_name
        )
    except Exception as exc:  # noqa: BLE001 - optional logger must not crash setup
        result["error"] = (
            f"{type(exc).__name__}: "
            f"{exc}"
        )

    return result


In [ ]:
def prepare_kaggle_training(
    algorithm="matd3",
    configure_wandb=True,
):
    normalized = str(
        algorithm
    ).strip().lower()

    if normalized not in {
        "matd3",
        "masac",
    }:
        raise ValueError(
            "algorithm must be "
            "'matd3' or 'masac'"
        )

    status = (
        kaggle_runtime_status()
    )

    if running_on_kaggle():
        output_root = Path(
            "/kaggle/working"
        )

        if normalized == "matd3":
            CONFIG[
                "matd3_training_output_dir"
            ] = str(
                output_root
                / "outputs"
                / "matd3"
            )
        else:
            CONFIG[
                "training_output_dir"
            ] = str(
                output_root
                / "outputs"
                / "masac"
            )

    wandb_status = None

    if configure_wandb:
        wandb_status = (
            configure_kaggle_wandb()
        )

        if (
            wandb_status.get(
                "configured"
            )
        ):
            CONFIG[
                "training_enable_wandb"
            ] = True
            CONFIG[
                "training_wandb_mode"
            ] = "online"

    return {
        **status,
        "algorithm": normalized,
        "wandb": wandb_status,
        "output_dir": (
            CONFIG[
                "matd3_training_output_dir"
            ]
            if normalized == "matd3"
            else CONFIG[
                "training_output_dir"
            ]
        ),
    }


In [ ]:
def run_kaggle_matd3_experiment(
    total_steps=100_000,
    seed=None,
    run_name=None,
    backend_name="simple",
    enable_wandb=True,
):
    if not running_on_kaggle():
        raise RuntimeError(
            "run_kaggle_matd3_experiment "
            "must be called inside a "
            "Kaggle notebook"
        )

    if seed is None:
        seed = int(
            CONFIG["seed"]
        )

    seed = int(seed)

    if run_name is None:
        run_name = (
            make_experiment_run_name(
                "matd3",
                seed,
            )
        )

    preparation = (
        prepare_kaggle_training(
            algorithm="matd3",
            configure_wandb=(
                enable_wandb
            ),
        )
    )

    if (
        enable_wandb
        and not preparation[
            "wandb"
        ].get(
            "configured",
            False,
        )
    ):
        raise RuntimeError(
            "W&B was requested but no "
            "attached Kaggle secret could "
            "be configured. Add a Kaggle "
            "Secret named WANDB_API_KEY "
            "or wandb_key."
        )

    run_dir = (
        Path(
            preparation[
                "output_dir"
            ]
        )
        / run_name
    )

    result = run_training_experiment(
        "matd3",
        total_steps=total_steps,
        seed=seed,
        backend_name=backend_name,
        run_name=run_name,
        run_dir=run_dir,
        device=preparation[
            "device"
        ],
        enable_csv=True,
        enable_tensorboard=(
            CONFIG[
                "training_enable_tensorboard"
            ]
        ),
        enable_wandb=enable_wandb,
    )

    return {
        **result,
        "kaggle_status": preparation,
    }


In [ ]:
def _select_vector_policy_actions(
    trainer,
    observations_by_env,
    algorithm,
    exploration_steps,
):
    num_envs = len(observations_by_env)
    arrays = [
        observations_to_masac_arrays(
            observations,
            trainer.agent_ids,
        )
        for observations in observations_by_env
    ]
    observation_vectors = np.stack(
        [item[0] for item in arrays],
        axis=0,
    )
    destination_masks = np.stack(
        [item[1] for item in arrays],
        axis=0,
    )
    observation_tensor = torch.as_tensor(
        observation_vectors,
        dtype=torch.float32,
        device=trainer.device,
    )
    mask_tensor = torch.as_tensor(
        destination_masks,
        dtype=torch.float32,
        device=trainer.device,
    )

    if algorithm == "masac":
        with torch.no_grad():
            sampled = trainer._sample_joint_policy(
                observation_tensor,
                mask_tensor,
                deterministic=False,
            )
        continuous = (
            sampled["continuous"]
            .cpu()
            .numpy()
            .astype(np.float32)
        )
        destination_indices = (
            sampled["destination_indices"]
            .cpu()
            .numpy()
            .astype(np.int64)
        )
    else:
        flat_observations = observation_tensor.reshape(
            num_envs * trainer.num_agents,
            trainer.observation_dim,
        )
        flat_masks = mask_tensor.reshape(
            num_envs * trainer.num_agents,
            trainer.discrete_dim,
        )
        with torch.no_grad():
            sampled = trainer.actor.actions(
                flat_observations,
                flat_masks,
                straight_through=False,
            )
        continuous = (
            sampled["continuous"]
            .reshape(
                num_envs,
                trainer.num_agents,
                trainer.continuous_dim,
            )
            .cpu()
            .numpy()
            .astype(np.float32)
        )
        destination_indices = (
            sampled["destination_indices"]
            .reshape(
                num_envs,
                trainer.num_agents,
            )
            .cpu()
            .numpy()
            .astype(np.int64)
        )
        noise = trainer.rng.normal(
            loc=0.0,
            scale=trainer.exploration_noise,
            size=continuous.shape,
        ).astype(np.float32)
        continuous = np.clip(
            continuous + noise,
            -1.0,
            1.0,
        ).astype(np.float32)

        for env_index in range(num_envs):
            epsilon = trainer._discrete_epsilon(
                exploration_steps[env_index]
            )
            for agent_index in range(
                trainer.num_agents
            ):
                if trainer.rng.random() >= epsilon:
                    continue
                valid = np.flatnonzero(
                    destination_masks[
                        env_index,
                        agent_index,
                    ]
                    > 0.5
                )
                destination_indices[
                    env_index,
                    agent_index,
                ] = int(
                    trainer.rng.choice(valid)
                )
        trainer.action_count += num_envs

    return [
        masac_arrays_to_env_actions(
            continuous[index],
            destination_indices[index],
            trainer.agent_ids,
        )
        for index in range(num_envs)
    ]


In [ ]:
def _vector_batch_item(value, index, num_envs):
    if isinstance(value, dict):
        result = {}
        for key, item in value.items():
            if str(key).startswith("_"):
                continue
            mask = value.get(f"_{key}")
            if (
                isinstance(mask, np.ndarray)
                and mask.shape == (num_envs,)
                and not bool(mask[index])
            ):
                continue
            result[key] = _vector_batch_item(
                item,
                index,
                num_envs,
            )
        return result

    if (
        isinstance(value, np.ndarray)
        and value.ndim >= 1
        and value.shape[0] == num_envs
    ):
        return copy.deepcopy(value[index])

    return copy.deepcopy(value)


In [ ]:
def _vector_observation_at(
    observations,
    index,
    num_envs,
):
    return _vector_batch_item(
        observations,
        index,
        num_envs,
    )


In [ ]:
def _vector_info_at(
    infos,
    index,
    num_envs,
):
    return _vector_batch_item(
        infos,
        index,
        num_envs,
    )


In [ ]:
def _stack_vector_actions(
    actions_by_env,
    agent_ids,
):
    return {
        agent_id: {
            "motion": np.stack(
                [
                    np.asarray(
                        actions[agent_id]["motion"],
                        dtype=np.float32,
                    )
                    for actions in actions_by_env
                ],
                axis=0,
            ),
            "destination": np.asarray(
                [
                    int(
                        actions[agent_id][
                            "destination"
                        ]
                    )
                    for actions in actions_by_env
                ],
                dtype=np.int64,
            ),
            "power": np.stack(
                [
                    np.asarray(
                        actions[agent_id]["power"],
                        dtype=np.float32,
                    )
                    for actions in actions_by_env
                ],
                axis=0,
            ),
        }
        for agent_id in agent_ids
    }


In [ ]:
def _new_vector_episode_tracker(
    episode_id,
):
    return {
        "episode_id": int(episode_id),
        "return": 0.0,
        "length": 0,
        "diagnostics": (
            new_episode_diagnostics()
        ),
        "confirmed_target_ids": set(),
        "delivered_target_ids": set(),
    }


In [ ]:
def _update_vector_episode_tracker(
    tracker,
    info,
    reward,
):
    tracker["return"] += float(reward)
    tracker["length"] += 1

    update_episode_diagnostics(
        tracker["diagnostics"],
        info,
    )

    tracker[
        "confirmed_target_ids"
    ].update(
        int(target_id)
        for target_id in info.get(
            "newly_confirmed_target_ids",
            [],
        )
    )
    tracker[
        "delivered_target_ids"
    ].update(
        int(target_id)
        for target_id in info.get(
            "newly_delivered_target_ids",
            [],
        )
    )


In [ ]:
def _finalize_vector_episode_tracker(
    tracker,
    success,
):
    diagnostics = tracker["diagnostics"]
    target_count = max(
        1,
        int(CONFIG["num_targets"]),
    )
    possible_agent_steps = max(
        1,
        int(tracker["length"])
        * int(CONFIG["num_uavs"]),
    )

    metrics = {
        "return": float(
            tracker["return"]
        ),
        "length": int(
            tracker["length"]
        ),
        "success": float(
            bool(success)
        ),
        "delivery_rate": float(
            len(
                tracker[
                    "delivered_target_ids"
                ]
            )
            / target_count
        ),
        "confirmation_rate": float(
            len(
                tracker[
                    "confirmed_target_ids"
                ]
            )
            / target_count
        ),
        "false_confirmations": int(
            diagnostics[
                "false_confirmations"
            ]
        ),
        "expired_reports": int(
            diagnostics["expired_reports"]
        ),
        "dropped_reports": int(
            diagnostics["dropped_reports"]
        ),
        "obstacle_block_rate": float(
            diagnostics["obstacle_blocks"]
            / possible_agent_steps
        ),
        "peer_safety_block_rate": float(
            diagnostics["peer_safety_blocks"]
            / possible_agent_steps
        ),
        "boundary_clip_rate": float(
            diagnostics["boundary_clips"]
            / possible_agent_steps
        ),
        "total_energy_j": float(
            diagnostics["total_energy_j"]
        ),
        "communication_energy_j": float(
            diagnostics[
                "communication_energy_j"
            ]
        ),
        "information_gain_bits": float(
            diagnostics[
                "information_gain_bits"
            ]
        ),
    }
    metrics.update(
        extract_network_kpis(
            diagnostics[
                "network_snapshot"
            ]
        )
    )
    return metrics


In [ ]:
def _vector_training_is_enabled(
    env,
    device=None,
):
    vector_runtime = (
        resolve_training_vector_runtime(
            device
        )
    )
    return bool(
        env.backend_name == "simple"
        and int(
            vector_runtime["num_envs"]
        )
        > 1
    )


In [ ]:
def _count_vector_updates_due(
    previous_global_step,
    num_envs,
    learning_starts,
    replay_len_before,
    replay_capacity,
    batch_size,
    updates_per_step,
):
    updates_due = 0

    for offset in range(
        int(num_envs)
    ):
        transition_step = (
            int(previous_global_step)
            + offset
            + 1
        )
        replay_size_after = min(
            int(replay_capacity),
            int(replay_len_before)
            + offset
            + 1,
        )

        if (
            transition_step
            > int(learning_starts)
            and replay_size_after
            >= int(batch_size)
        ):
            updates_due += int(
                updates_per_step
            )

    return int(updates_due)


In [ ]:
def _vector_system_metrics(
    trainer,
    global_step,
    wall_start,
    evaluation_seconds,
    env_collection_seconds,
    env_wait_seconds,
    policy_inference_seconds,
    update_seconds,
    num_envs,
    overlap,
    initial_global_step=0,
    prior_metrics=None,
):
    if prior_metrics is None:
        prior_metrics = {}

    session_wall_seconds = max(
        np.finfo(np.float64).eps,
        time.perf_counter()
        - float(wall_start),
    )
    session_core_seconds = max(
        np.finfo(np.float64).eps,
        session_wall_seconds
        - float(evaluation_seconds),
    )

    total_wall_seconds = (
        float(
            prior_metrics.get(
                "training_wall_time_seconds",
                0.0,
            )
        )
        + session_wall_seconds
    )
    total_evaluation_seconds = (
        float(
            prior_metrics.get(
                "evaluation_seconds",
                0.0,
            )
        )
        + float(evaluation_seconds)
    )
    core_training_seconds = max(
        np.finfo(np.float64).eps,
        total_wall_seconds
        - total_evaluation_seconds,
    )
    session_transitions = max(
        0,
        int(global_step)
        - int(initial_global_step),
    )

    metrics = {
        "transitions_per_second": (
            float(global_step)
            / core_training_seconds
        ),
        "session_transitions_per_second": (
            float(session_transitions)
            / session_core_seconds
        ),
        "training_wall_time_seconds": (
            total_wall_seconds
        ),
        "core_training_seconds": (
            core_training_seconds
        ),
        "evaluation_seconds": (
            total_evaluation_seconds
        ),
        "env_collection_seconds": (
            float(
                prior_metrics.get(
                    "env_collection_seconds",
                    0.0,
                )
            )
            + float(
                env_collection_seconds
            )
        ),
        "env_wait_seconds": (
            float(
                prior_metrics.get(
                    "env_wait_seconds",
                    0.0,
                )
            )
            + float(env_wait_seconds)
        ),
        "policy_inference_seconds": (
            float(
                prior_metrics.get(
                    "policy_inference_seconds",
                    0.0,
                )
            )
            + float(
                policy_inference_seconds
            )
        ),
        "update_seconds": (
            float(
                prior_metrics.get(
                    "update_seconds",
                    0.0,
                )
            )
            + float(update_seconds)
        ),
        "num_envs": float(num_envs),
        "overlap_enabled": float(
            bool(overlap)
        ),
    }

    device = getattr(
        trainer,
        "device",
        None,
    )

    if (
        isinstance(
            device,
            torch.device,
        )
        and device.type == "cuda"
    ):
        device_index = (
            device.index
            if device.index is not None
            else torch.cuda.current_device()
        )
        metrics[
            "gpu_memory_allocated_mb"
        ] = float(
            torch.cuda.memory_allocated(
                device_index
            )
            / (1024.0 ** 2)
        )
        metrics[
            "gpu_peak_memory_allocated_mb"
        ] = float(
            torch.cuda.max_memory_allocated(
                device_index
            )
            / (1024.0 ** 2)
        )

    return metrics


In [ ]:
def _capture_vector_env_states(
    vector_env,
):
    if isinstance(
        vector_env,
        TorchThreadedSimpleVectorEnv,
    ):
        return [
            env.checkpoint_state()
            for env in vector_env.envs
        ]

    # Forked CPU workers must not enter the torch runtime. Environment
    # checkpoint arrays are therefore encoded as dtype/shape/raw bytes.
    return list(
        vector_env.call(
            "checkpoint_state"
        )
    )


In [ ]:
def _restore_vector_env_states(
    vector_env,
    env_states,
    num_envs,
):
    if len(env_states) != int(
        num_envs
    ):
        raise ValueError(
            "checkpoint environment count "
            "does not match vector runtime"
        )

    if isinstance(
        vector_env,
        TorchThreadedSimpleVectorEnv,
    ):
        restored = [
            env.restore_checkpoint_state(
                state
            )
            for env, state
            in zip(
                vector_env.envs,
                env_states,
            )
        ]
        vector_env._last_observations = [
            item[0]
            for item in restored
        ]
        vector_env._last_infos = [
            item[1]
            for item in restored
        ]
    else:
        vector_env.set_attr(
            "_pending_checkpoint_state",
            tuple(env_states),
        )
        restored = list(
            vector_env.call(
                "restore_pending_checkpoint_state"
            )
        )

    observations = _stack_vector_dicts(
        [
            item[0]
            for item in restored
        ]
    )
    states = np.stack(
        [
            np.asarray(
                item[1]["state"],
                dtype=np.float32,
            )
            for item in restored
        ],
        axis=0,
    )
    return observations, states


In [ ]:
def _make_vector_checkpoint_state(
    *,
    algorithm,
    seed,
    global_step,
    best_evaluation,
    latest_evaluation,
    last_eval_step,
    last_checkpoint_step,
    rng,
    num_envs,
    vector_backend,
    vector_env,
    trackers,
    completed_episodes,
    next_episode_id,
    system_metrics,
    wandb_run_id,
    resume_safe,
):
    return {
        "seed": int(seed),
        "algorithm": str(algorithm),
        "global_step": int(
            global_step
        ),
        "episode_index": len(
            completed_episodes
        ),
        "completed_episodes": (
            _checkpoint_pack(
                completed_episodes
            )
        ),
        "trackers": _checkpoint_pack(
            trackers
        ),
        "next_episode_id": int(
            next_episode_id
        ),
        "best_evaluation": (
            copy.deepcopy(
                best_evaluation
            )
        ),
        "latest_evaluation": (
            copy.deepcopy(
                latest_evaluation
            )
        ),
        "last_eval_step": int(
            last_eval_step
        ),
        "last_checkpoint_step": int(
            last_checkpoint_step
        ),
        "training_rng_state_json": (
            json.dumps(
                rng.bit_generator.state
            )
        ),
        "vector_env_states": (
            _capture_vector_env_states(
                vector_env
            )
        ),
        "system_metrics": (
            copy.deepcopy(
                system_metrics
            )
        ),
        "wandb_run_id": wandb_run_id,
        "resume_safe": bool(
            resume_safe
        ),
        "vectorized": True,
        "num_envs": int(num_envs),
        "vector_backend": str(
            vector_backend
        ),
    }


In [ ]:
def train_vectorized_experiment(
    algorithm,
    env,
    total_steps,
    seed=None,
    run_dir=None,
    run_name=None,
    resume_checkpoint=None,
    device=None,
    enable_csv=None,
    enable_tensorboard=None,
    enable_wandb=None,
):
    normalized = str(
        algorithm
    ).strip().lower()

    if normalized not in {
        "masac",
        "matd3",
    }:
        raise ValueError(
            "algorithm must be "
            "'masac' or 'matd3'"
        )

    if not isinstance(
        env,
        UAVSearchEnv,
    ):
        raise TypeError(
            "env must be UAVSearchEnv"
        )

    if env.backend_name != "simple":
        raise ValueError(
            "vectorized training currently "
            "supports simple backend only"
        )

    total_steps = int(total_steps)
    vector_runtime = (
        resolve_training_vector_runtime(
            device
        )
    )
    num_envs = int(
        vector_runtime["num_envs"]
    )

    if total_steps < num_envs:
        raise ValueError(
            "total_steps must be >= "
            f"the selected num_envs ({num_envs})"
        )

    if total_steps % num_envs != 0:
        raise ValueError(
            "vectorized total_steps must be "
            "divisible by selected num_envs "
            f"({num_envs})"
        )

    if seed is None:
        seed = int(CONFIG["seed"])
    seed = int(seed)

    if resume_checkpoint is not None:
        resume_checkpoint = Path(
            resume_checkpoint
        )
        if run_dir is None:
            run_dir = (
                resume_checkpoint
                .resolve()
                .parent
                .parent
            )

    if run_name is None:
        if (
            resume_checkpoint is not None
            and run_dir is not None
        ):
            run_name = Path(
                run_dir
            ).name
        else:
            run_name = (
                make_experiment_run_name(
                    normalized,
                    seed,
                )
            )

    if run_dir is None:
        output_key = (
            "matd3_training_output_dir"
            if normalized == "matd3"
            else "training_output_dir"
        )
        run_dir = (
            Path(CONFIG[output_key])
            / run_name
        )

    run_dir = Path(run_dir)
    checkpoint_dir = (
        run_dir / "checkpoints"
    )
    checkpoint_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    context = str(
        CONFIG.get(
            "training_vector_context",
            "fork",
        )
    )
    shared_memory = bool(
        CONFIG.get(
            "training_vector_shared_memory",
            True,
        )
    )
    overlap = bool(
        vector_runtime["overlap"]
    )
    vector_backend = str(
        vector_runtime["backend"]
    )

    env_fns = [
        lambda: UAVSearchEnv(
            backend_name="simple",
            network_backend_kwargs=(
                env.network_backend_kwargs
            ),
        )
        for _ in range(num_envs)
    ]

    if vector_runtime[
        "use_torch_vector"
    ]:
        vector_env = (
            TorchThreadedSimpleVectorEnv(
                env_fns,
                sensing_device=(
                    vector_runtime[
                        "sensing_device"
                    ]
                ),
                sensing_dtype=(
                    vector_runtime[
                        "sensing_dtype"
                    ]
                ),
                sensing_batch_timeout_s=(
                    vector_runtime[
                        "sensing_batch_timeout_s"
                    ]
                ),
            )
        )
    else:
        vector_env = (
            gym.vector.AsyncVectorEnv(
                env_fns,
                context=context,
                shared_memory=(
                    shared_memory
                ),
                autoreset_mode=(
                    gym.vector
                    .AutoresetMode.DISABLED
                ),
            )
        )

    logger = None
    eval_env = None

    try:
        set_seed(seed)

        if normalized == "masac":
            batch_size = int(
                CONFIG[
                    "masac_batch_size"
                ]
            )
            learning_starts = int(
                CONFIG[
                    "masac_learning_starts"
                ]
            )
            updates_per_step = int(
                CONFIG[
                    "masac_updates_per_step"
                ]
            )
            evaluate_fn = (
                evaluate_masac
            )
            save_fn = (
                save_masac_checkpoint
            )
            load_fn = (
                load_masac_checkpoint
            )
            wandb_project = CONFIG[
                "training_wandb_project"
            ]
        else:
            batch_size = int(
                CONFIG[
                    "matd3_batch_size"
                ]
            )
            learning_starts = int(
                CONFIG[
                    "matd3_learning_starts"
                ]
            )
            updates_per_step = int(
                CONFIG[
                    "matd3_updates_per_step"
                ]
            )
            evaluate_fn = (
                evaluate_matd3
            )
            save_fn = (
                save_matd3_checkpoint
            )
            load_fn = (
                load_matd3_checkpoint
            )
            wandb_project = CONFIG[
                "matd3_wandb_project"
            ]

        resume_state = None
        if resume_checkpoint is None:
            if normalized == "masac":
                trainer = HybridMASAC(
                    env,
                    device=device,
                )
            else:
                trainer = HybridMATD3(
                    env,
                    device=device,
                    seed=seed,
                )
            replay_buffer = (
                trainer.make_replay_buffer(
                    seed=seed
                )
            )
        else:
            loaded = load_fn(
                resume_checkpoint,
                env,
                device=device,
                load_replay=True,
                restore_torch_rng=True,
            )
            trainer = loaded["trainer"]
            replay_buffer = loaded[
                "replay_buffer"
            ]
            resume_state = loaded[
                "training_state"
            ]

            _validate_resume_config(
                loaded["config_snapshot"],
                normalized,
            )
            saved_config = loaded["config_snapshot"]
            runtime_keys = (
                ("training_torch_sensing_dtype",
                 "training_torch_overlap_env_and_updates")
                if vector_backend == "torch_threaded"
                else ("training_overlap_env_and_updates",)
            )
            for key in runtime_keys:
                if saved_config.get(key) != CONFIG.get(key):
                    raise ValueError(
                        f"exact resume vector runtime mismatch: {key}"
                    )

            if replay_buffer is None:
                raise ValueError(
                    "resume checkpoint does "
                    "not contain replay state"
                )
            if not bool(
                resume_state.get(
                    "resume_safe",
                    False,
                )
            ):
                raise ValueError(
                    "checkpoint is not marked "
                    "resume_safe"
                )
            if not bool(
                resume_state.get(
                    "vectorized",
                    False,
                )
            ):
                raise ValueError(
                    "checkpoint is not a "
                    "vectorized training state"
                )
            if str(
                resume_state.get(
                    "algorithm",
                    "",
                )
            ) != normalized:
                raise ValueError(
                    "checkpoint algorithm "
                    "does not match"
                )
            if int(
                resume_state.get(
                    "seed",
                    -1,
                )
            ) != seed:
                raise ValueError(
                    "checkpoint seed does "
                    "not match"
                )
            if int(
                resume_state.get(
                    "num_envs",
                    -1,
                )
            ) != num_envs:
                raise ValueError(
                    "checkpoint num_envs "
                    "does not match runtime"
                )
            if str(
                resume_state.get(
                    "vector_backend",
                    "",
                )
            ) != vector_backend:
                raise ValueError(
                    "checkpoint vector backend "
                    "does not match runtime"
                )

        wandb_run_id = (
            None
            if resume_state is None
            else resume_state.get(
                "wandb_run_id"
            )
        )
        logger = MASACExperimentLogger(
            run_dir=run_dir,
            run_name=run_name,
            enable_csv=enable_csv,
            enable_tensorboard=(
                enable_tensorboard
            ),
            enable_wandb=(
                enable_wandb
            ),
            wandb_entity=CONFIG.get(
                "training_wandb_entity"
            ),
            wandb_project=(
                wandb_project
            ),
            wandb_run_id=(
                wandb_run_id
            ),
            wandb_resume=(
                "allow"
                if wandb_run_id
                else None
            ),
            algorithm=normalized,
            seed=seed,
            backend_name=(
                env.backend_name
            ),
            config_snapshot=(
                _checkpoint_config_snapshot()
            ),
        )

        eval_env = UAVSearchEnv(
            backend_name=(
                env.backend_name
            ),
            network_backend_kwargs=(
                env.network_backend_kwargs
            ),
        )
        rng = np.random.default_rng(
            seed
        )

        if resume_state is None:
            initial_episode_ids = list(
                range(num_envs)
            )
            next_episode_id = (
                num_envs
            )
            trackers = [
                _new_vector_episode_tracker(
                    episode_id
                )
                for episode_id
                in initial_episode_ids
            ]
            completed_episodes = []
            latest_evaluation = None
            best_evaluation = None
            global_step = 0
            last_eval_step = 0
            last_checkpoint_step = 0
            prior_system_metrics = {}

            observations, infos = (
                vector_env.reset(
                    seed=[
                        seed + episode_id
                        for episode_id
                        in initial_episode_ids
                    ]
                )
            )
            states = np.asarray(
                infos["state"],
                dtype=np.float32,
            ).copy()
        else:
            global_step = int(
                resume_state[
                    "global_step"
                ]
            )
            if global_step > total_steps:
                raise ValueError(
                    "total_steps is below "
                    "checkpoint global_step"
                )
            if global_step % num_envs != 0:
                raise ValueError(
                    "checkpoint global_step "
                    "is not a vector-step boundary"
                )

            rng.bit_generator.state = (
                json.loads(
                    resume_state[
                        "training_rng_state_json"
                    ]
                )
            )
            trackers = _checkpoint_unpack(
                resume_state["trackers"]
            )
            completed_episodes = (
                _checkpoint_unpack(
                    resume_state[
                        "completed_episodes"
                    ]
                )
            )
            next_episode_id = int(
                resume_state[
                    "next_episode_id"
                ]
            )
            latest_evaluation = (
                copy.deepcopy(
                    resume_state.get(
                        "latest_evaluation"
                    )
                )
            )
            best_evaluation = (
                copy.deepcopy(
                    resume_state.get(
                        "best_evaluation"
                    )
                )
            )
            last_eval_step = int(
                resume_state.get(
                    "last_eval_step",
                    0,
                )
            )
            last_checkpoint_step = (
                int(
                    resume_state.get(
                        "last_checkpoint_step",
                        global_step,
                    )
                )
            )
            prior_system_metrics = (
                copy.deepcopy(
                    resume_state.get(
                        "system_metrics",
                        {},
                    )
                )
            )

            if len(trackers) != num_envs:
                raise ValueError(
                    "checkpoint tracker count "
                    "does not match runtime"
                )
            observations, states = (
                _restore_vector_env_states(
                    vector_env,
                    resume_state[
                        "vector_env_states"
                    ],
                    num_envs,
                )
            )

        initial_global_step = (
            global_step
        )
        latest_update_metrics = None
        pending_updates = 0

        log_interval = max(
            1,
            int(
                CONFIG[
                    "training_log_interval_steps"
                ]
            ),
        )
        eval_interval = max(
            1,
            int(
                CONFIG[
                    "training_eval_interval_steps"
                ]
            ),
        )
        checkpoint_interval = max(
            1,
            int(
                CONFIG[
                    "training_checkpoint_interval_steps"
                ]
            ),
        )
        eval_episodes = max(
            1,
            int(
                CONFIG[
                    "training_eval_episodes"
                ]
            ),
        )
        include_replay = bool(
            CONFIG[
                "training_checkpoint_include_replay"
            ]
        )

        wall_start = time.perf_counter()
        evaluation_seconds = 0.0
        env_collection_seconds = 0.0
        env_wait_seconds = 0.0
        policy_inference_seconds = 0.0
        update_seconds = 0.0

        while global_step < total_steps:
            observations_by_env = [
                _vector_observation_at(
                    observations,
                    index,
                    num_envs,
                )
                for index in range(
                    num_envs
                )
            ]

            exploration_steps = [
                global_step + index + 1
                for index in range(
                    num_envs
                )
            ]
            policy_inference_start = (
                time.perf_counter()
                if any(
                    step > learning_starts
                    for step
                    in exploration_steps
                )
                else None
            )

            if all(
                step <= learning_starts
                for step
                in exploration_steps
            ):
                actions_by_env = [
                    sample_valid_random_actions(
                        observations_by_env[
                            index
                        ],
                        env.agent_ids,
                        rng,
                    )
                    for index in range(
                        num_envs
                    )
                ]
            elif all(
                step > learning_starts
                for step
                in exploration_steps
            ):
                actions_by_env = (
                    _select_vector_policy_actions(
                        trainer,
                        observations_by_env,
                        normalized,
                        exploration_steps,
                    )
                )
            else:
                actions_by_env = []
                for (
                    index,
                    env_observations,
                ) in enumerate(
                    observations_by_env
                ):
                    transition_step = (
                        exploration_steps[
                            index
                        ]
                    )
                    if (
                        transition_step
                        <= learning_starts
                    ):
                        actions = (
                            sample_valid_random_actions(
                                env_observations,
                                env.agent_ids,
                                rng,
                            )
                        )
                    elif normalized == "masac":
                        actions = (
                            trainer.select_actions(
                                env_observations,
                                deterministic=False,
                            )
                        )
                    else:
                        actions = (
                            trainer.select_actions(
                                env_observations,
                                deterministic=False,
                                exploration_step=(
                                    transition_step
                                ),
                            )
                        )
                    actions_by_env.append(
                        actions
                    )

            if (
                policy_inference_start
                is not None
            ):
                policy_inference_seconds += (
                    time.perf_counter()
                    - policy_inference_start
                )

            vector_actions = (
                _stack_vector_actions(
                    actions_by_env,
                    env.agent_ids,
                )
            )
            env_collection_start = (
                time.perf_counter()
            )

            if overlap:
                vector_env.step_async(
                    vector_actions
                )
            else:
                next_step_result = (
                    vector_env.step(
                        vector_actions
                    )
                )

            if (
                pending_updates > 0
                and len(replay_buffer)
                >= batch_size
            ):
                update_start = (
                    time.perf_counter()
                )
                for _ in range(
                    pending_updates
                ):
                    latest_update_metrics = (
                        trainer.update(
                            replay_buffer,
                            batch_size=(
                                batch_size
                            ),
                        )
                    )
                update_seconds += (
                    time.perf_counter()
                    - update_start
                )
                pending_updates = 0

            if overlap:
                wait_start = (
                    time.perf_counter()
                )
                next_step_result = (
                    vector_env.step_wait()
                )
                env_wait_seconds += (
                    time.perf_counter()
                    - wait_start
                )

            env_collection_seconds += (
                time.perf_counter()
                - env_collection_start
            )

            (
                next_observations,
                rewards,
                terminated,
                truncated,
                next_infos,
            ) = next_step_result
            next_states = np.asarray(
                next_infos["state"],
                dtype=np.float32,
            ).copy()

            previous_global_step = (
                global_step
            )
            replay_len_before = len(
                replay_buffer
            )

            for index in range(
                num_envs
            ):
                next_env_observations = (
                    _vector_observation_at(
                        next_observations,
                        index,
                        num_envs,
                    )
                )
                env_info = (
                    _vector_info_at(
                        next_infos,
                        index,
                        num_envs,
                    )
                )

                add_environment_transition_to_replay(
                    replay_buffer,
                    observations_by_env[
                        index
                    ],
                    states[index],
                    actions_by_env[index],
                    float(
                        rewards[index]
                    ),
                    next_env_observations,
                    next_states[index],
                    bool(
                        terminated[index]
                    ),
                    bool(
                        truncated[index]
                    ),
                    env.agent_ids,
                )
                _update_vector_episode_tracker(
                    trackers[index],
                    env_info,
                    rewards[index],
                )

            global_step += num_envs
            pending_updates += (
                _count_vector_updates_due(
                    previous_global_step=(
                        previous_global_step
                    ),
                    num_envs=num_envs,
                    learning_starts=(
                        learning_starts
                    ),
                    replay_len_before=(
                        replay_len_before
                    ),
                    replay_capacity=(
                        replay_buffer.capacity
                    ),
                    batch_size=batch_size,
                    updates_per_step=(
                        updates_per_step
                    ),
                )
            )

            if (
                not overlap
                and pending_updates > 0
            ):
                update_start = (
                    time.perf_counter()
                )
                for _ in range(
                    pending_updates
                ):
                    latest_update_metrics = (
                        trainer.update(
                            replay_buffer,
                            batch_size=(
                                batch_size
                            ),
                        )
                    )
                update_seconds += (
                    time.perf_counter()
                    - update_start
                )
                pending_updates = 0

            if (
                global_step
                % log_interval
                == 0
                or global_step
                >= total_steps
            ):
                logger.log_metrics(
                    "system",
                    _vector_system_metrics(
                        trainer=trainer,
                        global_step=(
                            global_step
                        ),
                        wall_start=(
                            wall_start
                        ),
                        evaluation_seconds=(
                            evaluation_seconds
                        ),
                        env_collection_seconds=(
                            env_collection_seconds
                        ),
                        env_wait_seconds=(
                            env_wait_seconds
                        ),
                        policy_inference_seconds=(
                            policy_inference_seconds
                        ),
                        update_seconds=(
                            update_seconds
                        ),
                        num_envs=num_envs,
                        overlap=overlap,
                        initial_global_step=(
                            initial_global_step
                        ),
                        prior_metrics=(
                            prior_system_metrics
                        ),
                    ),
                    step=global_step,
                    episode=len(
                        completed_episodes
                    ),
                )

            done_mask = np.logical_or(
                terminated,
                truncated,
            )

            for index in np.flatnonzero(
                done_mask
            ):
                index = int(index)
                env_info = (
                    _vector_info_at(
                        next_infos,
                        index,
                        num_envs,
                    )
                )
                episode_metrics = (
                    _finalize_vector_episode_tracker(
                        trackers[index],
                        env_info.get(
                            "success",
                            False,
                        ),
                    )
                )
                episode_id = int(
                    trackers[index][
                        "episode_id"
                    ]
                )

                logger.log_metrics(
                    "episode",
                    episode_metrics,
                    step=global_step,
                    episode=episode_id,
                )
                completed_episodes.append(
                    {
                        "episode": (
                            episode_id
                        ),
                        **episode_metrics,
                        "terminated": bool(
                            terminated[index]
                        ),
                        "truncated": bool(
                            truncated[index]
                        ),
                    }
                )

            observations = (
                next_observations
            )
            states = next_states

            if np.any(done_mask):
                reset_seeds = [
                    None
                ] * num_envs

                for index in np.flatnonzero(
                    done_mask
                ):
                    index = int(index)
                    reset_seeds[index] = (
                        seed
                        + next_episode_id
                    )
                    trackers[index] = (
                        _new_vector_episode_tracker(
                            next_episode_id
                        )
                    )
                    next_episode_id += 1

                observations, reset_infos = (
                    vector_env.reset(
                        seed=reset_seeds,
                        options={
                            "reset_mask": (
                                done_mask
                                .astype(
                                    np.bool_
                                )
                                .copy()
                            )
                        },
                    )
                )
                states = next_states.copy()
                reset_states = np.asarray(
                    reset_infos["state"],
                    dtype=np.float32,
                )
                reset_state_mask = np.asarray(
                    reset_infos.get(
                        "_state",
                        done_mask,
                    ),
                    dtype=np.bool_,
                )
                for index in np.flatnonzero(
                    reset_state_mask
                ):
                    states[int(index)] = (
                        reset_states[
                            int(index)
                        ]
                    )

            should_evaluate = bool(
                global_step
                - last_eval_step
                >= eval_interval
                or global_step
                >= total_steps
            )
            should_checkpoint = bool(
                global_step
                - last_checkpoint_step
                >= checkpoint_interval
                and global_step
                < total_steps
            )

            if (
                overlap
                and pending_updates > 0
                and (
                    should_evaluate
                    or should_checkpoint
                )
            ):
                update_start = (
                    time.perf_counter()
                )
                for _ in range(
                    pending_updates
                ):
                    latest_update_metrics = (
                        trainer.update(
                            replay_buffer,
                            batch_size=(
                                batch_size
                            ),
                        )
                    )
                update_seconds += (
                    time.perf_counter()
                    - update_start
                )
                pending_updates = 0

            if (
                latest_update_metrics
                is not None
                and (
                    global_step % log_interval == 0
                    or global_step >= total_steps
                )
            ):
                logger.log_metrics(
                    "train",
                    filter_training_metrics_for_wandb(
                        normalized,
                        latest_update_metrics,
                    ),
                    step=global_step,
                    episode=len(
                        completed_episodes
                    ),
                )

            if should_evaluate:
                evaluation_start = (
                    time.perf_counter()
                )
                evaluation_results = (
                    evaluate_fn(
                        eval_env,
                        trainer,
                        episodes=(
                            eval_episodes
                        ),
                        seed=(
                            seed
                            + int(
                                CONFIG[
                                    "training_eval_seed_offset"
                                ]
                            )
                        ),
                    )
                )
                latest_evaluation = (
                    summarize_evaluation_results(
                        evaluation_results
                    )
                )
                evaluation_seconds += (
                    time.perf_counter()
                    - evaluation_start
                )

                logger.log_metrics(
                    "evaluation",
                    latest_evaluation,
                    step=global_step,
                    episode=len(
                        completed_episodes
                    ),
                )
                logger.update_summary(
                    latest_evaluation,
                    prefix=(
                        "evaluation_latest"
                    ),
                )
                last_eval_step = (
                    global_step
                )

                if evaluation_is_better(
                    latest_evaluation,
                    best_evaluation,
                ):
                    best_evaluation = (
                        copy.deepcopy(
                            latest_evaluation
                        )
                    )
                    best_state = {
                        "seed": seed,
                        "algorithm": (
                            normalized
                        ),
                        "global_step": (
                            global_step
                        ),
                        "episode_index": len(
                            completed_episodes
                        ),
                        "best_evaluation": (
                            copy.deepcopy(
                                best_evaluation
                            )
                        ),
                        "last_eval_step": (
                            last_eval_step
                        ),
                        "last_checkpoint_step": (
                            last_checkpoint_step
                        ),
                        "resume_safe": False,
                        "vectorized": True,
                        "num_envs": (
                            num_envs
                        ),
                        "vector_backend": (
                            vector_backend
                        ),
                    }
                    best_path = save_fn(
                        checkpoint_dir
                        / "best.pt",
                        trainer,
                        replay_buffer=(
                            replay_buffer
                        ),
                        training_state=(
                            best_state
                        ),
                        include_replay=False,
                    )
                    logger.log_model_artifact(
                        best_path,
                        aliases=["best"],
                        metadata={
                            "global_step": (
                                global_step
                            ),
                            "episode_index": len(
                                completed_episodes
                            ),
                            "num_envs": (
                                num_envs
                            ),
                        },
                    )

            if should_checkpoint:
                last_checkpoint_step = (
                    global_step
                )
                system_snapshot = (
                    _vector_system_metrics(
                        trainer=trainer,
                        global_step=(
                            global_step
                        ),
                        wall_start=(
                            wall_start
                        ),
                        evaluation_seconds=(
                            evaluation_seconds
                        ),
                        env_collection_seconds=(
                            env_collection_seconds
                        ),
                        env_wait_seconds=(
                            env_wait_seconds
                        ),
                        policy_inference_seconds=(
                            policy_inference_seconds
                        ),
                        update_seconds=(
                            update_seconds
                        ),
                        num_envs=num_envs,
                        overlap=overlap,
                        initial_global_step=(
                            initial_global_step
                        ),
                        prior_metrics=(
                            prior_system_metrics
                        ),
                    )
                )
                checkpoint_state = (
                    _make_vector_checkpoint_state(
                        algorithm=normalized,
                        seed=seed,
                        global_step=(
                            global_step
                        ),
                        best_evaluation=(
                            best_evaluation
                        ),
                        latest_evaluation=(
                            latest_evaluation
                        ),
                        last_eval_step=(
                            last_eval_step
                        ),
                        last_checkpoint_step=(
                            last_checkpoint_step
                        ),
                        rng=rng,
                        num_envs=num_envs,
                        vector_backend=(
                            vector_backend
                        ),
                        vector_env=(
                            vector_env
                        ),
                        trackers=trackers,
                        completed_episodes=(
                            completed_episodes
                        ),
                        next_episode_id=(
                            next_episode_id
                        ),
                        system_metrics=(
                            system_snapshot
                        ),
                        wandb_run_id=(
                            logger.wandb_run_id
                        ),
                        resume_safe=(
                            include_replay
                        ),
                    )
                )
                save_fn(
                    checkpoint_dir
                    / "latest.pt",
                    trainer,
                    replay_buffer=(
                        replay_buffer
                    ),
                    training_state=(
                        checkpoint_state
                    ),
                    include_replay=(
                        include_replay
                    ),
                )

        if (
            overlap
            and pending_updates > 0
        ):
            update_start = (
                time.perf_counter()
            )
            for _ in range(
                pending_updates
            ):
                latest_update_metrics = (
                    trainer.update(
                        replay_buffer,
                        batch_size=(
                            batch_size
                        ),
                    )
                )
            update_seconds += (
                time.perf_counter()
                - update_start
            )
            pending_updates = 0

        system_metrics = (
            _vector_system_metrics(
                trainer=trainer,
                global_step=global_step,
                wall_start=wall_start,
                evaluation_seconds=(
                    evaluation_seconds
                ),
                env_collection_seconds=(
                    env_collection_seconds
                ),
                env_wait_seconds=(
                    env_wait_seconds
                ),
                policy_inference_seconds=(
                    policy_inference_seconds
                ),
                update_seconds=(
                    update_seconds
                ),
                num_envs=num_envs,
                overlap=overlap,
                initial_global_step=(
                    initial_global_step
                ),
                prior_metrics=(
                    prior_system_metrics
                ),
            )
        )
        last_checkpoint_step = (
            global_step
        )
        exact_state = (
            _make_vector_checkpoint_state(
                algorithm=normalized,
                seed=seed,
                global_step=global_step,
                best_evaluation=(
                    best_evaluation
                ),
                latest_evaluation=(
                    latest_evaluation
                ),
                last_eval_step=(
                    last_eval_step
                ),
                last_checkpoint_step=(
                    last_checkpoint_step
                ),
                rng=rng,
                num_envs=num_envs,
                vector_backend=(
                    vector_backend
                ),
                vector_env=vector_env,
                trackers=trackers,
                completed_episodes=(
                    completed_episodes
                ),
                next_episode_id=(
                    next_episode_id
                ),
                system_metrics=(
                    system_metrics
                ),
                wandb_run_id=(
                    logger.wandb_run_id
                ),
                resume_safe=(
                    include_replay
                ),
            )
        )
        latest_checkpoint_path = (
            save_fn(
                checkpoint_dir
                / "latest.pt",
                trainer,
                replay_buffer=(
                    replay_buffer
                ),
                training_state=(
                    exact_state
                ),
                include_replay=(
                    include_replay
                ),
            )
        )

        final_state = {
            "seed": seed,
            "algorithm": normalized,
            "global_step": (
                global_step
            ),
            "episode_index": len(
                completed_episodes
            ),
            "best_evaluation": (
                copy.deepcopy(
                    best_evaluation
                )
            ),
            "latest_evaluation": (
                copy.deepcopy(
                    latest_evaluation
                )
            ),
            "last_eval_step": (
                last_eval_step
            ),
            "last_checkpoint_step": (
                last_checkpoint_step
            ),
            "resume_safe": False,
            "vectorized": True,
            "num_envs": num_envs,
            "vector_backend": (
                vector_backend
            ),
        }
        final_path = save_fn(
            checkpoint_dir
            / "final.pt",
            trainer,
            replay_buffer=(
                replay_buffer
            ),
            training_state=(
                final_state
            ),
            include_replay=False,
        )
        logger.log_model_artifact(
            final_path,
            aliases=["final"],
            metadata={
                "global_step": (
                    global_step
                ),
                "episode_index": len(
                    completed_episodes
                ),
                "num_envs": num_envs,
            },
        )
        logger.update_summary(
            {
                "global_step": (
                    global_step
                ),
                "episodes_completed": len(
                    completed_episodes
                ),
                "trainer_updates": int(
                    trainer.update_count
                ),
                "num_envs": num_envs,
                **system_metrics,
            },
            prefix="training_final",
        )
        logger.log_metrics(
            "system",
            system_metrics,
            step=global_step,
            episode=len(
                completed_episodes
            ),
        )

        return {
            "trainer": trainer,
            "replay_buffer": (
                replay_buffer
            ),
            "completed_episodes": (
                completed_episodes
            ),
            "latest_update_metrics": (
                latest_update_metrics
            ),
            "latest_evaluation": (
                latest_evaluation
            ),
            "best_evaluation": (
                best_evaluation
            ),
            "training_state": (
                exact_state
            ),
            "system_metrics": (
                system_metrics
            ),
            "logger_status": (
                logger.status()
            ),
            "run_dir": run_dir,
            "checkpoint_dir": (
                checkpoint_dir
            ),
            "latest_checkpoint_path": (
                latest_checkpoint_path
            ),
        }
    finally:
        vector_env.close()
        if eval_env is not None:
            eval_env.close()
        if logger is not None:
            logger.close()


In [ ]:
def make_experiment_run_name(
    algorithm,
    seed,
    run_suffix=None,
):
    algorithm = str(
        algorithm
    ).strip().upper()

    if algorithm not in {
        "MASAC",
        "MATD3",
    }:
        raise ValueError(
            "algorithm must be MASAC or MATD3"
        )

    seed = int(seed)

    if run_suffix is None:
        # Uses OS randomness, not Python/NumPy/Torch RNG,
        # so display-name uniqueness does not change training determinism.
        run_suffix = secrets.token_hex(
            4
        )

    run_suffix = str(
        run_suffix
    ).strip()

    if not run_suffix:
        raise ValueError(
            "run_suffix must not be empty"
        )

    return (
        f"{algorithm}-"
        f"seed{seed}-"
        f"{run_suffix}"
    )


In [ ]:
def run_training_experiment(
    algorithm,
    total_steps,
    seed=44,
    backend_name="simple",
    run_name=None,
    run_dir=None,
    resume_checkpoint=None,
    device=None,
    enable_csv=None,
    enable_tensorboard=None,
    enable_wandb=None,
):
    normalized = str(
        algorithm
    ).strip().lower()

    if normalized not in {
        "masac",
        "matd3",
    }:
        raise ValueError(
            "algorithm must be "
            "'masac' or 'matd3'"
        )

    seed = int(seed)

    if (
        resume_checkpoint is not None
        and run_dir is None
    ):
        run_dir = (
            Path(resume_checkpoint)
            .resolve()
            .parent
            .parent
        )

    if run_name is None:
        if (
            resume_checkpoint is not None
            and run_dir is not None
        ):
            run_name = Path(
                run_dir
            ).name
        else:
            run_name = (
                make_experiment_run_name(
                    normalized,
                    seed,
                )
            )

    env = UAVSearchEnv(
        backend_name=backend_name
    )

    try:
        if _vector_training_is_enabled(
            env,
            device,
        ):
            return train_vectorized_experiment(
                normalized,
                env,
                total_steps=total_steps,
                seed=seed,
                run_dir=run_dir,
                run_name=run_name,
                resume_checkpoint=(
                    resume_checkpoint
                ),
                device=device,
                enable_csv=enable_csv,
                enable_tensorboard=enable_tensorboard,
                enable_wandb=enable_wandb,
            )

        if normalized == "masac":
            return train_masac_experiment(
                env,
                total_steps=total_steps,
                seed=seed,
                run_dir=run_dir,
                run_name=run_name,
                resume_checkpoint=(
                    resume_checkpoint
                ),
                device=device,
                enable_csv=enable_csv,
                enable_tensorboard=(
                    enable_tensorboard
                ),
                enable_wandb=(
                    enable_wandb
                ),
            )

        return train_matd3_experiment(
            env,
            total_steps=total_steps,
            seed=seed,
            run_dir=run_dir,
            run_name=run_name,
            resume_checkpoint=(
                resume_checkpoint
            ),
            device=device,
            enable_csv=enable_csv,
            enable_tensorboard=(
                enable_tensorboard
            ),
            enable_wandb=(
                enable_wandb
            ),
        )
    finally:
        env.close()


In [ ]:
DEFAULT_WANDB_COMPARISON_METRICS = (
    "evaluation/success_rate",
    "evaluation/delivery_rate",
    "evaluation/confirmation_rate",
    "evaluation/return",
    "evaluation/length",
    "evaluation/false_confirmations",
    "evaluation/expired_reports",
    "evaluation/dropped_reports",
    "evaluation/obstacle_block_rate",
    "evaluation/peer_safety_block_rate",
    "evaluation/boundary_clip_rate",
    "evaluation/total_energy_j",
    "evaluation/communication_energy_j",
    "evaluation/information_gain_bits",
    "evaluation/network_pdr_percent",
    "evaluation/network_e2e_delay_ms",
    "evaluation/network_throughput_kbps",
    "evaluation/network_average_hops",
    "evaluation/network_payload_delivery_ratio",
)


In [ ]:
def _select_latest_wandb_training_runs(
    seed,
    algorithms=("masac", "matd3"),
    entity=None,
    project=None,
):
    wandb = importlib.import_module(
        "wandb"
    )

    if entity is None:
        entity = CONFIG.get(
            "training_wandb_entity"
        )

    if project is None:
        project = CONFIG[
            "training_wandb_project"
        ]

    api = wandb.Api(
        timeout=30
    )
    all_runs = list(
        api.runs(
            f"{entity}/{project}",
            order="-created_at",
            per_page=200,
        )
    )

    selected = {}

    for algorithm in algorithms:
        normalized = str(
            algorithm
        ).strip().lower()

        for run in all_runs:
            if run.job_type != "training":
                continue

            config = dict(
                run.config
            )

            if str(
                config.get(
                    "monitor/algorithm",
                    "",
                )
            ).lower() != normalized:
                continue

            if int(
                config.get(
                    "monitor/seed",
                    -1,
                )
            ) != int(seed):
                continue

            if run.state != "finished":
                continue

            selected[
                normalized
            ] = run
            break

    missing = [
        str(algorithm)
        for algorithm in algorithms
        if str(
            algorithm
        ).strip().lower()
        not in selected
    ]

    if missing:
        raise RuntimeError(
            "no finished W&B training run "
            "found for: "
            + ", ".join(
                missing
            )
        )

    return selected


In [ ]:
def log_wandb_seed_comparison_plots(
    seed=44,
    algorithms=("masac", "matd3"),
    metrics=None,
    entity=None,
    project=None,
    mode=None,
):
    wandb = importlib.import_module(
        "wandb"
    )

    seed = int(seed)

    if entity is None:
        entity = CONFIG.get(
            "training_wandb_entity"
        )

    if project is None:
        project = CONFIG[
            "training_wandb_project"
        ]

    if mode is None:
        mode = CONFIG[
            "training_wandb_mode"
        ]

    if metrics is None:
        metrics = (
            DEFAULT_WANDB_COMPARISON_METRICS
        )

    selected = (
        _select_latest_wandb_training_runs(
            seed=seed,
            algorithms=algorithms,
            entity=entity,
            project=project,
        )
    )

    analysis_name = (
        make_experiment_run_name(
            "MASAC",
            seed,
            run_suffix=(
                "comparison-"
                + secrets.token_hex(
                    3
                )
            ),
        )
        .replace(
            "MASAC",
            "COMPARE",
            1,
        )
    )

    with wandb.init(
        entity=entity,
        project=str(project),
        name=analysis_name,
        group=f"comparison-seed{seed}",
        job_type="analysis",
        tags=[
            "comparison",
            f"seed-{seed}",
            *[
                str(
                    algorithm
                ).strip().lower()
                for algorithm
                in algorithms
            ],
        ],
        config={
            "seed": seed,
            "selected_runs": {
                algorithm: {
                    "id": run.id,
                    "name": run.name,
                    "url": run.url,
                }
                for algorithm, run
                in selected.items()
            },
            **git_experiment_metadata(),
        },
        mode=str(mode),
        reinit="finish_previous",
        settings=wandb.Settings(
            x_disable_stats=bool(
                CONFIG.get(
                    "training_wandb_disable_system_stats",
                    True,
                )
            )
        ),
    ) as analysis_run:
        logged_metrics = []

        for metric in metrics:
            rows = []

            for (
                algorithm,
                source_run,
            ) in selected.items():
                history = (
                    source_run.history(
                        keys=[
                            "global_step",
                            metric,
                        ],
                        pandas=False,
                    )
                )

                for point in history:
                    step = point.get(
                        "global_step"
                    )
                    value = point.get(
                        metric
                    )

                    if (
                        step is None
                        or value is None
                    ):
                        continue

                    if not np.isfinite(
                        float(value)
                    ):
                        continue

                    rows.append(
                        [
                            float(step),
                            float(value),
                            algorithm.upper(),
                        ]
                    )

            if not rows:
                continue

            table = wandb.Table(
                data=rows,
                columns=[
                    "global_step",
                    "value",
                    "algorithm",
                ],
            )

            key = (
                metric.replace(
                    "/",
                    "_",
                )
            )

            chart = wandb.plot.line(
                table=table,
                x="global_step",
                y="value",
                stroke="algorithm",
                title=(
                    f"{metric} | seed {seed}"
                ),
                split_table=True,
            )

            analysis_run.log(
                {
                    f"comparison/{key}": (
                        chart
                    )
                }
            )
            logged_metrics.append(
                metric
            )

        return {
            "run_id": (
                analysis_run.id
            ),
            "run_url": (
                analysis_run.url
            ),
            "run_name": (
                analysis_run.name
            ),
            "seed": seed,
            "source_runs": {
                algorithm: {
                    "id": run.id,
                    "name": run.name,
                    "url": run.url,
                }
                for algorithm, run
                in selected.items()
            },
            "logged_metrics": (
                logged_metrics
            ),
        }


In [ ]:
class FullGpuUAVBatchEnv:
    """Fixed-shape batched Simple backend implemented with CUDA tensors."""

    observation_dim = 333
    state_dim = 2875
    continuous_dim = 4

    def __init__(self, num_envs: int, device: str = "cuda:0", seed: int = 44, strict_cuda: bool = True):
        self.device = torch.device(device)
        if strict_cuda:
            _require_cuda(self.device)
        self.num_envs = int(num_envs)
        self.num_uavs = int(CONFIG["num_uavs"])
        self.num_targets = int(CONFIG["num_targets"])
        self.num_obstacles = int(CONFIG["num_obstacles"])
        self.discrete_dim = self.num_uavs + 1
        self.grid_n = math.ceil(float(CONFIG["map_size"]) / float(CONFIG["grid_cell_m"]))
        nearest_obstacles = int(CONFIG["observation_nearest_obstacles"])
        patch_cells = int(CONFIG["belief_patch_cells"])
        coarse_cells = int(CONFIG["belief_coarse_cells"])
        critic_cells = int(CONFIG["critic_belief_grid_cells"])
        buffer_slots = int(
            float(CONFIG["buffer_bytes"])
            // float(CONFIG["report_bytes"])
        )
        pending_slots = int(
            float(CONFIG["pending_buffer_bytes"])
            // float(CONFIG["report_bytes"])
        )
        self.observation_dim = (
            9
            + 3
            + 5 * (self.num_uavs - 1)
            + 5 * nearest_obstacles
            + patch_cells * patch_cells
            + coarse_cells * coarse_cells
            + 5
            + self.discrete_dim
        )
        per_uav_state_dim = (
            self.observation_dim
            + 4 * buffer_slots
            + 4 * pending_slots
            + 2
            + critic_cells * critic_cells
        )
        self.state_dim = (
            self.num_uavs * per_uav_state_dim
            + 4 * self.num_targets
            + 4 * self.num_obstacles
            + 1
        )
        self.seed = int(seed)
        self.generator = torch.Generator(device=self.device)
        self.generator.manual_seed(self.seed)

        self.map_size = float(CONFIG["map_size"])
        self.altitude_min = float(CONFIG["altitude_min"])
        self.altitude_max = float(CONFIG["altitude_max"])
        self.dt = float(CONFIG["dt"])
        self.max_speed = float(CONFIG["max_speed"])
        self.max_accel = float(CONFIG["max_accel"])
        self.cell_size = float(CONFIG["grid_cell_m"])
        self.max_steps = int(CONFIG["max_steps"])
        self.gcs = torch.tensor(CONFIG["gcs_position"], device=self.device, dtype=torch.float32)
        # Optional external mission-network adapter. The default Simple backend
        # remains fully tensorized; UavNetSim is attached explicitly by the
        # benchmark/training helpers because UavNetSim itself is CPU/SimPy.
        self.external_network = None
        self.network_model = str(
            CONFIG.get("full_gpu_network_model", "simple")
        ).strip().lower()
        if self.network_model not in {"simple", "packet_approx", "uavnetsim_gpu"}:
            raise ValueError(
                "full_gpu_network_model must be simple, packet_approx, or uavnetsim_gpu"
            )

        # At altitude_max with 90-degree full FOV, radius <= altitude.
        max_fov_radius = self.altitude_max * math.tan(
            math.radians(float(CONFIG["camera_full_fov_deg"])) / 2.0
        )
        max_radius_cells = math.ceil(max_fov_radius / self.cell_size) + 1
        offsets = torch.arange(-max_radius_cells, max_radius_cells + 1, device=self.device)
        oy, ox = torch.meshgrid(offsets, offsets, indexing="ij")
        self.cell_offsets = torch.stack((ox.reshape(-1), oy.reshape(-1)), dim=-1).long()

        # Communication choice table: 0=silent, last=GCS, middle entries are peers.
        choices = torch.full(
            (self.num_uavs, self.discrete_dim),
            -2,
            device=self.device,
            dtype=torch.long,
        )
        for sender in range(self.num_uavs):
            peers = [p for p in range(self.num_uavs) if p != sender]
            choices[sender, 1 : self.num_uavs] = torch.tensor(
                peers, device=self.device, dtype=torch.long
            )
            choices[sender, self.num_uavs] = -1
        self.choices = choices

        # Immutable tensors reused by every observation/network step.
        self.xyz_scale = torch.tensor(
            [
                self.map_size,
                self.map_size,
                max(self.altitude_max, 1.0),
            ],
            device=self.device,
            dtype=torch.float32,
        )
        peer_rows = [
            [peer for peer in range(self.num_uavs) if peer != sender]
            for sender in range(self.num_uavs)
        ]
        self.peer_index = torch.tensor(
            peer_rows,
            device=self.device,
            dtype=torch.long,
        )
        patch_n = int(CONFIG["belief_patch_cells"])
        patch_radius = patch_n // 2
        patch_axis = torch.arange(
            -patch_radius,
            patch_radius + 1,
            device=self.device,
            dtype=torch.long,
        )
        patch_y, patch_x = torch.meshgrid(
            patch_axis,
            patch_axis,
            indexing="ij",
        )
        self.belief_patch_offsets = torch.stack(
            (
                patch_x.reshape(-1),
                patch_y.reshape(-1),
            ),
            dim=-1,
        )
        self.target_ids = torch.arange(
            self.num_targets,
            device=self.device,
            dtype=torch.long,
        ).view(1, self.num_targets)

        self.reset()

    def attach_external_network(self, adapter):
        if self.num_envs != 1:
            raise ValueError(
                "external packet-level networking currently requires num_envs=1"
            )
        if self.external_network is not None:
            self.external_network.close()
        self.external_network = adapter
        self.external_network.reset_from_env()
        return self

    def detach_external_network(self):
        if self.external_network is not None:
            self.external_network.close()
        self.external_network = None

    def close(self):
        self.detach_external_network()

    def _rand(self, *shape):
        return torch.rand(*shape, device=self.device, generator=self.generator)


    def _sample_world(self):
        E, U, T, O = (
            self.num_envs,
            self.num_uavs,
            self.num_targets,
            self.num_obstacles,
        )
        pool = max(
            64,
            int(CONFIG.get("full_gpu_world_candidates", 256))
            * int(CONFIG.get("full_gpu_world_sampling_rounds", 4)),
        )

        # GPU rejection sampling that mirrors create_uav(): random positions
        # inside the launch disk, clipped only by map bounds and minimum
        # pairwise launch spacing.
        launch_radius = float(CONFIG["launch_radius_m"])
        launch_spacing = float(CONFIG["launch_min_spacing_m"])
        positions = self.gcs.view(1, 1, 3).repeat(E, U, 1)
        positions[..., 2] = self.altitude_min
        for uav_index in range(U):
            offsets = (self._rand(E, pool, 2) * 2.0 - 1.0) * launch_radius
            candidates = self.gcs[:2].view(1, 1, 2) + offsets
            base_valid = (
                (offsets.square().sum(-1) <= launch_radius * launch_radius)
                & (candidates[..., 0] >= 0.0)
                & (candidates[..., 0] <= self.map_size)
                & (candidates[..., 1] >= 0.0)
                & (candidates[..., 1] <= self.map_size)
            )
            if uav_index > 0:
                distances = torch.linalg.vector_norm(
                    candidates[:, :, None, :]
                    - positions[:, None, :uav_index, :2],
                    dim=-1,
                )
                spacing_score = distances.amin(-1)
                valid = base_valid & (spacing_score >= launch_spacing)
            else:
                spacing_score = torch.full(
                    (E, pool),
                    launch_spacing,
                    device=self.device,
                )
                valid = base_valid

            first_valid = valid.float().argmax(-1)
            # The fallback is still fully GPU-side and chooses the
            # best-separated in-disk candidate. With the project geometry the
            # configured pool has abundant valid samples; this only prevents a
            # silent all-false argmax from selecting an arbitrary invalid point.
            fallback_score = torch.where(
                base_valid,
                spacing_score,
                torch.full_like(spacing_score, -1.0),
            )
            fallback = fallback_score.argmax(-1)
            chosen_index = torch.where(valid.any(-1), first_valid, fallback)
            chosen = torch.gather(
                candidates,
                1,
                chosen_index[:, None, None].expand(-1, 1, 2),
            ).squeeze(1)
            positions[:, uav_index, :2] = chosen

        velocities = torch.zeros(E, U, 3, device=self.device)
        battery = torch.full(
            (E, U),
            float(CONFIG["battery_j"]),
            device=self.device,
        )
        active = torch.ones(E, U, device=self.device, dtype=torch.bool)

        rmin = float(CONFIG["obstacle_radius_min_m"])
        rmax = float(CONFIG["obstacle_radius_max_m"])
        hmin = float(CONFIG["obstacle_height_min_m"])
        hmax = float(CONFIG["obstacle_height_max_m"])
        obstacle_radius = rmin + (rmax - rmin) * self._rand(E, O)
        obstacle_height = hmin + (hmax - hmin) * self._rand(E, O)
        obstacle_xy = torch.zeros(E, O, 2, device=self.device)
        gcs_exclusion = float(CONFIG["gcs_exclusion_radius_m"])
        launch_margin = float(CONFIG["safety_distance"])

        for obstacle_index in range(O):
            radius = obstacle_radius[:, obstacle_index]
            candidates = (
                radius[:, None, None]
                + self._rand(E, pool, 2)
                * (self.map_size - 2.0 * radius).clamp_min(1.0)[:, None, None]
            )
            gcs_distance = torch.linalg.vector_norm(
                candidates - self.gcs[:2].view(1, 1, 2),
                dim=-1,
            )
            launch_distance = torch.linalg.vector_norm(
                candidates[:, :, None, :]
                - positions[:, None, :, :2],
                dim=-1,
            )
            valid = (
                (gcs_distance > (gcs_exclusion + radius)[:, None])
                & (
                    launch_distance
                    > (radius[:, None, None] + launch_margin)
                ).all(-1)
            )
            margin_score = torch.minimum(
                gcs_distance - (gcs_exclusion + radius)[:, None],
                (
                    launch_distance
                    - (radius[:, None, None] + launch_margin)
                ).amin(-1),
            )
            if obstacle_index > 0:
                previous_distance = torch.linalg.vector_norm(
                    candidates[:, :, None, :]
                    - obstacle_xy[:, None, :obstacle_index, :],
                    dim=-1,
                )
                previous_margin = (
                    previous_distance
                    - radius[:, None, None]
                    - obstacle_radius[:, None, :obstacle_index]
                )
                valid &= (previous_margin > 0.0).all(-1)
                margin_score = torch.minimum(
                    margin_score,
                    previous_margin.amin(-1),
                )

            first_valid = valid.float().argmax(-1)
            fallback = margin_score.argmax(-1)
            chosen_index = torch.where(valid.any(-1), first_valid, fallback)
            obstacle_xy[:, obstacle_index] = torch.gather(
                candidates,
                1,
                chosen_index[:, None, None].expand(-1, 1, 2),
            ).squeeze(1)

        target_xy = torch.zeros(E, T, 2, device=self.device)
        target_exclusion = float(CONFIG["target_exclusion_radius_m"])
        for target_index in range(T):
            candidates = self._rand(E, pool, 2) * self.map_size
            gcs_distance = torch.linalg.vector_norm(
                candidates - self.gcs[:2].view(1, 1, 2),
                dim=-1,
            )
            obstacle_distance = torch.linalg.vector_norm(
                candidates[:, :, None, :] - obstacle_xy[:, None, :, :],
                dim=-1,
            )
            valid = (
                (gcs_distance > target_exclusion)
                & (
                    obstacle_distance
                    > obstacle_radius[:, None, :]
                ).all(-1)
            )
            obstacle_margin = (
                (
                    obstacle_distance - obstacle_radius[:, None, :]
                ).amin(-1)
                if O > 0
                else torch.full_like(gcs_distance, float("inf"))
            )
            margin_score = torch.minimum(
                gcs_distance - target_exclusion,
                obstacle_margin,
            )
            candidate_cells = torch.floor(candidates / self.cell_size).long()
            if target_index > 0:
                previous_cells = torch.floor(
                    target_xy[:, :target_index] / self.cell_size
                ).long()
                duplicate_cell = (
                    candidate_cells[:, :, None, :]
                    == previous_cells[:, None, :, :]
                ).all(-1).any(-1)
                valid &= ~duplicate_cell
                margin_score = torch.where(
                    duplicate_cell,
                    torch.full_like(margin_score, -1.0),
                    margin_score,
                )

            first_valid = valid.float().argmax(-1)
            fallback = margin_score.argmax(-1)
            chosen_index = torch.where(valid.any(-1), first_valid, fallback)
            target_xy[:, target_index] = torch.gather(
                candidates,
                1,
                chosen_index[:, None, None].expand(-1, 1, 2),
            ).squeeze(1)

        return (
            positions,
            velocities,
            battery,
            active,
            target_xy,
            obstacle_xy,
            obstacle_radius,
            obstacle_height,
        )

    def reset(self, mask: torch.Tensor | None = None):
        sampled = self._sample_world()
        E, U, T = self.num_envs, self.num_uavs, self.num_targets

        if mask is None:
            self.positions, self.velocities, self.battery, self.active = sampled[:4]
            self.target_xy, self.obstacle_xy, self.obstacle_radius, self.obstacle_height = sampled[4:]
            self.confirmed = torch.zeros(E, T, device=self.device, dtype=torch.bool)
            self.delivered = torch.zeros(E, T, device=self.device, dtype=torch.bool)
            self.belief = torch.full(
                (E, U, self.grid_n, self.grid_n),
                float(CONFIG["belief_prior"]),
                device=self.device,
            )
            self.report_valid = torch.zeros(E, U, T, device=self.device, dtype=torch.bool)
            self.report_remaining = torch.zeros(E, U, T, device=self.device)
            self.report_created = torch.full(
                (E, U, T), self.max_steps + int(CONFIG["report_ttl"]) + 1,
                device=self.device, dtype=torch.long
            )
            self.report_source = torch.full(
                (E, U, T), -1, device=self.device, dtype=torch.long
            )
            self.peer_received = torch.zeros(
                E, U, U, T, device=self.device
            )
            self.peer_source = torch.full(
                (E, U, U, T), -1, device=self.device, dtype=torch.long
            )
            self.peer_created = torch.full(
                (E, U, U, T),
                self.max_steps + int(CONFIG["report_ttl"]) + 1,
                device=self.device,
                dtype=torch.long,
            )
            self.pending_valid = torch.zeros(E, U, T, device=self.device, dtype=torch.bool)
            self.pending_remaining = torch.zeros(E, U, T, device=self.device)
            self.pending_created = torch.full_like(
                self.report_created,
                self.max_steps + int(CONFIG["report_ttl"]) + 1,
            )
            self.step_count = torch.zeros(E, device=self.device, dtype=torch.long)
        else:
            mask = mask.to(device=self.device, dtype=torch.bool)
            if mask.ndim != 1 or mask.shape[0] != E:
                raise ValueError("reset mask must have shape (num_envs,)")
            m3 = mask[:, None, None]
            self.positions = torch.where(m3, sampled[0], self.positions)
            self.velocities = torch.where(m3, sampled[1], self.velocities)
            self.battery = torch.where(mask[:, None], sampled[2], self.battery)
            self.active = torch.where(mask[:, None], sampled[3], self.active)
            self.target_xy = torch.where(m3[:, :1].expand(-1, T, 2), sampled[4], self.target_xy)
            mo2 = mask[:, None, None]
            self.obstacle_xy = torch.where(mo2, sampled[5], self.obstacle_xy)
            self.obstacle_radius = torch.where(mask[:, None], sampled[6], self.obstacle_radius)
            self.obstacle_height = torch.where(mask[:, None], sampled[7], self.obstacle_height)
            self.confirmed = torch.where(mask[:, None], torch.zeros_like(self.confirmed), self.confirmed)
            self.delivered = torch.where(mask[:, None], torch.zeros_like(self.delivered), self.delivered)
            self.belief = torch.where(
                mask[:, None, None, None],
                torch.full_like(self.belief, float(CONFIG["belief_prior"])),
                self.belief,
            )
            self.report_valid = torch.where(
                mask[:, None, None], torch.zeros_like(self.report_valid), self.report_valid
            )
            self.report_remaining = torch.where(
                mask[:, None, None], torch.zeros_like(self.report_remaining), self.report_remaining
            )
            self.report_created = torch.where(
                mask[:, None, None],
                torch.full_like(self.report_created, self.max_steps + int(CONFIG["report_ttl"]) + 1),
                self.report_created,
            )
            self.report_source = torch.where(
                mask[:, None, None],
                torch.full_like(self.report_source, -1),
                self.report_source,
            )
            self.peer_received = torch.where(
                mask[:, None, None, None],
                torch.zeros_like(self.peer_received),
                self.peer_received,
            )
            self.peer_source = torch.where(
                mask[:, None, None, None],
                torch.full_like(self.peer_source, -1),
                self.peer_source,
            )
            self.peer_created = torch.where(
                mask[:, None, None, None],
                torch.full_like(
                    self.peer_created,
                    self.max_steps + int(CONFIG["report_ttl"]) + 1,
                ),
                self.peer_created,
            )
            self.pending_valid = torch.where(
                mask[:, None, None], torch.zeros_like(self.pending_valid), self.pending_valid
            )
            self.pending_remaining = torch.where(
                mask[:, None, None], torch.zeros_like(self.pending_remaining), self.pending_remaining
            )
            self.pending_created = torch.where(
                mask[:, None, None],
                torch.full_like(self.pending_created, self.max_steps + int(CONFIG["report_ttl"]) + 1),
                self.pending_created,
            )
            self.step_count = torch.where(mask, torch.zeros_like(self.step_count), self.step_count)

        entropy_map = self._belief_entropy()
        self.belief_potential = self._belief_potential(
            entropy_map
        )
        if self.external_network is not None:
            self.external_network.reset_from_env()
        obs, mask_out = self._observation(
            entropy_map=entropy_map
        )
        state = self._state(
            obs,
            entropy_map=entropy_map,
        )
        return obs, state, mask_out

    def _belief_entropy(self):
        p = self.belief.clamp(1e-6, 1.0 - 1e-6)
        return (
            -p * torch.log2(p)
            - (1.0 - p) * torch.log2(1.0 - p)
        )

    def _belief_potential(self, entropy_map=None):
        if entropy_map is None:
            entropy_map = self._belief_entropy()
        return (
            1.0
            - entropy_map.mean(dim=(1, 2, 3))
        ).clamp(0.0, 1.0)

    @staticmethod
    def _project_unit_ball(x):
        x = x.clamp(-1.0, 1.0)
        norm = torch.linalg.vector_norm(x, dim=-1, keepdim=True).clamp_min(1.0)
        return x / norm

    def _segment_cylinder_blocked(
        self,
        start,
        end,
        obstacle_xy=None,
        obstacle_radius=None,
        obstacle_height=None,
    ):
        """Exact vectorized counterpart of CPU segment_intersects_obstacle().

        start/end have shape [E, ..., 3]. Obstacles are the environment's
        vertical finite cylinders [E, O]. The returned tensor has shape
        [E, ...] and is True when the segment intersects any cylinder.
        """
        if obstacle_xy is None:
            obstacle_xy = self.obstacle_xy
        if obstacle_radius is None:
            obstacle_radius = self.obstacle_radius
        if obstacle_height is None:
            obstacle_height = self.obstacle_height

        start = start.to(dtype=self.positions.dtype)
        end = end.to(dtype=self.positions.dtype)
        extra = start.ndim - 2
        E = start.shape[0]
        O = obstacle_xy.shape[1]
        obs_xy = obstacle_xy.reshape(
            E, *([1] * extra), O, 2
        )
        obs_r = obstacle_radius.reshape(
            E, *([1] * extra), O
        )
        obs_h = obstacle_height.reshape(
            E, *([1] * extra), O
        )
        s = start.unsqueeze(-2)
        e = end.unsqueeze(-2)
        d = e - s
        dxy = d[..., :2]
        dz = d[..., 2]
        rel_xy = s[..., :2] - obs_xy
        eps = torch.finfo(start.dtype).eps

        dz_small = dz.abs() <= eps
        horizontal_z_ok = (s[..., 2] >= 0.0) & (s[..., 2] <= obs_h)
        safe_dz = torch.where(dz_small, torch.ones_like(dz), dz)
        t_ground = (0.0 - s[..., 2]) / safe_dz
        t_top = (obs_h - s[..., 2]) / safe_dz
        z_enter = torch.maximum(
            torch.zeros_like(t_ground),
            torch.minimum(t_ground, t_top),
        )
        z_exit = torch.minimum(
            torch.ones_like(t_ground),
            torch.maximum(t_ground, t_top),
        )
        z_enter = torch.where(dz_small, torch.zeros_like(z_enter), z_enter)
        z_exit = torch.where(dz_small, torch.ones_like(z_exit), z_exit)
        z_valid = torch.where(
            dz_small,
            horizontal_z_ok,
            z_enter <= z_exit,
        )

        a = (dxy * dxy).sum(-1)
        b = 2.0 * (rel_xy * dxy).sum(-1)
        c = (rel_xy * rel_xy).sum(-1) - obs_r * obs_r
        a_small = a <= eps
        discriminant = b * b - 4.0 * a * c
        root = discriminant.clamp_min(0.0).sqrt()
        safe_two_a = torch.where(
            a_small, torch.ones_like(a), 2.0 * a
        )
        t1 = (-b - root) / safe_two_a
        t2 = (-b + root) / safe_two_a
        xy_enter = torch.maximum(
            torch.zeros_like(t1),
            torch.minimum(t1, t2),
        )
        xy_exit = torch.minimum(
            torch.ones_like(t1),
            torch.maximum(t1, t2),
        )
        xy_enter = torch.where(a_small, torch.zeros_like(xy_enter), xy_enter)
        xy_exit = torch.where(a_small, torch.ones_like(xy_exit), xy_exit)
        xy_valid = torch.where(
            a_small,
            c <= 0.0,
            (discriminant >= 0.0) & (xy_enter <= xy_exit),
        )

        enter = torch.maximum(z_enter, xy_enter)
        exit_ = torch.minimum(z_exit, xy_exit)
        intersects = z_valid & xy_valid & (enter <= exit_)
        return intersects.any(-1)


    def _motion(self, motion):
        E, U, T = self.num_envs, self.num_uavs, self.num_targets
        old_pos = self.positions
        old_vel = self.velocities
        projected = self._project_unit_ball(motion)
        accel = projected * self.max_accel
        candidate_velocity = old_vel + accel * self.dt
        speed = torch.linalg.vector_norm(
            candidate_velocity,
            dim=-1,
            keepdim=True,
        )
        candidate_velocity = candidate_velocity * torch.minimum(
            torch.ones_like(speed),
            torch.tensor(self.max_speed, device=self.device)
            / speed.clamp_min(1e-8),
        )
        raw_pos = old_pos + candidate_velocity * self.dt
        candidate_pos = raw_pos.clone()
        candidate_pos[..., 0].clamp_(0.0, self.map_size)
        candidate_pos[..., 1].clamp_(0.0, self.map_size)
        candidate_pos[..., 2].clamp_(
            self.altitude_min,
            self.altitude_max,
        )
        hclip = (
            (raw_pos[..., :2] - candidate_pos[..., :2])
            .abs()
            .amax(-1)
            > 1e-6
        )
        zclip = (
            (raw_pos[..., 2] - candidate_pos[..., 2]).abs()
            > 1e-6
        )
        hclip &= self.active
        zclip &= self.active
        # CPU reference uses the realized velocity after boundary clipping.
        candidate_velocity = (
            candidate_pos - old_pos
        ) / self.dt

        # Exact CPU-equivalent segment/finite-cylinder collision, including
        # horizontal and vertical clearance.
        clearance = float(CONFIG["obstacle_clearance_m"])
        blocked_by_obstacle = self._segment_cylinder_blocked(
            old_pos,
            candidate_pos,
            obstacle_radius=self.obstacle_radius + clearance,
            obstacle_height=self.obstacle_height + clearance,
        ) & self.active

        # Match apply_swarm_motion(): collision is checked over the entire
        # simultaneous segment, not only at the candidate endpoints. Blocking
        # is monotone, so U fixed iterations are sufficient for convergence
        # without a CPU-side convergence check.
        blocked = blocked_by_obstacle.clone()
        blocked_by_peer = torch.zeros_like(blocked)
        eye = torch.eye(
            U,
            device=self.device,
            dtype=torch.bool,
        ).unsqueeze(0)
        active_pair = (
            self.active.unsqueeze(2)
            & self.active.unsqueeze(1)
            & ~eye
        )
        safety = float(CONFIG["safety_distance"])

        for _ in range(U):
            effective = torch.where(
                blocked.unsqueeze(-1),
                old_pos,
                candidate_pos,
            )
            relative_start = (
                old_pos.unsqueeze(2) - old_pos.unsqueeze(1)
            )
            displacement = effective - old_pos
            relative_motion = (
                displacement.unsqueeze(2)
                - displacement.unsqueeze(1)
            )
            denominator = (
                relative_motion * relative_motion
            ).sum(-1)
            t_closest = torch.where(
                denominator > 1e-12,
                -(
                    relative_start * relative_motion
                ).sum(-1)
                / denominator.clamp_min(1e-12),
                torch.zeros_like(denominator),
            ).clamp(0.0, 1.0)
            relative_at_closest = (
                relative_start
                + t_closest.unsqueeze(-1) * relative_motion
            )
            min_distance = torch.linalg.vector_norm(
                relative_at_closest,
                dim=-1,
            )
            pair_bad = (
                (min_distance < safety)
                & active_pair
            )
            peer_now = pair_bad.any(-1)
            blocked_by_peer |= peer_now
            blocked |= peer_now

        new_pos = torch.where(
            blocked.unsqueeze(-1),
            old_pos,
            candidate_pos,
        )
        new_vel = torch.where(
            blocked.unsqueeze(-1),
            torch.zeros_like(candidate_velocity),
            candidate_velocity,
        )
        new_vel = torch.where(
            self.active.unsqueeze(-1),
            new_vel,
            torch.zeros_like(new_vel),
        )
        new_pos = torch.where(
            self.active.unsqueeze(-1),
            new_pos,
            old_pos,
        )
        realized_accel = (
            new_vel - old_vel
        ) / self.dt

        self.positions = new_pos
        self.velocities = new_vel
        return {
            "blocked": blocked,
            "blocked_by_obstacle": blocked_by_obstacle,
            "blocked_by_peer": blocked_by_peer,
            "boundary": hclip | zclip,
            "horizontal_boundary": hclip,
            "altitude_boundary": zclip,
            "realized_accel": realized_accel,
            "commanded_motion": projected,
        }

    def _sensing_profile(self):
        z = self.positions[..., 2]
        anchors = torch.tensor(CONFIG["altitude_anchors"], device=self.device)
        pdv = torch.tensor(CONFIG["pd"], device=self.device)
        pfv = torch.tensor(CONFIG["pf"], device=self.device)
        # Piecewise linear interpolation over four anchors.
        idx = torch.bucketize(z.contiguous(), anchors[1:-1])
        z0, z1 = anchors[idx], anchors[idx + 1]
        w = ((z - z0) / (z1 - z0).clamp_min(1e-6)).clamp(0.0, 1.0)
        pd = pdv[idx] + w * (pdv[idx + 1] - pdv[idx])
        pf = pfv[idx] + w * (pfv[idx + 1] - pfv[idx])
        half = math.radians(float(CONFIG["camera_full_fov_deg"]) / 2.0)
        radius = z * math.tan(half)
        return pd, pf, radius

    @staticmethod
    def _circle_sqrt_integral(x, radius):
        radius_safe = radius.clamp_min(torch.finfo(x.dtype).eps)
        x = torch.maximum(torch.minimum(x, radius), -radius)
        root = (radius * radius - x * x).clamp_min(0.0).sqrt()
        ratio = (x / radius_safe).clamp(-1.0, 1.0)
        return 0.5 * (x * root + radius * radius * torch.asin(ratio))

    def _cell_coverage_fraction(self, circle_x, circle_y, radius, gx, gy):
        original_shape = gx.shape
        circle_x = circle_x.reshape(-1)
        circle_y = circle_y.reshape(-1)
        radius = radius.reshape(-1)
        gx = gx.reshape(-1).float()
        gy = gy.reshape(-1).float()

        x_min = gx * self.cell_size - circle_x
        x_max = ((gx + 1.0) * self.cell_size).clamp(max=self.map_size) - circle_x
        y_min = gy * self.cell_size - circle_y
        y_max = ((gy + 1.0) * self.cell_size).clamp(max=self.map_size) - circle_y
        left = torch.maximum(x_min, -radius)
        right = torch.minimum(x_max, radius)
        base_valid = (
            (radius > 0.0)
            & (right > left)
            & (y_max > -radius)
            & (y_min < radius)
        )

        def crossings(y_edge):
            has = y_edge.abs() < radius
            x_cross = (radius * radius - y_edge * y_edge).clamp_min(0.0).sqrt()
            negative = torch.where(has, -x_cross, left)
            positive = torch.where(has, x_cross, left)
            negative = torch.maximum(left, torch.minimum(right, negative))
            positive = torch.maximum(left, torch.minimum(right, positive))
            return negative, positive

        y_min_neg, y_min_pos = crossings(y_min)
        y_max_neg, y_max_pos = crossings(y_max)
        cuts = torch.stack(
            (left, right, y_min_neg, y_min_pos, y_max_neg, y_max_pos), dim=-1
        ).sort(dim=-1).values
        a, b = cuts[:, :-1], cuts[:, 1:]
        width = b - a
        midpoint = 0.5 * (a + b)
        radius_2d = radius[:, None]
        y_min_2d = y_min[:, None]
        y_max_2d = y_max[:, None]
        half_height = (
            radius_2d * radius_2d - midpoint * midpoint
        ).clamp_min(0.0).sqrt()
        upper = torch.minimum(y_max_2d, half_height)
        lower = torch.maximum(y_min_2d, -half_height)
        arc = self._circle_sqrt_integral(b, radius_2d) - self._circle_sqrt_integral(
            a, radius_2d
        )
        upper_integral = torch.where(
            y_max_2d < half_height, y_max_2d * width, arc
        )
        lower_integral = torch.where(
            y_min_2d > -half_height, y_min_2d * width, -arc
        )
        area = torch.where(
            (width > 0.0) & (upper > lower),
            upper_integral - lower_integral,
            torch.zeros_like(width),
        ).sum(-1).clamp_min(0.0)
        cell_area = ((x_max - x_min) * (y_max - y_min)).clamp_min(
            torch.finfo(area.dtype).eps
        )
        fraction = (area / cell_area).clamp(0.0, 1.0)
        return torch.where(base_valid, fraction, torch.zeros_like(fraction)).reshape(
            original_shape
        )

    def _sense(self):
        E, U, T = self.num_envs, self.num_uavs, self.num_targets
        pd, pf, radius = self._sensing_profile()
        base = torch.floor(self.positions[..., :2] / self.cell_size).long()
        cand = base.unsqueeze(2) + self.cell_offsets.view(1, 1, -1, 2)
        gx, gy = cand[..., 0], cand[..., 1]
        valid = (gx >= 0) & (gx < self.grid_n) & (gy >= 0) & (gy < self.grid_n)
        gx_c = gx.clamp(0, self.grid_n - 1)
        gy_c = gy.clamp(0, self.grid_n - 1)
        centers = (torch.stack((gx_c, gy_c), dim=-1).float() + 0.5) * self.cell_size

        coverage = self._cell_coverage_fraction(
            self.positions[..., 0].unsqueeze(-1).expand_as(gx_c),
            self.positions[..., 1].unsqueeze(-1).expand_as(gy_c),
            radius.unsqueeze(-1).expand_as(gx_c),
            gx_c,
            gy_c,
        )
        visible = valid & (coverage > 0.0)
        visible &= (self.positions[..., 2] > 0.0).unsqueeze(-1) & self.active.unsqueeze(-1)

        if bool(CONFIG["sensing_obstacle_occlusion"]):
            start = self.positions.unsqueeze(2).expand(-1, -1, centers.shape[2], -1)
            end = torch.cat(
                (centers, torch.zeros(*centers.shape[:-1], 1, device=self.device)), dim=-1
            )
            blocked = self._segment_cylinder_blocked(start, end)
            visible &= ~blocked

        target_cell = torch.floor(self.target_xy / self.cell_size).long().clamp(0, self.grid_n - 1)
        cell_match = (
            (gx_c.unsqueeze(-1) == target_cell[:, None, None, :, 0])
            & (gy_c.unsqueeze(-1) == target_cell[:, None, None, :, 1])
        )
        target_distance = torch.linalg.vector_norm(
            self.target_xy[:, None, :, :] - self.positions[:, :, None, :2], dim=-1
        )
        target_in_fov = target_distance <= radius[:, :, None]

        if bool(CONFIG["sensing_obstacle_occlusion"]):
            target_end = torch.cat(
                (
                    self.target_xy[:, None, :, :].expand(-1, U, -1, -1),
                    torch.zeros(E, U, self.num_targets, 1, device=self.device),
                ),
                dim=-1,
            )
            target_start = self.positions[:, :, None, :].expand(
                -1, -1, self.num_targets, -1
            )
            target_blocked = self._segment_cylinder_blocked(
                target_start, target_end
            )
            target_in_fov &= ~target_blocked

        target_visible_per_cell = (
            cell_match
            & target_in_fov[:, :, None, :]
        )
        has_target = target_visible_per_cell.any(-1) & visible

        effective_pf = (
            1.0 - torch.pow(1.0 - pf.unsqueeze(-1), coverage)
        ).clamp(0.0, 1.0)
        effective_pd = (
            coverage * pd.unsqueeze(-1)
            + (1.0 - coverage) * effective_pf
        ).clamp(0.0, 1.0)
        positive_prob = torch.where(
            has_target, pd.unsqueeze(-1).expand_as(effective_pf), effective_pf
        )
        observation = self._rand(*positive_prob.shape) < positive_prob
        observation &= visible

        flat_idx = gy_c * self.grid_n + gx_c
        belief_flat = self.belief.view(E, U, -1)
        sentinel_index = self.grid_n * self.grid_n
        padded_belief = torch.cat(
            (
                belief_flat,
                torch.full(
                    (E, U, 1),
                    float(CONFIG["belief_prior"]),
                    device=self.device,
                    dtype=belief_flat.dtype,
                ),
            ),
            dim=2,
        )
        scatter_idx = torch.where(
            valid,
            flat_idx,
            torch.full_like(flat_idx, sentinel_index),
        )
        prior = torch.gather(
            padded_belief,
            2,
            scatter_idx,
        )
        eps = 1e-6
        prior_c = prior.clamp(eps, 1.0 - eps)
        pos_num = effective_pd * prior_c
        pos_den = pos_num + effective_pf * (1.0 - prior_c)
        neg_num = (1.0 - effective_pd) * prior_c
        neg_den = neg_num + (1.0 - effective_pf) * (1.0 - prior_c)
        posterior = torch.where(
            observation,
            pos_num / pos_den.clamp_min(eps),
            neg_num / neg_den.clamp_min(eps),
        ).clamp(eps, 1.0 - eps)
        posterior = torch.where(visible, posterior, prior)
        padded_belief.scatter_(
            2,
            scatter_idx,
            posterior,
        )
        belief_flat.copy_(
            padded_belief[..., :sentinel_index]
        )

        fine = self.positions[..., 2] <= float(CONFIG["fine_altitude"])
        confirm_cell = (
            visible
            & observation
            & (posterior >= float(CONFIG["confirmation_threshold"]))
            & fine.unsqueeze(-1)
            & has_target
        )
        confirm_target = (
            confirm_cell.unsqueeze(-1) & target_visible_per_cell
        ).any(2)  # [E,U,T]
        any_confirm = confirm_target.any(1)
        newly_confirmed = any_confirm & ~self.confirmed
        self.confirmed |= any_confirm

        # CPU semantics permit each confirming source UAV to create its own
        # report copy. A later confirmation can also create a new source copy
        # if that UAV no longer has that target buffered/pending.
        already_local = self.report_valid | self.pending_valid
        create_mask = (
            confirm_target
            & ~self.delivered[:, None, :]
            & ~already_local
        )
        capacity = int(
            float(CONFIG["buffer_bytes"])
            // float(CONFIG["report_bytes"])
        )
        pending_capacity = int(
            float(CONFIG["pending_buffer_bytes"])
            // float(CONFIG["report_bytes"])
        )
        free_slots = (
            capacity - self.report_valid.sum(-1)
        ).clamp_min(0)
        direct_rank = create_mask.long().cumsum(-1)
        direct = (
            create_mask
            & (direct_rank <= free_slots.unsqueeze(-1))
        )
        remaining_create = create_mask & ~direct
        pending_free_slots = (
            pending_capacity - self.pending_valid.sum(-1)
        ).clamp_min(0)
        pending_rank = remaining_create.long().cumsum(-1)
        pending = (
            remaining_create
            & (
                pending_rank
                <= pending_free_slots.unsqueeze(-1)
            )
        )
        dropped = create_mask & ~direct & ~pending

        self.report_valid |= direct
        self.report_remaining = torch.where(
            direct,
            torch.full_like(
                self.report_remaining,
                float(CONFIG["report_bytes"]),
            ),
            self.report_remaining,
        )
        self.report_created = torch.where(
            direct,
            self.step_count[:, None, None].expand_as(
                self.report_created
            ),
            self.report_created,
        )
        holder_ids = torch.arange(
            U,
            device=self.device,
            dtype=torch.long,
        ).view(1, U, 1).expand(E, U, T)
        self.report_source = torch.where(
            direct,
            holder_ids,
            self.report_source,
        )
        self.pending_valid |= pending
        self.pending_remaining = torch.where(
            pending,
            torch.full_like(
                self.pending_remaining,
                float(CONFIG["report_bytes"]),
            ),
            self.pending_remaining,
        )
        self.pending_created = torch.where(
            pending,
            self.step_count[:, None, None].expand_as(
                self.pending_created
            ),
            self.pending_created,
        )

        false_mask = (
            visible
            & observation
            & (
                posterior
                >= float(CONFIG["confirmation_threshold"])
            )
            & fine.unsqueeze(-1)
            & ~has_target
        )
        false_confirmation = false_mask.sum(dim=(1, 2))
        corrected = torch.where(
            false_mask,
            torch.full_like(
                posterior,
                float(CONFIG["verified_empty_belief"]),
            ),
            torch.gather(
                padded_belief,
                2,
                scatter_idx,
            ),
        )
        padded_belief.scatter_(
            2,
            scatter_idx,
            corrected,
        )
        belief_flat.copy_(
            padded_belief[..., :sentinel_index]
        )

        return {
            "new_confirmed": newly_confirmed.sum(-1),
            "false_confirmation": false_confirmation,
            "dropped": dropped.sum(dim=(1, 2)),
        }


    def _flush_pending(self):
        capacity = int(
            float(CONFIG["buffer_bytes"])
            // float(CONFIG["report_bytes"])
        )
        sentinel = (
            self.max_steps
            + int(CONFIG["report_ttl"])
            + 1
        )
        holder_ids = torch.arange(
            self.num_uavs,
            device=self.device,
            dtype=torch.long,
        ).view(1, self.num_uavs, 1).expand(
            self.num_envs,
            self.num_uavs,
            self.num_targets,
        )
        # A completed relay may have filled this target slot while its local
        # confirmation was pending. Preserve the existing copy and its GCS
        # progress, matching flush_pending_reports() duplicate handling.
        source_already_buffered = (
            self.report_valid[:, None, :, :]
            & (
                self.report_source[:, None, :, :]
                == holder_ids[:, :, None, :]
            )
        ).any(dim=2)
        discard = (
            self.report_valid
            | source_already_buffered
            | self.delivered[:, None, :]
        )
        self.pending_valid &= ~discard
        self.pending_remaining = torch.where(
            self.pending_valid,
            self.pending_remaining,
            torch.zeros_like(self.pending_remaining),
        )
        for _ in range(capacity):
            free = (
                (self.report_valid.sum(-1) < capacity)
                & self.active
            )
            created = torch.where(
                self.pending_valid,
                self.pending_created,
                torch.full_like(
                    self.pending_created,
                    sentinel,
                ),
            )
            target = created.argmin(-1)
            selected = torch.gather(
                self.pending_valid,
                2,
                target.unsqueeze(-1),
            ).squeeze(-1)
            move = free & selected
            target_oh = F.one_hot(
                target,
                self.num_targets,
            ).bool()
            move_mask = move.unsqueeze(-1) & target_oh
            self.report_valid |= move_mask
            self.report_remaining = torch.where(
                move_mask,
                self.pending_remaining,
                self.report_remaining,
            )
            self.report_created = torch.where(
                move_mask,
                self.pending_created,
                self.report_created,
            )
            self.report_source = torch.where(
                move_mask,
                holder_ids,
                self.report_source,
            )
            self.pending_valid &= ~move_mask
            self.pending_remaining = torch.where(
                self.pending_valid,
                self.pending_remaining,
                torch.zeros_like(
                    self.pending_remaining
                ),
            )

    def _destination_mask(self):
        E, U = self.num_envs, self.num_uavs
        has_report = self.report_valid.any(-1) & self.active
        mask = torch.zeros(E, U, self.discrete_dim, device=self.device)
        mask[..., 0] = 1.0

        diff = self.positions.unsqueeze(2) - self.positions.unsqueeze(1)
        peer_dist = torch.linalg.vector_norm(diff, dim=-1)
        for sender in range(U):
            peers = self.choices[sender, 1:U]
            ok = (
                peer_dist[:, sender, peers] <= float(CONFIG["peer_contact_range_m"])
            ) & self.active[:, peers] & has_report[:, sender, None]
            mask[:, sender, 1:U] = ok.float()
        gcs_dist = torch.linalg.vector_norm(self.positions - self.gcs.view(1, 1, 3), dim=-1)
        mask[..., U] = (
            (gcs_dist <= float(CONFIG["gcs_contact_range_m"])) & has_report
        ).float()
        return mask


    def _commit_completed_peer_receipts(self):
        """Retry complete peer transfers when receiver buffer space exists."""
        E, U, T = (
            self.num_envs,
            self.num_uavs,
            self.num_targets,
        )
        report_size = float(CONFIG["report_bytes"])
        capacity = int(
            float(CONFIG["buffer_bytes"]) // report_size
        )
        for sender in range(U):
            for receiver in range(U):
                if sender == receiver:
                    continue
                for target in range(T):
                    complete = (
                        self.peer_received[
                            :, sender, receiver, target
                        ]
                        >= report_size
                    )
                    source = self.peer_source[
                        :, sender, receiver, target
                    ]
                    created = self.peer_created[
                        :, sender, receiver, target
                    ]
                    generation_valid = (
                        source >= 0
                    ) & (
                        created
                        <= self.step_count
                    )
                    free = (
                        self.report_valid[:, receiver, :]
                        .sum(-1)
                        < capacity
                    )
                    add = (
                        complete
                        & generation_valid
                        & ~self.delivered[:, target]
                        & ~self.report_valid[
                            :, receiver, target
                        ]
                        & free
                    )
                    # known GCS progress is the best progress among all
                    # copies of the same report generation.
                    same_generation = (
                        self.report_valid[:, :, target]
                        & (
                            self.report_source[
                                :, :, target
                            ]
                            == source[:, None]
                        )
                        & (
                            self.report_created[
                                :, :, target
                            ]
                            == created[:, None]
                        )
                    )
                    known_remaining = torch.where(
                        same_generation,
                        self.report_remaining[
                            :, :, target
                        ],
                        torch.full_like(
                            self.report_remaining[
                                :, :, target
                            ],
                            report_size,
                        ),
                    ).amin(-1)
                    self.report_valid[
                        :, receiver, target
                    ] |= add
                    self.report_remaining[
                        :, receiver, target
                    ] = torch.where(
                        add,
                        known_remaining,
                        self.report_remaining[
                            :, receiver, target
                        ],
                    )
                    self.report_created[
                        :, receiver, target
                    ] = torch.where(
                        add,
                        created,
                        self.report_created[
                            :, receiver, target
                        ],
                    )
                    self.report_source[
                        :, receiver, target
                    ] = torch.where(
                        add,
                        source,
                        self.report_source[
                            :, receiver, target
                        ],
                    )

    def _uavnetsim_gpu_radio(
        self,
        selected_dest,
        tx_power,
        attempted_mask,
    ):
        """Vectorized UavNetSim-v2 A2A PHY/SINR surrogate on one CUDA device.

        The external simulator remains a SimPy discrete-event engine. This
        routine mirrors the configured A2A path-gain law and simultaneous-link
        SINR calculation in tensor form so hundreds/thousands of training
        environments do not leave CUDA for packet/radio calculations.
        """
        E, U = self.num_envs, self.num_uavs
        is_gcs = selected_dest == -1
        is_peer = selected_dest >= 0
        receiver = selected_dest.clamp(0, max(0, U - 1))
        peer_pos = torch.gather(
            self.positions,
            1,
            receiver.unsqueeze(-1).expand(-1, -1, 3),
        )
        dest_pos = torch.where(
            is_gcs.unsqueeze(-1),
            self.gcs.view(1, 1, 3),
            peer_pos,
        )
        min_distance = 1.0
        distance = torch.linalg.vector_norm(
            dest_pos - self.positions,
            dim=-1,
        ).clamp_min(min_distance)
        receiver_active = torch.gather(
            self.active,
            1,
            receiver,
        )
        contact = torch.where(
            is_gcs,
            distance <= float(CONFIG["gcs_contact_range_m"]),
            torch.where(
                is_peer,
                distance <= float(CONFIG["peer_contact_range_m"]),
                torch.zeros_like(is_peer),
            ),
        )
        contact &= torch.where(
            is_peer,
            receiver_active,
            torch.ones_like(receiver_active),
        )

        frequency = float(CONFIG["uavnetsim_carrier_frequency_hz"])
        wavelength = 299_792_458.0 / frequency
        reference_gain = (wavelength / (4.0 * math.pi)) ** 2
        los_model = str(CONFIG["uavnetsim_los_model"]).lower()
        nlos_model = str(CONFIG["uavnetsim_nlos_model"]).lower()
        los_params = {
            "free_space": (2.0, 0.0),
            "log_distance": (2.1, 1.0),
        }
        nlos_params = {
            "urban": (3.0, 12.0),
            "suburban": (2.6, 8.0),
        }
        if los_model not in los_params:
            raise ValueError(f"unsupported UavNetSim LoS model: {los_model}")
        if nlos_model not in nlos_params:
            raise ValueError(f"unsupported UavNetSim NLoS model: {nlos_model}")
        los_exp, los_excess = los_params[los_model]
        nlos_exp, nlos_excess = nlos_params[nlos_model]

        # CSMA/CA normally serializes contenders. Applying every attempted
        # sender as simultaneous interference would badly under-estimate its
        # throughput; the expected backoff/contention model below accounts for
        # sharing. Explicit simultaneous interference is reserved for ALOHA-like
        # access where overlapping transmissions are part of the MAC semantics.
        mac_name = str(CONFIG["uavnetsim_mac_protocol"]).strip().upper()
        interference_enabled = (
            bool(CONFIG.get("full_gpu_network_interference", True))
            and mac_name in {"PURE_ALOHA", "ALOHA"}
        )
        if interference_enabled:
            tx_start = self.positions[:, :, None, :].expand(
                -1, -1, U, -1
            )
            rx_end = dest_pos[:, None, :, :].expand(
                -1, U, -1, -1
            )
            pair_distance = torch.linalg.vector_norm(
                rx_end - tx_start,
                dim=-1,
            ).clamp_min(min_distance)
            pair_nlos = self._segment_cylinder_blocked(
                tx_start,
                rx_end,
            )
            exponent = torch.where(
                pair_nlos,
                torch.full_like(pair_distance, nlos_exp),
                torch.full_like(pair_distance, los_exp),
            )
            excess_db = torch.where(
                pair_nlos,
                torch.full_like(pair_distance, nlos_excess),
                torch.full_like(pair_distance, los_excess),
            )
            gain = (
                reference_gain
                * torch.pow(pair_distance, -exponent)
                * torch.pow(
                    torch.full_like(pair_distance, 10.0),
                    -excess_db / 10.0,
                )
            )
            # UavNetSim treats a transmitter that is itself the selected
            # receiver as unit-gain self-interference (half-duplex conflict).
            tx_ids = torch.arange(U, device=self.device).view(1, U, 1)
            selected_receivers = receiver[:, None, :]
            self_interference = is_peer[:, None, :] & (
                tx_ids == selected_receivers
            )
            gain = torch.where(
                self_interference,
                torch.ones_like(gain),
                gain,
            )
            rx_matrix = tx_power[:, :, None] * gain
            desired = torch.diagonal(
                rx_matrix, dim1=1, dim2=2
            )
            active_tx = attempted_mask[:, :, None].to(rx_matrix.dtype)
            total_received = (rx_matrix * active_tx).sum(dim=1)
            interference = (
                total_received
                - desired * attempted_mask.to(desired.dtype)
            ).clamp_min(0.0)
            selected_nlos = torch.diagonal(
                pair_nlos, dim1=1, dim2=2
            )
        else:
            selected_nlos = self._segment_cylinder_blocked(
                self.positions,
                dest_pos,
            )
            exponent = torch.where(
                selected_nlos,
                torch.full_like(distance, nlos_exp),
                torch.full_like(distance, los_exp),
            )
            excess_db = torch.where(
                selected_nlos,
                torch.full_like(distance, nlos_excess),
                torch.full_like(distance, los_excess),
            )
            gain = (
                reference_gain
                * torch.pow(distance, -exponent)
                * torch.pow(
                    torch.full_like(distance, 10.0),
                    -excess_db / 10.0,
                )
            )
            desired = tx_power * gain
            interference = torch.zeros_like(desired)

        thermal_dbm = (
            float(CONFIG["uavnetsim_thermal_noise_density_dbm_hz"])
            + 10.0 * math.log10(float(CONFIG["uavnetsim_bandwidth_hz"]))
            + float(CONFIG["uavnetsim_receiver_noise_figure_db"])
        )
        noise_w = 10.0 ** ((thermal_dbm - 30.0) / 10.0)
        sinr = desired / (noise_w + interference).clamp_min(1e-30)
        sinr_db = 10.0 * torch.log10(sinr.clamp_min(1e-20))
        link_ok = (
            attempted_mask
            & contact
            & (
                sinr_db
                >= float(CONFIG["uavnetsim_sinr_threshold_db"])
            )
        )
        rate = torch.where(
            link_ok,
            torch.full_like(
                sinr_db,
                float(CONFIG["uavnetsim_bit_rate_bps"]),
            ),
            torch.zeros_like(sinr_db),
        )
        return {
            "receiver": receiver,
            "dest_pos": dest_pos,
            "distance": distance,
            "contact": contact,
            "nlos": selected_nlos,
            "signal_w": desired,
            "interference_w": interference,
            "sinr_db": sinr_db,
            "link_ok": link_ok,
            "rate_bps": rate,
        }


    def _network(
        self,
        destination_idx,
        power_action,
        energy_budget_j=None,
    ):
        E, U, T = (
            self.num_envs,
            self.num_uavs,
            self.num_targets,
        )
        report_size = float(CONFIG["report_bytes"])
        selected_dest = torch.gather(
            self.choices.unsqueeze(0).expand(
                E, -1, -1
            ),
            2,
            destination_idx.unsqueeze(-1),
        ).squeeze(-1)
        tx_power = float(CONFIG["tx_power_min_w"]) + (
            power_action.squeeze(-1)
            .clamp(-1.0, 1.0)
            + 1.0
        ) * 0.5 * (
            float(CONFIG["tx_power_max_w"])
            - float(CONFIG["tx_power_min_w"])
        )

        if energy_budget_j is None:
            energy_budget = torch.full(
                (E, U),
                float("inf"),
                device=self.device,
                dtype=self.positions.dtype,
            )
        else:
            energy_budget = torch.as_tensor(
                energy_budget_j,
                device=self.device,
                dtype=self.positions.dtype,
            )
            if energy_budget.shape != (E, U):
                raise ValueError(
                    "energy_budget_j has wrong shape"
                )
            if (
                not torch.isfinite(
                    energy_budget
                ).all()
                or (energy_budget < 0.0).any()
            ):
                raise ValueError(
                    "energy_budget_j must be finite and >= 0"
                )

        affordable_duration = torch.minimum(
            torch.full_like(
                tx_power,
                self.dt,
            ),
            energy_budget
            / tx_power.clamp_min(1e-12),
        ).clamp_min(0.0)
        energy_byte_budget = torch.floor(
            affordable_duration
            * float(
                CONFIG[
                    "comm_max_link_rate_bps"
                ]
            )
            / 8.0
        )

        attempted_all = torch.zeros(
            E, U, device=self.device, dtype=torch.bool
        )
        actual_all = torch.zeros(
            E, U, device=self.device
        )
        comm_energy_all = torch.zeros(
            E, U, device=self.device
        )
        new_delivery_total = torch.zeros(
            E, device=self.device, dtype=torch.long
        )
        sentinel = (
            self.max_steps
            + int(CONFIG["report_ttl"])
            + 1
        )
        target_ids = self.target_ids

        # Freeze the head report for every sender before any same-slot mutation.
        all_is_gcs = selected_dest == -1
        all_is_peer = selected_dest >= 0
        all_receivers = selected_dest.clamp(0, max(0, U - 1))
        receipt_index = all_receivers[:, :, None, None].expand(-1, -1, 1, T)
        initial_peer_received = torch.gather(
            self.peer_received, 2, receipt_index
        ).squeeze(2)
        initial_peer_source = torch.gather(
            self.peer_source, 2, receipt_index
        ).squeeze(2)
        initial_peer_created = torch.gather(
            self.peer_created, 2, receipt_index
        ).squeeze(2)
        initial_peer_complete = (
            (initial_peer_source == self.report_source)
            & (initial_peer_created == self.report_created)
            & (initial_peer_received >= report_size)
        )
        candidates = torch.where(
            all_is_gcs[:, :, None],
            self.report_valid & (self.report_remaining > 0.0),
            torch.where(
                all_is_peer[:, :, None],
                self.report_valid & ~initial_peer_complete,
                torch.zeros_like(self.report_valid),
            ),
        )
        intent_key = torch.where(
            candidates,
            self.report_created * (T + 1) + target_ids,
            torch.full_like(self.report_created, sentinel * (T + 1) + T),
        )
        intent_targets = intent_key.argmin(-1)
        intent_valid = (
            candidates.any(-1)
            & self.active
            & (energy_byte_budget > 0.0)
        )
        self._commit_completed_peer_receipts()

        network_model = self.network_model

        if network_model in {"packet_approx", "uavnetsim_gpu"}:
            pre_attempt = intent_valid
            channel_contenders = (
                pre_attempt.sum(-1)
                .float()
            )
            packet_payload_bytes = float(
                CONFIG[
                    "uavnetsim_payload_bytes"
                ]
            )
            queue_budget_bytes = (
                packet_payload_bytes
                * float(
                    CONFIG[
                        "uavnetsim_max_queue_size"
                    ]
                )
            )
            bit_rate = float(
                CONFIG[
                    "comm_max_link_rate_bps"
                ]
            )
            data_airtime_s = (
                packet_payload_bytes
                * 8.0
                / bit_rate
            )
            ack_airtime_s = (
                float(
                    CONFIG[
                        "full_gpu_packet_ack_bits"
                    ]
                )
                / bit_rate
            )
            avg_backoff_s = (
                (
                    float(
                        CONFIG[
                            "full_gpu_packet_cw_min"
                        ]
                    )
                    - 1.0
                )
                * 0.5
                * float(
                    CONFIG[
                        "full_gpu_packet_slot_us"
                    ]
                )
                * 1e-6
            )
            fixed_mac_overhead_s = (
                (
                    float(
                        CONFIG[
                            "full_gpu_packet_difs_us"
                        ]
                    )
                    + float(
                        CONFIG[
                            "full_gpu_packet_sifs_us"
                        ]
                    )
                )
                * 1e-6
                + ack_airtime_s
                + avg_backoff_s
            )
            packet_service_time_s = (
                data_airtime_s
                + fixed_mac_overhead_s
            )
            mac_service_budget_bytes = (
                math.floor(
                    self.dt
                    / max(
                        packet_service_time_s,
                        1e-12,
                    )
                )
                * packet_payload_bytes
            )
        else:
            channel_contenders = intent_valid.sum(-1).float()
            packet_payload_bytes = float(
                CONFIG[
                    "uavnetsim_payload_bytes"
                ]
            )
            queue_budget_bytes = float("inf")
            mac_service_budget_bytes = float(
                "inf"
            )

        uavnetsim_radio = None
        if network_model == "uavnetsim_gpu":
            # Pre-compute all selected-link radio quantities in one batched
            # CUDA pass. Report mutation below remains ordered per sender so
            # custody/delivery semantics stay identical and deterministic.
            uavnetsim_radio = self._uavnetsim_gpu_radio(
                selected_dest,
                tx_power,
                pre_attempt,
            )

        for sender in range(U):
            dest = selected_dest[:, sender]
            is_gcs = dest == -1
            is_peer = dest >= 0
            receiver = dest.clamp(
                0,
                max(0, U - 1),
            )

            report_valid = self.report_valid[
                :, sender, :
            ]
            peer_received = torch.gather(
                self.peer_received[:, sender, :, :],
                1,
                receiver[:, None, None].expand(
                    -1, 1, T
                ),
            ).squeeze(1)
            peer_source = torch.gather(
                self.peer_source[:, sender, :, :],
                1,
                receiver[:, None, None].expand(
                    -1, 1, T
                ),
            ).squeeze(1)
            peer_created = torch.gather(
                self.peer_created[:, sender, :, :],
                1,
                receiver[:, None, None].expand(
                    -1, 1, T
                ),
            ).squeeze(1)
            target = intent_targets[:, sender]
            selected_valid = (
                intent_valid[:, sender]
                & torch.gather(
                    report_valid, 1, target[:, None]
                ).squeeze(1)
            )
            attempted = (
                selected_valid
                & self.active[:, sender]
                & (is_gcs | is_peer)
            )
            attempted_all[:, sender] = attempted

            if uavnetsim_radio is not None:
                receiver = uavnetsim_radio["receiver"][:, sender]
                dest_pos = uavnetsim_radio["dest_pos"][:, sender, :]
                distance = uavnetsim_radio["distance"][:, sender]
                contact = uavnetsim_radio["contact"][:, sender]
                nlos = uavnetsim_radio["nlos"][:, sender]
                snr_db = uavnetsim_radio["sinr_db"][:, sender]
                rate = uavnetsim_radio["rate_bps"][:, sender]
                link_ok = (
                    uavnetsim_radio["link_ok"][:, sender]
                    & attempted
                )
                capacity_bytes = torch.where(
                    link_ok,
                    torch.floor(rate * self.dt / 8.0),
                    torch.zeros_like(rate),
                )
            else:
                peer_pos = torch.gather(
                    self.positions,
                    1,
                    receiver[:, None, None].expand(
                        -1, 1, 3
                    ),
                ).squeeze(1)
                dest_pos = torch.where(
                    is_gcs[:, None],
                    self.gcs.view(1, 3),
                    peer_pos,
                )
                distance = torch.linalg.vector_norm(
                    dest_pos - self.positions[:, sender, :],
                    dim=-1,
                ).clamp_min(
                    float(CONFIG["comm_reference_distance_m"])
                )
                contact = torch.where(
                    is_gcs,
                    distance <= float(CONFIG["gcs_contact_range_m"]),
                    torch.where(
                        is_peer,
                        distance <= float(CONFIG["peer_contact_range_m"]),
                        torch.zeros_like(is_peer),
                    ),
                )
                receiver_active = torch.gather(
                    self.active,
                    1,
                    receiver[:, None],
                ).squeeze(1)
                contact &= torch.where(
                    is_peer,
                    receiver_active,
                    torch.ones_like(receiver_active),
                )
                nlos = self._segment_cylinder_blocked(
                    self.positions[:, sender, :],
                    dest_pos,
                )
                gain_db = (
                    float(CONFIG["comm_reference_gain_db"])
                    - 10.0
                    * float(CONFIG["comm_path_loss_exponent"])
                    * torch.log10(
                        distance
                        / float(CONFIG["comm_reference_distance_m"])
                    )
                    - nlos.float()
                    * float(CONFIG["comm_nlos_additional_loss_db"])
                )
                rx_w = tx_power[:, sender] * torch.pow(
                    torch.tensor(10.0, device=self.device),
                    gain_db / 10.0,
                )
                noise_w = 10.0 ** (
                    (float(CONFIG["comm_noise_power_dbm"]) - 30.0)
                    / 10.0
                )
                snr = rx_w / noise_w
                snr_db = 10.0 * torch.log10(
                    snr.clamp_min(1e-20)
                )
                rate = float(CONFIG["comm_bandwidth_hz"]) * torch.log2(
                    1.0 + snr
                )
                rate = rate.clamp_max(
                    float(CONFIG["comm_max_link_rate_bps"])
                )
                link_ok = (
                    attempted
                    & contact
                    & (
                        snr_db
                        >= float(CONFIG["comm_snr_threshold_db"])
                    )
                )
                capacity_bytes = torch.where(
                    link_ok,
                    torch.floor(rate * self.dt / 8.0),
                    torch.zeros_like(rate),
                )
            if network_model in {"packet_approx", "uavnetsim_gpu"}:
                capacity_bytes = torch.floor(
                    torch.minimum(
                        torch.minimum(
                            capacity_bytes,
                            torch.full_like(
                                capacity_bytes,
                                queue_budget_bytes,
                            ),
                        ),
                        torch.full_like(
                            capacity_bytes,
                            float(
                                mac_service_budget_bytes
                            ),
                        ),
                    )
                    * torch.pow(
                        torch.full_like(
                            channel_contenders,
                            float(
                                CONFIG[
                                    "full_gpu_packet_contention_decay"
                                ]
                            ),
                        ),
                        (
                            channel_contenders
                            - 1.0
                        ).clamp_min(0.0),
                    )
                )

            capacity_bytes = torch.minimum(
                capacity_bytes,
                energy_byte_budget[:, sender],
            )

            target_oh = F.one_hot(
                target,
                T,
            ).bool()
            selected_source = torch.gather(
                self.report_source[
                    :, sender, :
                ],
                1,
                target[:, None],
            ).squeeze(1)
            selected_created = torch.gather(
                self.report_created[
                    :, sender, :
                ],
                1,
                target[:, None],
            ).squeeze(1)

            # GCS transfer: delivered-byte progress is shared by report copies
            # of the same generation.
            selected_remaining = torch.gather(
                self.report_remaining[
                    :, sender, :
                ],
                1,
                target[:, None],
            ).squeeze(1)
            gcs_actual = torch.minimum(
                capacity_bytes,
                selected_remaining,
            )
            gcs_actual = torch.where(
                is_gcs & selected_valid,
                gcs_actual,
                torch.zeros_like(gcs_actual),
            )
            gcs_new_remaining = (
                selected_remaining - gcs_actual
            ).clamp_min(0.0)
            copy_match = (
                self.report_valid
                & target_oh[:, None, :]
                & (
                    self.report_source
                    == selected_source[
                        :, None, None
                    ]
                )
                & (
                    self.report_created
                    == selected_created[
                        :, None, None
                    ]
                )
            )
            gcs_update = (
                is_gcs
                & link_ok
                & selected_valid
            )
            self.report_remaining = torch.where(
                copy_match
                & gcs_update[:, None, None],
                gcs_new_remaining[
                    :, None, None
                ],
                self.report_remaining,
            )
            gcs_complete = (
                gcs_update
                & (gcs_new_remaining <= 0.0)
            )
            delivered_target = (
                target_oh
                & gcs_complete[:, None]
            )
            new_delivery = (
                delivered_target
                & ~self.delivered
            )
            new_delivery_total += (
                new_delivery.sum(-1)
            )
            self.delivered |= delivered_target

            # Keep mission cleanup entirely tensorized: no CUDA->CPU scalar
            # synchronization is needed when no delivery completed.
            clear_target = delivered_target
            self.report_valid &= ~clear_target[
                :, None, :
            ]
            self.report_remaining = torch.where(
                self.report_valid,
                self.report_remaining,
                torch.zeros_like(
                    self.report_remaining
                ),
            )
            self.report_source = torch.where(
                self.report_valid,
                self.report_source,
                torch.full_like(
                    self.report_source,
                    -1,
                ),
            )
            self.pending_valid &= ~clear_target[
                :, None, :
            ]
            self.pending_remaining = torch.where(
                self.pending_valid,
                self.pending_remaining,
                torch.zeros_like(
                    self.pending_remaining
                ),
            )
            peer_clear = clear_target[
                :, None, None, :
            ]
            self.peer_received = torch.where(
                peer_clear,
                torch.zeros_like(
                    self.peer_received
                ),
                self.peer_received,
            )
            self.peer_source = torch.where(
                peer_clear,
                torch.full_like(
                    self.peer_source,
                    -1,
                ),
                self.peer_source,
            )

            # Peer transfer: progress is per sender-recipient-target and does
            # not consume/remove the sender's report.
            current_peer_received = torch.gather(
                peer_received,
                1,
                target[:, None],
            ).squeeze(1)
            current_peer_source = torch.gather(
                peer_source,
                1,
                target[:, None],
            ).squeeze(1)
            current_peer_created = torch.gather(
                peer_created,
                1,
                target[:, None],
            ).squeeze(1)
            generation_match = (
                (
                    current_peer_source
                    == selected_source
                )
                & (
                    current_peer_created
                    == selected_created
                )
            )
            base_peer_received = torch.where(
                generation_match,
                current_peer_received,
                torch.zeros_like(
                    current_peer_received
                ),
            )
            peer_remaining = (
                report_size - base_peer_received
            ).clamp_min(0.0)

            requested_bytes = torch.where(
                is_gcs,
                selected_remaining,
                torch.where(
                    is_peer,
                    peer_remaining,
                    torch.zeros_like(
                        selected_remaining
                    ),
                ),
            )
            max_attempt_bytes = float(
                math.floor(
                    float(
                        CONFIG[
                            "comm_max_link_rate_bps"
                        ]
                    )
                    * self.dt
                    / 8.0
                )
            )
            attempted_bytes = torch.minimum(
                torch.minimum(
                    requested_bytes,
                    torch.full_like(
                        requested_bytes,
                        max_attempt_bytes,
                    ),
                ),
                energy_byte_budget[:, sender],
            )
            tx_duration = torch.minimum(
                torch.full_like(
                    attempted_bytes,
                    self.dt,
                ),
                attempted_bytes
                * 8.0
                / float(
                    CONFIG[
                        "comm_max_link_rate_bps"
                    ]
                ),
            )
            comm_energy_all[:, sender] = torch.where(
                attempted,
                tx_power[:, sender]
                * tx_duration,
                torch.zeros_like(tx_duration),
            )

            peer_actual = torch.minimum(
                capacity_bytes,
                peer_remaining,
            )
            peer_actual = torch.where(
                is_peer
                & link_ok
                & selected_valid,
                peer_actual,
                torch.zeros_like(peer_actual),
            )
            new_peer_received = (
                base_peer_received + peer_actual
            ).clamp_max(report_size)
            receiver_oh = F.one_hot(
                receiver,
                U,
            ).bool()
            pair_mask = (
                receiver_oh[:, :, None]
                & target_oh[:, None, :]
                & (
                    is_peer
                    & selected_valid
                )[:, None, None]
            )
            peer_slice = self.peer_received[
                :, sender, :, :
            ]
            source_slice = self.peer_source[
                :, sender, :, :
            ]
            created_slice = self.peer_created[
                :, sender, :, :
            ]
            self.peer_received[
                :, sender, :, :
            ] = torch.where(
                pair_mask,
                new_peer_received[
                    :, None, None
                ],
                peer_slice,
            )
            self.peer_source[
                :, sender, :, :
            ] = torch.where(
                pair_mask,
                selected_source[
                    :, None, None
                ],
                source_slice,
            )
            self.peer_created[
                :, sender, :, :
            ] = torch.where(
                pair_mask,
                selected_created[
                    :, None, None
                ],
                created_slice,
            )
            actual_all[:, sender] = (
                gcs_actual + peer_actual
            )

        # Same-step custody commit after all radio transfers.
        self._commit_completed_peer_receipts()
        packets_tx = torch.where(
            actual_all > 0.0,
            torch.ceil(
                actual_all
                / packet_payload_bytes
            ),
            torch.zeros_like(actual_all),
        )
        return {
            "new_delivery": new_delivery_total,
            "comm_energy": comm_energy_all,
            "attempted": attempted_all,
            "bytes_tx": actual_all,
            "packets_tx": packets_tx,
            "channel_contenders": (
                channel_contenders
            ),
            "link_ok": (
                (uavnetsim_radio["link_ok"] & attempted_all)
                if uavnetsim_radio is not None
                else (actual_all > 0.0)
            ),
            "sinr_db": (
                uavnetsim_radio["sinr_db"]
                if uavnetsim_radio is not None
                else torch.full_like(actual_all, float("nan"))
            ),
            "nlos": (
                uavnetsim_radio["nlos"]
                if uavnetsim_radio is not None
                else torch.zeros_like(attempted_all)
            ),
            "interference_w": (
                uavnetsim_radio["interference_w"]
                if uavnetsim_radio is not None
                else torch.zeros_like(actual_all)
            ),
        }


    def _propulsion_energy(
        self,
        motion_result,
        active_mask=None,
    ):
        speed2 = (
            self.velocities * self.velocities
        ).sum(-1)
        realized = motion_result[
            "realized_accel"
        ]
        realized_norm = torch.linalg.vector_norm(
            realized,
            dim=-1,
            keepdim=True,
        )
        realized = realized * torch.minimum(
            torch.ones_like(realized_norm),
            torch.tensor(
                self.max_accel,
                device=self.device,
            )
            / realized_norm.clamp_min(1e-8),
        )
        accel2 = (realized * realized).sum(-1)
        commanded_accel = (
            torch.linalg.vector_norm(
                motion_result["commanded_motion"],
                dim=-1,
            )
            * self.max_accel
        )
        ground_idle = (
            (self.positions[..., 2] <= 0.0)
            & (speed2 <= 1e-12)
            & (commanded_accel <= 1e-12)
        )
        base = torch.where(
            ground_idle,
            torch.full_like(
                speed2,
                float(CONFIG["energy_idle_power_w"]),
            ),
            torch.full_like(
                speed2,
                float(CONFIG["energy_hover_power_w"]),
            ),
        )
        propulsion = (
            base
            + float(
                CONFIG["energy_speed_sq_coeff"]
            )
            * speed2
            + float(
                CONFIG["energy_accel_sq_coeff"]
            )
            * accel2
        ) * self.dt
        propulsion = torch.where(
            ground_idle,
            torch.full_like(
                speed2,
                float(
                    CONFIG["energy_idle_power_w"]
                )
                * self.dt,
            ),
            propulsion,
        )
        if active_mask is None:
            active_mask = self.active
        return (
            propulsion
            * active_mask.float()
        )

    def _energy(
        self,
        motion_result,
        comm_energy,
        propulsion_override=None,
        active_before=None,
    ):
        if active_before is None:
            active_before = self.active
        active_before = active_before.bool()

        if propulsion_override is None:
            propulsion = self._propulsion_energy(
                motion_result
            )
        else:
            propulsion = torch.as_tensor(
                propulsion_override,
                device=self.device,
                dtype=self.positions.dtype,
            )
            if propulsion.shape != (
                self.num_envs,
                self.num_uavs,
            ):
                raise ValueError(
                    "propulsion_override has wrong shape"
                )
            if (
                not torch.isfinite(
                    propulsion
                ).all()
                or (propulsion < 0.0).any()
            ):
                raise ValueError(
                    "propulsion_override must be finite and >= 0"
                )

        comm_energy = torch.as_tensor(
            comm_energy,
            device=self.device,
            dtype=self.positions.dtype,
        )
        if comm_energy.shape != (
            self.num_envs,
            self.num_uavs,
        ):
            raise ValueError(
                "comm_energy has wrong shape"
            )
        if (
            not torch.isfinite(
                comm_energy
            ).all()
            or (comm_energy < 0.0).any()
        ):
            raise ValueError(
                "comm_energy must be finite and >= 0"
            )

        available = self.battery.clamp_min(0.0)
        propulsion = torch.minimum(
            propulsion,
            available,
        )
        comm_energy = torch.minimum(
            comm_energy,
            (
                available
                - propulsion
            ).clamp_min(0.0),
        )
        total = (
            propulsion + comm_energy
        ) * active_before.float()
        self.battery = (
            self.battery - total
        ).clamp_min(0.0)
        self.active &= self.battery > 1e-12
        self.velocities = torch.where(
            self.active.unsqueeze(-1),
            self.velocities,
            torch.zeros_like(self.velocities),
        )
        return total


    def _expire_reports(self):
        ttl = float(CONFIG["report_ttl"])
        age = (
            self.step_count[:, None, None]
            - self.report_created
        )
        expired = (
            self.report_valid
            & (age.float() * self.dt >= ttl)
        )
        pending_age = (
            self.step_count[:, None, None]
            - self.pending_created
        )
        pending_expired = (
            self.pending_valid
            & (
                pending_age.float()
                * self.dt
                >= ttl
            )
        )
        expired_targets = (
            expired | pending_expired
        ).any(dim=1).sum(-1)

        self.report_valid &= ~expired
        self.pending_valid &= ~pending_expired
        self.report_remaining = torch.where(
            self.report_valid,
            self.report_remaining,
            torch.zeros_like(
                self.report_remaining
            ),
        )
        self.report_source = torch.where(
            self.report_valid,
            self.report_source,
            torch.full_like(
                self.report_source,
                -1,
            ),
        )
        self.pending_remaining = torch.where(
            self.pending_valid,
            self.pending_remaining,
            torch.zeros_like(
                self.pending_remaining
            ),
        )

        peer_age = (
            self.step_count[:, None, None, None]
            - self.peer_created
        )
        peer_expired = (
            (self.peer_received > 0.0)
            & (
                peer_age.float()
                * self.dt
                >= ttl
            )
        )
        self.peer_received = torch.where(
            peer_expired,
            torch.zeros_like(self.peer_received),
            self.peer_received,
        )
        self.peer_source = torch.where(
            peer_expired,
            torch.full_like(
                self.peer_source,
                -1,
            ),
            self.peer_source,
        )
        return expired_targets

    def _observation(self, entropy_map=None):
        E, U, O = self.num_envs, self.num_uavs, self.num_obstacles
        p = self.positions
        v = self.velocities
        time_frac = (self.step_count.float() / max(1, self.max_steps)).clamp(0.0, 1.0)
        self_state = torch.cat(
            (
                (p[..., :2] / self.map_size).clamp(0.0, 1.0),
                ((p[..., 2:3] - self.altitude_min) / max(1e-6, self.altitude_max - self.altitude_min)).clamp(0.0, 1.0),
                (v / self.max_speed).clamp(-1.0, 1.0),
                (self.battery / float(CONFIG["battery_j"])).unsqueeze(-1).clamp(0.0, 1.0),
                self.active.float().unsqueeze(-1),
                time_frac[:, None, None].expand(E, U, 1),
            ),
            dim=-1,
        )

        gcs_rel = (
            (self.gcs.view(1, 1, 3) - p)
            / self.xyz_scale.view(1, 1, 3)
        ).clamp(-1.0, 1.0)

        # Vectorized peer features while preserving CPU peer-ID order.
        all_relative = (
            p[:, None, :, :]
            - p[:, :, None, :]
        )
        gather_index = self.peer_index.view(
            1, U, U - 1, 1
        ).expand(E, -1, -1, 3)
        relative = torch.gather(
            all_relative,
            2,
            gather_index,
        )
        distance = torch.linalg.vector_norm(
            relative,
            dim=-1,
        )
        peer_active = torch.gather(
            self.active[:, None, :].expand(
                E, U, U
            ),
            2,
            self.peer_index.view(
                1, U, U - 1
            ).expand(E, -1, -1),
        )
        visible = (
            self.active[:, :, None]
            & peer_active
            & (
                distance
                <= float(
                    CONFIG["peer_contact_range_m"]
                )
            )
        )
        neighbors = torch.cat(
            (
                (
                    relative
                    / self.xyz_scale.view(
                        1, 1, 1, 3
                    )
                ).clamp(-1.0, 1.0),
                (
                    distance
                    / float(
                        CONFIG[
                            "peer_contact_range_m"
                        ]
                    )
                )
                .clamp(0.0, 1.0)
                .unsqueeze(-1),
                visible.float().unsqueeze(-1),
            ),
            dim=-1,
        )
        neighbors = torch.where(
            visible.unsqueeze(-1),
            neighbors,
            torch.zeros_like(neighbors),
        )

        # Nearest 3 obstacles.
        odelta = self.obstacle_xy[:, None, :, :] - p[:, :, None, :2]
        odist = torch.linalg.vector_norm(odelta, dim=-1)
        ok = min(int(CONFIG["observation_nearest_obstacles"]), O)
        _, oi = torch.topk(odist, k=ok, dim=-1, largest=False)
        oxy = torch.gather(
            self.obstacle_xy[:, None, :, :].expand(E, U, O, 2),
            2, oi.unsqueeze(-1).expand(-1, -1, -1, 2)
        )
        orad = torch.gather(self.obstacle_radius[:, None, :].expand(E, U, O), 2, oi)
        oh = torch.gather(self.obstacle_height[:, None, :].expand(E, U, O), 2, oi)
        orel = oxy - p[:, :, None, :2]
        center_distance = torch.linalg.vector_norm(orel, dim=-1)
        obstacles = torch.cat(
            (
                (orel / self.map_size).clamp(-1.0, 1.0),
                (center_distance / (math.sqrt(2.0) * self.map_size))
                .clamp(0.0, 1.0)
                .unsqueeze(-1),
                (orad / self.map_size).clamp(0.0, 1.0).unsqueeze(-1),
                (oh / max(self.altitude_max, 1.0)).clamp(0.0, 1.0).unsqueeze(-1),
            ),
            dim=-1,
        )

        missing_obstacles = int(CONFIG["observation_nearest_obstacles"]) - ok
        if missing_obstacles > 0:
            obstacles = F.pad(obstacles, (0, 0, 0, missing_obstacles))

        # 13x13 local belief patch.
        center = torch.floor(
            p[..., :2] / self.cell_size
        ).long()
        pc = (
            center.unsqueeze(2)
            + self.belief_patch_offsets.view(
                1, 1, -1, 2
            )
        )
        pxg, pyg = pc[..., 0], pc[..., 1]
        pvalid = (pxg >= 0) & (pxg < self.grid_n) & (pyg >= 0) & (pyg < self.grid_n)
        pidx = pyg.clamp(0, self.grid_n - 1) * self.grid_n + pxg.clamp(0, self.grid_n - 1)
        patch = torch.gather(self.belief.view(E, U, -1), 2, pidx)
        patch = torch.where(pvalid, patch, torch.full_like(patch, float(CONFIG["belief_prior"])))

        coarse_n = int(CONFIG["belief_coarse_cells"])
        if entropy_map is None:
            entropy_map = self._belief_entropy()
        coarse = F.adaptive_avg_pool2d(
            entropy_map.view(E * U, 1, self.grid_n, self.grid_n),
            (coarse_n, coarse_n),
        ).view(E, U, -1).clamp(0.0, 1.0)

        valid_count = self.report_valid.sum(-1).float()
        oldest = torch.where(
            self.report_valid,
            self.report_created,
            torch.full_like(self.report_created, self.max_steps + int(CONFIG["report_ttl"]) + 1),
        ).amin(-1)
        age = torch.where(
            valid_count > 0,
            (self.step_count[:, None] - oldest).float() * self.dt / max(float(CONFIG["report_ttl"]), 1.0),
            torch.zeros_like(valid_count),
        ).clamp(0.0, 1.0)
        capacity_reports = max(
            1.0, float(CONFIG["buffer_bytes"]) / float(CONFIG["report_bytes"])
        )
        used_fraction = (valid_count / capacity_reports).clamp(0.0, 1.0)
        report_fraction = used_fraction
        pending_fraction = (
            self.pending_valid.sum(-1).float()
            / max(
                1.0,
                float(CONFIG["pending_buffer_bytes"]) / float(CONFIG["report_bytes"]),
            )
        ).clamp(0.0, 1.0)
        ordered_created = torch.where(
            self.report_valid,
            self.report_created,
            torch.full_like(
                self.report_created,
                self.max_steps + int(CONFIG["report_ttl"]) + 1,
            ),
        )
        head_target = ordered_created.argmin(-1)
        head_remaining = torch.gather(
            self.report_remaining, 2, head_target.unsqueeze(-1)
        ).squeeze(-1)
        head_delivery = torch.where(
            valid_count > 0,
            (1.0 - head_remaining / float(CONFIG["report_bytes"])).clamp(0.0, 1.0),
            torch.zeros_like(valid_count),
        )
        buffer_features = torch.stack(
            (used_fraction, report_fraction, age, pending_fraction, head_delivery),
            dim=-1,
        )

        dest_mask = self._destination_mask()
        obs = torch.cat(
            (
                self_state,
                gcs_rel,
                neighbors.reshape(E, U, -1),
                obstacles.reshape(E, U, -1),
                patch,
                coarse,
                buffer_features,
                dest_mask,
            ),
            dim=-1,
        ).float()
        if obs.shape[-1] != self.observation_dim:
            raise RuntimeError(f"GPU observation dim {obs.shape[-1]} != {self.observation_dim}")
        return obs, dest_mask

    def _report_slots(self, valid, remaining, created, slot_count=None):
        T = valid.shape[-1]
        if slot_count is None:
            slot_count = int(float(CONFIG["buffer_bytes"]) // float(CONFIG["report_bytes"]))
        target_ids = torch.arange(T, device=self.device).view(1, 1, T)
        sentinel = self.max_steps + int(CONFIG["report_ttl"]) + 1
        key = torch.where(
            valid,
            created * (T + 1) + target_ids,
            torch.full_like(created, sentinel * (T + 1) + T),
        )
        order = torch.argsort(key, dim=-1)[..., :slot_count]
        chosen_valid = torch.gather(valid, 2, order)
        chosen_remaining = torch.gather(remaining, 2, order)
        chosen_created = torch.gather(created, 2, order)
        age = (
            (self.step_count[:, None, None] - chosen_created).float()
            * self.dt
            / max(float(CONFIG["report_ttl"]), 1.0)
        ).clamp(0.0, 1.0)
        delivery = (1.0 - chosen_remaining / float(CONFIG["report_bytes"])).clamp(
            0.0, 1.0
        )
        target_fraction = order.float() / max(1.0, float(T - 1))
        slots = torch.stack(
            (
                chosen_valid.float(),
                target_fraction,
                age,
                delivery,
            ),
            dim=-1,
        )
        slots = torch.where(chosen_valid.unsqueeze(-1), slots, torch.zeros_like(slots))
        if slots.shape[-2] < slot_count:
            slots = F.pad(slots, (0, 0, 0, slot_count - slots.shape[-2]))
        return slots

    def _state(self, obs, entropy_map=None):
        E, U = self.num_envs, self.num_uavs
        buffer_slots = self._report_slots(
            self.report_valid, self.report_remaining, self.report_created
        )
        pending_slots = self._report_slots(
            self.pending_valid, self.pending_remaining, self.pending_created,
            slot_count=int(float(CONFIG["pending_buffer_bytes"]) // float(CONFIG["report_bytes"])),
        )
        if entropy_map is None:
            entropy_map = self._belief_entropy()
        entropy = entropy_map
        summary = torch.stack(
            (
                entropy.mean(dim=(-2, -1)).clamp(0.0, 1.0),
                ((self.belief - float(CONFIG["belief_prior"])).abs() > 0.05)
                .float()
                .mean(dim=(-2, -1)),
            ),
            dim=-1,
        )
        critic_cells = int(CONFIG["critic_belief_grid_cells"])
        pooled_belief = F.adaptive_avg_pool2d(
            self.belief.reshape(E * U, 1, self.grid_n, self.grid_n),
            (critic_cells, critic_cells),
        ).reshape(E, U, -1)

        per_uav = torch.cat(
            (
                obs,
                buffer_slots.reshape(E, U, -1),
                pending_slots.reshape(E, U, -1),
                summary,
                pooled_belief,
            ),
            dim=-1,
        ).reshape(E, -1)

        targets = torch.cat(
            (
                (self.target_xy / self.map_size).clamp(0.0, 1.0),
                self.confirmed.float().unsqueeze(-1),
                self.delivered.float().unsqueeze(-1),
            ),
            dim=-1,
        ).reshape(E, -1)
        obstacles = torch.cat(
            (
                (self.obstacle_xy / self.map_size).clamp(0.0, 1.0),
                (self.obstacle_radius / self.map_size).clamp(0.0, 1.0).unsqueeze(-1),
                (self.obstacle_height / max(self.altitude_max, 1.0))
                .clamp(0.0, 1.0)
                .unsqueeze(-1),
            ),
            dim=-1,
        ).reshape(E, -1)
        time_fraction = (
            self.step_count.float() / max(1, self.max_steps)
        ).clamp(0.0, 1.0).unsqueeze(-1)
        state = torch.cat((per_uav, targets, obstacles, time_fraction), dim=-1)
        if state.shape[-1] != self.state_dim:
            raise RuntimeError(
                f"GPU global state dim {state.shape[-1]} != {self.state_dim}"
            )
        return state.float()

    @torch.no_grad()
    def step(self, continuous_action, destination_idx):
        if continuous_action.shape != (
            self.num_envs,
            self.num_uavs,
            self.continuous_dim,
        ):
            raise ValueError("continuous_action has wrong shape")
        if destination_idx.shape != (self.num_envs, self.num_uavs):
            raise ValueError("destination_idx has wrong shape")

        self.step_count += 1
        expired = self._expire_reports()
        self._flush_pending()
        old_potential = self.belief_potential
        old_confirmed = self.confirmed.clone()
        old_delivered = self.delivered.clone()

        positions_before_motion = self.positions.clone()
        velocities_before_motion = self.velocities.clone()
        battery_before_step = self.battery.clone()
        active_before_step = self.active.clone()

        motion = self._motion(
            continuous_action[..., :3]
        )
        propulsion_required_full = (
            self._propulsion_energy(
                motion,
                active_mask=(
                    active_before_step
                ),
            )
        )
        battery_fraction = torch.where(
            propulsion_required_full > 1e-12,
            (
                battery_before_step
                / propulsion_required_full.clamp_min(
                    1e-12
                )
            ).clamp(
                0.0,
                1.0,
            ),
            torch.ones_like(
                propulsion_required_full
            ),
        )
        battery_limited = (
            active_before_step
            & (
                propulsion_required_full
                > battery_before_step + 1e-6
            )
        )
        self.positions = torch.where(
            battery_limited.unsqueeze(-1),
            positions_before_motion
            + battery_fraction.unsqueeze(-1)
            * (
                self.positions
                - positions_before_motion
            ),
            self.positions,
        )
        self.velocities = torch.where(
            battery_limited.unsqueeze(-1),
            self.velocities
            * battery_fraction.unsqueeze(-1),
            self.velocities,
        )

        # Re-run monotone continuous-time peer safety on the scaled segments.
        blocked = motion[
            "blocked"
        ].clone()
        blocked_by_peer = motion[
            "blocked_by_peer"
        ].clone()
        eye = torch.eye(
            self.num_uavs,
            device=self.device,
            dtype=torch.bool,
        ).unsqueeze(0)
        active_pair = (
            active_before_step.unsqueeze(2)
            & active_before_step.unsqueeze(1)
            & ~eye
        )
        safety = float(
            CONFIG["safety_distance"]
        )

        for _ in range(
            self.num_uavs
        ):
            effective = torch.where(
                blocked.unsqueeze(-1),
                positions_before_motion,
                self.positions,
            )
            relative_start = (
                positions_before_motion.unsqueeze(2)
                - positions_before_motion.unsqueeze(1)
            )
            displacement = (
                effective
                - positions_before_motion
            )
            relative_motion = (
                displacement.unsqueeze(2)
                - displacement.unsqueeze(1)
            )
            denominator = (
                relative_motion
                * relative_motion
            ).sum(-1)
            t_closest = torch.where(
                denominator > 1e-12,
                -(
                    relative_start
                    * relative_motion
                ).sum(-1)
                / denominator.clamp_min(
                    1e-12
                ),
                torch.zeros_like(
                    denominator
                ),
            ).clamp(
                0.0,
                1.0,
            )
            relative_at_closest = (
                relative_start
                + t_closest.unsqueeze(-1)
                * relative_motion
            )
            min_distance = (
                torch.linalg.vector_norm(
                    relative_at_closest,
                    dim=-1,
                )
            )
            pair_bad = (
                (min_distance < safety)
                & active_pair
            )
            peer_now = pair_bad.any(-1)
            blocked_by_peer |= peer_now
            blocked |= peer_now

        self.positions = torch.where(
            blocked.unsqueeze(-1),
            positions_before_motion,
            self.positions,
        )
        self.velocities = torch.where(
            blocked.unsqueeze(-1),
            torch.zeros_like(
                self.velocities
            ),
            self.velocities,
        )
        motion["blocked"] = blocked
        motion[
            "blocked_by_peer"
        ] = blocked_by_peer
        motion[
            "realized_accel"
        ] = (
            self.velocities
            - velocities_before_motion
        ) / self.dt

        propulsion_required_final = (
            self._propulsion_energy(
                motion,
                active_mask=(
                    active_before_step
                ),
            )
        )
        newly_energy_limited = (
            active_before_step
            & ~battery_limited
            & (
                propulsion_required_final
                > battery_before_step
                + 1e-6
            )
        )
        battery_limited |= (
            newly_energy_limited
        )
        battery_fraction = torch.where(
            newly_energy_limited,
            torch.zeros_like(
                battery_fraction
            ),
            battery_fraction,
        )
        self.positions = torch.where(
            newly_energy_limited.unsqueeze(-1),
            positions_before_motion,
            self.positions,
        )
        self.velocities = torch.where(
            newly_energy_limited.unsqueeze(-1),
            torch.zeros_like(
                self.velocities
            ),
            self.velocities,
        )
        motion[
            "realized_accel"
        ] = (
            self.velocities
            - velocities_before_motion
        ) / self.dt
        motion[
            "battery_limited"
        ] = battery_limited
        motion[
            "battery_fraction"
        ] = battery_fraction

        propulsion_charged = torch.where(
            battery_limited,
            battery_before_step,
            torch.minimum(
                propulsion_required_final,
                battery_before_step,
            ),
        )
        self.active = (
            active_before_step
            & ~battery_limited
        )
        communication_budget = (
            battery_before_step
            - propulsion_charged
        ).clamp_min(0.0)
        communication_budget = torch.where(
            self.active,
            communication_budget,
            torch.zeros_like(
                communication_budget
            ),
        )

        sensing = self._sense()
        if self.external_network is None:
            network = self._network(
                destination_idx.long(),
                continuous_action[..., 3:4],
                energy_budget_j=(
                    communication_budget
                ),
            )
        else:
            network = self.external_network.step(
                destination_idx.long(),
                continuous_action[..., 3:4],
                energy_budget_j=(
                    communication_budget
                ),
            )
        network["comm_energy"] = torch.minimum(
            network["comm_energy"],
            communication_budget,
        )
        self._flush_pending()
        total_energy = self._energy(
            motion,
            network["comm_energy"],
            propulsion_override=(
                propulsion_charged
            ),
            active_before=(
                active_before_step
            ),
        )
        entropy_map = self._belief_entropy()
        new_potential = self._belief_potential(
            entropy_map
        )
        self.belief_potential = new_potential

        newly_confirmed = (self.confirmed & ~old_confirmed).sum(-1)
        newly_delivered = (self.delivered & ~old_delivered).sum(-1)
        success = self.delivered.all(-1)
        inactive = ~self.active.any(-1)
        horizon = self.step_count >= self.max_steps
        done = success | inactive | horizon
        next_phi = torch.where(done, torch.zeros_like(new_potential), new_potential)
        shaping = float(CONFIG["reward_info_gain"]) * (
            float(CONFIG["reward_shaping_gamma"]) * next_phi - old_potential
        )
        reward = (
            shaping
            + float(CONFIG["reward_confirmation"]) * newly_confirmed.float()
            + float(CONFIG["reward_delivery"]) * newly_delivered.float()
            - float(CONFIG["reward_false_confirmation"]) * sensing["false_confirmation"].float()
            - float(CONFIG["reward_blocked"]) * motion["blocked"].sum(-1).float()
            - float(CONFIG["reward_boundary"]) * motion["boundary"].sum(-1).float()
            - float(CONFIG["reward_expired_report"]) * expired.float()
            - float(CONFIG["reward_dropped_report"]) * sensing["dropped"].float()
            - float(CONFIG["reward_energy_per_kj"]) * total_energy.sum(-1) / 1000.0
            - float(CONFIG["reward_step_penalty"])
            + torch.where(success, torch.full_like(shaping, float(CONFIG["reward_all_delivered_bonus"])), torch.zeros_like(shaping))
        )

        obs, mask = self._observation(
            entropy_map=entropy_map
        )
        state = self._state(
            obs,
            entropy_map=entropy_map,
        )
        metrics = {
            "success": success,
            "new_confirmed": newly_confirmed,
            "new_delivered": newly_delivered,
            "blocked": motion["blocked"].sum(-1),
            "boundary": motion["boundary"].sum(-1),
            "energy_j": total_energy.sum(-1),
        }
        return obs, state, mask, reward.float(), done, metrics

    def assert_all_state_on_cuda(self):
        tensors = {
            k: v
            for k, v in self.__dict__.items()
            if isinstance(v, torch.Tensor)
        }
        bad = [k for k, v in tensors.items() if v.device.type != "cuda"]
        if bad:
            raise AssertionError(f"state tensors not on CUDA: {bad}")
        return len(tensors)


## GPU replay buffer

GpuReplayBuffer stores transitions directly on the selected torch.device and samples training batches without intentionally moving replay data back to CPU.

The class remains one cell for the same reason as FullGpuUAVBatchEnv.


In [ ]:
class GpuReplayBuffer:
    """Tensor replay storage with portable checkpoints and unique ring writes."""

    _tensor_fields = (
        "observations", "next_observations", "states", "next_states",
        "continuous_actions", "destination_indices", "destination_masks",
        "next_destination_masks", "rewards", "terminated", "truncated",
    )
    _dimension_fields = (
        "capacity", "num_agents", "observation_dim", "state_dim",
        "continuous_dim", "discrete_dim",
    )

    def __init__(self, capacity, num_agents, observation_dim, state_dim,
                 continuous_dim, discrete_dim, device, seed=44, strict_cuda=True):
        self.capacity = int(capacity)
        self.num_agents = int(num_agents)
        self.observation_dim = int(observation_dim)
        self.state_dim = int(state_dim)
        self.continuous_dim = int(continuous_dim)
        self.discrete_dim = int(discrete_dim)
        if any(getattr(self, name) < 1 for name in self._dimension_fields):
            raise ValueError("replay dimensions and capacity must be >= 1")
        self.device = torch.device(device)
        if strict_cuda:
            _require_cuda(self.device)
        C, U, O, S, A, D = (self.capacity, self.num_agents, self.observation_dim,
                            self.state_dim, self.continuous_dim, self.discrete_dim)
        self.observations = torch.empty(C, U, O, device=self.device)
        self.next_observations = torch.empty_like(self.observations)
        self.states = torch.empty(C, S, device=self.device)
        self.next_states = torch.empty_like(self.states)
        self.continuous_actions = torch.empty(C, U, A, device=self.device)
        self.destination_indices = torch.empty(C, U, device=self.device, dtype=torch.long)
        self.destination_masks = torch.empty(C, U, D, device=self.device)
        self.next_destination_masks = torch.empty_like(self.destination_masks)
        self.rewards = torch.empty(C, 1, device=self.device)
        self.terminated = torch.empty(C, 1, device=self.device)
        self.truncated = torch.zeros(C, 1, device=self.device)
        self.position = 0
        self.size = 0
        self.generator = torch.Generator(device=self.device)
        self.generator.manual_seed(int(seed))

    def __len__(self):
        return int(self.size)

    @torch.no_grad()
    def add_batch(self, observations, states, continuous_actions, destination_indices,
                  destination_masks, rewards, next_observations, next_states,
                  next_destination_masks, terminated, truncated=None):
        batch_size = int(observations.shape[0])
        if batch_size == 0:
            return
        if truncated is None:
            truncated = torch.zeros_like(terminated)
        values = {
            "observations": observations,
            "next_observations": next_observations,
            "states": states,
            "next_states": next_states,
            "continuous_actions": continuous_actions,
            "destination_indices": destination_indices,
            "destination_masks": destination_masks,
            "next_destination_masks": next_destination_masks,
            "rewards": rewards.reshape(batch_size, 1),
            "terminated": terminated.reshape(batch_size, 1),
            "truncated": truncated.reshape(batch_size, 1),
        }
        for name, value in values.items():
            expected = (batch_size,) + tuple(getattr(self, name).shape[1:])
            if tuple(value.shape) != expected:
                raise ValueError(f"replay {name} has wrong shape: {tuple(value.shape)} != {expected}")
        # An oversized insertion must retain the newest capacity rows. Dropping
        # its overwritten prefix makes destination indices unique on CUDA.
        skip = max(0, batch_size - self.capacity)
        retained = batch_size - skip
        indices = (torch.arange(retained, device=self.device)
                   + self.position + skip) % self.capacity
        for name, value in values.items():
            destination = getattr(self, name)
            destination[indices] = value[skip:].to(device=self.device, dtype=destination.dtype)
        self.position = (self.position + batch_size) % self.capacity
        self.size = min(self.capacity, self.size + batch_size)

    def sample(self, batch_size, device):
        device = torch.device(device)
        if device != self.device:
            raise ValueError("GpuReplayBuffer sample device must equal storage device")
        batch_size = int(batch_size)
        if batch_size < 1:
            raise ValueError("batch_size must be >= 1")
        if self.size < batch_size:
            raise ValueError("not enough replay samples")
        indices = torch.randint(0, self.size, (batch_size,),
                                generator=self.generator, device=self.device)
        return {name: getattr(self, name)[indices] for name in self._tensor_fields}

    def state_dict(self):
        state = {name: int(getattr(self, name)) for name in self._dimension_fields}
        state.update(format_version=2, storage="torch", position=int(self.position),
                     size=int(self.size), generator_device_type=self.device.type,
                     generator_state=self.generator.get_state().cpu().clone())
        for name in self._tensor_fields:
            state[name] = getattr(self, name)[:self.size].detach().cpu().clone()
        return state

    @torch.no_grad()
    def load_state_dict(self, state):
        if not isinstance(state, dict):
            raise TypeError("replay state must be a dict")
        if int(state.get("format_version", -1)) != 2 or state.get("storage") != "torch":
            raise ValueError("unsupported GPU replay checkpoint format")
        for name in self._dimension_fields:
            if int(state[name]) != int(getattr(self, name)):
                raise ValueError(f"replay {name} mismatch")
        size, position = int(state["size"]), int(state["position"])
        if not 0 <= size <= self.capacity or not 0 <= position < self.capacity:
            raise ValueError("invalid replay size or position")
        if size < self.capacity and position != size:
            raise ValueError("partial replay position must equal size")
        if state.get("generator_device_type") != self.device.type:
            raise ValueError("exact replay RNG restoration requires the same device type")
        generator_state = state["generator_state"]
        if not isinstance(generator_state, torch.Tensor):
            raise TypeError("generator_state must be a tensor")
        # Validate the complete checkpoint before mutating storage.
        restored_generator = torch.Generator(device=self.device)
        restored_generator.set_state(generator_state.detach().cpu())
        for name in self._tensor_fields:
            value = state[name]
            if not isinstance(value, torch.Tensor):
                raise TypeError(f"replay {name} must be a tensor")
            if tuple(value.shape) != tuple(getattr(self, name)[:size].shape):
                raise ValueError(f"replay {name} has wrong shape")
        for name in self._tensor_fields:
            destination = getattr(self, name)
            destination[:size].copy_(state[name].to(device=self.device, dtype=destination.dtype))
        self.size, self.position = size, position
        self.generator = restored_generator


## Training and benchmark helpers

From this point onward, each **top-level function** or **top-level class** is placed in a separate code cell. This keeps execution order explicit and makes helpers easier to read and test independently.


In [ ]:
import importlib
import os


### Project/reference loading


In [ ]:
def load_project_namespace(repo: Path | None = None):
    """Return an explicit embedded project namespace without reading files."""
    _ = repo
    ns = {
        "gym": gym,
        "np": np,
        "threading": threading,
        "UAVSearchEnv": UAVSearchEnv,
        "HybridMASAC": HybridMASAC,
        "HybridMATD3": HybridMATD3,
        "HybridReplayBuffer": HybridReplayBuffer,
        "Report": Report,
        "UAV": UAV,
        "Obstacle": Obstacle,
        "HybridAction": HybridAction,
        "UavNetSimBackend": UavNetSimBackend,
        "project_motion_action_tensor": project_motion_action_tensor,
        "radial_squash_motion_action": radial_squash_motion_action,
        "straight_through_categorical_sample": straight_through_categorical_sample,
        "straight_through_masked_argmax": straight_through_masked_argmax,
        "train_masac": train_masac,
        "train_matd3": train_matd3,
        "set_seed": set_seed,
        "uavnetsim_available": uavnetsim_available,
        "build_transmission_intent": build_transmission_intent,
        "sample_valid_random_actions": sample_valid_random_actions,
        "add_environment_transition_to_replay": add_environment_transition_to_replay,
        "_select_vector_policy_actions": _select_vector_policy_actions,
        "_vector_observation_at": _vector_observation_at,
        "_stack_vector_actions": _stack_vector_actions,
        "_count_vector_updates_due": _count_vector_updates_due,
        "running_on_kaggle": running_on_kaggle,
        "configure_kaggle_wandb": configure_kaggle_wandb,
    }
    ns["_TORCH_SENSING_TLS"] = _get_torch_sensing_tls()
    return ns



### Learner construction and action helpers


In [ ]:
def _trainer(
    repo: Path,
    algorithm: str,
    device: str,
    seed: int,
):
    """Build the existing MASAC/MATD3 learner without constructing a CPU env.

    Only lightweight Gymnasium space metadata is created on the host. No CPU
    mission reset/step/sensing/network simulation is executed by the full-GPU
    training path.
    """
    random.seed(int(seed))
    torch.manual_seed(int(seed))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(seed))

    _ = repo
    gym_module = __import__("gymnasium")
    np_module = __import__("numpy")
    ns = {
        "gym": gym_module,
        "np": np_module,
        "UAVSearchEnv": UAVSearchEnv,
        "HybridMASAC": HybridMASAC,
        "HybridMATD3": HybridMATD3,
        "HybridReplayBuffer": HybridReplayBuffer,
        "UAV": UAV,
        "Obstacle": Obstacle,
        "Report": Report,
        "UavNetSimBackend": UavNetSimBackend,
        "HybridAction": HybridAction,
        "build_transmission_intent": build_transmission_intent,
        "project_motion_action_tensor": project_motion_action_tensor,
        "radial_squash_motion_action": radial_squash_motion_action,
        "straight_through_categorical_sample": straight_through_categorical_sample,
        "straight_through_masked_argmax": straight_through_masked_argmax,
    }
    gym = ns["gym"]
    np = ns["np"]
    env_cls = ns["UAVSearchEnv"]
    dummy = object.__new__(env_cls)

    U = int(CONFIG["num_uavs"])
    T = int(CONFIG["num_targets"])
    O = int(CONFIG["num_obstacles"])
    D = U + 1
    nearest = int(CONFIG["observation_nearest_obstacles"])
    patch = int(CONFIG["belief_patch_cells"])
    coarse = int(CONFIG["belief_coarse_cells"])
    critic = int(CONFIG["critic_belief_grid_cells"])
    agent_ids = tuple(f"uav_{index}" for index in range(U))
    dummy.agent_ids = agent_ids

    agent_space = gym.spaces.Dict(
        {
            "self_state": gym.spaces.Box(-1.0, 1.0, shape=(9,), dtype=np.float32),
            "gcs_relative": gym.spaces.Box(-1.0, 1.0, shape=(3,), dtype=np.float32),
            "neighbors": gym.spaces.Box(
                -1.0, 1.0, shape=(U - 1, 5), dtype=np.float32
            ),
            "obstacles": gym.spaces.Box(
                -1.0, 1.0, shape=(nearest, 5), dtype=np.float32
            ),
            "belief_patch": gym.spaces.Box(
                0.0, 1.0, shape=(patch, patch), dtype=np.float32
            ),
            "belief_coarse": gym.spaces.Box(
                0.0, 1.0, shape=(coarse, coarse), dtype=np.float32
            ),
            "buffer": gym.spaces.Box(0.0, 1.0, shape=(5,), dtype=np.float32),
            "destination_mask": gym.spaces.Box(
                0.0, 1.0, shape=(D,), dtype=np.float32
            ),
        }
    )
    dummy.observation_space = gym.spaces.Dict(
        {agent_id: agent_space for agent_id in agent_ids}
    )
    dummy.action_space = gym.spaces.Dict(
        {
            agent_id: gym.spaces.Dict(
                {"destination": gym.spaces.Discrete(D)}
            )
            for agent_id in agent_ids
        }
    )

    observation_dim = (
        9 + 3 + 5 * (U - 1) + 5 * nearest
        + patch * patch + coarse * coarse + 5 + D
    )
    buffer_slots = int(
        float(CONFIG["buffer_bytes"]) // float(CONFIG["report_bytes"])
    )
    pending_slots = int(
        float(CONFIG["pending_buffer_bytes"]) // float(CONFIG["report_bytes"])
    )
    per_uav_state = (
        observation_dim
        + 4 * buffer_slots
        + 4 * pending_slots
        + 2
        + critic * critic
    )
    state_dim = U * per_uav_state + 4 * T + 4 * O + 1
    dummy.state_space = gym.spaces.Box(
        low=-np.inf,
        high=np.inf,
        shape=(state_dim,),
        dtype=np.float32,
    )

    if algorithm == "masac":
        trainer = ns["HybridMASAC"](
            dummy,
            device=device,
        )
    else:
        trainer = ns["HybridMATD3"](
            dummy,
            device=device,
            seed=int(seed),
        )
    trainer._project_namespace = ns
    return trainer


In [ ]:
def make_project_gpu_replay_buffer(
    trainer,
    *,
    capacity,
    num_agents,
    observation_dim,
    state_dim,
    continuous_dim,
    discrete_dim,
    device,
    seed,
    strict_cuda=True,
):
    project_ns = getattr(
        trainer,
        "_project_namespace",
        None,
    )
    if not isinstance(project_ns, dict):
        raise RuntimeError(
            "trainer is missing project namespace"
        )
    base = project_ns[
        "HybridReplayBuffer"
    ]

    class ProjectGpuReplayBuffer(
        GpuReplayBuffer,
        base,
    ):
        pass

    return ProjectGpuReplayBuffer(
        capacity=capacity,
        num_agents=num_agents,
        observation_dim=observation_dim,
        state_dim=state_dim,
        continuous_dim=continuous_dim,
        discrete_dim=discrete_dim,
        device=device,
        seed=seed,
        strict_cuda=strict_cuda,
    )


In [ ]:
def _matd3_actions(
    trainer,
    obs,
    mask,
    transition_steps,
    generator,
):
    with torch.no_grad():
        sampled = trainer._actor_joint_actions(
            obs,
            mask,
            straight_through=False,
            target_actor=False,
        )
        continuous = sampled["continuous"]
        destination = sampled[
            "destination_indices"
        ]
        noise = (
            torch.randn(
                continuous.shape,
                device=continuous.device,
                generator=generator,
            )
            * float(
                CONFIG[
                    "matd3_exploration_noise"
                ]
            )
        )
        continuous = (
            continuous + noise
        ).clamp(-1.0, 1.0)
        motion = continuous[..., :3]
        norm = torch.linalg.vector_norm(
            motion,
            dim=-1,
            keepdim=True,
        ).clamp_min(1.0)
        continuous = torch.cat(
            (
                motion / norm,
                continuous[..., 3:4],
            ),
            dim=-1,
        )

        start = float(
            CONFIG[
                "matd3_discrete_epsilon_start"
            ]
        )
        end = float(
            CONFIG[
                "matd3_discrete_epsilon_end"
            ]
        )
        decay = max(
            1.0,
            float(
                CONFIG[
                    "matd3_discrete_epsilon_decay_steps"
                ]
            ),
        )
        fraction = (
            transition_steps.float()
            / decay
        ).clamp(0.0, 1.0)
        epsilon = (
            start
            + fraction * (end - start)
        )
        explore = (
            torch.rand(
                destination.shape,
                device=destination.device,
                generator=generator,
            )
            < epsilon[:, None]
        )
        random_logits = torch.rand(
            mask.shape,
            device=mask.device,
            generator=generator,
        ).masked_fill(
            mask <= 0.5,
            -1.0,
        )
        random_destination = (
            random_logits.argmax(-1)
        )
        destination = torch.where(
            explore,
            random_destination,
            destination,
        )
    return continuous, destination


In [ ]:
def _random_exploration_actions(
    env,
    mask,
):
    shape = (
        env.num_envs,
        env.num_uavs,
        3,
    )
    direction = torch.randn(
        shape,
        device=env.device,
        generator=env.generator,
    )
    direction = direction / (
        torch.linalg.vector_norm(
            direction,
            dim=-1,
            keepdim=True,
        ).clamp_min(1e-12)
    )
    radius = torch.pow(
        torch.rand(
            env.num_envs,
            env.num_uavs,
            1,
            device=env.device,
            generator=env.generator,
        ),
        1.0 / 3.0,
    )
    motion = direction * radius
    power = (
        torch.rand(
            env.num_envs,
            env.num_uavs,
            1,
            device=env.device,
            generator=env.generator,
        )
        * 2.0
        - 1.0
    )
    continuous = torch.cat(
        (motion, power),
        dim=-1,
    )
    logits = torch.rand(
        mask.shape,
        device=env.device,
        generator=env.generator,
    ).masked_fill(
        mask <= 0.5,
        -1.0,
    )
    destination = logits.argmax(-1)
    return continuous, destination


### Tensor environment benchmark


In [ ]:
def benchmark_env(num_envs: int, steps: int, seed: int, device: str, strict_cuda: bool = True):
    env = FullGpuUAVBatchEnv(num_envs=num_envs, device=device, seed=seed, strict_cuda=strict_cuda)
    n_cuda_tensors = (
        env.assert_all_state_on_cuda()
        if env.device.type == "cuda"
        else sum(isinstance(v, torch.Tensor) for v in env.__dict__.values())
    )
    _obs, _state, mask = env.reset()
    if env.device.type == "cuda":
        torch.cuda.synchronize()
    start = time.perf_counter()
    transitions = 0
    for _ in range(int(steps)):
        continuous, destination = (
            _random_exploration_actions(
                env,
                mask,
            )
        )
        _obs, _state, mask, _reward, _done, _ = env.step(continuous, destination)
        transitions += num_envs
    if env.device.type == "cuda":
        torch.cuda.synchronize(env.device)
    wall = time.perf_counter() - start
    return {
        "kind": "full_gpu_env",
        "num_envs": num_envs,
        "vector_steps": int(steps),
        "transitions": transitions,
        "wall_seconds": wall,
        "transitions_per_second": transitions / wall,
        "cuda_state_tensors": n_cuda_tensors,
        "gpu_name": torch.cuda.get_device_name(env.device) if env.device.type == "cuda" else "CPU_DEBUG",
        "peak_memory_mb": torch.cuda.max_memory_allocated(env.device) / 2**20 if env.device.type == "cuda" else 0.0,
    }


### CPU-vector / GPU-learner hybrid reference path


In [ ]:
def _make_hybrid_vector_env(
    project_ns,
    num_envs,
):
    gym = project_ns["gym"]
    env_fns = [
        lambda: project_ns[
            "UAVSearchEnv"
        ](backend_name="simple")
        for _ in range(int(num_envs))
    ]
    return gym.vector.AsyncVectorEnv(
        env_fns,
        context=str(
            CONFIG.get(
                "training_vector_context",
                "fork",
            )
        ),
        shared_memory=bool(
            CONFIG.get(
                "training_vector_shared_memory",
                True,
            )
        ),
        autoreset_mode=(
            gym.vector.AutoresetMode.DISABLED
        ),
    )


In [ ]:
def benchmark_hybrid_env(
    repo: Path,
    num_envs: int,
    steps: int,
    seed: int,
):
    """CPU vectorized reference env throughput (no learner update)."""
    project_ns = load_project_namespace(
        repo
    )
    np = project_ns["np"]
    vector_env = _make_hybrid_vector_env(
        project_ns,
        num_envs,
    )
    template = project_ns[
        "UAVSearchEnv"
    ](backend_name="simple")
    rng = np.random.default_rng(
        int(seed) + 991
    )
    try:
        observations, _ = vector_env.reset(
            seed=[
                int(seed) + index
                for index in range(num_envs)
            ]
        )
        start = time.perf_counter()
        transitions = 0
        next_episode_seed = (
            int(seed) + 100_000
        )
        for _ in range(int(steps)):
            observations_by_env = [
                project_ns[
                    "_vector_observation_at"
                ](
                    observations,
                    index,
                    num_envs,
                )
                for index in range(
                    num_envs
                )
            ]
            actions_by_env = [
                project_ns[
                    "sample_valid_random_actions"
                ](
                    observations_by_env[index],
                    template.agent_ids,
                    rng,
                )
                for index in range(
                    num_envs
                )
            ]
            vector_actions = project_ns[
                "_stack_vector_actions"
            ](
                actions_by_env,
                template.agent_ids,
            )
            (
                next_observations,
                _rewards,
                terminated,
                truncated,
                _infos,
            ) = vector_env.step(
                vector_actions
            )
            transitions += num_envs
            done = np.logical_or(
                terminated,
                truncated,
            )
            observations = next_observations
            if np.any(done):
                seeds = [None] * num_envs
                for index in np.flatnonzero(
                    done
                ):
                    seeds[int(index)] = (
                        next_episode_seed
                    )
                    next_episode_seed += 1
                observations, _ = (
                    vector_env.reset(
                        seed=seeds,
                        options={
                            "reset_mask": (
                                done.astype(
                                    np.bool_
                                )
                            )
                        },
                    )
                )
        wall = time.perf_counter() - start
        return {
            "kind": "hybrid_cpu_env",
            "num_envs": int(num_envs),
            "vector_steps": int(steps),
            "transitions": int(
                transitions
            ),
            "wall_seconds": wall,
            "transitions_per_second": (
                transitions / wall
            ),
        }
    finally:
        vector_env.close()
        template.close()


In [ ]:
def benchmark_hybrid_train(
    repo: Path,
    algorithm: str,
    num_envs: int,
    total_transitions: int,
    seed: int,
    device: str,
):
    """Best current CPU-env + CUDA-learner path, including overlap."""
    algorithm = str(
        algorithm
    ).strip().lower()
    if algorithm not in {
        "masac",
        "matd3",
    }:
        raise ValueError(
            "algorithm must be masac or matd3"
        )
    if (
        int(total_transitions)
        % int(num_envs)
        != 0
    ):
        raise ValueError(
            "total_transitions must be "
            "divisible by num_envs"
        )

    project_ns = load_project_namespace(
        repo
    )
    np = project_ns["np"]
    template = project_ns[
        "UAVSearchEnv"
    ](backend_name="simple")
    trainer = _trainer(
        repo,
        algorithm,
        device,
        seed,
    )
    replay_capacity = min(
        int(
            CONFIG[
                f"{algorithm}_replay_capacity"
            ]
        ),
        max(
            4096,
            int(total_transitions),
        ),
    )
    replay = trainer._project_namespace[
        "HybridReplayBuffer"
    ](
        capacity=replay_capacity,
        num_agents=trainer.num_agents,
        observation_dim=(
            trainer.observation_dim
        ),
        state_dim=trainer.state_dim,
        continuous_dim=(
            trainer.continuous_dim
        ),
        discrete_dim=(
            trainer.discrete_dim
        ),
        seed=int(seed) + 123,
    )
    vector_env = _make_hybrid_vector_env(
        project_ns,
        num_envs,
    )

    batch_size = int(
        CONFIG[
            f"{algorithm}_batch_size"
        ]
    )
    learning_starts = int(
        CONFIG[
            f"{algorithm}_learning_starts"
        ]
    )
    updates_per_step = int(
        CONFIG[
            f"{algorithm}_updates_per_step"
        ]
    )
    rng = np.random.default_rng(
        int(seed)
    )
    overlap = bool(
        CONFIG.get(
            "training_overlap_env_and_updates",
            True,
        )
    )
    next_episode_seed = (
        int(seed) + 100_000
    )

    try:
        observations, infos = (
            vector_env.reset(
                seed=[
                    int(seed) + index
                    for index in range(
                        num_envs
                    )
                ]
            )
        )
        states = np.asarray(
            infos["state"],
            dtype=np.float32,
        ).copy()
        global_step = 0
        pending_updates = 0
        update_count = 0
        latest_metrics = None
        env_seconds = 0.0
        wait_seconds = 0.0
        update_seconds = 0.0
        policy_seconds = 0.0

        if (
            torch.device(device).type
            == "cuda"
        ):
            torch.cuda.synchronize()
            torch.cuda.reset_peak_memory_stats()

        wall_start = time.perf_counter()
        while global_step < int(
            total_transitions
        ):
            observations_by_env = [
                project_ns[
                    "_vector_observation_at"
                ](
                    observations,
                    index,
                    num_envs,
                )
                for index in range(
                    num_envs
                )
            ]
            exploration_steps = [
                global_step + index + 1
                for index in range(
                    num_envs
                )
            ]

            policy_start = time.perf_counter()
            if all(
                step <= learning_starts
                for step in exploration_steps
            ):
                actions_by_env = [
                    project_ns[
                        "sample_valid_random_actions"
                    ](
                        observations_by_env[
                            index
                        ],
                        template.agent_ids,
                        rng,
                    )
                    for index in range(
                        num_envs
                    )
                ]
            elif all(
                step > learning_starts
                for step in exploration_steps
            ):
                actions_by_env = project_ns[
                    "_select_vector_policy_actions"
                ](
                    trainer,
                    observations_by_env,
                    algorithm,
                    exploration_steps,
                )
            else:
                actions_by_env = []
                for index in range(
                    num_envs
                ):
                    transition_step = (
                        exploration_steps[index]
                    )
                    if (
                        transition_step
                        <= learning_starts
                    ):
                        action = project_ns[
                            "sample_valid_random_actions"
                        ](
                            observations_by_env[
                                index
                            ],
                            template.agent_ids,
                            rng,
                        )
                    elif algorithm == "masac":
                        action = (
                            trainer.select_actions(
                                observations_by_env[
                                    index
                                ],
                                deterministic=False,
                            )
                        )
                    else:
                        action = (
                            trainer.select_actions(
                                observations_by_env[
                                    index
                                ],
                                deterministic=False,
                                exploration_step=(
                                    transition_step
                                ),
                            )
                        )
                    actions_by_env.append(
                        action
                    )
            policy_seconds += (
                time.perf_counter()
                - policy_start
            )

            vector_actions = project_ns[
                "_stack_vector_actions"
            ](
                actions_by_env,
                template.agent_ids,
            )
            env_start = time.perf_counter()
            if overlap:
                vector_env.step_async(
                    vector_actions
                )
            else:
                result = vector_env.step(
                    vector_actions
                )

            if (
                pending_updates > 0
                and len(replay)
                >= batch_size
            ):
                update_start = (
                    time.perf_counter()
                )
                for _ in range(
                    pending_updates
                ):
                    latest_metrics = (
                        trainer.update(
                            replay,
                            batch_size=(
                                batch_size
                            ),
                        )
                    )
                    update_count += 1
                update_seconds += (
                    time.perf_counter()
                    - update_start
                )
                pending_updates = 0

            if overlap:
                wait_start = (
                    time.perf_counter()
                )
                result = (
                    vector_env.step_wait()
                )
                wait_seconds += (
                    time.perf_counter()
                    - wait_start
                )
            env_seconds += (
                time.perf_counter()
                - env_start
            )

            (
                next_observations,
                rewards,
                terminated,
                truncated,
                next_infos,
            ) = result
            next_states = np.asarray(
                next_infos["state"],
                dtype=np.float32,
            ).copy()

            previous_global_step = (
                global_step
            )
            replay_len_before = len(
                replay
            )
            for index in range(
                num_envs
            ):
                next_env_observations = (
                    project_ns[
                        "_vector_observation_at"
                    ](
                        next_observations,
                        index,
                        num_envs,
                    )
                )
                project_ns[
                    "add_environment_transition_to_replay"
                ](
                    replay,
                    observations_by_env[
                        index
                    ],
                    states[index],
                    actions_by_env[index],
                    float(rewards[index]),
                    next_env_observations,
                    next_states[index],
                    bool(
                        terminated[index]
                    ),
                    bool(
                        truncated[index]
                    ),
                    template.agent_ids,
                )

            global_step += num_envs
            pending_updates += (
                project_ns[
                    "_count_vector_updates_due"
                ](
                    previous_global_step=(
                        previous_global_step
                    ),
                    num_envs=num_envs,
                    learning_starts=(
                        learning_starts
                    ),
                    replay_len_before=(
                        replay_len_before
                    ),
                    replay_capacity=(
                        replay.capacity
                    ),
                    batch_size=batch_size,
                    updates_per_step=(
                        updates_per_step
                    ),
                )
            )

            if (
                not overlap
                and pending_updates > 0
            ):
                update_start = (
                    time.perf_counter()
                )
                for _ in range(
                    pending_updates
                ):
                    latest_metrics = (
                        trainer.update(
                            replay,
                            batch_size=(
                                batch_size
                            ),
                        )
                    )
                    update_count += 1
                update_seconds += (
                    time.perf_counter()
                    - update_start
                )
                pending_updates = 0

            done = np.logical_or(
                terminated,
                truncated,
            )
            observations = (
                next_observations
            )
            states = next_states
            if np.any(done):
                seeds = [None] * num_envs
                for index in np.flatnonzero(
                    done
                ):
                    seeds[int(index)] = (
                        next_episode_seed
                    )
                    next_episode_seed += 1
                (
                    observations,
                    reset_infos,
                ) = vector_env.reset(
                    seed=seeds,
                    options={
                        "reset_mask": done.astype(
                            np.bool_
                        )
                    },
                )
                reset_states = np.asarray(
                    reset_infos["state"],
                    dtype=np.float32,
                )
                reset_mask = np.asarray(
                    reset_infos.get(
                        "_state",
                        done,
                    ),
                    dtype=np.bool_,
                )
                for index in np.flatnonzero(
                    reset_mask
                ):
                    states[int(index)] = (
                        reset_states[
                            int(index)
                        ]
                    )

        if pending_updates > 0:
            update_start = (
                time.perf_counter()
            )
            for _ in range(
                pending_updates
            ):
                latest_metrics = (
                    trainer.update(
                        replay,
                        batch_size=batch_size,
                    )
                )
                update_count += 1
            update_seconds += (
                time.perf_counter()
                - update_start
            )

        if (
            torch.device(device).type
            == "cuda"
        ):
            torch.cuda.synchronize()
        wall = (
            time.perf_counter()
            - wall_start
        )

        return {
            "kind": "hybrid_cpu_env_cuda_learner",
            "algorithm": algorithm,
            "num_envs": int(
                num_envs
            ),
            "transitions": int(
                global_step
            ),
            "updates": int(
                update_count
            ),
            "wall_seconds": wall,
            "transitions_per_second": (
                global_step / wall
            ),
            "updates_per_second": (
                update_count / wall
            ),
            "env_collection_seconds": (
                env_seconds
            ),
            "env_wait_seconds": (
                wait_seconds
            ),
            "policy_inference_seconds": (
                policy_seconds
            ),
            "update_seconds": (
                update_seconds
            ),
            "overlap_enabled": bool(
                overlap
            ),
            "gpu_name": (
                torch.cuda.get_device_name()
                if (
                    torch.device(
                        device
                    ).type
                    == "cuda"
                    and torch.cuda.is_available()
                )
                else "CPU"
            ),
            "peak_memory_mb": (
                torch.cuda.max_memory_allocated()
                / 2**20
                if (
                    torch.device(
                        device
                    ).type
                    == "cuda"
                    and torch.cuda.is_available()
                )
                else 0.0
            ),
        }
    finally:
        vector_env.close()
        template.close()


### Official UavNetSim bridge and parity validation


In [ ]:
class GpuUavNetSimBridge:
    """Packet-level UavNetSim adapter for one tensorized mission environment."""

    def __init__(self, env, project_ns, *, repo_path=None, seed=44):
        if env.num_envs != 1:
            raise ValueError(
                "GpuUavNetSimBridge currently supports exactly one mission env"
            )
        self.env = env
        self.ns = project_ns
        self.repo_path = repo_path
        self.seed = int(seed)
        self.backend = None
        self.uavs = []
        self.obstacles = []
        self.report_buffers = []
        self.pending_reports = []
        self.gcs_received_target_ids = set()
        self._last_results = []
        self._last_metrics = None
        self._timing_totals = {
            "sync_gpu_to_cpu_seconds": 0.0,
            "uavnetsim_step_seconds": 0.0,
            "sync_cpu_to_gpu_seconds": 0.0,
            "calls": 0,
        }

    @property
    def np(self):
        return self.ns["np"]

    def _tensor_numpy(self, value, dtype=None):
        array = value.detach().cpu().numpy()
        if dtype is not None:
            array = array.astype(dtype, copy=False)
        return array

    def _build_mission_objects(self):
        np = self.np
        UAV = self.ns["UAV"]
        Obstacle = self.ns["Obstacle"]
        positions = self._tensor_numpy(self.env.positions[0], np.float64)
        velocities = self._tensor_numpy(self.env.velocities[0], np.float64)
        batteries = self._tensor_numpy(self.env.battery[0], np.float64)
        active = self._tensor_numpy(self.env.active[0], bool)
        self.uavs = [
            UAV(
                id=index,
                position=positions[index].copy(),
                velocity=velocities[index].copy(),
                battery_j=float(batteries[index]),
                active=bool(active[index]),
            )
            for index in range(self.env.num_uavs)
        ]
        obstacle_xy = self._tensor_numpy(self.env.obstacle_xy[0], np.float64)
        obstacle_radius = self._tensor_numpy(self.env.obstacle_radius[0], np.float64)
        obstacle_height = self._tensor_numpy(self.env.obstacle_height[0], np.float64)
        self.obstacles = [
            Obstacle(
                position=obstacle_xy[index].copy(),
                radius=float(obstacle_radius[index]),
                height=float(obstacle_height[index]),
            )
            for index in range(self.env.num_obstacles)
        ]
        self.report_buffers = [[] for _ in range(self.env.num_uavs)]
        self.pending_reports = []
        self.gcs_received_target_ids = set()
        self._sync_gpu_to_cpu()

    def _sync_gpu_to_cpu(self):
        np = self.np
        Report = self.ns["Report"]
        positions = self._tensor_numpy(self.env.positions[0], np.float64)
        velocities = self._tensor_numpy(self.env.velocities[0], np.float64)
        batteries = self._tensor_numpy(self.env.battery[0], np.float64)
        active = self._tensor_numpy(self.env.active[0], bool)
        for index, uav in enumerate(self.uavs):
            uav.position = positions[index].copy()
            uav.velocity = velocities[index].copy()
            uav.battery_j = float(batteries[index])
            uav.active = bool(active[index])

        valid = self._tensor_numpy(self.env.report_valid[0], bool)
        remaining = self._tensor_numpy(self.env.report_remaining[0], np.float64)
        created = self._tensor_numpy(self.env.report_created[0], np.int64)
        source = self._tensor_numpy(self.env.report_source[0], np.int64)
        report_size = int(CONFIG["report_bytes"])
        ttl_s = float(CONFIG["report_ttl"])
        for holder in range(self.env.num_uavs):
            buffer = self.report_buffers[holder]
            buffer.clear()
            for target in range(self.env.num_targets):
                if not valid[holder, target]:
                    continue
                delivered_bytes = int(round(report_size - float(remaining[holder, target])))
                delivered_bytes = max(0, min(report_size, delivered_bytes))
                buffer.append(
                    Report(
                        target_id=int(target),
                        source_uav=int(source[holder, target]),
                        created_step=int(created[holder, target]),
                        size_bytes=report_size,
                        ttl_s=ttl_s,
                        delivered_bytes=delivered_bytes,
                    )
                )

        pvalid = self._tensor_numpy(self.env.pending_valid[0], bool)
        premaining = self._tensor_numpy(self.env.pending_remaining[0], np.float64)
        pcreated = self._tensor_numpy(self.env.pending_created[0], np.int64)
        self.pending_reports.clear()
        for holder in range(self.env.num_uavs):
            for target in range(self.env.num_targets):
                if not pvalid[holder, target]:
                    continue
                delivered_bytes = int(round(report_size - float(premaining[holder, target])))
                delivered_bytes = max(0, min(report_size, delivered_bytes))
                self.pending_reports.append(
                    Report(
                        target_id=int(target),
                        source_uav=int(holder),
                        created_step=int(pcreated[holder, target]),
                        size_bytes=report_size,
                        ttl_s=ttl_s,
                        delivered_bytes=delivered_bytes,
                    )
                )
        self.pending_reports.sort(
            key=lambda report: (
                int(report.created_step),
                int(report.source_uav),
                int(report.target_id),
            )
        )

        delivered = self._tensor_numpy(self.env.delivered[0], bool)
        self.gcs_received_target_ids.clear()
        self.gcs_received_target_ids.update(
            int(index) for index, flag in enumerate(delivered) if flag
        )

    def _sync_cpu_to_gpu(self):
        env = self.env
        device = env.device
        valid = torch.zeros_like(env.report_valid)
        remaining = torch.zeros_like(env.report_remaining)
        created = torch.full_like(
            env.report_created,
            env.max_steps + int(CONFIG["report_ttl"]) + 1,
        )
        source = torch.full_like(env.report_source, -1)
        for holder, buffer in enumerate(self.report_buffers):
            for report in buffer:
                target = int(report.target_id)
                valid[0, holder, target] = True
                remaining[0, holder, target] = float(
                    max(0, int(report.size_bytes) - int(report.delivered_bytes))
                )
                created[0, holder, target] = int(report.created_step)
                source[0, holder, target] = int(report.source_uav)

        pvalid = torch.zeros_like(env.pending_valid)
        premaining = torch.zeros_like(env.pending_remaining)
        pcreated = torch.full_like(
            env.pending_created,
            env.max_steps + int(CONFIG["report_ttl"]) + 1,
        )
        for report in self.pending_reports:
            holder = int(report.source_uav)
            target = int(report.target_id)
            if not (0 <= holder < env.num_uavs):
                continue
            if pvalid[0, holder, target]:
                continue
            pvalid[0, holder, target] = True
            premaining[0, holder, target] = float(
                max(0, int(report.size_bytes) - int(report.delivered_bytes))
            )
            pcreated[0, holder, target] = int(report.created_step)

        delivered = torch.zeros_like(env.delivered)
        for target in self.gcs_received_target_ids:
            if 0 <= int(target) < env.num_targets:
                delivered[0, int(target)] = True

        env.report_valid = valid.to(device)
        env.report_remaining = remaining.to(device)
        env.report_created = created.to(device)
        env.report_source = source.to(device)
        env.pending_valid = pvalid.to(device)
        env.pending_remaining = premaining.to(device)
        env.pending_created = pcreated.to(device)
        env.delivered = delivered.to(device)

    def reset_from_env(self):
        if self.backend is not None:
            self.backend.close()
        self._build_mission_objects()
        self.backend = self.ns["UavNetSimBackend"](
            repo_path=self.repo_path,
            seed=self.seed,
        )
        self.backend.reset(
            uavs=self.uavs,
            gcs_position=CONFIG["gcs_position"],
            obstacles=self.obstacles,
            report_buffers=self.report_buffers,
            pending_reports=self.pending_reports,
            gcs_received_target_ids=self.gcs_received_target_ids,
        )
        self._last_results = []
        self._last_metrics = self.backend.metrics()
        self._timing_totals = {
            "sync_gpu_to_cpu_seconds": 0.0,
            "uavnetsim_step_seconds": 0.0,
            "sync_cpu_to_gpu_seconds": 0.0,
            "calls": 0,
        }

    def step(
        self,
        destination_idx,
        power_action,
        energy_budget_j=None,
    ):
        if self.backend is None:
            raise RuntimeError("GpuUavNetSimBridge must be reset before step")
        timing_start = time.perf_counter()
        self._sync_gpu_to_cpu()
        after_gpu_to_cpu = time.perf_counter()
        np = self.np
        current_step = int(self.env.step_count[0].detach().cpu().item())
        destinations = self._tensor_numpy(destination_idx[0], np.int64)
        powers = self._tensor_numpy(power_action[0, :, 0], np.float64)
        if energy_budget_j is None:
            energy_budget = np.full(
                self.env.num_uavs,
                np.inf,
                dtype=np.float64,
            )
        else:
            energy_budget = self._tensor_numpy(
                torch.as_tensor(
                    energy_budget_j,
                    device=self.env.device,
                    dtype=self.env.positions.dtype,
                )[0],
                np.float64,
            )
            if energy_budget.shape != (
                self.env.num_uavs,
            ):
                raise ValueError(
                    "energy_budget_j has wrong shape"
                )
            if (
                not np.all(
                    np.isfinite(
                        energy_budget
                    )
                )
                or np.any(
                    energy_budget < 0.0
                )
            ):
                raise ValueError(
                    "energy_budget_j must be finite and >= 0"
                )

        HybridAction = self.ns["HybridAction"]
        build_intent = self.ns["build_transmission_intent"]
        intents = []
        sender_for_result = []
        for sender in range(self.env.num_uavs):
            choice_index = int(destinations[sender])
            if not 0 <= choice_index < self.env.discrete_dim:
                raise ValueError("destination index outside communication choices")
            destination = int(
                self.env.choices[sender, choice_index].detach().cpu().item()
            )
            if destination == -2:
                decoded_destination = None
                tx_power_w = 0.0
            else:
                decoded_destination = destination
                tx_power_w = (
                    float(CONFIG["tx_power_min_w"])
                    + (float(max(-1.0, min(1.0, powers[sender]))) + 1.0)
                    * 0.5
                    * (
                        float(CONFIG["tx_power_max_w"])
                        - float(CONFIG["tx_power_min_w"])
                    )
                )
            hybrid_action = HybridAction(
                movement=np.zeros(3, dtype=np.float64),
                destination=decoded_destination,
                tx_power_w=float(tx_power_w),
            )
            if decoded_destination is None:
                continue
            intent = build_intent(
                sender,
                hybrid_action,
                self.report_buffers[sender],
                current_step,
                peer_transfer_states=self.backend.peer_transfer_states,
            )
            if intent is not None:
                budget = float(
                    energy_budget[sender]
                )
                full_step_energy = (
                    float(tx_power_w)
                    * float(self.env.dt)
                )
                if budget <= 0.0:
                    intent = None
                elif budget < (
                    full_step_energy - 1e-12
                ):
                    max_bytes = int(
                        np.floor(
                            float(
                                CONFIG[
                                    "uavnetsim_bit_rate_bps"
                                ]
                            )
                            * min(
                                float(self.env.dt),
                                budget
                                / max(
                                    float(tx_power_w),
                                    1e-12,
                                ),
                            )
                            / 8.0
                            + 1e-9
                        )
                    )
                    if max_bytes <= 0:
                        intent = None
                    elif max_bytes < int(
                        intent.requested_bytes
                    ):
                        intent = type(intent)(
                            sender=int(
                                intent.sender
                            ),
                            recipient=int(
                                intent.recipient
                            ),
                            target_id=int(
                                intent.target_id
                            ),
                            requested_bytes=(
                                max_bytes
                            ),
                            tx_power_w=float(
                                intent.tx_power_w
                            ),
                        )

            if intent is not None:
                intents.append(intent)
                sender_for_result.append(sender)

        before_delivered = set(self.gcs_received_target_ids)
        self.backend.sync_positions(self.uavs)
        self.backend.set_step_energy_budget_j(
            energy_budget
        )
        results = self.backend.step(
            intents,
            dt=self.env.dt,
            current_step=current_step,
        )
        after_uavnetsim = time.perf_counter()
        communication_energy = np.minimum(
            self.backend.last_step_communication_energy_by_uav(),
            energy_budget,
        )
        attempted = torch.zeros(
            1, self.env.num_uavs, device=self.env.device, dtype=torch.bool
        )
        bytes_tx = torch.zeros(
            1, self.env.num_uavs, device=self.env.device
        )
        for sender, result in zip(sender_for_result, results):
            attempted[0, sender] = True
            bytes_tx[0, sender] = float(result.get("tx_bytes", 0))

        self._sync_cpu_to_gpu()
        after_cpu_to_gpu = time.perf_counter()
        self._timing_totals["sync_gpu_to_cpu_seconds"] += (
            after_gpu_to_cpu - timing_start
        )
        self._timing_totals["uavnetsim_step_seconds"] += (
            after_uavnetsim - after_gpu_to_cpu
        )
        self._timing_totals["sync_cpu_to_gpu_seconds"] += (
            after_cpu_to_gpu - after_uavnetsim
        )
        self._timing_totals["calls"] += 1
        after_delivered = set(self.gcs_received_target_ids)
        new_delivery = len(after_delivered - before_delivered)
        comm_energy = torch.as_tensor(
            communication_energy,
            device=self.env.device,
            dtype=self.env.positions.dtype,
        ).view(1, self.env.num_uavs)
        self._last_results = results
        self._last_metrics = self.backend.metrics()
        return {
            "new_delivery": torch.tensor(
                [new_delivery], device=self.env.device, dtype=torch.long
            ),
            "comm_energy": comm_energy,
            "attempted": attempted,
            "bytes_tx": bytes_tx,
        }

    def metrics(self):
        if self.backend is None:
            return {}
        self._last_metrics = self.backend.metrics()
        return dict(self._last_metrics)

    def timing_metrics(self):
        result = dict(self._timing_totals)
        calls = max(1, int(result["calls"]))
        total = (
            float(result["sync_gpu_to_cpu_seconds"])
            + float(result["uavnetsim_step_seconds"])
            + float(result["sync_cpu_to_gpu_seconds"])
        )
        result["bridge_total_seconds"] = total
        result["uavnetsim_fraction"] = (
            float(result["uavnetsim_step_seconds"]) / total
            if total > 0.0
            else 0.0
        )
        result["avg_uavnetsim_step_seconds"] = (
            float(result["uavnetsim_step_seconds"]) / calls
        )
        return result

    def close(self):
        if self.backend is not None:
            self.backend.close()
            self.backend = None


In [ ]:
def copy_cpu_reference_state_to_gpu(cpu_env, gpu_env):
    """Copy one CPU mission state into one tensor env for apples-to-apples runs."""
    if gpu_env.num_envs != 1:
        raise ValueError("copy_cpu_reference_state_to_gpu requires num_envs=1")
    device = gpu_env.device
    np = __import__("numpy")

    positions = np.stack([uav.position for uav in cpu_env.uavs])
    velocities = np.stack([uav.velocity for uav in cpu_env.uavs])
    battery = np.asarray([uav.battery_j for uav in cpu_env.uavs], dtype=np.float32)
    active = np.asarray([uav.active for uav in cpu_env.uavs], dtype=bool)
    targets = np.stack([target.position for target in cpu_env.targets])
    if cpu_env.obstacles:
        obstacles = np.stack(
            [obstacle.position for obstacle in cpu_env.obstacles]
        )
    else:
        obstacles = np.empty((0, 2), dtype=np.float32)
    obstacle_radius = np.asarray(
        [obstacle.radius for obstacle in cpu_env.obstacles], dtype=np.float32
    )
    obstacle_height = np.asarray(
        [obstacle.height for obstacle in cpu_env.obstacles], dtype=np.float32
    )

    gpu_env.positions = torch.as_tensor(
        positions, device=device, dtype=gpu_env.positions.dtype
    ).unsqueeze(0)
    gpu_env.velocities = torch.as_tensor(
        velocities, device=device, dtype=gpu_env.velocities.dtype
    ).unsqueeze(0)
    gpu_env.battery = torch.as_tensor(
        battery, device=device, dtype=gpu_env.battery.dtype
    ).unsqueeze(0)
    gpu_env.active = torch.as_tensor(
        active, device=device, dtype=torch.bool
    ).unsqueeze(0)
    gpu_env.target_xy = torch.as_tensor(
        targets, device=device, dtype=gpu_env.target_xy.dtype
    ).unsqueeze(0)
    gpu_env.obstacle_xy = torch.as_tensor(
        obstacles, device=device, dtype=gpu_env.obstacle_xy.dtype
    ).unsqueeze(0)
    gpu_env.obstacle_radius = torch.as_tensor(
        obstacle_radius, device=device, dtype=gpu_env.obstacle_radius.dtype
    ).unsqueeze(0)
    gpu_env.obstacle_height = torch.as_tensor(
        obstacle_height, device=device, dtype=gpu_env.obstacle_height.dtype
    ).unsqueeze(0)
    gpu_env.confirmed = torch.as_tensor(
        [[target.confirmed for target in cpu_env.targets]],
        device=device,
        dtype=torch.bool,
    )
    gpu_env.delivered.zero_()
    for target in cpu_env.gcs_received_target_ids:
        gpu_env.delivered[0, int(target)] = True
    gpu_env.belief = torch.as_tensor(
        cpu_env.belief_maps,
        device=device,
        dtype=gpu_env.belief.dtype,
    ).unsqueeze(0)
    gpu_env.step_count.fill_(int(cpu_env.current_step))
    gpu_env.report_valid.zero_()
    gpu_env.report_remaining.zero_()
    gpu_env.report_created.fill_(
        gpu_env.max_steps + int(CONFIG["report_ttl"]) + 1
    )
    gpu_env.report_source.fill_(-1)
    for holder, buffer in enumerate(cpu_env.report_buffers):
        for report in buffer:
            target = int(report.target_id)
            gpu_env.report_valid[0, holder, target] = True
            gpu_env.report_remaining[0, holder, target] = float(
                max(0, int(report.size_bytes) - int(report.delivered_bytes))
            )
            gpu_env.report_created[0, holder, target] = int(report.created_step)
            gpu_env.report_source[0, holder, target] = int(report.source_uav)
    gpu_env.pending_valid.zero_()
    gpu_env.pending_remaining.zero_()
    gpu_env.pending_created.fill_(
        gpu_env.max_steps + int(CONFIG["report_ttl"]) + 1
    )
    for report in cpu_env.pending_reports:
        holder = int(report.source_uav)
        target = int(report.target_id)
        if 0 <= holder < gpu_env.num_uavs:
            gpu_env.pending_valid[0, holder, target] = True
            gpu_env.pending_remaining[0, holder, target] = float(
                max(0, int(report.size_bytes) - int(report.delivered_bytes))
            )
            gpu_env.pending_created[0, holder, target] = int(report.created_step)
    gpu_env.peer_received.zero_()
    gpu_env.peer_source.fill_(-1)
    gpu_env.peer_created.fill_(
        gpu_env.max_steps + int(CONFIG["report_ttl"]) + 1
    )
    gpu_env.belief_potential = gpu_env._belief_potential()
    obs, mask = gpu_env._observation()
    state = gpu_env._state(obs)
    return obs, state, mask


In [ ]:
def benchmark_uavnetsim_full_flow(
    repo: Path,
    *,
    repo_path,
    steps=20,
    seed=44,
    device="cuda:0",
    strict_cuda=True,
    communication_loaded=True,
):
    """CPU-UavNetSim vs tensor-env+UavNetSim using one scenario and action trace."""
    project_ns = load_project_namespace(repo)
    if not project_ns["uavnetsim_available"](repo_path):
        raise FileNotFoundError("UavNetSim repository is unavailable")

    CPUEnv = project_ns["UAVSearchEnv"]
    cpu_env = CPUEnv(
        backend_name="uavnetsim",
        network_backend_kwargs={"repo_path": repo_path},
    )
    cpu_env.reset(seed=int(seed))
    if communication_loaded:
        Report = project_ns["Report"]
        cpu_env.targets[0].confirmed = True
        cpu_env.report_buffers[0].clear()
        cpu_env.report_buffers[0].append(
            Report(
                target_id=0,
                source_uav=0,
                created_step=0,
                size_bytes=int(CONFIG["report_bytes"]),
                ttl_s=float(CONFIG["report_ttl"]),
                delivered_bytes=0,
            )
        )

    gpu_env = FullGpuUAVBatchEnv(
        num_envs=1,
        device=device,
        seed=int(seed),
        strict_cuda=bool(strict_cuda),
    )
    copy_cpu_reference_state_to_gpu(cpu_env, gpu_env)
    bridge = GpuUavNetSimBridge(
        gpu_env,
        project_ns,
        repo_path=repo_path,
        seed=int(seed),
    )
    gpu_env.attach_external_network(bridge)

    rng = project_ns["np"].random.default_rng(int(seed) + 99173)
    actions = rng.uniform(
        -1.0,
        1.0,
        size=(int(steps), gpu_env.num_uavs, gpu_env.continuous_dim),
    ).astype("float32")
    destinations = rng.integers(
        0,
        gpu_env.discrete_dim,
        size=(int(steps), gpu_env.num_uavs),
        endpoint=False,
        dtype="int64",
    )
    if communication_loaded:
        actions[:, 0, 3] = 1.0
        destinations[:, 0] = gpu_env.num_uavs

    def cpu_action_at(index):
        result = {}
        for sender, agent_id in enumerate(cpu_env.agent_ids):
            result[agent_id] = {
                "motion": actions[index, sender, :3].copy(),
                "destination": int(destinations[index, sender]),
                "power": project_ns["np"].array(
                    [actions[index, sender, 3]],
                    dtype=project_ns["np"].float32,
                ),
            }
        return result

    cpu_rewards = []
    cpu_start = time.perf_counter()
    cpu_steps = 0
    for index in range(int(steps)):
        out = cpu_env.step(cpu_action_at(index))
        cpu_rewards.append(float(out[1]))
        cpu_steps += 1
        if bool(out[2] or out[3]):
            break
    cpu_wall = time.perf_counter() - cpu_start
    cpu_network = cpu_env.network_backend.metrics()

    gpu_rewards = []
    if gpu_env.device.type == "cuda":
        torch.cuda.synchronize(gpu_env.device)
    gpu_start = time.perf_counter()
    gpu_steps = 0
    for index in range(int(steps)):
        continuous = torch.as_tensor(
            actions[index:index + 1],
            device=gpu_env.device,
            dtype=torch.float32,
        )
        destination = torch.as_tensor(
            destinations[index:index + 1],
            device=gpu_env.device,
            dtype=torch.long,
        )
        out = gpu_env.step(continuous, destination)
        gpu_rewards.append(float(out[3][0].detach().cpu().item()))
        gpu_steps += 1
        if bool(out[4][0].detach().cpu().item()):
            break
    if gpu_env.device.type == "cuda":
        torch.cuda.synchronize(gpu_env.device)
    gpu_wall = time.perf_counter() - gpu_start
    gpu_network = bridge.metrics()

    np = project_ns["np"]
    cpu_positions = np.stack(
        [uav.position for uav in cpu_env.uavs]
    )
    gpu_positions = (
        gpu_env.positions[0]
        .detach()
        .cpu()
        .numpy()
    )
    cpu_belief = np.asarray(
        cpu_env.belief_maps,
        dtype=np.float32,
    )
    gpu_belief = (
        gpu_env.belief[0]
        .detach()
        .cpu()
        .numpy()
    )
    common_steps = min(
        len(cpu_rewards),
        len(gpu_rewards),
    )
    reward_abs_diff = [
        abs(
            float(cpu_rewards[index])
            - float(gpu_rewards[index])
        )
        for index in range(common_steps)
    ]
    network_compare_keys = (
        "packets_injected",
        "packets_delivered",
        "packets_failed",
        "queue_limit_events",
        "payload_bytes_injected",
        "payload_bytes_delivered",
    )
    network_core_equal = all(
        cpu_network.get(key)
        == gpu_network.get(key)
        for key in network_compare_keys
    )
    final_cpu_confirmed = {
        int(target.id)
        for target in cpu_env.targets
        if target.confirmed
    }
    final_gpu_confirmed = {
        int(index)
        for index, flag in enumerate(
            gpu_env.confirmed[0]
            .detach()
            .cpu()
            .tolist()
        )
        if flag
    }
    final_cpu_delivered = set(
        int(value)
        for value in cpu_env.gcs_received_target_ids
    )
    final_gpu_delivered = {
        int(index)
        for index, flag in enumerate(
            gpu_env.delivered[0]
            .detach()
            .cpu()
            .tolist()
        )
        if flag
    }

    result = {
        "kind": "uavnetsim_full_flow_compare",
        "seed": int(seed),
        "requested_steps": int(steps),
        "same_initial_scenario": True,
        "same_action_trace": True,
        "parity": {
            "final_position_max_abs_diff_m": float(
                np.max(
                    np.abs(
                        cpu_positions
                        - gpu_positions
                    )
                )
            ),
            "final_belief_max_abs_diff": float(
                np.max(
                    np.abs(
                        cpu_belief
                        - gpu_belief
                    )
                )
            ),
            "reward_max_abs_diff": (
                float(max(reward_abs_diff))
                if reward_abs_diff
                else 0.0
            ),
            "network_core_equal": bool(
                network_core_equal
            ),
            "confirmed_equal": (
                final_cpu_confirmed
                == final_gpu_confirmed
            ),
            "delivered_equal": (
                final_cpu_delivered
                == final_gpu_delivered
            ),
            "cpu_confirmed": sorted(
                final_cpu_confirmed
            ),
            "gpu_confirmed": sorted(
                final_gpu_confirmed
            ),
            "cpu_delivered": sorted(
                final_cpu_delivered
            ),
            "gpu_delivered": sorted(
                final_gpu_delivered
            ),
        },
        "cpu": {
            "steps": cpu_steps,
            "wall_seconds": cpu_wall,
            "steps_per_second": cpu_steps / cpu_wall if cpu_wall > 0.0 else float("inf"),
            "reward_sum": float(sum(cpu_rewards)),
            "network": cpu_network,
        },
        "gpu_mission_plus_uavnetsim": {
            "steps": gpu_steps,
            "wall_seconds": gpu_wall,
            "steps_per_second": gpu_steps / gpu_wall if gpu_wall > 0.0 else float("inf"),
            "reward_sum": float(sum(gpu_rewards)),
            "network": gpu_network,
            "network_timing": bridge.timing_metrics(),
            "device": str(gpu_env.device),
        },
    }
    cpu_env.close()
    gpu_env.close()
    return result


In [ ]:
def validate_uavnetsim_gpu_direct_link(
    repo: Path,
    *,
    repo_path=None,
    seed=44,
    steps=6,
    device="cpu",
    strict_cuda=False,
):
    """Controlled direct-to-GCS validation against the real SimPy backend.

    This is intentionally a single-link case: CSMA has no collision ambiguity,
    so queue cap, payload packetization, service budget, LoS/NLoS PHY contract,
    delivery timing, reward timing and report cleanup are directly comparable.
    """
    project_ns = load_project_namespace(repo)
    if not project_ns["uavnetsim_available"](repo_path):
        raise FileNotFoundError("UavNetSim repository is unavailable")
    np = project_ns["np"]
    CPUEnv = project_ns["UAVSearchEnv"]
    Report = project_ns["Report"]
    cpu_env = CPUEnv(
        backend_name="uavnetsim",
        network_backend_kwargs={"repo_path": repo_path},
    )
    cpu_env.reset(seed=int(seed))
    cpu_env.targets[0].confirmed = True
    cpu_env.report_buffers[0].clear()
    cpu_env.report_buffers[0].append(
        Report(
            target_id=0,
            source_uav=0,
            created_step=0,
            size_bytes=int(CONFIG["report_bytes"]),
            ttl_s=float(CONFIG["report_ttl"]),
            delivered_bytes=0,
        )
    )

    gpu_env = FullGpuUAVBatchEnv(
        num_envs=1,
        device=device,
        seed=int(seed),
        strict_cuda=bool(strict_cuda),
    )
    copy_cpu_reference_state_to_gpu(cpu_env, gpu_env)
    gpu_env.network_model = "uavnetsim_gpu"

    per_step = []
    try:
        for index in range(int(steps)):
            action = np.zeros(
                (gpu_env.num_uavs, gpu_env.continuous_dim),
                dtype=np.float32,
            )
            destination = np.zeros(
                gpu_env.num_uavs,
                dtype=np.int64,
            )
            action[0, 3] = 1.0
            destination[0] = gpu_env.num_uavs

            cpu_action = {}
            for sender, agent_id in enumerate(cpu_env.agent_ids):
                cpu_action[agent_id] = {
                    "motion": action[sender, :3].copy(),
                    "destination": int(destination[sender]),
                    "power": np.asarray(
                        [action[sender, 3]],
                        dtype=np.float32,
                    ),
                }
            cpu_out = cpu_env.step(cpu_action)
            gpu_out = gpu_env.step(
                torch.as_tensor(
                    action[None],
                    device=gpu_env.device,
                    dtype=torch.float32,
                ),
                torch.as_tensor(
                    destination[None],
                    device=gpu_env.device,
                    dtype=torch.long,
                ),
            )
            cpu_network = cpu_env.network_backend.metrics()
            gpu_delivered = bool(gpu_env.delivered[0, 0].item())
            gpu_remaining = float(
                gpu_env.report_remaining[0, 0, 0].item()
            )
            gpu_delivered_bytes = int(
                round(float(CONFIG["report_bytes"]) - gpu_remaining)
            )
            per_step.append(
                {
                    "step": index + 1,
                    "cpu_payload_bytes_delivered": int(
                        cpu_network["payload_bytes_delivered"]
                    ),
                    "gpu_payload_bytes_delivered": gpu_delivered_bytes,
                    "cpu_delivered": 0 in cpu_env.gcs_received_target_ids,
                    "gpu_delivered": gpu_delivered,
                    "cpu_reward": float(cpu_out[1]),
                    "gpu_reward": float(gpu_out[3][0].item()),
                }
            )

        byte_parity = all(
            row["cpu_payload_bytes_delivered"]
            == row["gpu_payload_bytes_delivered"]
            for row in per_step
        )
        delivery_parity = all(
            row["cpu_delivered"] == row["gpu_delivered"]
            for row in per_step
        )
        reward_max_abs_diff = max(
            (
                abs(row["cpu_reward"] - row["gpu_reward"])
                for row in per_step
            ),
            default=0.0,
        )
        return {
            "passed": bool(
                byte_parity
                and delivery_parity
                and reward_max_abs_diff < 1e-4
            ),
            "byte_parity": bool(byte_parity),
            "delivery_parity": bool(delivery_parity),
            "reward_max_abs_diff": float(reward_max_abs_diff),
            "per_step": per_step,
            "cpu_network_metrics": cpu_env.network_backend.metrics(),
        }
    finally:
        cpu_env.close()
        gpu_env.close()


In [ ]:
def validate_uavnetsim_network_parity(
    repo: Path,
    *,
    repo_path,
    seed=44,
    device="cpu",
    strict_cuda=False,
    steps=4,
):
    """Drive two independent UavNetSim backends with identical state/intents."""
    project_ns = load_project_namespace(repo)
    CPUEnv = project_ns["UAVSearchEnv"]
    cpu_env = CPUEnv(
        backend_name="uavnetsim",
        network_backend_kwargs={"repo_path": repo_path},
    )
    cpu_env.reset(seed=int(seed))
    Report = project_ns["Report"]
    cpu_env.targets[0].confirmed = True
    cpu_env.report_buffers[0].append(
        Report(
            target_id=0,
            source_uav=0,
            created_step=0,
            size_bytes=int(CONFIG["report_bytes"]),
            ttl_s=float(CONFIG["report_ttl"]),
            delivered_bytes=0,
        )
    )

    gpu_env = FullGpuUAVBatchEnv(
        1,
        device=device,
        seed=int(seed),
        strict_cuda=bool(strict_cuda),
    )
    copy_cpu_reference_state_to_gpu(cpu_env, gpu_env)
    bridge = GpuUavNetSimBridge(
        gpu_env,
        project_ns,
        repo_path=repo_path,
        seed=int(seed),
    )
    gpu_env.attach_external_network(bridge)

    destination = torch.zeros(
        1, gpu_env.num_uavs, device=gpu_env.device, dtype=torch.long
    )
    destination[0, 0] = gpu_env.num_uavs
    power = torch.zeros(
        1, gpu_env.num_uavs, 1, device=gpu_env.device
    )
    power[0, 0, 0] = 1.0

    per_step = []
    for current_step in range(1, int(steps) + 1):
        cpu_env.current_step = current_step - 1
        gpu_env.step_count.fill_(current_step)
        action = {}
        for sender, agent_id in enumerate(cpu_env.agent_ids):
            action[agent_id] = {
                "motion": project_ns["np"].zeros(
                    3, dtype=project_ns["np"].float32
                ),
                "destination": gpu_env.num_uavs if sender == 0 else 0,
                "power": project_ns["np"].array(
                    [1.0 if sender == 0 else 0.0],
                    dtype=project_ns["np"].float32,
                ),
            }
        hybrid = cpu_env._decode_actions(cpu_env._canonical_actions(action))
        intents = []
        peer_states = cpu_env.network_backend.peer_transfer_states
        for sender, haction in enumerate(hybrid):
            intent = project_ns["build_transmission_intent"](
                sender,
                haction,
                cpu_env.report_buffers[sender],
                current_step,
                peer_transfer_states=peer_states,
            )
            if intent is not None:
                intents.append(intent)
        cpu_env.network_backend.sync_positions(cpu_env.uavs)
        cpu_results = cpu_env.network_backend.step(
            intents,
            dt=float(CONFIG["dt"]),
            current_step=current_step,
        )
        gpu_result = bridge.step(destination, power)
        cpu_metrics = cpu_env.network_backend.metrics()
        gpu_metrics = bridge.metrics()
        comparable = {
            "packets_injected": (
                cpu_metrics["packets_injected"],
                gpu_metrics["packets_injected"],
            ),
            "packets_delivered": (
                cpu_metrics["packets_delivered"],
                gpu_metrics["packets_delivered"],
            ),
            "packets_failed": (
                cpu_metrics["packets_failed"],
                gpu_metrics["packets_failed"],
            ),
            "payload_bytes_delivered": (
                cpu_metrics["payload_bytes_delivered"],
                gpu_metrics["payload_bytes_delivered"],
            ),
            "queue_limit_events": (
                cpu_metrics["queue_limit_events"],
                gpu_metrics["queue_limit_events"],
            ),
        }
        equal = all(left == right for left, right in comparable.values())
        per_step.append(
            {
                "step": current_step,
                "equal": equal,
                "metrics": comparable,
                "cpu_results": cpu_results,
                "gpu_bytes_tx": gpu_result["bytes_tx"].detach().cpu().tolist(),
            }
        )
        if not equal:
            break

    passed = all(item["equal"] for item in per_step)
    cpu_env.close()
    gpu_env.close()
    return {
        "passed": passed,
        "steps_checked": len(per_step),
        "per_step": per_step,
    }


### CUDA learner optimization helpers


In [ ]:
def _rebuild_cuda_optimizers_with_fused_adam(
    trainer,
    algorithm,
):
    """Opt-in benchmark path; safe only before the first optimizer step."""
    if trainer.device.type != "cuda":
        return False
    algorithm = str(algorithm).strip().lower()
    if algorithm == "masac":
        trainer.actor_optimizer = torch.optim.Adam(
            trainer.actor.parameters(),
            lr=float(CONFIG["masac_actor_lr"]),
            fused=True,
        )
        trainer.critic_optimizer = torch.optim.Adam(
            list(trainer.critic_1.parameters())
            + list(trainer.critic_2.parameters()),
            lr=float(CONFIG["masac_critic_lr"]),
            fused=True,
        )
        trainer.alpha_continuous_optimizer = torch.optim.Adam(
            [trainer.log_alpha_continuous],
            lr=float(CONFIG["masac_alpha_lr"]),
            fused=True,
        )
        trainer.alpha_discrete_optimizer = torch.optim.Adam(
            [trainer.log_alpha_discrete],
            lr=float(CONFIG["masac_alpha_lr"]),
            fused=True,
        )
    elif algorithm == "matd3":
        trainer.actor_optimizer = torch.optim.Adam(
            trainer.actor.parameters(),
            lr=float(CONFIG["matd3_actor_lr"]),
            fused=True,
        )
        trainer.critic_optimizer = torch.optim.Adam(
            list(trainer.critic_1.parameters())
            + list(trainer.critic_2.parameters()),
            lr=float(CONFIG["matd3_critic_lr"]),
            fused=True,
        )
    else:
        raise ValueError("unsupported algorithm")
    return True


In [ ]:
def _compile_trainer_modules(
    trainer,
    algorithm,
    mode="default",
):
    """Compile fixed-shape actor/critic modules in-place for benchmarking."""
    if trainer.device.type != "cuda":
        return []
    names = [
        "actor",
        "critic_1",
        "critic_2",
        "target_critic_1",
        "target_critic_2",
    ]
    if str(algorithm).strip().lower() == "matd3":
        names.append("target_actor")
    compiled = []
    for name in names:
        module = getattr(trainer, name, None)
        if isinstance(module, torch.nn.Module):
            module.compile(
                mode=str(mode),
                fullgraph=False,
            )
            compiled.append(name)
    return compiled


### Single-device tensor training


In [ ]:
def ensure_uavnetsim_reference_available(
    repo_path=None,
):
    source_available = bool(
        uavnetsim_available(
            repo_path
        )
    )
    missing_runtime = []
    for import_name in (
        "simpy",
        "trimesh",
    ):
        try:
            importlib.import_module(
                import_name
            )
        except ModuleNotFoundError:
            missing_runtime.append(
                import_name
            )

    if (
        source_available
        and not missing_runtime
    ):
        return (
            resolve_uavnetsim_repo_path(
                repo_path
            )
        )

    if not running_on_kaggle():
        if (
            source_available
            and missing_runtime
        ):
            raise FileNotFoundError(
                "UavNetSim runtime dependencies are missing: "
                f"{missing_runtime}. Install requirements.txt."
            )
        raise FileNotFoundError(
            "UavNetSim is required for authoritative "
            "network evaluation. Install requirements.txt "
            "or pass uavnetsim_repo_path."
        )

    # Keep Kaggle evaluation reproducible. UavNetSim is installed with
    # --no-deps so pip cannot replace CUDA/PyTorch/NumPy. Only lightweight
    # dependencies required by the A2A adapter are bootstrapped separately.
    install_spec = (
        "git+https://github.com/"
        "Zihao-Felix-Zhou/UavNetSim.git@"
        "04daafb815eb377409b40b285574eeb62b9a8d58"
    )
    runtime_specs = {
        "simpy": "simpy==4.1.1",
        "trimesh": "trimesh==4.12.2",
    }
    from importlib.metadata import (
        PackageNotFoundError,
        version as package_version,
    )

    protected_versions = {}
    for package_name in (
        "numpy",
        "scipy",
        "torch",
    ):
        try:
            protected_versions[
                package_name
            ] = package_version(
                package_name
            )
        except PackageNotFoundError:
            protected_versions[
                package_name
            ] = None

    dependency_results = []
    for import_name in missing_runtime:
        spec = runtime_specs[
            import_name
        ]
        dependency_result = subprocess.run(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "--disable-pip-version-check",
                "--no-input",
                "--quiet",
                "--no-deps",
                spec,
            ],
            capture_output=True,
            text=True,
            check=False,
        )
        dependency_results.append(
            (
                import_name,
                dependency_result,
            )
        )
        importlib.invalidate_caches()
        try:
            importlib.import_module(
                import_name
            )
        except ModuleNotFoundError as exc:
            detail = (
                dependency_result.stderr.strip()
                or dependency_result.stdout.strip()
                or "unknown pip failure"
            )
            raise RuntimeError(
                f"failed to install {spec} required by "
                f"UavNetSim: {detail}"
            ) from exc

    if source_available:
        result = None
    else:
        result = subprocess.run(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "--disable-pip-version-check",
                "--no-input",
                "--quiet",
                "--no-deps",
                install_spec,
            ],
            capture_output=True,
            text=True,
            check=False,
        )
        importlib.invalidate_caches()

    changed_packages = {}
    for (
        package_name,
        before_version,
    ) in protected_versions.items():
        try:
            after_version = package_version(
                package_name
            )
        except PackageNotFoundError:
            after_version = None
        if after_version != before_version:
            changed_packages[
                package_name
            ] = {
                "before": before_version,
                "after": after_version,
            }

    if changed_packages:
        raise RuntimeError(
            "UavNetSim bootstrap unexpectedly changed "
            "protected runtime packages: "
            f"{changed_packages}"
        )

    for (
        import_name,
        dependency_result,
    ) in dependency_results:
        if dependency_result.returncode != 0:
            detail = (
                dependency_result.stderr.strip()
                or dependency_result.stdout.strip()
                or "unknown pip failure"
            )
            raise RuntimeError(
                "failed to install pinned UavNetSim runtime "
                f"dependency {import_name}: {detail}"
            )

    if (
        result is not None
        and result.returncode != 0
    ):
        detail = (
            result.stderr.strip()
            or result.stdout.strip()
            or "unknown pip failure"
        )
        raise RuntimeError(
            "failed to install pinned UavNetSim "
            f"for Kaggle evaluation: {detail}"
        )

    if not uavnetsim_available(
        repo_path
    ):
        raise RuntimeError(
            "pinned UavNetSim installation completed "
            "but the simulator source is still unavailable"
        )

    return resolve_uavnetsim_repo_path(
        repo_path
    )


In [ ]:
def evaluate_full_gpu_reference(
    trainer,
    algorithm,
    network_backend,
    seed,
    episodes=None,
    uavnetsim_repo_path=None,
):
    algorithm = str(algorithm).strip().lower()
    if algorithm not in {"masac", "matd3"}:
        raise ValueError(
            "algorithm must be masac or matd3"
        )
    if episodes is None:
        episodes = int(
            CONFIG["training_eval_episodes"]
        )
    episodes = int(episodes)
    if episodes < 1:
        raise ValueError(
            "episodes must be >= 1"
        )

    normalized_backend = str(
        network_backend
    ).strip().lower()
    if normalized_backend in {
        "uavnetsim",
        "uavnetsim_gpu",
    }:
        eval_backend = "uavnetsim"
        resolved_uavnetsim_path = (
            ensure_uavnetsim_reference_available(
                uavnetsim_repo_path
            )
        )
        backend_kwargs = {
            "repo_path": (
                resolved_uavnetsim_path
            )
        }
    else:
        # packet_approx has no event-level CPU counterpart. Evaluate its
        # learned policy in the deterministic Simple reference backend and
        # report the backend explicitly instead of inventing packet KPIs.
        eval_backend = "simple"
        backend_kwargs = {}

    python_state = random.getstate()
    numpy_state = np.random.get_state()
    torch_state = torch.get_rng_state()
    cuda_states = (
        torch.cuda.get_rng_state_all()
        if torch.cuda.is_available()
        else []
    )
    trainer_rng_state = None
    if hasattr(trainer, "rng"):
        trainer_rng_state = copy.deepcopy(
            trainer.rng.bit_generator.state
        )
    actor_training = bool(
        trainer.actor.training
    )

    env = None
    try:
        trainer.actor.eval()
        env = UAVSearchEnv(
            backend_name=eval_backend,
            network_backend_kwargs=(
                backend_kwargs
            ),
        )
        evaluation_seed = (
            int(seed)
            + int(
                CONFIG[
                    "training_eval_seed_offset"
                ]
            )
        )
        if algorithm == "masac":
            rows = evaluate_masac(
                env,
                trainer,
                episodes=episodes,
                seed=evaluation_seed,
            )
        else:
            rows = evaluate_matd3(
                env,
                trainer,
                episodes=episodes,
                seed=evaluation_seed,
            )
        summary = summarize_evaluation_results(
            rows
        )
        return {
            "backend": eval_backend,
            "summary": summary,
            "episodes": rows,
            "seed": evaluation_seed,
        }
    finally:
        if env is not None:
            env.close()
        trainer.actor.train(
            actor_training
        )
        random.setstate(
            python_state
        )
        np.random.set_state(
            numpy_state
        )
        torch.set_rng_state(
            torch_state
        )
        if (
            torch.cuda.is_available()
            and cuda_states
        ):
            torch.cuda.set_rng_state_all(
                cuda_states
            )
        if (
            trainer_rng_state is not None
            and hasattr(trainer, "rng")
        ):
            trainer.rng.bit_generator.state = (
                trainer_rng_state
            )


In [ ]:
def train_full_gpu(
    repo: Path,
    algorithm: str,
    num_envs: int,
    total_transitions: int,
    seed: int,
    device: str,
    network_backend: str = "simple",
    uavnetsim_repo_path=None,
    strict_cuda: bool = True,
    compile_mode=None,
    fused_adam: bool = False,
    update_to_data_ratio=None,
    enable_wandb: bool = False,
    enable_evaluation=None,
):
    algorithm = str(algorithm).strip().lower()
    if algorithm not in {"masac", "matd3"}:
        raise ValueError(
            "algorithm must be masac or matd3"
        )
    if int(total_transitions) < 1:
        raise ValueError(
            "total_transitions must be >= 1"
        )
    network_backend = str(network_backend).strip().lower()
    if network_backend not in {
        "simple",
        "packet_approx",
        "uavnetsim_gpu",
        "uavnetsim",
    }:
        raise ValueError(
            "network_backend must be simple, packet_approx, uavnetsim_gpu, or uavnetsim"
        )
    if network_backend == "uavnetsim" and int(num_envs) != 1:
        raise ValueError(
            "UavNetSim packet-level training currently requires num_envs=1"
        )

    torch.use_deterministic_algorithms(True)
    random.seed(int(seed))
    torch.manual_seed(int(seed))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(seed))

    env = FullGpuUAVBatchEnv(
        num_envs=num_envs,
        device=device,
        seed=seed,
        strict_cuda=bool(strict_cuda),
    )
    if network_backend in {
        "simple",
        "packet_approx",
        "uavnetsim_gpu",
    }:
        env.network_model = network_backend
    if env.device.type == "cuda":
        env.assert_all_state_on_cuda()
    trainer = _trainer(
        repo,
        algorithm,
        device,
        seed,
    )
    wandb_run = _init_gpu_wandb_run(
        algorithm,
        seed,
        enabled=enable_wandb,
        network_backend=network_backend,
        num_envs=num_envs,
        gpu_count=(
            1
            if env.device.type == "cuda"
            else 0
        ),
    )
    wandb_last_log_step = -1
    if enable_evaluation is None:
        enable_evaluation = bool(
            enable_wandb
        )
    enable_evaluation = bool(
        enable_evaluation
    )
    eval_interval = max(
        1,
        int(
            CONFIG[
                "training_eval_interval_steps"
            ]
        ),
    )
    eval_episodes = max(
        1,
        int(
            CONFIG[
                "training_eval_episodes"
            ]
        ),
    )
    last_eval_step = 0
    latest_evaluation = None
    evaluation_backend = None
    fused_adam_enabled = False
    if bool(fused_adam):
        fused_adam_enabled = (
            _rebuild_cuda_optimizers_with_fused_adam(
                trainer,
                algorithm,
            )
        )
    compiled_modules = []
    if compile_mode is not None:
        compiled_modules = _compile_trainer_modules(
            trainer,
            algorithm,
            mode=compile_mode,
        )
    if network_backend == "uavnetsim":
        bridge = GpuUavNetSimBridge(
            env,
            trainer._project_namespace,
            repo_path=uavnetsim_repo_path,
            seed=seed,
        )
        env.attach_external_network(bridge)
    else:
        bridge = None
    if (
        env.observation_dim
        != trainer.observation_dim
        or env.state_dim
        != trainer.state_dim
        or env.discrete_dim
        != trainer.discrete_dim
    ):
        raise RuntimeError(
            "full-GPU environment dimensions do not match "
            "the CPU-reference learner contract"
        )

    capacity = min(
        max(
            int(
                CONFIG[
                    f"{algorithm}_replay_capacity"
                ]
            ),
            4096,
        ),
        max(
            int(total_transitions),
            4096,
        ),
    )
    replay = make_project_gpu_replay_buffer(
        trainer,
        capacity=capacity,
        num_agents=env.num_uavs,
        observation_dim=env.observation_dim,
        state_dim=env.state_dim,
        continuous_dim=env.continuous_dim,
        discrete_dim=env.discrete_dim,
        device=device,
        seed=seed + 123,
        strict_cuda=bool(strict_cuda),
    )
    batch_size = int(
        CONFIG[f"{algorithm}_batch_size"]
    )
    learning_starts = int(
        CONFIG[
            f"{algorithm}_learning_starts"
        ]
    )
    if update_to_data_ratio is None:
        update_to_data_ratio = float(
            CONFIG[
                f"{algorithm}_updates_per_step"
            ]
        )
    else:
        update_to_data_ratio = float(
            update_to_data_ratio
        )
    if update_to_data_ratio < 0.0:
        raise ValueError(
            "update_to_data_ratio must be >= 0"
        )
    update_credit = 0.0

    obs, state, mask = env.reset()
    global_step = 0
    update_count = 0
    latest_metrics = None
    completed_episodes = torch.zeros(
        num_envs,
        device=env.device,
        dtype=torch.long,
    )
    transition_offsets = torch.arange(
        num_envs,
        device=env.device,
        dtype=torch.long,
    )
    total_transitions_int = int(
        total_transitions
    )

    if env.device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(
            env.device
        )
        torch.cuda.synchronize(env.device)
    start = time.perf_counter()

    while global_step < total_transitions_int:
        transition_steps = (
            transition_offsets
            + global_step
            + 1
        )
        warmup = (
            transition_steps
            <= learning_starts
        )

        batch_first_step = global_step + 1
        batch_last_step = global_step + num_envs
        all_warmup = batch_last_step <= learning_starts
        all_policy = batch_first_step > learning_starts

        if all_warmup:
            continuous, destination = (
                _random_exploration_actions(
                    env,
                    mask,
                )
            )
        elif all_policy:
            if algorithm == "masac":
                with torch.no_grad():
                    sampled = (
                        trainer._sample_joint_policy(
                            obs,
                            mask,
                            deterministic=False,
                        )
                    )
                    continuous = sampled[
                        "continuous"
                    ]
                    destination = sampled[
                        "destination_indices"
                    ]
            else:
                continuous, destination = (
                    _matd3_actions(
                        trainer,
                        obs,
                        mask,
                        transition_steps,
                        env.generator,
                    )
                )
        else:
            # Only one vector batch can straddle learning_starts. In that
            # boundary batch both paths are needed to preserve per-transition
            # semantics exactly.
            random_continuous, random_destination = (
                _random_exploration_actions(
                    env,
                    mask,
                )
            )
            if algorithm == "masac":
                with torch.no_grad():
                    sampled = (
                        trainer._sample_joint_policy(
                            obs,
                            mask,
                            deterministic=False,
                        )
                    )
                    policy_continuous = sampled[
                        "continuous"
                    ]
                    policy_destination = sampled[
                        "destination_indices"
                    ]
            else:
                (
                    policy_continuous,
                    policy_destination,
                ) = _matd3_actions(
                    trainer,
                    obs,
                    mask,
                    transition_steps,
                    env.generator,
                )
            continuous = torch.where(
                warmup[:, None, None],
                random_continuous,
                policy_continuous,
            )
            destination = torch.where(
                warmup[:, None],
                random_destination,
                policy_destination,
            )

        (
            next_obs,
            next_state,
            next_mask,
            reward,
            done,
            _,
        ) = env.step(
            continuous,
            destination,
        )
        take = min(
            num_envs,
            int(total_transitions)
            - global_step,
        )
        replay.add_batch(
            obs[:take],
            state[:take],
            continuous[:take],
            destination[:take],
            mask[:take],
            reward[:take],
            next_obs[:take],
            next_state[:take],
            next_mask[:take],
            done[:take],
        )
        previous = global_step
        global_step += take

        if len(replay) >= batch_size:
            first_train_step = max(
                previous,
                learning_starts,
            )
            eligible = max(
                0,
                global_step - first_train_step,
            )
            update_credit += (
                eligible
                * update_to_data_ratio
            )
            updates_this_step = int(
                update_credit
            )
            update_credit -= (
                updates_this_step
            )
            for local_update in range(
                updates_this_step
            ):
                collected = trainer.update(
                    replay,
                    batch_size=batch_size,
                    collect_metrics=(
                        local_update
                        == updates_this_step - 1
                    ),
                )
                if collected is not None:
                    latest_metrics = collected
                update_count += 1

        should_evaluate = (
            enable_evaluation
            and (
                global_step
                - last_eval_step
                >= eval_interval
                or global_step
                >= total_transitions_int
            )
        )
        if should_evaluate:
            evaluation = (
                evaluate_full_gpu_reference(
                    trainer=trainer,
                    algorithm=algorithm,
                    network_backend=(
                        network_backend
                    ),
                    seed=seed,
                    episodes=eval_episodes,
                    uavnetsim_repo_path=(
                        uavnetsim_repo_path
                    ),
                )
            )
            latest_evaluation = (
                evaluation["summary"]
            )
            evaluation_backend = (
                evaluation["backend"]
            )
            last_eval_step = global_step

            if wandb_run is not None:
                eval_payload = {
                    "global_step": int(
                        global_step
                    ),
                    **{
                        f"evaluation/{key}": float(
                            value
                        )
                        for key, value
                        in latest_evaluation.items()
                        if isinstance(
                            value,
                            (bool, int, float),
                        )
                    },
                }
                wandb_run.log(
                    eval_payload
                )
                for key, value in (
                    latest_evaluation.items()
                ):
                    if isinstance(
                        value,
                        (bool, int, float),
                    ):
                        wandb_run.summary[
                            f"evaluation_latest/{key}"
                        ] = float(value)

        if (
            wandb_run is not None
            and (
                global_step - wandb_last_log_step
                >= max(1, int(CONFIG.get("training_log_interval_steps", 100)))
                or global_step >= total_transitions_int
            )
        ):
            elapsed_for_log = max(
                1e-12, time.perf_counter() - start
            )
            payload = {
                "global_step": int(global_step),
                "train/transitions_per_second": (
                    global_step / elapsed_for_log
                ),
                "train/updates_per_second": (
                    update_count / elapsed_for_log
                ),
            }
            if latest_metrics is not None:
                for key, value in latest_metrics.items():
                    if isinstance(value, torch.Tensor):
                        if value.numel() != 1:
                            continue
                        value = value.detach().cpu().item()
                    if isinstance(value, (bool, int, float)):
                        payload[f"train/{key}"] = float(value)
            wandb_run.log(payload)
            wandb_last_log_step = global_step

        # A scalar host check only orchestrates rare episode reset. State
        # generation/reset and all environment math remain CUDA tensor work.
        if bool(done.any().item()):
            completed_episodes += (
                done.long()
            )
            if global_step < int(
                total_transitions
            ):
                obs, state, mask = env.reset(
                    mask=done
                )
            else:
                obs, state, mask = (
                    next_obs,
                    next_state,
                    next_mask,
                )
        else:
            obs, state, mask = (
                next_obs,
                next_state,
                next_mask,
            )

    if env.device.type == "cuda":
        torch.cuda.synchronize(env.device)
    wall = time.perf_counter() - start

    def metric_value(value):
        if isinstance(value, torch.Tensor):
            if value.numel() == 1:
                return float(
                    value.detach().cpu().item()
                )
            return None
        if isinstance(
            value,
            (int, float, bool),
        ):
            return value
        return None

    metrics = None
    if latest_metrics is not None:
        metrics = {
            key: metric_value(value)
            for key, value
            in latest_metrics.items()
            if metric_value(value)
            is not None
        }

    network_metrics = (
        bridge.metrics()
        if bridge is not None
        else None
    )
    result = {
        "kind": (
            "gpu_mission_uavnetsim_train"
            if network_backend == "uavnetsim"
            else (
                "full_gpu_uavnetsim_surrogate_train"
                if network_backend == "uavnetsim_gpu"
                else (
                    "full_gpu_packet_train"
                    if network_backend == "packet_approx"
                    else "full_gpu_train"
                )
            )
        ),
        "network_backend": network_backend,
        "algorithm": algorithm,
        "compile_mode": compile_mode,
        "compiled_modules": compiled_modules,
        "fused_adam": bool(
            fused_adam_enabled
        ),
        "num_envs": num_envs,
        "transitions": global_step,
        "updates": update_count,
        "update_to_data_ratio": float(
            update_to_data_ratio
        ),
        "episodes_completed": int(
            completed_episodes.sum()
            .detach()
            .cpu()
            .item()
        ),
        "wall_seconds": wall,
        "transitions_per_second": (
            global_step / wall
        ),
        "updates_per_second": (
            update_count / wall
        ),
        "gpu_name": (
            torch.cuda.get_device_name(
                env.device
            )
            if env.device.type == "cuda"
            else None
        ),
        "peak_memory_mb": (
            torch.cuda.max_memory_allocated(
                env.device
            )
            / 2**20
            if env.device.type == "cuda"
            else None
        ),
        "latest_update_metrics": metrics,
        "model_fingerprint_sha256": (
            _trainer_parameter_fingerprint(
                trainer
            )
        ),
        "all_env_state_cuda": (
            env.device.type == "cuda"
        ),
        "network_metrics": network_metrics,
        "latest_evaluation": (
            latest_evaluation
        ),
        "evaluation_backend": (
            evaluation_backend
        ),
    }
    result["wandb_run_id"] = (
        wandb_run.id if wandb_run is not None else None
    )
    result["wandb_run_name"] = (
        wandb_run.name if wandb_run is not None else None
    )
    result["wandb_run_url"] = (
        wandb_run.url if wandb_run is not None else None
    )
    if wandb_run is not None:
        wandb_run.summary["final/transitions_per_second"] = result[
            "transitions_per_second"
        ]
        wandb_run.summary["final/updates_per_second"] = result[
            "updates_per_second"
        ]
        wandb_run.summary["final/gpu_count"] = (
            1
            if env.device.type == "cuda"
            else 0
        )
        wandb_run.finish()
    env.close()
    return result


### Distributed Data Parallel (DDP) helpers


In [ ]:
class _MASACDDPActorForward(torch.nn.Module):
    def __init__(self, actor, project_ns):
        super().__init__()
        self.actor = actor
        self._radial_squash = project_ns["radial_squash_motion_action"]
        self._categorical_sample = project_ns["straight_through_categorical_sample"]

    def forward(self, observations, destination_mask):
        mean, log_std, logits = self.actor(observations)
        std = torch.exp(log_std)
        distribution = torch.distributions.Normal(mean, std)
        pre_tanh = distribution.rsample()
        motion_action, motion_log_det = self._radial_squash(pre_tanh[..., :3])
        power_action = torch.tanh(pre_tanh[..., 3:4])
        continuous_action = torch.cat((motion_action, power_action), dim=-1)
        power_log_det = torch.log(
            1.0 - power_action.pow(2) + 1e-6
        ).sum(dim=-1, keepdim=True)
        log_probability = (
            distribution.log_prob(pre_tanh).sum(dim=-1, keepdim=True)
            - motion_log_det
            - power_log_det
        )
        discrete = self._categorical_sample(
            logits,
            destination_mask,
            temperature=CONFIG["masac_gumbel_temperature"],
            deterministic=False,
        )
        return (
            continuous_action,
            log_probability,
            discrete["action"],
            discrete["indices"],
            discrete["log_probability"],
            discrete["entropy"],
        )


In [ ]:
class _MATD3DDPActorForward(torch.nn.Module):
    def __init__(self, actor, project_ns):
        super().__init__()
        self.actor = actor
        self._masked_argmax = project_ns["straight_through_masked_argmax"]

    def forward(self, observations, destination_mask):
        continuous, logits = self.actor(observations)
        discrete = self._masked_argmax(logits, destination_mask)
        return (
            continuous,
            discrete["action"],
            discrete["indices"],
            discrete["probabilities"],
        )


### W&B integration for GPU/DDP runs


In [ ]:
def _gpu_wandb_api_key_from_kaggle():
    if os.environ.get("WANDB_API_KEY"):
        return os.environ["WANDB_API_KEY"]
    try:
        from kaggle_secrets import UserSecretsClient
        client = UserSecretsClient()
        for secret_name in CONFIG.get("kaggle_wandb_secret_names", ()):
            try:
                value = client.get_secret(str(secret_name))
            except Exception:
                continue
            if value:
                os.environ["WANDB_API_KEY"] = str(value)
                return str(value)
    except Exception:
        pass
    return None


In [ ]:
def _init_gpu_wandb_run(
    algorithm,
    seed,
    enabled=None,
    network_backend=None,
    num_envs=None,
    gpu_count=None,
):
    if enabled is None:
        enabled = bool(CONFIG.get("training_enable_wandb", True))
    if not enabled:
        return None
    mode = str(
        CONFIG.get(
            "training_wandb_mode",
            "online",
        )
    ).strip().lower()
    if (
        mode == "online"
        and not os.environ.get(
            "WANDB_API_KEY"
        )
    ):
        resolved_key = (
            _gpu_wandb_api_key_from_kaggle()
        )
        if (
            running_on_kaggle()
            and not resolved_key
        ):
            if bool(
                CONFIG.get(
                    "training_wandb_required",
                    True,
                )
            ):
                raise RuntimeError(
                    "W&B online mode requires a "
                    "WANDB_API_KEY Kaggle Secret"
                )
            return None
    wandb = importlib.import_module("wandb")
    import secrets
    run_id = secrets.token_hex(4)
    name_template = str(
        CONFIG.get(
            "training_wandb_name_format",
            "{algorithm}-seed{seed}-{id}",
        )
    )
    run_name = name_template.format(
        algorithm=str(algorithm).upper(),
        seed=int(seed),
        id=run_id,
    )
    try:
        run = wandb.init(
            entity=CONFIG.get("training_wandb_entity"),
            project=str(CONFIG["training_wandb_project"]),
            name=run_name,
            id=run_id,
            mode=mode,
            reinit="finish_previous",
            settings=wandb.Settings(
                x_disable_stats=bool(
                    CONFIG.get(
                        "training_wandb_disable_system_stats",
                        True,
                    )
                )
            ),
            save_code=False,
            tags=None,
            group=None,
            job_type=None,
        )
    except Exception:
        if bool(CONFIG.get("training_wandb_required", True)):
            raise
        return None

    run.define_metric(
        "global_step"
    )
    for pattern in (
        "train/*",
        "evaluation/*",
    ):
        run.define_metric(
            pattern,
            step_metric="global_step",
        )
    run.config.update(
        {
            "algorithm": str(
                algorithm
            ).lower(),
            "seed": int(seed),
            "network_backend": str(
                (
                    network_backend
                    if network_backend is not None
                    else CONFIG.get(
                        "full_gpu_network_model",
                        "simple",
                    )
                )
            ),
            **(
                {
                    "num_envs": int(
                        num_envs
                    )
                }
                if num_envs is not None
                else {}
            ),
            **(
                {
                    "gpu_count": int(
                        gpu_count
                    )
                }
                if gpu_count is not None
                else {}
            ),
            "num_uavs": int(
                CONFIG["num_uavs"]
            ),
            "num_targets": int(
                CONFIG["num_targets"]
            ),
            "num_obstacles": int(
                CONFIG["num_obstacles"]
            ),
        },
        allow_val_change=True,
    )
    return run


In [ ]:
def _ddp_actor_sample_masac(trainer, actor_ddp, observations, masks):
    batch_size = int(observations.shape[0])
    flat_obs = observations.reshape(
        batch_size * trainer.num_agents,
        trainer.observation_dim,
    )
    flat_mask = masks.reshape(
        batch_size * trainer.num_agents,
        trainer.discrete_dim,
    )
    (
        continuous,
        continuous_log_probability,
        destination_one_hot,
        destination_indices,
        destination_log_probability,
        destination_entropy,
    ) = actor_ddp(flat_obs, flat_mask)
    return {
        "continuous": continuous.reshape(
            batch_size, trainer.num_agents, trainer.continuous_dim
        ),
        "continuous_log_probability": continuous_log_probability.reshape(
            batch_size, trainer.num_agents, 1
        ).sum(dim=1),
        "destination_one_hot": destination_one_hot.reshape(
            batch_size, trainer.num_agents, trainer.discrete_dim
        ),
        "destination_indices": destination_indices.reshape(
            batch_size, trainer.num_agents
        ),
        "destination_log_probability": destination_log_probability.reshape(
            batch_size, trainer.num_agents, 1
        ).sum(dim=1),
        "destination_entropy": destination_entropy.reshape(
            batch_size, trainer.num_agents, 1
        ).sum(dim=1),
    }


In [ ]:
def _ddp_actor_actions_matd3(
    trainer, actor_ddp, observations, masks
):
    batch_size = int(observations.shape[0])
    flat_obs = observations.reshape(
        batch_size * trainer.num_agents,
        trainer.observation_dim,
    )
    flat_mask = masks.reshape(
        batch_size * trainer.num_agents,
        trainer.discrete_dim,
    )
    continuous, one_hot, indices, probabilities = actor_ddp(
        flat_obs, flat_mask
    )
    return {
        "continuous": continuous.reshape(
            batch_size, trainer.num_agents, trainer.continuous_dim
        ),
        "destination_one_hot": one_hot.reshape(
            batch_size, trainer.num_agents, trainer.discrete_dim
        ),
        "destination_indices": indices.reshape(
            batch_size, trainer.num_agents
        ),
        "destination_probabilities": probabilities.reshape(
            batch_size, trainer.num_agents, trainer.discrete_dim
        ),
    }


In [ ]:
def _allreduce_parameter_grad(parameter):
    import torch.distributed as dist
    if parameter.grad is None:
        return
    dist.all_reduce(parameter.grad, op=dist.ReduceOp.SUM)
    parameter.grad.div_(dist.get_world_size())


In [ ]:
def _ddp_update_masac(
    trainer,
    replay,
    batch_size,
    actor_ddp,
    critic_1_ddp,
    critic_2_ddp,
    collect_metrics=True,
):
    batch = replay.sample(batch_size, trainer.device)
    destination_one_hot = torch.nn.functional.one_hot(
        batch["destination_indices"],
        num_classes=trainer.discrete_dim,
    ).to(torch.float32)
    replay_joint_action = trainer._joint_action_tensor(
        batch["continuous_actions"],
        destination_one_hot,
    )

    with torch.no_grad(), trainer._autocast():
        next_policy = trainer._sample_joint_policy(
            batch["next_observations"],
            batch["next_destination_masks"],
            deterministic=False,
        )
        next_joint_action = trainer._joint_action_tensor(
            next_policy["continuous"],
            next_policy["destination_one_hot"],
        )
        target_q = torch.minimum(
            trainer.target_critic_1(
                batch["next_states"], next_joint_action
            ),
            trainer.target_critic_2(
                batch["next_states"], next_joint_action
            ),
        )
        entropy_adjusted_target = (
            target_q
            - trainer.alpha_continuous.detach()
            * next_policy["continuous_log_probability"]
            - trainer.alpha_discrete.detach()
            * next_policy["destination_log_probability"]
        )
        critic_target = (
            batch["rewards"]
            + trainer.gamma
            * (1.0 - batch["terminated"])
            * entropy_adjusted_target
        )

    with trainer._autocast():
        critic_1_value = critic_1_ddp(
            batch["states"], replay_joint_action
        )
        critic_2_value = critic_2_ddp(
            batch["states"], replay_joint_action
        )
        critic_loss = (
            torch.nn.functional.mse_loss(
                critic_1_value, critic_target
            )
            + torch.nn.functional.mse_loss(
                critic_2_value, critic_target
            )
        )

    trainer.critic_optimizer.zero_grad(set_to_none=True)
    if trainer.amp_enabled:
        trainer.grad_scaler.scale(critic_loss).backward()
        trainer.grad_scaler.unscale_(trainer.critic_optimizer)
    else:
        critic_loss.backward()
    critic_grad_norm = torch.nn.utils.clip_grad_norm_(
        list(trainer.critic_1.parameters())
        + list(trainer.critic_2.parameters()),
        trainer.gradient_clip_norm,
    )
    if trainer.amp_enabled:
        trainer.grad_scaler.step(trainer.critic_optimizer)
    else:
        trainer.critic_optimizer.step()

    for critic in (trainer.critic_1, trainer.critic_2):
        for parameter in critic.parameters():
            parameter.requires_grad_(False)

    with trainer._autocast():
        policy = _ddp_actor_sample_masac(
            trainer,
            actor_ddp,
            batch["observations"],
            batch["destination_masks"],
        )
        policy_joint_action = trainer._joint_action_tensor(
            policy["continuous"],
            policy["destination_one_hot"],
        )
        policy_q = torch.minimum(
            trainer.critic_1(
                batch["states"], policy_joint_action
            ),
            trainer.critic_2(
                batch["states"], policy_joint_action
            ),
        )
        actor_loss = (
            trainer.alpha_continuous.detach()
            * policy["continuous_log_probability"]
            - trainer.alpha_discrete.detach()
            * policy["destination_entropy"]
            - policy_q
        ).mean()

    trainer.actor_optimizer.zero_grad(set_to_none=True)
    if trainer.amp_enabled:
        trainer.grad_scaler.scale(actor_loss).backward()
        trainer.grad_scaler.unscale_(trainer.actor_optimizer)
    else:
        actor_loss.backward()
    actor_grad_norm = torch.nn.utils.clip_grad_norm_(
        trainer.actor.parameters(),
        trainer.gradient_clip_norm,
    )
    if trainer.amp_enabled:
        trainer.grad_scaler.step(trainer.actor_optimizer)
        trainer.grad_scaler.update()
    else:
        trainer.actor_optimizer.step()

    for critic in (trainer.critic_1, trainer.critic_2):
        for parameter in critic.parameters():
            parameter.requires_grad_(True)

    continuous_entropy = -policy[
        "continuous_log_probability"
    ].detach()
    continuous_target = torch.full_like(
        continuous_entropy,
        float(trainer.continuous_target_entropy),
    )
    valid_counts = batch["destination_masks"].sum(
        dim=-1
    ).clamp(min=1.0)
    discrete_target = (
        trainer.discrete_target_entropy_ratio
        * torch.log(valid_counts).sum(
            dim=1, keepdim=True
        )
    )
    discrete_entropy = policy["destination_entropy"].detach()
    alpha_continuous_loss = (
        trainer.log_alpha_continuous
        * (continuous_entropy - continuous_target)
    ).mean()
    alpha_discrete_loss = (
        trainer.log_alpha_discrete
        * (discrete_entropy - discrete_target)
    ).mean()

    trainer.alpha_continuous_optimizer.zero_grad(set_to_none=True)
    alpha_continuous_loss.backward()
    _allreduce_parameter_grad(trainer.log_alpha_continuous)
    trainer.alpha_continuous_optimizer.step()

    trainer.alpha_discrete_optimizer.zero_grad(set_to_none=True)
    alpha_discrete_loss.backward()
    _allreduce_parameter_grad(trainer.log_alpha_discrete)
    trainer.alpha_discrete_optimizer.step()

    trainer._soft_update_targets()
    trainer.update_count += 1

    if not collect_metrics:
        return None
    return {
        "critic_loss": float(critic_loss.detach().cpu()),
        "actor_loss": float(actor_loss.detach().cpu()),
        "alpha_continuous_loss": float(
            alpha_continuous_loss.detach().cpu()
        ),
        "alpha_discrete_loss": float(
            alpha_discrete_loss.detach().cpu()
        ),
        "alpha_continuous": float(
            trainer.alpha_continuous.detach().cpu()
        ),
        "alpha_discrete": float(
            trainer.alpha_discrete.detach().cpu()
        ),
        "continuous_entropy": float(
            continuous_entropy.mean().cpu()
        ),
        "discrete_entropy": float(
            discrete_entropy.mean().cpu()
        ),
        "target_q_mean": float(
            critic_target.mean().detach().cpu()
        ),
        "critic_grad_norm": float(
            torch.as_tensor(critic_grad_norm).detach().cpu()
        ),
        "actor_grad_norm": float(
            torch.as_tensor(actor_grad_norm).detach().cpu()
        ),
        "update_count": int(trainer.update_count),
    }


In [ ]:
def _ddp_update_matd3(
    trainer,
    replay,
    batch_size,
    actor_ddp,
    critic_1_ddp,
    critic_2_ddp,
    collect_metrics=True,
):
    batch = replay.sample(batch_size, trainer.device)
    replay_destination_one_hot = torch.nn.functional.one_hot(
        batch["destination_indices"],
        num_classes=trainer.discrete_dim,
    ).to(torch.float32)
    replay_joint_action = trainer._joint_action_tensor(
        batch["continuous_actions"],
        replay_destination_one_hot,
    )

    with torch.no_grad(), trainer._autocast():
        target_policy = trainer._actor_joint_actions(
            batch["next_observations"],
            batch["next_destination_masks"],
            straight_through=False,
            target_actor=True,
        )
        noise = (
            torch.randn_like(target_policy["continuous"])
            * trainer.target_policy_noise
        ).clamp(
            -trainer.target_noise_clip,
            trainer.target_noise_clip,
        )
        target_continuous = (
            target_policy["continuous"] + noise
        ).clamp(-1.0, 1.0)
        target_continuous = torch.cat(
            (
                trainer._project_namespace[
                    "project_motion_action_tensor"
                ](target_continuous[..., :3]),
                target_continuous[..., 3:4],
            ),
            dim=-1,
        )
        target_joint_action = trainer._joint_action_tensor(
            target_continuous,
            target_policy["destination_one_hot"],
        )
        target_q = torch.minimum(
            trainer.target_critic_1(
                batch["next_states"], target_joint_action
            ),
            trainer.target_critic_2(
                batch["next_states"], target_joint_action
            ),
        )
        critic_target = (
            batch["rewards"]
            + trainer.gamma
            * (1.0 - batch["terminated"])
            * target_q
        )

    with trainer._autocast():
        critic_1_value = critic_1_ddp(
            batch["states"], replay_joint_action
        )
        critic_2_value = critic_2_ddp(
            batch["states"], replay_joint_action
        )
        critic_loss = (
            torch.nn.functional.mse_loss(
                critic_1_value, critic_target
            )
            + torch.nn.functional.mse_loss(
                critic_2_value, critic_target
            )
        )

    trainer.critic_optimizer.zero_grad(set_to_none=True)
    if trainer.amp_enabled:
        trainer.grad_scaler.scale(critic_loss).backward()
        trainer.grad_scaler.unscale_(trainer.critic_optimizer)
    else:
        critic_loss.backward()
    critic_grad_norm = torch.nn.utils.clip_grad_norm_(
        list(trainer.critic_1.parameters())
        + list(trainer.critic_2.parameters()),
        trainer.gradient_clip_norm,
    )
    if trainer.amp_enabled:
        trainer.grad_scaler.step(trainer.critic_optimizer)
    else:
        trainer.critic_optimizer.step()

    trainer.update_count += 1
    actor_updated = bool(
        trainer.update_count % trainer.policy_delay == 0
    )
    actor_loss = None
    actor_grad_norm = None

    if actor_updated:
        for critic in (trainer.critic_1, trainer.critic_2):
            for parameter in critic.parameters():
                parameter.requires_grad_(False)

        with trainer._autocast():
            policy = _ddp_actor_actions_matd3(
                trainer,
                actor_ddp,
                batch["observations"],
                batch["destination_masks"],
            )
            policy_joint_action = trainer._joint_action_tensor(
                policy["continuous"],
                policy["destination_one_hot"],
            )
            actor_loss = -trainer.critic_1(
                batch["states"],
                policy_joint_action,
            ).mean()

        trainer.actor_optimizer.zero_grad(set_to_none=True)
        if trainer.amp_enabled:
            trainer.grad_scaler.scale(actor_loss).backward()
            trainer.grad_scaler.unscale_(trainer.actor_optimizer)
        else:
            actor_loss.backward()
        actor_grad_norm = torch.nn.utils.clip_grad_norm_(
            trainer.actor.parameters(),
            trainer.gradient_clip_norm,
        )
        if trainer.amp_enabled:
            trainer.grad_scaler.step(trainer.actor_optimizer)
        else:
            trainer.actor_optimizer.step()

        for critic in (trainer.critic_1, trainer.critic_2):
            for parameter in critic.parameters():
                parameter.requires_grad_(True)

        trainer._soft_update(
            trainer.target_actor, trainer.actor
        )
        trainer._soft_update(
            trainer.target_critic_1, trainer.critic_1
        )
        trainer._soft_update(
            trainer.target_critic_2, trainer.critic_2
        )

    if trainer.amp_enabled:
        trainer.grad_scaler.update()
    if not collect_metrics:
        return None
    metrics = {
        "critic_loss": float(critic_loss.detach().cpu()),
        "critic_1_q_mean": float(
            critic_1_value.detach().mean().cpu()
        ),
        "critic_2_q_mean": float(
            critic_2_value.detach().mean().cpu()
        ),
        "target_q_mean": float(
            critic_target.detach().mean().cpu()
        ),
        "critic_grad_norm": float(
            torch.as_tensor(critic_grad_norm).detach().cpu()
        ),
        "actor_updated": bool(actor_updated),
        "update_count": int(trainer.update_count),
    }
    if actor_loss is not None:
        metrics["actor_loss"] = float(
            actor_loss.detach().cpu()
        )
        metrics["actor_grad_norm"] = float(
            torch.as_tensor(actor_grad_norm).detach().cpu()
        )
    return metrics


In [ ]:
def _train_full_gpu_ddp_worker(
    repo,
    algorithm,
    num_envs,
    total_transitions,
    seed,
    network_backend,
    result_path,
    enable_wandb=True,
):
    import torch.distributed as dist
    from torch.nn.parallel import DistributedDataParallel as DDP
    from datetime import timedelta

    rank = int(os.environ["RANK"])
    local_rank = int(os.environ["LOCAL_RANK"])
    world_size = int(os.environ["WORLD_SIZE"])
    torch.cuda.set_device(local_rank)
    device = torch.device(f"cuda:{local_rank}")
    dist.init_process_group(
        backend="nccl",
        timeout=timedelta(
            seconds=max(
                3600,
                int(
                    CONFIG.get(
                        "full_gpu_ddp_timeout_s",
                        3600,
                    )
                ),
            )
        ),
    )

    algorithm = str(algorithm).lower()
    if algorithm not in {"masac", "matd3"}:
        raise ValueError("algorithm must be masac or matd3")
    if world_size != 2:
        raise ValueError("current automatic DDP implementation expects exactly 2 GPUs")

    torch.use_deterministic_algorithms(True)
    random.seed(int(seed))
    torch.manual_seed(int(seed))
    torch.cuda.manual_seed(int(seed))

    shard_counts = _split_env_counts(num_envs, world_size)
    local_envs = int(shard_counts[rank])
    env = FullGpuUAVBatchEnv(
        num_envs=local_envs,
        device=str(device),
        seed=int(seed) + rank * 1_000_003,
        strict_cuda=True,
    )
    env.network_model = str(network_backend)
    env.assert_all_state_on_cuda()

    trainer = _trainer(
        Path(repo), algorithm, str(device), int(seed)
    )
    project_ns = trainer._project_namespace

    if algorithm == "masac":
        actor_forward = _MASACDDPActorForward(
            trainer.actor, project_ns
        ).to(device)
    else:
        actor_forward = _MATD3DDPActorForward(
            trainer.actor, project_ns
        ).to(device)

    actor_ddp = DDP(
        actor_forward,
        device_ids=[local_rank],
        output_device=local_rank,
        broadcast_buffers=False,
        gradient_as_bucket_view=True,
    )
    critic_1_ddp = DDP(
        trainer.critic_1,
        device_ids=[local_rank],
        output_device=local_rank,
        broadcast_buffers=False,
        gradient_as_bucket_view=True,
    )
    critic_2_ddp = DDP(
        trainer.critic_2,
        device_ids=[local_rank],
        output_device=local_rank,
        broadcast_buffers=False,
        gradient_as_bucket_view=True,
    )

    # All source networks are synchronized by DDP construction. Mirror the
    # synchronized online critics/actor into the local targets before training.
    if algorithm == "masac":
        trainer.target_critic_1.load_state_dict(
            trainer.critic_1.state_dict()
        )
        trainer.target_critic_2.load_state_dict(
            trainer.critic_2.state_dict()
        )
    else:
        trainer.target_actor.load_state_dict(
            trainer.actor.state_dict()
        )
        trainer.target_critic_1.load_state_dict(
            trainer.critic_1.state_dict()
        )
        trainer.target_critic_2.load_state_dict(
            trainer.critic_2.state_dict()
        )

    # Rank-specific data RNG after synchronized model initialization.
    random.seed(int(seed) + rank * 97_409)
    torch.cuda.manual_seed(int(seed) + rank * 97_409)

    global_capacity = min(
        max(int(CONFIG[f"{algorithm}_replay_capacity"]), 4096),
        max(int(total_transitions), 4096),
    )
    local_capacity = max(
        4096,
        int(math.ceil(global_capacity / world_size)),
    )
    replay = make_project_gpu_replay_buffer(
        trainer,
        capacity=local_capacity,
        num_agents=env.num_uavs,
        observation_dim=env.observation_dim,
        state_dim=env.state_dim,
        continuous_dim=env.continuous_dim,
        discrete_dim=env.discrete_dim,
        device=str(device),
        seed=int(seed) + 123 + rank * 10_007,
        strict_cuda=True,
    )

    global_batch_size = int(
        CONFIG[f"{algorithm}_batch_size"]
    )
    local_batch_size = max(
        1, int(math.ceil(global_batch_size / world_size))
    )
    learning_starts = int(
        CONFIG[f"{algorithm}_learning_starts"]
    )
    update_to_data_ratio = float(
        CONFIG[f"{algorithm}_updates_per_step"]
    )

    wandb_run = (
        _init_gpu_wandb_run(
            algorithm,
            seed,
            enabled=enable_wandb,
            network_backend=network_backend,
            num_envs=num_envs,
            gpu_count=world_size,
        )
        if rank == 0
        else None
    )

    obs, state, mask = env.reset()
    global_step = 0
    update_count = 0
    update_credit = 0.0
    latest_metrics = None
    completed_episodes = 0
    shard_offsets = []
    running_offset = 0
    for count in shard_counts:
        shard_offsets.append(running_offset)
        running_offset += int(count)

    torch.cuda.reset_peak_memory_stats(device)
    torch.cuda.synchronize(device)
    start = time.perf_counter()
    log_interval = max(
        1, int(CONFIG.get("training_log_interval_steps", 100))
    )
    last_log_step = -1
    eval_interval = max(
        1,
        int(
            CONFIG[
                "training_eval_interval_steps"
            ]
        ),
    )
    eval_episodes = max(
        1,
        int(
            CONFIG[
                "training_eval_episodes"
            ]
        ),
    )
    last_eval_step = 0
    latest_evaluation = None
    evaluation_backend = None
    enable_evaluation = bool(
        enable_wandb
    )

    try:
        while global_step < int(total_transitions):
            local_offset = int(shard_offsets[rank])
            transition_steps = (
                torch.arange(
                    local_envs,
                    device=device,
                    dtype=torch.long,
                )
                + global_step
                + local_offset
                + 1
            )
            batch_first = global_step + 1
            batch_last = global_step + int(num_envs)
            all_warmup = batch_last <= learning_starts
            all_policy = batch_first > learning_starts

            if all_warmup:
                continuous, destination = (
                    _random_exploration_actions(env, mask)
                )
            elif all_policy:
                if algorithm == "masac":
                    with torch.no_grad():
                        sampled = trainer._sample_joint_policy(
                            obs,
                            mask,
                            deterministic=False,
                        )
                        continuous = sampled["continuous"]
                        destination = sampled[
                            "destination_indices"
                        ]
                else:
                    continuous, destination = _matd3_actions(
                        trainer,
                        obs,
                        mask,
                        transition_steps,
                        env.generator,
                    )
            else:
                random_continuous, random_destination = (
                    _random_exploration_actions(env, mask)
                )
                if algorithm == "masac":
                    with torch.no_grad():
                        sampled = trainer._sample_joint_policy(
                            obs,
                            mask,
                            deterministic=False,
                        )
                        policy_continuous = sampled["continuous"]
                        policy_destination = sampled[
                            "destination_indices"
                        ]
                else:
                    policy_continuous, policy_destination = (
                        _matd3_actions(
                            trainer,
                            obs,
                            mask,
                            transition_steps,
                            env.generator,
                        )
                    )
                warmup = transition_steps <= learning_starts
                continuous = torch.where(
                    warmup[:, None, None],
                    random_continuous,
                    policy_continuous,
                )
                destination = torch.where(
                    warmup[:, None],
                    random_destination,
                    policy_destination,
                )

            (
                next_obs,
                next_state,
                next_mask,
                reward,
                done,
                _,
            ) = env.step(continuous, destination)

            take_total = min(
                int(num_envs),
                int(total_transitions) - global_step,
            )
            local_take = max(
                0,
                min(
                    local_envs,
                    take_total - local_offset,
                ),
            )
            if local_take > 0:
                replay.add_batch(
                    obs[:local_take],
                    state[:local_take],
                    continuous[:local_take],
                    destination[:local_take],
                    mask[:local_take],
                    reward[:local_take],
                    next_obs[:local_take],
                    next_state[:local_take],
                    next_mask[:local_take],
                    done[:local_take],
                )

            previous = global_step
            global_step += take_total

            local_ready = torch.tensor(
                [int(len(replay) >= local_batch_size)],
                device=device,
                dtype=torch.int32,
            )
            dist.all_reduce(
                local_ready, op=dist.ReduceOp.MIN
            )
            if int(local_ready.item()) == 1:
                first_train_step = max(
                    previous, learning_starts
                )
                eligible = max(
                    0, global_step - first_train_step
                )
                update_credit += (
                    eligible * update_to_data_ratio
                )
                updates_this_step = int(update_credit)
                update_credit -= updates_this_step
                for local_update in range(
                    updates_this_step
                ):
                    collect = (
                        local_update
                        == updates_this_step - 1
                    )
                    if algorithm == "masac":
                        metrics = _ddp_update_masac(
                            trainer,
                            replay,
                            local_batch_size,
                            actor_ddp,
                            critic_1_ddp,
                            critic_2_ddp,
                            collect_metrics=collect,
                        )
                    else:
                        metrics = _ddp_update_matd3(
                            trainer,
                            replay,
                            local_batch_size,
                            actor_ddp,
                            critic_1_ddp,
                            critic_2_ddp,
                            collect_metrics=collect,
                        )
                    if metrics is not None:
                        latest_metrics = metrics
                    update_count += 1

            if bool(done.any().item()):
                completed_episodes += int(
                    done.long().sum().item()
                )
                if global_step < int(total_transitions):
                    obs, state, mask = env.reset(mask=done)
                else:
                    obs, state, mask = (
                        next_obs,
                        next_state,
                        next_mask,
                    )
            else:
                obs, state, mask = (
                    next_obs,
                    next_state,
                    next_mask,
                )

            if (
                rank == 0
                and wandb_run is not None
                and (
                    global_step - last_log_step
                    >= log_interval
                    or global_step
                    >= int(total_transitions)
                )
            ):
                elapsed = max(
                    1e-12, time.perf_counter() - start
                )
                payload = {
                    "global_step": int(global_step),
                    "train/transitions_per_second": (
                        global_step / elapsed
                    ),
                    "train/updates_per_second": (
                        update_count / elapsed
                    ),
                }
                if latest_metrics:
                    for key, value in latest_metrics.items():
                        if isinstance(
                            value, (bool, int, float)
                        ):
                            payload[
                                f"train/{key}"
                            ] = float(value)
                wandb_run.log(payload)
                last_log_step = global_step

            should_evaluate = (
                enable_evaluation
                and (
                    global_step
                    - last_eval_step
                    >= eval_interval
                    or global_step
                    >= int(
                        total_transitions
                    )
                )
            )
            if should_evaluate:
                dist.barrier(
                    device_ids=[local_rank]
                )
                if rank == 0:
                    evaluation = (
                        evaluate_full_gpu_reference(
                            trainer=trainer,
                            algorithm=algorithm,
                            network_backend=(
                                network_backend
                            ),
                            seed=seed,
                            episodes=eval_episodes,
                            uavnetsim_repo_path=None,
                        )
                    )
                    latest_evaluation = (
                        evaluation["summary"]
                    )
                    evaluation_backend = (
                        evaluation["backend"]
                    )
                    if wandb_run is not None:
                        wandb_run.log(
                            {
                                "global_step": int(
                                    global_step
                                ),
                                **{
                                    f"evaluation/{key}": float(
                                        value
                                    )
                                    for key, value
                                    in latest_evaluation.items()
                                    if isinstance(
                                        value,
                                        (
                                            bool,
                                            int,
                                            float,
                                        ),
                                    )
                                },
                            }
                        )
                        for key, value in (
                            latest_evaluation.items()
                        ):
                            if isinstance(
                                value,
                                (
                                    bool,
                                    int,
                                    float,
                                ),
                            ):
                                wandb_run.summary[
                                    f"evaluation_latest/{key}"
                                ] = float(
                                    value
                                )
                dist.barrier(
                    device_ids=[local_rank]
                )
                last_eval_step = (
                    global_step
                )

        torch.cuda.synchronize(device)
        wall = time.perf_counter() - start
        dist.barrier(device_ids=[rank])

        rank_result = {
            "rank": rank,
            "device": str(device),
            "gpu_name": torch.cuda.get_device_name(device),
            "local_envs": local_envs,
            "local_replay_size": int(len(replay)),
            "peak_memory_mb": (
                torch.cuda.max_memory_allocated(device)
                / 2**20
            ),
            "completed_episodes": completed_episodes,
        }
        gathered = [None] * world_size if rank == 0 else None
        dist.gather_object(
            rank_result,
            object_gather_list=gathered,
            dst=0,
        )

        if rank == 0:
            result = {
                "kind": "full_gpu_ddp",
                "multi_gpu_strategy": "ddp",
                "algorithm": algorithm,
                "network_backend": network_backend,
                "used_gpu_count": world_size,
                "num_envs": int(num_envs),
                "envs_per_device": {
                    item["device"]: item["local_envs"]
                    for item in gathered
                },
                "devices": [
                    item["device"] for item in gathered
                ],
                "gpu_names": {
                    item["device"]: item["gpu_name"]
                    for item in gathered
                },
                "peak_memory_mb_per_device": {
                    item["device"]: item[
                        "peak_memory_mb"
                    ]
                    for item in gathered
                },
                "transitions": int(global_step),
                "updates": int(update_count),
                "episodes_completed": int(
                    sum(
                        item["completed_episodes"]
                        for item in gathered
                    )
                ),
                "wall_seconds": float(wall),
                "transitions_per_second": (
                    global_step / wall
                ),
                "updates_per_second": (
                    update_count / wall
                ),
                "latest_update_metrics": (
                    latest_metrics
                ),
                "latest_evaluation": (
                    latest_evaluation
                ),
                "evaluation_backend": (
                    evaluation_backend
                ),
                "model_fingerprint_sha256": (
                    _trainer_parameter_fingerprint(
                        trainer
                    )
                ),
                "wandb_run_id": (
                    wandb_run.id
                    if wandb_run is not None
                    else None
                ),
                "wandb_run_name": (
                    wandb_run.name
                    if wandb_run is not None
                    else None
                ),
                "wandb_run_url": (
                    wandb_run.url
                    if wandb_run is not None
                    else None
                ),
            }
            Path(result_path).write_text(
                json.dumps(
                    result,
                    indent=2,
                    default=str,
                )
            )
            if wandb_run is not None:
                wandb_run.summary[
                    "final/transitions_per_second"
                ] = result[
                    "transitions_per_second"
                ]
                wandb_run.summary[
                    "final/updates_per_second"
                ] = result[
                    "updates_per_second"
                ]
                wandb_run.summary[
                    "final/gpu_count"
                ] = world_size
                wandb_run.finish()
        dist.barrier(device_ids=[rank])
    finally:
        env.close()
        if dist.is_initialized():
            dist.destroy_process_group()


In [ ]:
def _write_ddp_launcher_script(path):
    script = r"""import argparse
import cloudpickle

parser = argparse.ArgumentParser()
parser.add_argument("--payload", required=True)
parser.add_argument("--repo", required=True)
parser.add_argument("--algorithm", required=True)
parser.add_argument("--num-envs", type=int, required=True)
parser.add_argument("--total-transitions", type=int, required=True)
parser.add_argument("--seed", type=int, required=True)
parser.add_argument("--network-backend", required=True)
parser.add_argument("--result-path", required=True)
parser.add_argument("--enable-wandb", type=int, default=1)
args = parser.parse_args()

import sys
sys.path.insert(0, args.repo)

with open(args.payload, "rb") as handle:
    worker = cloudpickle.load(handle)

worker(
    repo=args.repo,
    algorithm=args.algorithm,
    num_envs=args.num_envs,
    total_transitions=args.total_transitions,
    seed=args.seed,
    network_backend=args.network_backend,
    result_path=args.result_path,
    enable_wandb=bool(args.enable_wandb),
)
"""
    Path(path).write_text(script)
    return Path(path)


In [ ]:
def train_full_gpu_ddp(
    repo: Path,
    algorithm: str,
    num_envs: int,
    total_transitions: int,
    seed: int,
    network_backend: str = "uavnetsim_gpu",
    enable_wandb=None,
):
    if torch.cuda.device_count() < 2:
        raise RuntimeError(
            "DDP requires at least two visible CUDA GPUs"
        )
    if enable_wandb is None:
        enable_wandb = bool(
            CONFIG.get("training_enable_wandb", True)
        )
    # Resolve/login once in the parent process so rank 0 cannot fail during
    # W&B setup while rank 1 is already waiting in NCCL collectives. The
    # WANDB_API_KEY environment variable is inherited by torchrun workers.
    if bool(enable_wandb):
        mode = str(CONFIG.get("training_wandb_mode", "online"))
        if mode == "online" and not os.environ.get("WANDB_API_KEY"):
            project_ns = load_project_namespace(Path(repo))
            if bool(project_ns.get("running_on_kaggle", lambda: False)()):
                status = project_ns["configure_kaggle_wandb"](
                    verify_login=True
                )
                if not bool(status.get("configured", False)):
                    raise RuntimeError(
                        "W&B online logging is required but no attached Kaggle "
                        "WANDB_API_KEY/wandb_key secret could be configured: "
                        + str(status.get("error"))
                    )
            else:
                raise RuntimeError(
                    "W&B online logging is required but WANDB_API_KEY is not set"
                )
    import subprocess
    import tempfile
    import gc

    # The notebook parent may have just finished a large single-GPU run. Its
    # caching allocator can otherwise keep most T4 VRAM reserved while the two
    # torchrun workers start. Release all inactive parent allocations first.
    gc.collect()
    for device_index in range(min(2, torch.cuda.device_count())):
        with torch.cuda.device(device_index):
            torch.cuda.synchronize(device_index)
            torch.cuda.empty_cache()

    temp_dir = Path(
        tempfile.mkdtemp(prefix="uav_ddp_")
    )
    import cloudpickle
    payload_path = temp_dir / "worker.cloudpickle"
    with payload_path.open("wb") as payload_file:
        cloudpickle.dump(
            _train_full_gpu_ddp_worker,
            payload_file,
        )
    launcher = _write_ddp_launcher_script(
        temp_dir / "uav_ddp_runner.py",
    )
    result_path = temp_dir / "result.json"
    command = [
        sys.executable,
        "-m",
        "torch.distributed.run",
        "--standalone",
        "--nnodes=1",
        "--nproc-per-node=2",
        str(launcher),
        "--payload",
        str(payload_path),
        "--repo",
        str(Path(repo).resolve()),
        "--algorithm",
        str(algorithm),
        "--num-envs",
        str(int(num_envs)),
        "--total-transitions",
        str(int(total_transitions)),
        "--seed",
        str(int(seed)),
        "--network-backend",
        str(network_backend),
        "--result-path",
        str(result_path),
        "--enable-wandb",
        "1" if enable_wandb else "0",
    ]
    stdout_path = temp_dir / "torchrun.stdout.log"
    stderr_path = temp_dir / "torchrun.stderr.log"
    with stdout_path.open("w") as stdout_file, stderr_path.open("w") as stderr_file:
        child_env = os.environ.copy()
        child_env.setdefault(
            "PYTORCH_CUDA_ALLOC_CONF",
            "expandable_segments:True",
        )
        completed = subprocess.run(
            command,
            cwd=str(Path(repo).resolve()),
            check=False,
            text=True,
            stdout=stdout_file,
            stderr=stderr_file,
            env=child_env,
        )
    stdout_text = stdout_path.read_text(errors="replace")
    stderr_text = stderr_path.read_text(errors="replace")
    if completed.returncode != 0:
        raise RuntimeError(
            "DDP training failed.\nSTDOUT:\n"
            + stdout_text[-12000:]
            + "\nSTDERR:\n"
            + stderr_text[-12000:]
        )
    if not result_path.is_file():
        raise RuntimeError(
            "DDP training completed without result.json"
        )
    result = json.loads(result_path.read_text())
    result["ddp_stdout_tail"] = stdout_text[-4000:]
    result["ddp_stderr_tail"] = stderr_text[-4000:]
    return result


### CPU / GPU / auto execution-mode selection


In [ ]:
def resolve_full_execution_mode(execution_mode=None):
    """Resolve cpu/gpu/auto while keeping one tensor implementation."""
    mode = (
        CONFIG.get("full_gpu_execution_mode", "auto")
        if execution_mode is None
        else execution_mode
    )
    mode = str(mode).strip().lower()
    if mode not in {"cpu", "gpu", "auto"}:
        raise ValueError("execution_mode must be cpu, gpu, or auto")
    if mode == "auto":
        mode = "gpu" if torch.cuda.is_available() else "cpu"
    if mode == "gpu" and not torch.cuda.is_available():
        raise RuntimeError(
            "execution_mode='gpu' requires an available CUDA GPU"
        )
    return mode


In [ ]:
def resolve_full_gpu_devices(max_gpus=None):
    """Return the CUDA devices used by automatic training, capped at two."""
    if not torch.cuda.is_available():
        return []
    visible = int(torch.cuda.device_count())
    if max_gpus is None:
        max_gpus = int(CONFIG.get("full_gpu_max_gpus", 2))
    limit = max(1, min(int(max_gpus), 2))
    return [torch.device(f"cuda:{index}") for index in range(min(visible, limit))]


In [ ]:
def _split_env_counts(total_envs, num_shards):
    total_envs = int(total_envs)
    num_shards = int(num_shards)
    if total_envs < 1:
        raise ValueError("total_envs must be >= 1")
    if num_shards < 1:
        raise ValueError("num_shards must be >= 1")
    num_shards = min(num_shards, total_envs)
    base, remainder = divmod(total_envs, num_shards)
    return [base + (1 if index < remainder else 0) for index in range(num_shards)]


### Multi-GPU training strategies


In [ ]:
def train_full_gpu_sharded(
    repo: Path,
    algorithm: str,
    num_envs: int,
    total_transitions: int,
    seed: int,
    devices=None,
    network_backend: str = "uavnetsim_gpu",
    compile_mode=None,
    fused_adam: bool = False,
    update_to_data_ratio=None,
    enable_wandb=None,
):
    """Use two CUDA devices by sharding environments and keeping one learner.

    GPU0 owns the replay buffer and learner/backward/optimizer. Environment
    shards are split across GPU0/GPU1 and stepped concurrently. Policy
    inference is centralized on GPU0 so there remains exactly one model state.
    This avoids two independently diverging MARL learners while still moving
    the dominant batched environment/network simulation onto both GPUs.
    """
    algorithm = str(algorithm).strip().lower()
    if algorithm not in {"masac", "matd3"}:
        raise ValueError("algorithm must be masac or matd3")
    network_backend = str(network_backend).strip().lower()
    if network_backend not in {"simple", "packet_approx", "uavnetsim_gpu"}:
        raise ValueError(
            "multi-GPU sharding supports simple, packet_approx, or uavnetsim_gpu"
        )
    if devices is None:
        devices = resolve_full_gpu_devices()
    devices = [torch.device(device) for device in devices]
    if len(devices) < 2:
        raise ValueError("train_full_gpu_sharded requires at least two CUDA devices")
    devices = devices[:2]
    for device in devices:
        if device.type != "cuda":
            raise ValueError("all sharded devices must be CUDA devices")
    if len({device.index for device in devices}) != len(devices):
        raise ValueError("sharded CUDA devices must be distinct")

    torch.use_deterministic_algorithms(True)
    random.seed(int(seed))
    torch.manual_seed(int(seed))
    torch.cuda.manual_seed_all(int(seed))

    shard_counts = _split_env_counts(num_envs, len(devices))
    primary = devices[0]
    trainer = _trainer(repo, algorithm, str(primary), seed)
    if enable_wandb is None:
        enable_wandb = bool(CONFIG.get("training_enable_wandb", True))
    wandb_run = _init_gpu_wandb_run(
        algorithm,
        seed,
        enabled=bool(enable_wandb),
        network_backend=network_backend,
        num_envs=num_envs,
        gpu_count=len(devices),
    )
    wandb_last_log_step = -1
    eval_interval = max(
        1,
        int(CONFIG["training_eval_interval_steps"]),
    )
    eval_episodes = max(
        1,
        int(CONFIG["training_eval_episodes"]),
    )
    last_eval_step = 0
    latest_evaluation = None
    evaluation_backend = None
    enable_evaluation = bool(enable_wandb)
    fused_adam_enabled = False
    if bool(fused_adam):
        fused_adam_enabled = _rebuild_cuda_optimizers_with_fused_adam(
            trainer, algorithm
        )
    compiled_modules = []
    if compile_mode is not None:
        compiled_modules = _compile_trainer_modules(
            trainer, algorithm, mode=compile_mode
        )

    envs = []
    shard_states = []
    offsets = []
    running_offset = 0
    for rank, (device, count) in enumerate(zip(devices, shard_counts)):
        env_seed = int(seed) + rank * 1_000_003
        env = FullGpuUAVBatchEnv(
            num_envs=int(count),
            device=str(device),
            seed=env_seed,
            strict_cuda=True,
        )
        env.network_model = network_backend
        env.assert_all_state_on_cuda()
        if (
            env.observation_dim != trainer.observation_dim
            or env.state_dim != trainer.state_dim
            or env.discrete_dim != trainer.discrete_dim
        ):
            raise RuntimeError("multi-GPU environment dimensions do not match learner")
        obs, state, mask = env.reset()
        envs.append(env)
        shard_states.append([obs, state, mask])
        offsets.append(running_offset)
        running_offset += int(count)

    capacity = min(
        max(int(CONFIG[f"{algorithm}_replay_capacity"]), 4096),
        max(int(total_transitions), 4096),
    )
    reference_env = envs[0]
    replay = make_project_gpu_replay_buffer(
        trainer,
        capacity=capacity,
        num_agents=reference_env.num_uavs,
        observation_dim=reference_env.observation_dim,
        state_dim=reference_env.state_dim,
        continuous_dim=reference_env.continuous_dim,
        discrete_dim=reference_env.discrete_dim,
        device=str(primary),
        seed=int(seed) + 123,
        strict_cuda=True,
    )
    batch_size = int(CONFIG[f"{algorithm}_batch_size"])
    learning_starts = int(CONFIG[f"{algorithm}_learning_starts"])
    if update_to_data_ratio is None:
        update_to_data_ratio = float(CONFIG[f"{algorithm}_updates_per_step"])
    else:
        update_to_data_ratio = float(update_to_data_ratio)
    if update_to_data_ratio < 0.0:
        raise ValueError("update_to_data_ratio must be >= 0")

    policy_generator = torch.Generator(device=primary)
    policy_generator.manual_seed(int(seed) + 77_777)
    completed_episodes = 0
    global_step = 0
    update_count = 0
    update_credit = 0.0
    latest_metrics = None
    total_transitions_int = int(total_transitions)
    per_device_env_seconds = [0.0 for _ in devices]
    transfer_seconds = 0.0
    policy_seconds = 0.0
    update_seconds = 0.0

    for device in devices:
        torch.cuda.reset_peak_memory_stats(device)
        torch.cuda.synchronize(device)

    import concurrent.futures

    def move_to_primary(tensor):
        nonlocal transfer_seconds
        if tensor.device == primary:
            return tensor
        started = time.perf_counter()
        result = tensor.to(primary, non_blocking=True)
        transfer_seconds += time.perf_counter() - started
        return result

    def policy_actions_for_all(transition_steps_all):
        nonlocal policy_seconds, transfer_seconds
        started = time.perf_counter()
        obs_parts = []
        mask_parts = []
        for obs, _state, mask in shard_states:
            transfer_start = time.perf_counter()
            obs_parts.append(obs if obs.device == primary else obs.to(primary, non_blocking=True))
            mask_parts.append(mask if mask.device == primary else mask.to(primary, non_blocking=True))
            transfer_seconds += time.perf_counter() - transfer_start
        obs_all = torch.cat(obs_parts, dim=0)
        mask_all = torch.cat(mask_parts, dim=0)
        if algorithm == "masac":
            with torch.no_grad():
                sampled = trainer._sample_joint_policy(
                    obs_all, mask_all, deterministic=False
                )
                continuous_all = sampled["continuous"]
                destination_all = sampled["destination_indices"]
        else:
            continuous_all, destination_all = _matd3_actions(
                trainer,
                obs_all,
                mask_all,
                transition_steps_all,
                policy_generator,
            )
        continuous_parts = []
        destination_parts = []
        start_index = 0
        for device, count in zip(devices, shard_counts):
            stop_index = start_index + int(count)
            c = continuous_all[start_index:stop_index]
            d = destination_all[start_index:stop_index]
            transfer_start = time.perf_counter()
            if device != primary:
                c = c.to(device, non_blocking=True)
                d = d.to(device, non_blocking=True)
            transfer_seconds += time.perf_counter() - transfer_start
            continuous_parts.append(c)
            destination_parts.append(d)
            start_index = stop_index
        policy_seconds += time.perf_counter() - started
        return continuous_parts, destination_parts

    wall_start = time.perf_counter()
    executor = concurrent.futures.ThreadPoolExecutor(max_workers=len(envs))
    try:
        while global_step < total_transitions_int:
            transition_steps_all = (
                torch.arange(int(num_envs), device=primary, dtype=torch.long)
                + global_step
                + 1
            )
            batch_first_step = global_step + 1
            batch_last_step = global_step + int(num_envs)
            all_warmup = batch_last_step <= learning_starts
            all_policy = batch_first_step > learning_starts

            if all_warmup:
                action_parts = [
                    _random_exploration_actions(env, shard_states[index][2])
                    for index, env in enumerate(envs)
                ]
                continuous_parts = [item[0] for item in action_parts]
                destination_parts = [item[1] for item in action_parts]
            elif all_policy:
                continuous_parts, destination_parts = policy_actions_for_all(
                    transition_steps_all
                )
            else:
                policy_continuous, policy_destination = policy_actions_for_all(
                    transition_steps_all
                )
                continuous_parts = []
                destination_parts = []
                for index, env in enumerate(envs):
                    random_continuous, random_destination = _random_exploration_actions(
                        env, shard_states[index][2]
                    )
                    local_start = offsets[index]
                    local_stop = local_start + shard_counts[index]
                    warmup_local = (
                        transition_steps_all[local_start:local_stop]
                        <= learning_starts
                    ).to(env.device)
                    continuous_parts.append(
                        torch.where(
                            warmup_local[:, None, None],
                            random_continuous,
                            policy_continuous[index],
                        )
                    )
                    destination_parts.append(
                        torch.where(
                            warmup_local[:, None],
                            random_destination,
                            policy_destination[index],
                        )
                    )

            def step_one(index):
                env = envs[index]
                with torch.cuda.device(env.device):
                    started = time.perf_counter()
                    result = env.step(
                        continuous_parts[index], destination_parts[index]
                    )
                    elapsed = time.perf_counter() - started
                return index, result, elapsed

            futures = [executor.submit(step_one, index) for index in range(len(envs))]
            step_results = [None] * len(envs)
            for future in futures:
                index, result, elapsed = future.result()
                step_results[index] = result
                per_device_env_seconds[index] += elapsed

            take_total = min(
                int(num_envs), total_transitions_int - global_step
            )
            remaining_take = take_total
            for index, env in enumerate(envs):
                (
                    next_obs,
                    next_state,
                    next_mask,
                    reward,
                    done,
                    _metrics,
                ) = step_results[index]
                obs, state, mask = shard_states[index]
                take = min(int(shard_counts[index]), remaining_take)
                if take > 0:
                    replay.add_batch(
                        move_to_primary(obs[:take]),
                        move_to_primary(state[:take]),
                        move_to_primary(continuous_parts[index][:take]),
                        move_to_primary(destination_parts[index][:take]),
                        move_to_primary(mask[:take]),
                        move_to_primary(reward[:take]),
                        move_to_primary(next_obs[:take]),
                        move_to_primary(next_state[:take]),
                        move_to_primary(next_mask[:take]),
                        move_to_primary(done[:take]),
                    )
                    remaining_take -= take
                shard_states[index] = [next_obs, next_state, next_mask]

            previous = global_step
            global_step += take_total
            if len(replay) >= batch_size:
                first_train_step = max(previous, learning_starts)
                eligible = max(0, global_step - first_train_step)
                update_credit += eligible * update_to_data_ratio
                updates_this_step = int(update_credit)
                update_credit -= updates_this_step
                update_start = time.perf_counter()
                for local_update in range(updates_this_step):
                    collected = trainer.update(
                        replay,
                        batch_size=batch_size,
                        collect_metrics=(local_update == updates_this_step - 1),
                    )
                    if collected is not None:
                        latest_metrics = collected
                    update_count += 1
                update_seconds += time.perf_counter() - update_start

            should_evaluate = (
                enable_evaluation
                and (
                    global_step - last_eval_step >= eval_interval
                    or global_step >= total_transitions_int
                )
            )
            if should_evaluate:
                evaluation = evaluate_full_gpu_reference(
                    trainer=trainer,
                    algorithm=algorithm,
                    network_backend=network_backend,
                    seed=seed,
                    episodes=eval_episodes,
                    uavnetsim_repo_path=None,
                )
                latest_evaluation = evaluation["summary"]
                evaluation_backend = evaluation["backend"]
                last_eval_step = global_step
                if wandb_run is not None:
                    wandb_run.log(
                        {
                            "global_step": int(global_step),
                            **{
                                f"evaluation/{key}": float(value)
                                for key, value in latest_evaluation.items()
                                if isinstance(value, (bool, int, float))
                            },
                        }
                    )
                    for key, value in latest_evaluation.items():
                        if isinstance(value, (bool, int, float)):
                            wandb_run.summary[
                                f"evaluation_latest/{key}"
                            ] = float(value)

            if (
                wandb_run is not None
                and (
                    global_step - wandb_last_log_step
                    >= max(1, int(CONFIG.get("training_log_interval_steps", 100)))
                    or global_step >= total_transitions_int
                )
            ):
                elapsed_for_log = max(
                    1e-12, time.perf_counter() - wall_start
                )
                payload = {
                    "global_step": int(global_step),
                    "train/transitions_per_second": global_step / elapsed_for_log,
                    "train/updates_per_second": update_count / elapsed_for_log,
                }
                if latest_metrics:
                    for key, value in latest_metrics.items():
                        if isinstance(value, torch.Tensor):
                            if value.numel() != 1:
                                continue
                            value = value.detach().cpu().item()
                        if isinstance(value, (bool, int, float)):
                            payload[f"train/{key}"] = float(value)
                wandb_run.log(payload)
                wandb_last_log_step = global_step

            for index, env in enumerate(envs):
                done = step_results[index][4]
                if bool(done.any().item()):
                    completed_episodes += int(done.long().sum().item())
                    if global_step < total_transitions_int:
                        obs, state, mask = env.reset(mask=done)
                        shard_states[index] = [obs, state, mask]

        for device in devices:
            torch.cuda.synchronize(device)
        wall = time.perf_counter() - wall_start
    finally:
        executor.shutdown(wait=True)

    def scalarize(metrics):
        if metrics is None:
            return None
        result = {}
        for key, value in metrics.items():
            if isinstance(value, torch.Tensor):
                if value.numel() == 1:
                    result[key] = float(value.detach().cpu().item())
            elif isinstance(value, (int, float, bool)):
                result[key] = value
        return result

    peak_memory = {
        str(device): torch.cuda.max_memory_allocated(device) / 2**20
        for device in devices
    }
    gpu_names = {
        str(device): torch.cuda.get_device_name(device)
        for device in devices
    }
    result = {
        "kind": "full_gpu_auto_multi",
        "multi_gpu_strategy": "env_shard_single_learner",
        "algorithm": algorithm,
        "network_backend": network_backend,
        "num_envs": int(num_envs),
        "envs_per_device": {
            str(device): int(count)
            for device, count in zip(devices, shard_counts)
        },
        "used_gpu_count": len(devices),
        "devices": [str(device) for device in devices],
        "learner_device": str(primary),
        "gpu_names": gpu_names,
        "transitions": int(global_step),
        "updates": int(update_count),
        "update_to_data_ratio": float(update_to_data_ratio),
        "episodes_completed": int(completed_episodes),
        "wall_seconds": float(wall),
        "transitions_per_second": global_step / wall,
        "updates_per_second": update_count / wall,
        "policy_seconds": float(policy_seconds),
        "update_seconds": float(update_seconds),
        "env_seconds_per_device": {
            str(device): float(seconds)
            for device, seconds in zip(devices, per_device_env_seconds)
        },
        "transfer_enqueue_seconds": float(transfer_seconds),
        "peak_memory_mb_per_device": peak_memory,
        "compile_mode": compile_mode,
        "compiled_modules": compiled_modules,
        "fused_adam": bool(fused_adam_enabled),
        "latest_update_metrics": scalarize(latest_metrics),
        "latest_evaluation": latest_evaluation,
        "evaluation_backend": evaluation_backend,
        "model_fingerprint_sha256": _trainer_parameter_fingerprint(trainer),
        "wandb_run_id": (
            wandb_run.id if wandb_run is not None else None
        ),
        "wandb_run_name": (
            wandb_run.name if wandb_run is not None else None
        ),
        "wandb_run_url": (
            wandb_run.url if wandb_run is not None else None
        ),
    }
    if wandb_run is not None:
        wandb_run.summary["final/transitions_per_second"] = result[
            "transitions_per_second"
        ]
        wandb_run.summary["final/updates_per_second"] = result[
            "updates_per_second"
        ]
        wandb_run.summary["final/gpu_count"] = len(devices)
        wandb_run.finish()
    for env in envs:
        env.close()
    return result


In [ ]:
def train_full_gpu_auto(
    repo: Path,
    algorithm: str,
    num_envs: int,
    total_transitions: int,
    seed: int,
    network_backend: str = "uavnetsim_gpu",
    max_gpus=None,
    auto_multi_gpu=None,
    compile_mode=None,
    fused_adam: bool = False,
    update_to_data_ratio=None,
    enable_wandb=None,
    execution_mode=None,
):
    """Run the same tensor training path in cpu/gpu/auto mode.

    cpu: force CPU for logic/regression testing.
    gpu: require CUDA and auto-select one GPU or 2-GPU DDP.
    auto: use GPU when CUDA is available, otherwise CPU.
    """
    if auto_multi_gpu is None:
        auto_multi_gpu = bool(
            CONFIG.get("full_gpu_auto_multi_gpu", True)
        )
    resolved_mode = resolve_full_execution_mode(
        execution_mode
    )
    if enable_wandb is None:
        enable_wandb = (
            False
            if resolved_mode == "cpu"
            else bool(CONFIG.get("training_enable_wandb", True))
        )

    if resolved_mode == "cpu":
        result = train_full_gpu(
            repo=repo,
            algorithm=algorithm,
            num_envs=int(num_envs),
            total_transitions=int(total_transitions),
            seed=int(seed),
            device="cpu",
            network_backend=network_backend,
            strict_cuda=False,
            compile_mode=compile_mode,
            fused_adam=False,
            update_to_data_ratio=update_to_data_ratio,
            enable_wandb=bool(enable_wandb),
        )
        result["execution_mode"] = "cpu"
        result["used_gpu_count"] = 0
        result["devices"] = ["cpu"]
        result["multi_gpu_strategy"] = "cpu"
        result["learner_device"] = "cpu"
        return result

    devices = resolve_full_gpu_devices(max_gpus=max_gpus)
    if not devices:
        raise RuntimeError(
            "GPU mode resolved but no CUDA device is available"
        )

    strategy = str(
        CONFIG.get("full_gpu_multi_gpu_strategy", "ddp")
    ).strip().lower()

    if (
        bool(auto_multi_gpu)
        and len(devices) >= 2
        and int(num_envs) >= 2
    ):
        if strategy == "ddp":
            if compile_mode is not None:
                raise ValueError(
                    "compile_mode is not enabled in the DDP path yet; "
                    "use compile_mode=None for deterministic DDP training"
                )
            if fused_adam:
                raise ValueError(
                    "fused_adam is not enabled in the DDP path yet; "
                    "use fused_adam=False until separately benchmarked"
                )
            if update_to_data_ratio is not None:
                configured = float(
                    CONFIG[
                        f"{str(algorithm).lower()}_updates_per_step"
                    ]
                )
                if float(update_to_data_ratio) != configured:
                    raise ValueError(
                        "custom update_to_data_ratio is not enabled in "
                        "the DDP launcher yet"
                    )
            result = train_full_gpu_ddp(
                repo=repo,
                algorithm=algorithm,
                num_envs=int(num_envs),
                total_transitions=int(total_transitions),
                seed=int(seed),
                network_backend=network_backend,
                enable_wandb=enable_wandb,
            )
            result["execution_mode"] = "gpu"
            return result
        if strategy == "env_shard_single_learner":
            result = train_full_gpu_sharded(
                repo=repo,
                algorithm=algorithm,
                num_envs=int(num_envs),
                total_transitions=int(total_transitions),
                seed=int(seed),
                devices=devices[:2],
                network_backend=network_backend,
                compile_mode=compile_mode,
                fused_adam=bool(fused_adam),
                update_to_data_ratio=update_to_data_ratio,
                enable_wandb=bool(enable_wandb),
            )
            result["execution_mode"] = "gpu"
            return result
        raise ValueError(
            "full_gpu_multi_gpu_strategy must be ddp or "
            "env_shard_single_learner"
        )

    result = train_full_gpu(
        repo=repo,
        algorithm=algorithm,
        num_envs=int(num_envs),
        total_transitions=int(total_transitions),
        seed=int(seed),
        device=str(devices[0]),
        network_backend=network_backend,
        strict_cuda=True,
        compile_mode=compile_mode,
        fused_adam=bool(fused_adam),
        update_to_data_ratio=update_to_data_ratio,
        enable_wandb=bool(enable_wandb),
    )
    result["execution_mode"] = "gpu"
    result["used_gpu_count"] = 1
    result["devices"] = [str(devices[0])]
    result["multi_gpu_strategy"] = "single_gpu"
    result["learner_device"] = str(devices[0])
    return result


### Profiling and optimization sweeps


In [ ]:
def benchmark_full_gpu_training_optimizations(
    repo: Path,
    *,
    algorithms=("masac", "matd3"),
    env_counts=(256, 512, 1024),
    total_transitions=8192,
    seed=44,
    device="cuda:0",
    network_backend="uavnetsim_gpu",
    test_compile=True,
    test_fused_adam=True,
    test_amp=True,
    update_to_data_ratios=(1.0,),
):
    """Short end-to-end optimization sweep without changing production defaults.

    Timing-only fields are expected to vary. Deterministic metrics/fingerprints
    are returned so any speed optimization can be rejected if it changes the
    numerical training path unexpectedly.
    """
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA is required for optimization benchmarking")

    previous_amp = bool(CONFIG.get("training_amp_enabled", False))
    cases = [
        {
            "name": "baseline",
            "compile_mode": None,
            "fused_adam": False,
            "amp": False,
        }
    ]
    if test_fused_adam:
        cases.append(
            {
                "name": "fused_adam",
                "compile_mode": None,
                "fused_adam": True,
                "amp": False,
            }
        )
    if test_compile:
        cases.append(
            {
                "name": "compile_default_fused",
                "compile_mode": "default",
                "fused_adam": bool(test_fused_adam),
                "amp": False,
            }
        )
    if test_amp:
        cases.append(
            {
                "name": "amp_fused",
                "compile_mode": None,
                "fused_adam": bool(test_fused_adam),
                "amp": True,
            }
        )
    if test_amp and test_compile:
        cases.append(
            {
                "name": "compile_amp_fused",
                "compile_mode": "default",
                "fused_adam": bool(test_fused_adam),
                "amp": True,
            }
        )

    results = []
    try:
        for algorithm in algorithms:
            for env_count in env_counts:
                for utd in update_to_data_ratios:
                    for case in cases:
                        CONFIG["training_amp_enabled"] = bool(
                            case["amp"]
                        )
                        torch.cuda.empty_cache()
                        torch.cuda.synchronize()
                        try:
                            item = train_full_gpu(
                                repo=repo,
                                algorithm=algorithm,
                                num_envs=int(env_count),
                                total_transitions=int(total_transitions),
                                seed=int(seed),
                                device=device,
                                network_backend=network_backend,
                                strict_cuda=True,
                                compile_mode=case["compile_mode"],
                                fused_adam=bool(case["fused_adam"]),
                                update_to_data_ratio=float(utd),
                            )
                            item["optimization_case"] = case["name"]
                            item["amp_enabled"] = bool(case["amp"])
                            results.append(item)
                        except torch.cuda.OutOfMemoryError as exc:
                            torch.cuda.empty_cache()
                            results.append(
                                {
                                    "algorithm": algorithm,
                                    "num_envs": int(env_count),
                                    "optimization_case": case["name"],
                                    "amp_enabled": bool(case["amp"]),
                                    "update_to_data_ratio": float(utd),
                                    "error": "CUDA OOM",
                                    "detail": str(exc),
                                }
                            )
        return results
    finally:
        CONFIG["training_amp_enabled"] = previous_amp


In [ ]:
def profile_full_gpu_step(
    repo: Path,
    *,
    algorithm="masac",
    num_envs=256,
    seed=44,
    device="cuda:0",
    network_backend="uavnetsim_gpu",
    warmup_steps=3,
    active_steps=5,
    output_trace=None,
):
    """Profile actor inference + environment/network step on CUDA.

    This intentionally excludes optimizer updates; use the end-to-end sweep for
    learner throughput. The profiler table makes environment/network hotspots
    visible before attempting more invasive rewrites.
    """
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA is required for profiling")
    algorithm = str(algorithm).strip().lower()
    env = FullGpuUAVBatchEnv(
        num_envs=int(num_envs),
        device=device,
        seed=int(seed),
        strict_cuda=True,
    )
    env.network_model = str(network_backend)
    trainer = _trainer(repo, algorithm, device, seed)
    obs, state, mask = env.reset()

    def one_step():
        nonlocal obs, state, mask
        if algorithm == "masac":
            with torch.no_grad():
                sample = trainer._sample_joint_policy(
                    obs,
                    mask,
                    deterministic=False,
                )
                continuous = sample["continuous"]
                destination = sample["destination_indices"]
        elif algorithm == "matd3":
            transition_steps = torch.ones(
                int(num_envs),
                dtype=torch.long,
                device=env.device,
            )
            continuous, destination = _matd3_actions(
                trainer,
                obs,
                mask,
                transition_steps,
                env.generator,
            )
        else:
            raise ValueError("algorithm must be masac or matd3")
        obs, state, mask, _, done, _ = env.step(
            continuous,
            destination,
        )
        if bool(done.any().item()):
            obs, state, mask = env.reset(mask=done)

    for _ in range(int(warmup_steps)):
        one_step()
    torch.cuda.synchronize(env.device)

    activities = [torch.profiler.ProfilerActivity.CPU]
    activities.append(torch.profiler.ProfilerActivity.CUDA)
    with torch.profiler.profile(
        activities=activities,
        record_shapes=True,
        profile_memory=True,
        with_stack=False,
    ) as prof:
        for _ in range(int(active_steps)):
            one_step()
    torch.cuda.synchronize(env.device)
    table = prof.key_averages().table(
        sort_by="self_cuda_time_total",
        row_limit=30,
    )
    if output_trace is not None:
        prof.export_chrome_trace(str(output_trace))
    env.close()
    return table


In [ ]:
def benchmark_gpu_network_models(
    *,
    network_models=("packet_approx", "uavnetsim_gpu"),
    env_counts=(256, 512, 1024),
    steps=40,
    warmup_steps=4,
    seed=44,
    device="cuda:0",
):
    """Measure the cost of richer network fidelity on the same GPU env."""
    previous = CONFIG.get("full_gpu_network_model", "simple")
    results = []
    try:
        for model in network_models:
            CONFIG["full_gpu_network_model"] = str(model)
            for count in env_counts:
                torch.cuda.empty_cache()
                if int(warmup_steps) > 0:
                    benchmark_env(
                        int(count),
                        int(warmup_steps),
                        int(seed),
                        device,
                        True,
                    )
                torch.cuda.empty_cache()
                item = benchmark_env(
                    int(count),
                    int(steps),
                    int(seed),
                    device,
                    True,
                )
                item["network_model"] = str(model)
                results.append(item)
        return results
    finally:
        CONFIG["full_gpu_network_model"] = previous


In [ ]:
def benchmark_simple_scaling(
    repo: Path,
    *,
    cpu_env_counts=(1, 2, 4, 8, 16),
    gpu_env_counts=(1, 2, 4, 8, 16, 32, 64, 128, 256),
    vector_steps=40,
    warmup_steps=4,
    seed=44,
    device="cuda:0",
):
    results = {"cpu": [], "gpu": []}

    for count in cpu_env_counts:
        if int(warmup_steps) > 0:
            benchmark_hybrid_env(
                repo,
                num_envs=int(count),
                steps=int(warmup_steps),
                seed=int(seed),
            )
        item = benchmark_hybrid_env(
            repo,
            num_envs=int(count),
            steps=int(vector_steps),
            seed=int(seed),
        )
        results["cpu"].append(item)

    for count in gpu_env_counts:
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats(torch.device(device))
        if int(warmup_steps) > 0:
            benchmark_env(
                num_envs=int(count),
                steps=int(warmup_steps),
                seed=int(seed),
                device=device,
                strict_cuda=True,
            )
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats(torch.device(device))
        item = benchmark_env(
            num_envs=int(count),
            steps=int(vector_steps),
            seed=int(seed),
            device=device,
            strict_cuda=True,
        )
        results["gpu"].append(item)

    cpu_by_count = {
        int(item["num_envs"]): item
        for item in results["cpu"]
    }
    gpu_by_count = {
        int(item["num_envs"]): item
        for item in results["gpu"]
    }
    same_count = []
    for count in sorted(set(cpu_by_count) & set(gpu_by_count)):
        cpu_rate = float(cpu_by_count[count]["transitions_per_second"])
        gpu_rate = float(gpu_by_count[count]["transitions_per_second"])
        same_count.append(
            {
                "num_envs": int(count),
                "cpu_transitions_per_second": cpu_rate,
                "gpu_transitions_per_second": gpu_rate,
                "gpu_over_cpu_speedup": (
                    gpu_rate / cpu_rate
                    if cpu_rate > 0.0
                    else float("inf")
                ),
            }
        )

    best_cpu = max(
        results["cpu"],
        key=lambda item: item["transitions_per_second"],
    )
    best_gpu = max(
        results["gpu"],
        key=lambda item: item["transitions_per_second"],
    )
    results["same_count_comparison"] = same_count
    results["best_cpu"] = best_cpu
    results["best_gpu"] = best_gpu
    results["best_gpu_over_best_cpu"] = (
        float(best_gpu["transitions_per_second"])
        / float(best_cpu["transitions_per_second"])
    )
    results["same_count_crossover_envs"] = next(
        (
            item["num_envs"]
            for item in same_count
            if item["gpu_over_cpu_speedup"] > 1.0
        ),
        None,
    )
    return results


### CPU UavNetSim worker scaling


In [ ]:
def _uavnetsim_cpu_worker(payload):
    repo_text, seed, steps, communication_loaded = payload
    repo = Path(repo_text)
    project_ns = load_project_namespace(repo)
    np = project_ns["np"]
    CPUEnv = project_ns["UAVSearchEnv"]
    env = CPUEnv(
        backend_name="uavnetsim",
        network_backend_kwargs={"repo_path": None},
    )
    try:
        env.reset(seed=int(seed))
        if communication_loaded:
            Report = project_ns["Report"]
            env.targets[0].confirmed = True
            env.report_buffers[0].clear()
            env.report_buffers[0].append(
                Report(
                    target_id=0,
                    source_uav=0,
                    created_step=0,
                    size_bytes=int(CONFIG["report_bytes"]),
                    ttl_s=float(CONFIG["report_ttl"]),
                    delivered_bytes=0,
                )
            )
        rng = np.random.default_rng(int(seed) + 99173)
        start = time.perf_counter()
        completed = 0
        for _ in range(int(steps)):
            action = {}
            for sender, agent_id in enumerate(env.agent_ids):
                continuous = rng.uniform(
                    -1.0, 1.0, size=4
                ).astype(np.float32)
                discrete_dim = int(
                    env.action_space[
                        agent_id
                    ]["destination"].n
                )
                destination = int(
                    rng.integers(0, discrete_dim)
                )
                if communication_loaded and sender == 0:
                    continuous[3] = 1.0
                    destination = discrete_dim - 1
                action[agent_id] = {
                    "motion": continuous[:3],
                    "destination": destination,
                    "power": np.asarray(
                        [continuous[3]],
                        dtype=np.float32,
                    ),
                }
            out = env.step(action)
            completed += 1
            if bool(out[2] or out[3]):
                break
        wall = time.perf_counter() - start
        return {
            "steps": int(completed),
            "wall_seconds": float(wall),
            "network": env.network_backend.metrics(),
        }
    finally:
        env.close()


In [ ]:
def benchmark_uavnetsim_cpu_worker_scaling(
    repo: Path,
    *,
    env_counts=(1, 2, 4, 8),
    steps=4,
    seed=44,
    communication_loaded=True,
):
    import multiprocessing as mp

    results = []
    context = mp.get_context("fork")

    def worker_entry(payload, queue):
        try:
            queue.put(
                ("ok", _uavnetsim_cpu_worker(payload))
            )
        except BaseException as exc:
            import traceback
            queue.put(
                (
                    "error",
                    (
                        type(exc).__name__,
                        str(exc),
                        traceback.format_exc(),
                    ),
                )
            )

    for count in env_counts:
        payloads = [
            (
                str(repo),
                int(seed) + index,
                int(steps),
                bool(communication_loaded),
            )
            for index in range(int(count))
        ]
        queue = context.Queue()
        processes = [
            context.Process(
                target=worker_entry,
                args=(payload, queue),
            )
            for payload in payloads
        ]
        start = time.perf_counter()
        for process in processes:
            process.start()
        messages = [
            queue.get()
            for _ in processes
        ]
        for process in processes:
            process.join()
        wall = time.perf_counter() - start
        errors = [
            payload
            for status, payload in messages
            if status != "ok"
        ]
        if errors:
            raise RuntimeError(
                "UavNetSim worker failed: "
                + repr(errors[0])
            )
        worker_results = [
            payload
            for status, payload in messages
            if status == "ok"
        ]
        transitions = sum(
            int(item["steps"])
            for item in worker_results
        )
        results.append(
            {
                "num_envs": int(count),
                "requested_steps_per_env": int(steps),
                "transitions": int(transitions),
                "wall_seconds": float(wall),
                "transitions_per_second": (
                    transitions / wall
                    if wall > 0.0
                    else float("inf")
                ),
                "worker_wall_seconds": [
                    float(item["wall_seconds"])
                    for item in worker_results
                ],
                "network_metrics": [
                    item["network"]
                    for item in worker_results
                ],
            }
        )
    return results


### Metrics, deterministic fingerprints, and end-to-end comparison


In [ ]:
def _scalarize_metrics(metrics):
    if metrics is None:
        return None
    result = {}
    for key, value in metrics.items():
        if isinstance(value, torch.Tensor):
            if value.numel() == 1:
                result[str(key)] = float(
                    value.detach().cpu().item()
                )
        elif isinstance(value, (int, float, bool)):
            result[str(key)] = value
    return result


In [ ]:
def _trainer_parameter_fingerprint(trainer):
    """SHA256 over all nn.Module parameters/buffers, independent of device."""
    import hashlib

    digest = hashlib.sha256()
    module_names = sorted(
        name
        for name, value in trainer.__dict__.items()
        if isinstance(value, torch.nn.Module)
    )
    for module_name in module_names:
        module = getattr(trainer, module_name)
        digest.update(module_name.encode("utf-8"))
        for key, tensor in sorted(
            module.state_dict().items()
        ):
            digest.update(key.encode("utf-8"))
            value = (
                tensor.detach()
                .cpu()
                .contiguous()
            )
            digest.update(
                str(value.dtype).encode("utf-8")
            )
            digest.update(
                str(tuple(value.shape)).encode(
                    "utf-8"
                )
            )
            digest.update(value.numpy().tobytes())
    return digest.hexdigest()


In [ ]:
def benchmark_uavnetsim_training_compare(
    repo: Path,
    *,
    repo_path,
    algorithm="masac",
    total_steps=8,
    seed=44,
    device="cuda:0",
    benchmark_learning_starts=4,
    benchmark_batch_size=4,
    benchmark_updates_per_step=1,
):
    """End-to-end CPU-mission vs GPU-mission training with UavNetSim on both.

    The benchmark overrides learning-start/batch settings only inside this
    function so a short run actually exercises replay sampling and gradients.
    Production CONFIG values are restored before returning.
    """
    algorithm = str(algorithm).strip().lower()
    if algorithm not in {"masac", "matd3"}:
        raise ValueError(
            "algorithm must be masac or matd3"
        )
    if not torch.cuda.is_available():
        raise RuntimeError(
            "training comparison requires CUDA"
        )
    total_steps = int(total_steps)
    if total_steps < 1:
        raise ValueError(
            "total_steps must be >= 1"
        )

    project_ns = load_project_namespace(repo)
    if not project_ns["uavnetsim_available"](
        repo_path
    ):
        raise FileNotFoundError(
            "UavNetSim repository is unavailable"
        )

    prefix = algorithm
    keys = {
        "learning": (
            f"{prefix}_learning_starts"
        ),
        "batch": f"{prefix}_batch_size",
        "updates": (
            f"{prefix}_updates_per_step"
        ),
    }
    previous = {
        key: CONFIG[name]
        for key, name in keys.items()
    }
    CONFIG[keys["learning"]] = int(
        benchmark_learning_starts
    )
    CONFIG[keys["batch"]] = int(
        benchmark_batch_size
    )
    CONFIG[keys["updates"]] = int(
        benchmark_updates_per_step
    )

    cpu_env = None
    try:
        CPUEnv = project_ns["UAVSearchEnv"]
        cpu_env = CPUEnv(
            backend_name="uavnetsim",
            network_backend_kwargs={
                "repo_path": repo_path,
            },
        )
        project_ns["set_seed"](int(seed))
        if algorithm == "masac":
            cpu_trainer = project_ns[
                "HybridMASAC"
            ](
                cpu_env,
                device=device,
            )
            train_fn = project_ns[
                "train_masac"
            ]
        else:
            cpu_trainer = project_ns[
                "HybridMATD3"
            ](
                cpu_env,
                device=device,
                seed=int(seed),
            )
            train_fn = project_ns[
                "train_matd3"
            ]

        torch.cuda.synchronize()
        cpu_start = time.perf_counter()
        cpu_result = train_fn(
            cpu_env,
            total_steps=total_steps,
            seed=int(seed),
            trainer=cpu_trainer,
        )
        torch.cuda.synchronize()
        cpu_wall = (
            time.perf_counter()
            - cpu_start
        )
        cpu_network = (
            cpu_env.network_backend.metrics()
        )

        torch.cuda.synchronize()
        gpu_result = train_full_gpu(
            repo=repo,
            algorithm=algorithm,
            num_envs=1,
            total_transitions=total_steps,
            seed=int(seed),
            device=device,
            network_backend="uavnetsim",
            uavnetsim_repo_path=repo_path,
        )

        return {
            "kind": (
                "uavnetsim_training_compare"
            ),
            "algorithm": algorithm,
            "seed": int(seed),
            "total_steps": total_steps,
            "benchmark_overrides": {
                keys["learning"]: int(
                    benchmark_learning_starts
                ),
                keys["batch"]: int(
                    benchmark_batch_size
                ),
                keys["updates"]: int(
                    benchmark_updates_per_step
                ),
            },
            "cpu_mission_gpu_learner": {
                "wall_seconds": cpu_wall,
                "steps_per_second": (
                    total_steps / cpu_wall
                ),
                "completed_episodes": len(
                    cpu_result[
                        "completed_episodes"
                    ]
                ),
                "completed_episode_metrics": (
                    cpu_result[
                        "completed_episodes"
                    ]
                ),
                "latest_update_metrics": (
                    _scalarize_metrics(
                        cpu_result[
                            "latest_update_metrics"
                        ]
                    )
                ),
                "model_fingerprint_sha256": (
                    _trainer_parameter_fingerprint(
                        cpu_result["trainer"]
                    )
                ),
                "network_metrics": (
                    cpu_network
                ),
            },
            "gpu_mission_uavnetsim": (
                gpu_result
            ),
        }
    finally:
        if cpu_env is not None:
            cpu_env.close()
        for key, name in keys.items():
            CONFIG[name] = previous[key]


## Usage

CUDA Simple benchmark: benchmark_env(num_envs=64, steps=100, seed=CONFIG["seed"], device="cuda:0")

Simple full-GPU training: train_full_gpu(repo=Path.cwd(), algorithm="masac", num_envs=64, total_transitions=10000, seed=CONFIG["seed"], device="cuda:0")

Packet-level UavNetSim training (mission math on GPU, UavNetSim on CPU/SimPy): train_full_gpu(repo=Path.cwd(), algorithm="masac", num_envs=1, total_transitions=10000, seed=CONFIG["seed"], device="cuda:0", network_backend="uavnetsim", uavnetsim_repo_path="/path/to/UavNetSim")

CPU-UavNetSim vs GPU-mission+UavNetSim full-flow comparison: benchmark_uavnetsim_full_flow(Path.cwd(), repo_path="/path/to/UavNetSim", steps=20, seed=CONFIG["seed"], device="cuda:0")

Network-only packet-level parity check: validate_uavnetsim_network_parity(Path.cwd(), repo_path="/path/to/UavNetSim").

Use algorithm="matd3" for MATD3. CPU debug is allowed for shape/integration checking via strict_cuda=False. UavNetSim is not CUDA-native, so any UavNetSim path necessarily includes a CPU/SimPy networking stage.

End-to-end short training comparison with UavNetSim on both paths: benchmark_uavnetsim_training_compare(Path.cwd(), repo_path="/path/to/UavNetSim", algorithm="masac", total_steps=8, device="cuda:0").

Scaling benchmark:
benchmark_simple_scaling(Path.cwd(), cpu_env_counts=(1,2,4,8,16), gpu_env_counts=(1,2,4,8,16,32,64,128,256), vector_steps=40, warmup_steps=4, seed=CONFIG["seed"], device="cuda:0")

Parallel packet-level CPU scaling:
benchmark_uavnetsim_cpu_worker_scaling(Path.cwd(), env_counts=(1,2,4,8), steps=4, seed=CONFIG["seed"])

benchmark_uavnetsim_full_flow now reports GPU-to-CPU sync time, UavNetSim step time, CPU-to-GPU sync time, and the UavNetSim fraction.


### GPU network modes

CONFIG["full_gpu_network_model"] = "simple" keeps the original tensor link model.

CONFIG["full_gpu_network_model"] = "packet_approx" keeps the mission/network state fully tensorized on CUDA and additionally models the UavNetSim-relevant packet payload/queue limit, 802.11b-style CSMA/CA airtime overhead, shared-channel contention, packet counts, LoS/NLoS, path loss/SNR/rate, report TTL/buffers, peer relay, GCS delivery, and communication energy. This is a calibrated training surrogate, not a bit-for-bit GPU port of SimPy UavNetSim.

Use the real UavNetSimBackend/GpuUavNetSimBridge for packet-level validation (routing/MAC event scheduling, collisions/interference, ACK/retry timing, delay/PDR/throughput).

### Optional learner speed experiments

train_full_gpu(..., fused_adam=True) benchmarks fused CUDA Adam for actor/critic optimizers.

train_full_gpu(..., compile_mode="default") benchmarks torch.compile without forcing the CUDA-graph-heavy reduce-overhead mode.

CONFIG["training_amp_enabled"] = True enables CUDA AMP (autocast + GradScaler) in MASAC/MATD3 learner updates; keep it disabled until same-seed reproducibility and numerical stability pass on the target GPU.

train_full_gpu(..., update_to_data_ratio=0.5) supports UTD experiments. Ratios below 1 change the learning schedule and therefore are hyperparameter experiments, not semantics-preserving code optimizations.

GPU-native packet-aware training:
train_full_gpu(repo=Path.cwd(), algorithm="masac", num_envs=256, total_transitions=10000, seed=CONFIG["seed"], device="cuda:0", network_backend="packet_approx")

This packet_approx path is CUDA-batched and mirrors the CURRENT project UavNetSim configuration (forced MARL next hop, 1024-byte payloads, queue cap, IEEE-802.11b CSMA/CA airtime/backoff/ACK cost, shared contention, LoS/NLoS A2A channel, relay/GCS delivery and communication energy). It is a training surrogate, not the entire upstream UavNetSim protocol library.

GPU-native UavNetSim surrogate:
`network_backend="uavnetsim_gpu"` keeps the current mission networking on CUDA: UavNetSim-v2 A2A LoS/NLoS path gain, simultaneous-link SINR/interference, packet queue cap, expected CSMA/CA backoff + ACK/SIFS/DIFS airtime, relay custody, GCS delivery, report TTL/buffers and communication energy. It is intentionally a deterministic time-slotted training surrogate; the official SimPy UavNetSim backend remains the packet-event validator.

Optimization sweep:
`benchmark_full_gpu_training_optimizations(Path.cwd(), algorithms=("masac","matd3"), env_counts=(256,512,1024), total_transitions=8192, network_backend="uavnetsim_gpu")` compares baseline, fused Adam, torch.compile, AMP, and combined paths without changing production defaults.

Profiler:
`profile_full_gpu_step(Path.cwd(), algorithm="masac", num_envs=256, network_backend="uavnetsim_gpu", output_trace="/tmp/uav_trace.json")` profiles actor + environment/network CUDA hotspots.

Direct CPU-UavNetSim vs GPU-surrogate validation: `validate_uavnetsim_gpu_direct_link(Path.cwd(), seed=CONFIG["seed"], steps=6, device="cuda:0", strict_cuda=True)`.

Network-fidelity scaling: `benchmark_gpu_network_models(network_models=("packet_approx","uavnetsim_gpu"), env_counts=(256,512,1024), steps=40, warmup_steps=4, seed=CONFIG["seed"], device="cuda:0")`.

Automatic GPU selection:
`train_full_gpu_auto(Path.cwd(), algorithm="masac", num_envs=2048, total_transitions=..., seed=CONFIG["seed"], network_backend="uavnetsim_gpu")` uses `cuda:0` only when one GPU is visible, and shards environments across `cuda:0` + `cuda:1` when two or more GPUs are visible. The cap is controlled by `full_gpu_max_gpus` and is currently 2. The learner/backward stays on `cuda:0`; GPU1 executes its environment/network shard in parallel.

Production automatic training:
train_full_gpu_auto(...) is the single entry point. It uses one GPU when only one is visible and switches to 2-process NCCL DDP when two or more GPUs are visible. Only rank 0 creates one W&B run, whose display name is exactly <ALGORITHM>-seed<SEED>-<RUN_ID>.
